# Rex trên RBA Dataset (Risk-Based Authentication) — train trên Kaggle
Mô hình Rex điều tra RBA đa vòng (**ModernBERT / Laya + 4 Semantic Tool Slots + PoE Refutation**).

**Registry & Domain**: Đồng bộ hoàn toàn với 2 file cấu hình chuyên dụng mới `rex/events/configs/registry_rba.json` và `rex/events/configs/registry_rba.yaml` (v2.0.0) — Domain `<TYPE_RBA_LOGIN_DEFENSE>` với 4 giả thuyết MECE: `H_botnet` (FRAUD), `H_sybil` (FRAUD), `H_shared` (LEGIT), `H_novel` (ESCALATE).

**Dữ liệu**: [RBA Dataset (Wiefling et al., 2020/2021) — Kaggle `dasgroup/rba-dataset`](https://www.kaggle.com/datasets/dasgroup/rba-dataset) / [Zenodo 6782156](https://zenodo.org/record/6782156)

Gồm 15 bước hoàn chỉnh: từ giải quyết dataset, nạp dữ liệu causal, tính chỉ số tool tầng B không rò rỉ tương lai, nạp và biên dịch registry Popperian từ 2 file mới, sinh quỹ đạo rollout, 3 stage huấn luyện (`heads` $\to$ `ponder` $\to$ `full`), hiệu chỉnh nhiệt độ & conformal, đánh giá offline, đến chạy `InvestigatorAgent` end-to-end.


## 1. Cấu hình
Mọi tham số nằm trong `CFG`. Hỗ trợ chạy thử nhanh bằng `DRY_RUN = True`, tự động tìm dataset RBA qua Kaggle input, kagglehub, hoặc Zenodo CDN.


In [ ]:
# ============================================================================================
# STEP 1/15 - Cấu hình chung, logger và seed
# Mọi tham số của notebook nằm trong CFG. Muốn chạy thử nhanh: đặt CFG["DRY_RUN"] = True.
# ============================================================================================
import os, sys, json, time, math, random, glob, re, gc, platform, datetime as _dt
import numpy as np
import pandas as pd

CFG = {
    # --- chế độ chạy ---
    "DRY_RUN": False,             # True: chỉ dùng tập con nhỏ để đo tốc độ / kiểm tra pipeline
    "RESUME_DIR": None,           # None: train từ đầu. "auto": tìm ckpt_*/model.safetensors trong /kaggle/input
    "TIME_BUDGET_HOURS": 11.3,    # Kaggle cắt phiên ở 12h; để dư cho phần lưu kết quả
    "SEED": 42,
    # --- dữ liệu RBA ---
    "DOMAIN": "<TYPE_RBA_LOGIN_DEFENSE>",
    "REGISTRY_JSON": "rex/events/configs/registry_rba.json",
    "REGISTRY_YAML": "rex/events/configs/registry_rba.yaml",
    "CSV_GLOB": "/kaggle/input/**/rba-dataset.csv",
    "OUT_DIR": "/kaggle/working",
    # --- bộ lọc alert ---
    "ALERT_VELOCITY_THRES": 3,   # IP velocity 1h >= 3 sự kiện
    "ALERT_STREAK_THRES": 2,     # Chuỗi thất bại liên tiếp >= 2
    "LEGIT_PER_ATTACK_TRAIN": 1.5, # train: mọi alert tấn công + alert hợp lệ theo tỉ lệ 1:4
    "LEGIT_FRAC_EVAL": 0.05,     # val/test: 5% alert hợp lệ ngẫu nhiên, trọng số 1/0.05 = 20
    # --- mô hình nền ---
    "BASE_MODEL": "convaiinnovations/laya",
    "BASE_REVISION": "55cf4c4ebb4ebe31b2550e8bdf3bd21b99753851",   # pin trong rex/revisions.py; lỗi thì dùng main
    "MAX_LEN": 192, "HEAD_MAX_LEN": 144, "EVIDENCE_MAX_TOKENS": 64,
    # --- train (tham số của rex_train.RexTrainConfig) ---
    "SHARDS": 2,
    "STAGES": {
        "heads":  {"batch_cases": 8, "grad_accum": 1, "lr_heads": 1e-4, "lr_encoder": 2.5e-5, "amp": True},
        "ponder": {"batch_cases": 8, "grad_accum": 1, "lr_heads": 5e-5, "lr_encoder": 2.5e-5, "amp": True},
        "full":   {"batch_cases": 8, "grad_accum": 2, "lr_heads": 1e-4, "lr_encoder": 2.5e-5, "amp": True},
    },
    "WARMUP_RATIO": 0.1,
    "WARMUP_RATIO_LATER": 0.03,
    "MAX_GRAD_NORM": 1.0, "AMP": True, "LOG_EVERY": 200,
    "NAN_FALLBACK_FP32": True,
    "EST_CASES_PER_SEC": {"heads": 25.0, "ponder": 9.0, "full": 8.0},
    # --- đánh giá ---
    "AGENT_EVAL_CASES": 3000,     # số case test chạy InvestigatorAgent end-to-end
    "LATENCY_RUNS": 50,
}
if CFG["DRY_RUN"]:
    CFG.update({"LEGIT_PER_ATTACK_TRAIN": 1.5, "AGENT_EVAL_CASES": 200, "LATENCY_RUNS": 10})

OUT = CFG["OUT_DIR"]
os.makedirs(OUT, exist_ok=True)
for _sub in ("data", "plots"):
    os.makedirs(os.path.join(OUT, _sub), exist_ok=True)

T_START = time.time()
LOG_PATH = os.path.join(OUT, "train_log.txt")
HISTORY = []          # mỗi dòng log train -> 1 hàng trong history.csv
METRICS = {"config": json.loads(json.dumps(CFG, default=str))}


def _now():
    return _dt.datetime.now().strftime("%H:%M:%S")


def _mem():
    parts = []
    try:
        import psutil
        parts.append(f"RAM {psutil.Process().memory_info().rss / 1e9:.1f} GB")
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            used = torch.cuda.max_memory_allocated() / 1e9
            tot = torch.cuda.get_device_properties(0).total_memory / 1e9
            parts.append(f"VRAM peak {used:.1f}/{tot:.1f} GB")
    except Exception:
        pass
    return " | ".join(parts)


def log(msg=""):
    line = str(msg)
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def fmt_secs(s):
    s = int(max(0, s))
    return f"{s // 3600}h{(s % 3600) // 60:02d}m{s % 60:02d}s" if s >= 3600 else f"{s // 60}m{s % 60:02d}s"


class Step:
    TOTAL = 15

    def __init__(self, i, title):
        self.i, self.title = i, title

    def __enter__(self):
        self.t0 = time.time()
        log("")
        log("=" * 100)
        log(f"[STEP {self.i:02d}/{self.TOTAL}] {self.title} | start {_now()} | elapsed {fmt_secs(time.time() - T_START)} | {_mem()}")
        log("=" * 100)
        return self

    def __exit__(self, exc_type, exc, tb):
        status = "FAILED" if exc_type else "done"
        log(f"[STEP {self.i:02d}/{self.TOTAL}] {status} in {fmt_secs(time.time() - self.t0)} | {_mem()}")
        return False


def table(rows, headers, title=None):
    rows = [[("" if v is None else (f"{v:,.4f}" if isinstance(v, float) else (f"{v:,}" if isinstance(v, (int, np.integer)) else str(v)))) for v in r] for r in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in rows)) if rows else len(str(h)) for i, h in enumerate(headers)]
    if title:
        log(f"  {title}")
    log("  " + " | ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    log("  " + "-+-".join("-" * w for w in widths))
    for r in rows:
        log("  " + " | ".join(v.ljust(w) for v, w in zip(r, widths)))


def time_left():
    return CFG["TIME_BUDGET_HOURS"] * 3600 - (time.time() - T_START)


random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])

with Step(1, "Cấu hình"):
    open(LOG_PATH, "w").close()
    log(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
    log(f"DRY_RUN={CFG['DRY_RUN']} | OUT_DIR={OUT} | time budget {CFG['TIME_BUDGET_HOURS']}h")
    log(json.dumps(CFG, indent=1, default=str))


## 2. Môi trường, mã nguồn Rex, checkpoint Laya
Kiểm tra GPU và thư viện. Trên GPU < Ampere (T4, P100) AMP dùng fp16 + GradScaler.

In [ ]:
# ============================================================================================
# STEP 2/15 - Môi trường, mã nguồn Rex và checkpoint nền Laya
# (a) kiểm tra GPU + thư viện; (b) các cell %%writefile ngay sau cell này ghi mã nguồn Rex
# (đúng các module train/inference của repo) vào REXLIB; (c) import và tải checkpoint Laya.
# ============================================================================================
import subprocess, importlib
STEP2 = Step(2, "Môi trường, mã nguồn Rex và checkpoint Laya")
STEP2.__enter__()

REXLIB = os.path.join(OUT, "rexlib")
os.makedirs(os.path.join(REXLIB, "rex", "events", "configs"), exist_ok=True)

import torch
assert torch.cuda.is_available(), "Cần GPU: Settings -> Accelerator -> GPU T4 x2 (hoặc P100)."
DEVICE = torch.device("cuda:0")
_cap = torch.cuda.get_device_capability(0)
log(f"  GPU: {torch.cuda.get_device_name(0)} | capability {_cap[0]}.{_cap[1]} | "
    f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB | số GPU: {torch.cuda.device_count()} (dùng 1)")
# T4/P100 không có bf16 phần cứng; torch mới vẫn báo is_bf16_supported()=True (giả lập, rất chậm).
# rex_train chọn bf16 theo hàm này -> ép về fp16 + GradScaler trên GPU < Ampere.
if _cap[0] < 8:
    torch.cuda.is_bf16_supported = lambda *a, **k: False
    log("  GPU < Ampere: AMP dùng fp16 + GradScaler (tắt bf16 giả lập)")
torch.backends.cuda.matmul.allow_tf32 = True
torch.manual_seed(CFG["SEED"])

import transformers
from packaging import version as _v
if _v.parse(transformers.__version__) < _v.parse("4.48.0"):
    log(f"  transformers {transformers.__version__} < 4.48 (chưa có ModernBERT) -> cài bản mới")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.48.0"], check=True)
    importlib.reload(transformers)
for _m in ["torch", "transformers", "safetensors", "huggingface_hub", "sklearn", "matplotlib"]:
    try:
        log(f"  {_m:16s} {importlib.import_module(_m).__version__}")
    except Exception as exc:
        log(f"  {_m:16s} THIẾU ({exc})")


### Mã nguồn Rex
Các cell `%%writefile` dưới đây ghi nguyên văn các module train/inference của repo Rex (snapshot lúc tạo notebook) vào `rexlib/rex/`. Không sửa các cell này: tên lớp và tham số giữ đúng repo để checkpoint nạp được bằng `rex.rex_model`.

In [ ]:
%%writefile {REXLIB}/rex/__init__.py
"""Rex - bản rút gọn cho notebook train: chỉ các module train/inference (arch='rex2')."""
import os as _os

_os.environ.setdefault("USE_TF", "0")
__version__ = "0.4.1"


In [ ]:
%%writefile {REXLIB}/rex/agent.py
"""Shim: chỉ giữ _fix_tokenizer_config (nguyên văn từ rex/agent.py) mà train.load_checkpoint cần."""
import json
import os
import tempfile
import warnings


def _fix_tokenizer_config(path: str):
    """Ensure tokenizer_config.json can be loaded across all transformers versions."""
    cfg_file = os.path.join(path, "tokenizer", "tokenizer_config.json")
    if not os.path.exists(cfg_file):
        return
    try:
        with open(cfg_file) as f:
            tcfg = json.load(f)
        changed = False
        if tcfg.get("tokenizer_class") in (None, "TokenizersBackend"):
            tcfg["tokenizer_class"] = "PreTrainedTokenizerFast"
            tcfg.pop("backend", None)
            tcfg.pop("is_local", None)
            changed = True
        # Checkpoints built on the mmBERT/Gemma tokenizer store extra_special_tokens as a list;
        # transformers expects a mapping and raises "'list' object has no attribute 'keys'",
        # which makes AutoTokenizer -- and so the whole model -- fail to load.
        extra = tcfg.get("extra_special_tokens")
        if isinstance(extra, list):
            tcfg["extra_special_tokens"] = {"extra_%d" % i: t for i, t in enumerate(extra)}
            changed = True
        if changed:
            # HuggingFace snapshots are symlinks into a shared blob store, so writing through the
            # link would truncate a file shared with other revisions and processes, race
            # concurrent loads, and desync the hub's cache metadata. Write to a temporary file in
            # the same directory and `os.replace` it into place: the snapshot entry becomes a
            # regular file and is swapped atomically, so a concurrent load never sees a missing
            # or half-written config.
            cfg_dir = os.path.dirname(cfg_file)
            fd, tmp_file = tempfile.mkstemp(dir=cfg_dir, prefix=".tokenizer_config.", suffix=".tmp")
            try:
                with os.fdopen(fd, "w") as f:
                    json.dump(tcfg, f, indent=2)
                # mkstemp creates the file 0600; keep the mode the cache file had so a shared
                # cache stays readable to the same users as before.
                try:
                    os.chmod(tmp_file, os.stat(cfg_file).st_mode & 0o777)
                except OSError:
                    pass
                os.replace(tmp_file, cfg_file)
            except BaseException:
                try:
                    os.unlink(tmp_file)
                except OSError:
                    pass
                raise
    except Exception as e:
        # Do not swallow this silently: if the patch did not apply, AutoTokenizer may fail later
        # with a confusing error and no hint that the config was the cause.
        warnings.warn(
            "laya: could not patch %s (%s); the tokenizer may fail to load with this "
            "transformers version." % (cfg_file, e),
            RuntimeWarning, stacklevel=2)


In [ ]:
%%writefile {REXLIB}/rex/router.py
"""Shim: notebook luôn truyền đường dẫn checkpoint cục bộ, không cần bảng alias của router gốc."""


def resolve_model_spec(name):
    return None


In [ ]:
%%writefile {REXLIB}/rex/common.py
"""Core model architecture, token sequence construction, and confidence estimation for laya."""
import json
import math
import os
import threading
import warnings
from contextlib import nullcontext
from contextvars import ContextVar
from functools import wraps
from typing import Any, Dict, List, Optional, Sequence, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

QTYPES = {"choice": 0, "score": 1, "noul": 2}
QTYPE_NAMES = {v: k for k, v in QTYPES.items()}
_DEFAULT_NOUL_LABELS = {"false": "false", "true": "true"}


def _autocast_enabled(device_type: str) -> bool:
    """Whether an autocast scope covers `device_type`.

    `torch.is_autocast_enabled` only accepted a `device_type` from torch 2.4; before that it
    reported CUDA alone and CPU and MPS had no autocast to report. This package declares
    `torch>=2.0.0`, so call the device-aware form where it exists and fall back otherwise,
    using torch's separate CPU predicate.
    """
    try:
        return torch.is_autocast_enabled(device_type)
    except TypeError:
        # torch < 2.4 took no `device_type`: the no-argument call reports CUDA alone, CPU has
        # its own predicate, and MPS had no autocast to report.
        if device_type == "cpu":
            return torch.is_autocast_cpu_enabled()
        return torch.is_autocast_enabled()


# A fast tokenizer is not read-only: `truncation=True` / `padding=True` make it call
# `enable_truncation` / `enable_padding`, which mutates the shared Rust object. One tokenizer is
# parsed per checkpoint directory and shared by every Agent that wants it, so concurrent
# `predict()` calls -- on one Agent or on two sharing the cache -- raced and raised
# `RuntimeError: Already borrowed`. Serialise encoding instead: it is a small fraction of a call
# next to the forward pass, and this keeps the cache's single parse.
_TOKENIZE_LOCK = threading.RLock()
_QUESTION_TOKEN_CACHE = ContextVar("laya_question_token_cache", default=None)


def encode_text(tok, text, **kwargs):
    """Tokenize `text` while holding the lock a shared fast tokenizer needs."""
    with _TOKENIZE_LOCK:
        return tok(text, **kwargs)


def _reuse_question_tokens(fn):
    @wraps(fn)
    def wrapped(*args, **kwargs):
        # One cache per prediction call: nested calls get their own scope, and exceptions
        # restore the outer scope. Nothing is retained on an Agent or shared across threads.
        scope = {"thread": threading.get_ident(), "tokens": {}}
        token = _QUESTION_TOKEN_CACHE.set(scope)
        try:
            return fn(*args, **kwargs)
        finally:
            # A timed hook can copy this context to a worker that outlives the call.
            scope["tokens"] = None
            _QUESTION_TOKEN_CACHE.reset(token)
    return wrapped


def _disable_question_token_reuse():
    scope = _QUESTION_TOKEN_CACHE.get()
    if scope is not None:
        scope["tokens"] = None


def _encode_question_text(tok, text, **kwargs):
    scope = _QUESTION_TOKEN_CACHE.get()
    cache = scope["tokens"] if scope is not None and scope["thread"] == threading.get_ident() else None
    if cache is None:
        return encode_text(tok, text, **kwargs)["input_ids"]
    # Key the rendered, mask-sanitized text and encoding settings, not a JSON question:
    # option order, structured criteria and custom noul labels must keep their meaning.
    key = (id(tok), text, tuple(kwargs.items()))
    if key not in cache:
        # Keep the tokenizer alive so its identity cannot be reused within this scope.
        cache[key] = (tok, tuple(encode_text(tok, text, **kwargs)["input_ids"]))
    # Sequence assembly must not mutate token lists retained for later states.
    return list(cache[key][1])


def serialize_state(state: Union[str, dict, list]) -> str:
    if isinstance(state, str):
        return state
    return json.dumps(state, ensure_ascii=False)


def render_criterion(value) -> str:
    """Render one criterion value as text.

    Strings pass through; anything structured (dict, list, number) becomes a single-line JSON
    document with the default separators -- ``", "`` between members, ``": "`` before a value --
    so a rubric reads as JSON rather than a Python repr. Without this a dict-valued criterion
    crashed `noul` outright and leaked `{'desc': ...}` into `choice` and `score` prompts.
    """
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, separators=(", ", ": "), default=str)


def _resolve_noul_labels(labels=None):
    if labels is None:
        labels = _DEFAULT_NOUL_LABELS
    if not isinstance(labels, dict) or set(labels) != {"false", "true"}:
        raise ValueError("noul labels must map exactly 'false' and 'true' to distinct non-empty strings")
    false_label, true_label = labels["false"], labels["true"]
    if not isinstance(false_label, str) or not isinstance(true_label, str):
        raise ValueError("noul labels must map exactly 'false' and 'true' to distinct non-empty strings")
    false_label, true_label = false_label.strip(), true_label.strip()
    if not false_label or not true_label or false_label == true_label:
        raise ValueError("noul labels must map exactly 'false' and 'true' to distinct non-empty strings")
    return false_label, true_label


def render_options(q: Dict) -> List[str]:
    """Render option texts in label-index order. Noul semantic order is always [false, true]."""
    t, crit = q["t"], q.get("crit")
    if t != "noul" and "labels" in q:
        raise ValueError("labels is only supported for noul questions")
    if t == "choice":
        # only None/"" mean "no description"; 0 and False are legitimate criterion values.
        # `str(k)` unconditionally: a label with no description is rendered as itself, so an int
        # label used to come back as an int from a function annotated `-> List[str]` and then
        # reached `build_sequence`, which calls `.replace` on it and raised an AttributeError
        # naming neither the question nor the label. With a description the same label already
        # went through `"%s: %s" %` and was a str, which is why only the undescribed form broke.
        # `structured._enum_field` stringifies labels the same way; the returned answer still
        # carries the caller's original label, which is unchanged.
        return [str(k) if v is None or v == "" else "%s: %s" % (k, render_criterion(v))
                for k, v in crit.items()]
    if t == "score":
        return ["level %d: %s" % (i, render_criterion(c)) for i, c in enumerate(crit)]
    crit = crit or {}
    false_label, true_label = _resolve_noul_labels(q.get("labels"))
    false_crit, true_crit = crit.get("false"), crit.get("true")
    return [
        false_label + ": "
        + (render_criterion(false_crit) if false_crit not in (None, "") else "no, the statement does not hold"),
        true_label + ": "
        + (render_criterion(true_crit) if true_crit not in (None, "") else "yes, the statement holds"),
    ]


def build_sequence(
    tok,
    state: Union[str, dict, list],
    q: Dict,
    max_len: int = 512,
    head_max_len: int = 192,
    option_order: Optional[List[int]] = None,
    truncate_left: bool = False,
    state_ids: Optional[List[int]] = None,
    return_stats: bool = False,
    return_truncation_stats: bool = False,
    return_layout: bool = False,
):
    """Format: [CLS] <type> instructions [SEP] [MASK] opt0 [MASK] opt1 ... [SEP] state [SEP].

    `state_ids` lets a caller tokenize the shared state once and reuse it across every question,
    instead of re-serializing and re-tokenizing the same document per question.

    `return_stats` adds a third return value describing what the head budget did to the options:
    `options` (how many the question defines), `options_distinct` (how many still have a token
    span of their own) and `tokens_per_option` (the cap applied to each, or None when none was).

    The state is clamped to whatever room is left after the head, so a long state loses tokens
    here silently. `return_truncation_stats=True` adds one more return value, after the option
    stats when both are asked for, reporting that clamp:

        {"state_tokens": int, "state_tokens_used": int, "state_tokens_dropped": int,
         "truncated": bool}

    Callers cannot reconstruct this from the outside. The budget is in tokens, not characters,
    and the room left for the state depends on `max_len`, `head_max_len`, the instruction and
    the rendered options - so it moves per checkpoint and per question. A caller guessing with a
    fixed character threshold is wrong in both directions: it reports truncation that did not
    happen, and stays silent while evidence is being dropped (issue #174).

    The question half is `build_head`; `state_room` reports how much of `max_len` is left for the
    state after it, which is what a caller must size a window against.

    `return_layout=True` adds one last return value, the parallel option layout from
    `parallel_layout`: `{"position_ids": [...], "option_ids": [...]}`, one entry per token.
    """
    ids, markers, stats = build_head(tok, q, head_max_len, option_order=option_order)
    head_len = len(ids)
    room = max(0, max_len - len(ids) - 1)
    if state_ids is None:
        state_ids = encode_text(tok, serialize_state(state).replace(tok.mask_token, " "),
                                add_special_tokens=False)["input_ids"]
    # not state_ids[-room:]: with no room left, state_ids[-0:] is the whole state rather than none of it
    st = state_ids[max(0, len(state_ids) - room):] if truncate_left else state_ids[:room]
    ids = ids + st + [tok.sep_token_id]
    # Laid out before the clamp: a dropped trailing option must not widen the last surviving span
    layout = parallel_layout(markers, head_len, len(ids)) if return_layout else None
    ids, markers = ids[:max_len], [m for m in markers if m < max_len]
    extra = ()
    if return_truncation_stats:
        # `room` leaves space for the closing [SEP], so every token in `st` survives the [:max_len] clamp
        extra = ({
            "state_tokens": len(state_ids),
            "state_tokens_used": len(st),
            "state_tokens_dropped": len(state_ids) - len(st),
            "truncated": len(st) < len(state_ids),
        },)
    if return_layout:
        extra += ({k: v[:max_len] for k, v in layout.items()},)
    if not return_stats:
        return (ids, markers) + extra
    return (ids, markers, stats) + extra


def build_head(tok, q: Dict, head_max_len: int = 192, option_order: Optional[List[int]] = None):
    """The question half of a sequence: `[CLS] <type> instructions [SEP] [MASK] opt0 [MASK] opt1 ... [SEP]`.

    `build_sequence` appends the state to this and `state_room` measures what is left over for it,
    so neither can disagree with the other about what the head costs.

    Returns `(ids, markers, stats)`: the head token ids, the `[MASK]` position of each option in
    `option_order`, and the stats dict `build_sequence` returns for `return_stats`.
    """
    mask_tok = tok.mask_token
    opts = render_options(q)
    order = option_order if option_order is not None else list(range(len(opts)))
    ins = str(q["ins"]).replace(mask_tok, " ")
    head_ids = _encode_question_text(tok, "%s question: %s" % (q["t"], ins), add_special_tokens=False)
    opt_ids = []
    for i in order:
        # Cap at the tokenizer, not after the fact: `[:48]` still makes the tokenizer process the
        # whole (possibly long) description. truncation=True, max_length=48 keeps the first 48
        # tokens, which is exactly what the previous slice produced.
        opt_tokens = _encode_question_text(
            tok,
            " " + opts[i].replace(mask_tok, " "),
            add_special_tokens=False,
            truncation=True,
            max_length=48,
        )
        opt_ids.append([tok.mask_token_id] + opt_tokens)
    opt_budget = head_max_len - sum(len(o) for o in opt_ids)
    per_option = None
    if opt_budget < 16:
        per = max(4, (head_max_len - 16) // max(1, len(opt_ids)))
        per_option = per
        opt_ids = [o[:per] for o in opt_ids]
        opt_budget = head_max_len - sum(len(o) for o in opt_ids)
    head_ids = head_ids[: max(8, opt_budget)]
    ids = [tok.cls_token_id] + head_ids + [tok.sep_token_id]
    markers = []
    for o in opt_ids:
        markers.append(len(ids))
        ids.extend(o)
    ids.append(tok.sep_token_id)
    # Two options that share a prefix can come out of the cut as the same token span: the marker
    # count still matches the option count, so the guard in `Agent._encode_state` passes and
    # nothing downstream can tell that the question lost the ability to name them apart. Counted
    # on the capped option ids, before assembly: re-slicing the finished sequence cannot close
    # the last option's span -- it runs on into the serialized state, which differs per request,
    # so the last option always looks distinguishable however it collided (#538).
    return ids, markers, {
        "options": len(opt_ids),
        "options_distinct": len({tuple(o) for o in opt_ids}),
        "tokens_per_option": per_option,
    }


def parallel_layout(markers: List[int], head_len: int, length: int) -> Dict[str, List[int]]:
    """Position ids and option ids that make the encoder blind to option order.

    In the sequential layout option `s` sits at positions after option `s-1`, and every option
    attends to every other, so where an option is listed changes its embedding: the slot logits
    of five identical options spread by 3.59 (`tests/test_option_order.py`). Here every option
    starts at the same position, the first one after the instruction's [SEP], and the head's
    closing [SEP] and the state continue after the longest option. Together with
    `parallel_option_masks`, which stops an option from attending to another, reordering the
    options only reorders the marker embeddings.

    `option_ids` is 0 for shared tokens (instruction, closing [SEP], state, padding) and `s + 1`
    for the tokens of slot `s`, which runs from its [MASK] up to the next marker or, for the last
    slot, up to the [SEP] at `head_len - 1`.
    """
    if not markers:
        return {"position_ids": list(range(length)), "option_ids": [0] * length}
    start = markers[0]
    spans = list(zip(markers, markers[1:] + [head_len - 1]))
    position_ids, option_ids = list(range(start)), [0] * start
    for s, (a, b) in enumerate(spans):
        position_ids += range(start, start + b - a)
        option_ids += [s + 1] * (b - a)
    after = start + max(b - a for a, b in spans)
    position_ids += range(after, after + length - len(position_ids))
    option_ids += [0] * (length - len(option_ids))
    return {"position_ids": position_ids, "option_ids": option_ids}


OPTION_LAYOUTS = ("sequential", "parallel")


def uses_parallel_layout(cfg: Dict) -> bool:
    """Whether a checkpoint config asks for the parallel option layout (`parallel_layout`).

    The layout is a property of the weights -- a checkpoint trained on one layout reads the other
    as a different input -- so it lives in the config, defaulting to the sequential layout every
    published checkpoint was trained on. The parallel masks go through ModernBERT's per-layer-type
    mask argument, which transformers 4.x does not have.
    """
    layout = cfg.get("option_layout", "sequential")
    if layout not in OPTION_LAYOUTS:
        raise ValueError("option_layout must be one of %s, got %r" % (", ".join(OPTION_LAYOUTS), layout))
    if layout == "parallel":
        import transformers
        if int(transformers.__version__.split(".")[0]) < 5:
            raise RuntimeError("option_layout='parallel' needs transformers>=5, found %s" % transformers.__version__)
    return layout == "parallel"


def parallel_option_masks(attention_mask: torch.Tensor, position_ids: torch.Tensor, option_ids: torch.Tensor,
                          sliding_window: Optional[int]) -> Dict[str, torch.Tensor]:
    """Per-layer-type boolean attention masks for the parallel layout (`parallel_layout`).

    A token attends to every real token except those of a *different* option. ModernBERT's local
    layers keep their window, measured in position ids rather than sequence index: options share
    positions, so a sequence-index window would let slot 0 see more of the state than slot 3.
    Returned in the `{"full_attention": ..., "sliding_attention": ...}` form ModernBERT takes
    in place of a padding mask; each mask is `[batch, 1, query, key]`, True where attention is allowed.
    """
    keys = attention_mask.bool()[:, None, :]
    other_option = (option_ids[:, :, None] > 0) & (option_ids[:, None, :] > 0) \
        & (option_ids[:, :, None] != option_ids[:, None, :])
    full = keys & ~other_option
    masks = {"full_attention": full[:, None]}
    if sliding_window is not None:
        near = (position_ids[:, :, None] - position_ids[:, None, :]).abs() <= sliding_window
        masks["sliding_attention"] = (full & near)[:, None]
    return masks


def state_room(tok, q: Dict, max_len: int = 512, head_max_len: int = 192) -> int:
    """How many state tokens `q` leaves inside `max_len`, which is what `build_sequence` keeps.

    The head is the question's own -- its instructions plus one `[MASK]`-prefixed span per option --
    so a question with many options leaves less room for the state than one with two, and two
    questions in the same request do not have to leave the same amount. Anything past the return
    value is cut off (the start is kept, or the end for a conversation turn list).
    """
    head, _, _ = build_head(tok, q, head_max_len)
    return max(0, max_len - len(head) - 1)          # -1 for the [SEP] that closes the state


#: How far the DEFAULT window may be cut before `window_budget` says so. A small clamp is ordinary
#: and warning about it would be noise; a large one multiplies the number of forward passes by the
#: same factor, which a caller needs to be told about because nothing in their code implies it. 2x
#: is the point where the scan costs at least twice what the caller would estimate from `max_len`.
_WINDOW_CLAMP_WARN_RATIO = 2

# How much capping the window may multiply the window count before `window_batch_cap` starts
# chunking. Under this, chunking would cost the single-shared-pass property for no real protection.
_WINDOW_BATCH_BLOWUP = 2

def window_budget(tok, questions, max_len: int = 512, head_max_len: int = 192,
                  window: Optional[int] = None, stride: Optional[int] = None):
    """Window size and stride for scanning a state that is longer than one sequence.

    `predict_long` decodes each token window back to text and scores it as an ordinary state, so a
    window wider than the room the questions leave is re-truncated by `build_sequence` on the way
    in: the tail of every window reaches no model, while the reported span says it did. The window
    is therefore capped at `state_room`, and at the *smallest* room of `questions`, because the
    windows are one list of states scored for every question in shared forward passes -- a window
    sized for the roomiest question would be cut short for the tightest one, and the offsets
    reported on its answers would mean something different per question. With no questions there is
    nothing to fit, so the caller's window (or the checkpoint default) stands. That default is
    `max(64, max_len - head_max_len - 8)`: the state budget the config leaves, with a 64-token
    floor under it, so a widened `head_max_len` stops shrinking it there.

    `questions` are internal question dicts, as `Agent._to_internal` returns them.

    An explicit `window` wider than the room is clamped to it with a `RuntimeWarning`, since a
    scan at the requested size cannot read what it claims to. A `stride` past the effective window
    is refused: the tokens between two windows would be read by neither, which is the failure
    `predict_long` exists to prevent. The default stride keeps its 50% overlap of the *effective*
    window, so a span near a boundary still lands whole inside some window; an explicit stride
    equal to the window still reads every token, with no overlap to catch a span that straddles a
    boundary.

    Returns `(window, stride, room)`.
    """
    rooms = [state_room(tok, q, max_len, head_max_len) for q in questions]
    requested = window if (window and window > 0) else max(64, max_len - head_max_len - 8)
    size = requested
    room = min(rooms) if rooms else size
    if room <= 0:
        raise ValueError(
            "predict_long: the questions' options fill the whole sequence (max_len=%d,"
            " head_max_len=%d), leaving no room for the state; no window can carry any of it."
            " A label set this large is what laya.shortlist.predict_shortlist is for"
            % (max_len, head_max_len))
    if size > room:
        if window and window > 0:
            warnings.warn(
                "laya: predict_long: window=%d is wider than the %d state tokens these questions"
                " leave inside max_len=%d, so every window would be truncated to %d on the way to"
                " the model; scanning with window=%d instead" % (size, room, max_len, room, room),
                RuntimeWarning, stacklevel=3)
        elif size >= room * _WINDOW_CLAMP_WARN_RATIO:
            # The DEFAULT window was cut, and cut hard. Capping it is what stops the tail of every
            # window reaching no model, but it is not free and it must not be silent: the scan now
            # needs about `size / room` times as many windows, each one a full forward pass, and
            # nothing in the caller's code says why. Measured on the English checkpoint with 100
            # four-word options: room 102 of max_len 512, so the question head alone is 409 tokens,
            # the window falls 312 -> 102 and the scan goes 11 windows -> 36, a 3.1x wall-clock
            # increase (1902 ms -> 5858 ms) for the same document.
            #
            # The cost is not the capping, it is the shape of the request: 80% of every sequence is
            # the question, and because the encoder is bidirectional the head cannot be computed
            # once and reused -- its representations depend on the state it is paired with. So the
            # warning names the real remedy rather than only reporting the clamp.
            warnings.warn(
                "laya: predict_long: these questions leave only %d of max_len=%d for the state"
                " (their heads take the rest), so the scan window is capped %d -> %d and roughly"
                " %.1fx as many windows -- each a full forward pass -- are needed to read the"
                " document. Fewer or shorter options, a larger max_len, or"
                " laya.shortlist.predict_shortlist for a large label set will all cost less than"
                " scanning at this width" % (room, max_len, size, room, size / room),
                RuntimeWarning, stacklevel=3)
        size = room
    step = stride if (stride and stride > 0) else max(1, size // 2)
    if step > size:
        if size < requested and stride and stride <= requested:
            # The window the caller asked for was reduced above, and their stride was valid for the
            # window they asked for -- so this is the library's clamp, not their mistake. Reducing
            # the stride to match keeps the no-gap guarantee without making a self-consistent pair
            # of arguments an error. A stride that overshot the *requested* window is still refused
            # below, because that one really is the caller's.
            warnings.warn(
                "laya: predict_long: stride=%d was a 50%% step for the window=%d you asked for, but"
                " the window was reduced to %d to fit the room these questions leave; scanning with"
                " stride=%d instead" % (step, requested, size, max(1, size // 2)),
                RuntimeWarning, stacklevel=3)
            step = max(1, size // 2)
        else:
            raise ValueError(
                "predict_long: stride=%d steps past the %d-token window%s, so %d tokens between"
                " every pair of windows would be read by no window at all; pass stride <= %d"
                % (step, size, " these questions leave room for" if size < requested else "",
                   step - size, size))
    return size, step, room



def window_batch_cap(n_windows: int, window: int, config_budget: int,
                     batch_size: Optional[int] = None) -> Optional[int]:
    """Keep one forward pass no wider than the un-capped scan's would have been -- but only when
    capping the window has multiplied the window count enough to matter.

    Capping the window at the room the questions leave multiplies the number of windows on exactly
    the inputs it targets: measured, a 4 561-token document at 120 options goes from 29 windows of
    312 tokens to roughly 413 of 23. `predict_batch` with `batch_size=None` puts every state in one
    forward pass, so a caller who passed no `batch_size` would go from a 29-row pass to a 413-row one
    at `max_len` width -- a plausible out-of-memory on an input that used to fit.

    Sending every window in one run is also a deliberate property (`ONNXAgent` asserts it), and a mild
    cap -- a 64-token budget reduced to 43, say -- multiplies the count by well under two. Chunking
    those would trade a real property for no real protection. So the cap only applies past
    `_WINDOW_BATCH_BLOWUP`x, and then bounds the pass at the count the un-capped budget would have
    produced: peak memory stays at parity with the behaviour before the cap, and the scan still reads
    the whole document, just in more passes. An explicit `batch_size` is always honoured.
    """
    if batch_size and batch_size > 0:
        return batch_size
    if window >= config_budget:
        return None                       # not capped: one pass, exactly as before
    unclamped = max(1, -(-n_windows * max(1, window // 2) // max(1, config_budget // 2)))
    if n_windows <= unclamped * _WINDOW_BATCH_BLOWUP:
        return None                       # a mild cap: keep the single shared pass
    return unclamped

def collapsed_options(qids, items) -> Dict[str, Dict[str, Optional[int]]]:
    """The questions whose options no longer have a token span each, from per-item stats.

    `total` is the number of options the question defines, not the number of markers that
    reached the sequence: a report counted from the markers would say "43/58" about a request
    where 28 options never made it into the input at all.
    """
    out = {}
    for qid, item in zip(qids, items):
        stats = item.get("options")
        if stats and stats["options_distinct"] < stats["options"]:
            out[qid] = {"total": stats["options"], "distinct": stats["options_distinct"],
                        "tokens_per_option": stats["tokens_per_option"]}
    return out


class _DynamicMultiheadAttention(nn.MultiheadAttention):
    """`nn.MultiheadAttention` that keeps the shapes it traces.

    The stock module reshapes the packed projection with sizes captured while tracing, so under
    the legacy ONNX exporter the traced sequence length becomes a constant and a model exported
    from a short dummy input only runs at that length. The parameters and the maths are the same
    here; the reshape uses only constant shape arguments (`chunk` / `unflatten` / `flatten`) and
    `scaled_dot_product_attention`, the kernel the stock path already uses when weights are not
    requested.
    """

    def forward(self, query, key, value, key_padding_mask=None, need_weights=True,
                attn_mask=None, average_attn_weights=True, is_causal=False):
        if (need_weights or self.in_proj_weight is None or self.bias_k is not None
                or self.bias_v is not None
                or (attn_mask is not None and attn_mask.dtype != torch.bool)):
            # Weight averaging, additive masks and the optional k/v bias are not on the traced
            # path; the stock implementation keeps them correct.
            return super().forward(query, key, value, key_padding_mask=key_padding_mask,
                                   need_weights=need_weights, attn_mask=attn_mask,
                                   average_attn_weights=average_attn_weights, is_causal=is_causal)
        if self.batch_first:
            query, key, value = query.transpose(0, 1), key.transpose(0, 1), value.transpose(0, 1)
        # (T, B, E) from here, matching the stock module's internals; attention runs on (B, H, T, D).
        if query is key is value:
            q, k, v = (part.unflatten(-1, (self.num_heads, self.head_dim)).permute(1, 2, 0, 3)
                       for part in F.linear(query, self.in_proj_weight, self.in_proj_bias).chunk(3, dim=-1))
        else:
            embed_dim = query.shape[-1]
            wq, wk, wv = self.in_proj_weight.split(embed_dim, dim=0)
            bq, bk, bv = ((None, None, None) if self.in_proj_bias is None
                          else self.in_proj_bias.split(embed_dim, dim=0))
            q, k, v = (
                F.linear(t, w, b).unflatten(-1, (self.num_heads, self.head_dim)).permute(1, 2, 0, 3)
                for t, w, b in ((query, wq, bq), (key, wk, bk), (value, wv, bv))
            )
        mask = None
        if attn_mask is not None:
            mask = ~attn_mask
        if key_padding_mask is not None:
            # `== 0` keeps this correct for a bool mask and for the 0 / -inf float mask the encoder
            # layer hands over (`F._canonical_mask`), where `~` would not be defined.
            keep = key_padding_mask[:, None, None, :] == 0
            mask = keep if mask is None else mask & keep
        attn = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal and mask is None,
                                              dropout_p=self.dropout if self.training else 0.0)
        attn = attn.permute(2, 0, 1, 3).flatten(-2)
        if self.batch_first:
            attn = attn.transpose(0, 1)
        return self.out_proj(attn), None


def unpermute_probs(p: np.ndarray, option_order: Optional[List[int]]) -> np.ndarray:
    """Put a slot-ordered probability row back into the caller's option order.

    `build_sequence` puts option `option_order[s]` in slot `s`, so a model row comes back
    indexed by slot. Everything downstream indexes by option -- `zip(keys, p)` for a choice,
    `arange(k) * p` for a score level, `p[1]` for noul-true -- so the row has to be inverted
    first or the probabilities end up attached to the wrong options, which is silent.

    A missing or mismatched order returns `p` untouched, so the canonical path is unaffected.
    """
    if option_order is None or len(option_order) != len(p):
        return p
    canonical = np.empty_like(p)
    canonical[np.asarray(option_order, dtype=int)] = p
    return canonical


class DecisionModel(nn.Module):
    """Bidirectional transformer encoder backbone + typed decision head."""

    def __init__(self, encoder: nn.Module, head_layers: int = 2, n_act: int = 2, dropout: float = 0.1,
                 no_init: bool = False):
        super().__init__()
        self.encoder = encoder
        d = encoder.config.hidden_size
        # `no_init` means the caller is about to load every parameter from a checkpoint, so the
        # head's initial values are pure overhead -- and not just time. transformers'
        # `no_init_weights()` patches the torch.nn.init functions, but something inside
        # nn.TransformerEncoderLayer draws from the RNG outside them, so building it still
        # advanced the global generator and made `load()` a visible side effect. On the meta
        # device no initialisation kernel runs at all; the layers are materialised empty and the
        # load fills them.
        with torch.device("meta") if no_init else nullcontext():
            nhead = max(1, d // 64)
            layer = nn.TransformerEncoderLayer(d, nhead, 4 * d, dropout, batch_first=True, norm_first=True)
            # The stock attention bakes the traced length into an exported graph; see
            # _DynamicMultiheadAttention. Same parameters, same maths, traceable shapes.
            layer.self_attn = _DynamicMultiheadAttention(d, nhead, dropout=dropout, batch_first=True)
            self.head = nn.TransformerEncoder(layer, head_layers, enable_nested_tensor=False) if head_layers > 0 else None
            self.type_emb = nn.Embedding(3, d)
            self.scorer = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))
            self.act_head = nn.Sequential(nn.Linear(d + 4, 256), nn.GELU(), nn.Linear(256, n_act))
            self.register_buffer("temperature", torch.ones(3))
        if no_init:
            # Only the modules created above are on the meta device; the encoder is already real
            # and may hold non-persistent buffers (RoPE frequencies) that to_empty would wipe.
            for module in (self.head, self.type_emb, self.scorer, self.act_head):
                if module is not None:
                    module.to_empty(device="cpu")
            self.temperature = torch.empty_like(self.temperature, device="cpu")
        self.head_checkpointing = False

    def _contextualize(self, input_ids, attention_mask, qtype, detach_encoder: bool = False,
                       position_ids=None, option_ids=None):
        if option_ids is None:
            h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        else:
            # Parallel option layout (`parallel_layout`). The head below has no positional
            # encoding, so with order-blind marker embeddings the logits permute with the options.
            masks = parallel_option_masks(attention_mask, position_ids, option_ids,
                                          getattr(self.encoder.config, "sliding_window", None))
            h = self.encoder(input_ids=input_ids, attention_mask=masks, position_ids=position_ids).last_hidden_state
        if detach_encoder:
            h = h.detach()
        h = h + self.type_emb(qtype)[:, None, :]
        if self.head is not None:
            pad = ~attention_mask.bool()
            for layer in self.head.layers:
                if self.head_checkpointing and self.training and torch.is_grad_enabled():
                    # Non-reentrant checkpointing also trains the head when its input
                    # is frozen. Default RNG preservation keeps dropout consistent.
                    h = checkpoint(layer, h, src_key_padding_mask=pad, use_reentrant=False)
                else:
                    h = layer(h, src_key_padding_mask=pad)
        return h

    def _decide(self, h, attention_mask, marker_pos, marker_mask):
        idx = marker_pos.clamp(min=0)[:, :, None].expand(-1, -1, h.size(-1))
        m = torch.gather(h, 1, idx)
        logits = self.scorer(m).squeeze(-1).float()
        logits = logits.masked_fill(~marker_mask, -1e4)

        p = torch.softmax(logits.detach(), -1)
        k = marker_mask.sum(-1).clamp(min=2).float()
        ent = -(p * torch.log(p.clamp_min(1e-9))).sum(-1) / torch.log(k)
        if p.size(-1) >= 2:
            top2 = p.topk(2, -1).values
        else:
            # A single-option question has exactly one marker, so p.topk(2, ...)
            # has nothing to select for the second slot and raises. The answer
            # is still well-defined: softmax over one logit is 1.0 regardless of
            # its value, so pad the missing second entry with 0.0 - that gives
            # the act head top1 - top2 == 1.0, the same "fully decided" signal
            # it would see for any other unambiguous top-1-vs-rest gap.
            top1 = p.topk(1, -1).values
            top2 = torch.cat([top1, torch.zeros_like(top1)], dim=-1)
        feats = torch.stack([top2[:, 0], top2[:, 0] - top2[:, 1], ent, k / 255.0], -1)
        pooled = h[:, 0].float()
        act_input = torch.cat([pooled, feats], -1)
        # Explicit-dtype exports run without autocast; keep the confidence maths in fp32,
        # then match the head's weights. Autocast already chooses the linear's input dtype.
        if not _autocast_enabled(h.device.type):
            act_input = act_input.to(self.act_head[0].weight.dtype)
        act_logits = self.act_head(act_input)
        return logits, act_logits

    def forward(self, input_ids, attention_mask, marker_pos, marker_mask, qtype, detach_encoder: bool = False,
                position_ids=None, option_ids=None):
        h = self._contextualize(input_ids, attention_mask, qtype, detach_encoder=detach_encoder,
                                position_ids=position_ids, option_ids=option_ids)
        return self._decide(h, attention_mask, marker_pos, marker_mask)


def _apply_rope_config(ecfg) -> None:
    """Carry transformers>=5 per-layer RoPE settings over to the attributes 4.x reads.

    A checkpoint re-saved by transformers 5 stores RoPE as
    `rope_parameters = {"full_attention": {"rope_theta": ...}, "sliding_attention": {...}}`.
    transformers 4.x does not know that key, so it keeps its own defaults (global 160000,
    local 10000) and any checkpoint whose sliding-attention theta differs silently runs the
    wrong RoPE base -- mmBERT is exactly that case, both of its thetas are 160000. Map the
    values onto `global_rope_theta` / `local_rope_theta`, which 4.x does read. On
    transformers 5 this is a no-op beyond re-setting the same numbers.
    """
    rope = getattr(ecfg, "rope_parameters", None)
    if not isinstance(rope, dict):
        return
    flat = rope.get("rope_theta")
    for layer_type, attr in (("full_attention", "global_rope_theta"),
                             ("sliding_attention", "local_rope_theta")):
        params = rope.get(layer_type)
        theta = params.get("rope_theta") if isinstance(params, dict) else flat
        if theta is not None and hasattr(ecfg, attr):
            setattr(ecfg, attr, float(theta))


def _no_init_weights():
    """`no_init_weights` lives in different modules across transformers versions."""
    try:
        from transformers.initialization import no_init_weights
    except ImportError:  # transformers 4.x
        from transformers.modeling_utils import no_init_weights
    return no_init_weights()


def build_model(cfg: Dict, encoder_dir: Optional[str] = None, pretrained: bool = True,
                revision: Optional[str] = None) -> DecisionModel:
    """Build the decision model described by `cfg`.

    With `pretrained=False`, or when `encoder_dir` holds a saved encoder config, nothing is
    downloaded and **no parameter is initialised**: the caller is expected to load a checkpoint
    into the result with `load_state_dict(..., strict=True)` immediately. Skipping initialisation
    keeps `load()` from spending time on, or consuming RNG for, weights it is about to overwrite.
    """
    from transformers import AutoConfig, AutoModel

    head_layers, n_act = cfg.get("head_layers", 2), len(cfg.get("act_costs", {})) + 1
    is_rex = cfg.get("arch") in ("rex", "rex2")
    if not pretrained or (encoder_dir and os.path.exists(encoder_dir)):
        ecfg = AutoConfig.from_pretrained(encoder_dir or cfg["encoder"])
        _apply_rope_config(ecfg)
        with _no_init_weights():
            enc = AutoModel.from_config(ecfg, attn_implementation="sdpa")
        if is_rex:
            from .rex_model import RexDecisionModel
            rex_cfg = cfg.get("rex", {})
            return RexDecisionModel(
                enc,
                tools=rex_cfg.get("tools", []),
                type_tokens=rex_cfg.get("type_tokens", {}),
                tool_hidden=rex_cfg.get("tool_hidden"),
                tool_dropout=rex_cfg.get("tool_dropout", 0.1),
                tool_prior=rex_cfg.get("tool_prior"),
                head_layers=head_layers,
                n_act=n_act,
                no_init=True,
                tool_meta=rex_cfg.get("tool_meta"),
            )
        return DecisionModel(enc, head_layers, n_act, no_init=True)
    # Training-time Hub load of the base encoder; allow pinning it like the checkpoints.
    kw = {"attn_implementation": "sdpa"}
    if revision:
        kw["revision"] = revision
    enc = AutoModel.from_pretrained(cfg["encoder"], **kw)
    if is_rex:
        from .rex_model import RexDecisionModel
        rex_cfg = cfg.get("rex", {})
        return RexDecisionModel(
            enc,
            tools=rex_cfg.get("tools", []),
            type_tokens=rex_cfg.get("type_tokens", {}),
            tool_hidden=rex_cfg.get("tool_hidden"),
            tool_dropout=rex_cfg.get("tool_dropout", 0.1),
            tool_prior=rex_cfg.get("tool_prior"),
            head_layers=head_layers,
            n_act=n_act,
            tool_meta=rex_cfg.get("tool_meta"),
        )
    return DecisionModel(enc, head_layers, n_act)


def proper_reward(
    q: torch.Tensor,
    target: torch.Tensor,
    qtype: torch.Tensor,
    mask: torch.Tensor,
    w_sph: float = 0.5,
    w_rps: float = 1.0,
    log_floor: float = -9.21,
) -> torch.Tensor:
    """Strictly proper scoring rule reward: log score + spherical score + ranked probability score.

    q: [..., N, K] reported distributions
    target: [N, K] (one-hot or soft target distributions)
    """
    q = q * mask
    logq = torch.log(q.clamp_min(1e-12)).clamp_min(log_floor)
    log_score = (target * logq).sum(-1)
    sph = (target * q).sum(-1) / q.norm(dim=-1).clamp_min(1e-9)
    r = log_score + w_sph * sph
    is_score = (qtype == QTYPES["score"]).float()
    if is_score.any():
        k = mask.sum(-1).clamp(min=2).float()
        cdf_q = torch.cumsum(q, -1)
        cdf_t = torch.cumsum(target, -1)
        rps = (((cdf_q - cdf_t) ** 2) * mask).sum(-1) / (k - 1)
        r = r - w_rps * rps * is_score
    return r


def td_lambda_targets(p_true: torch.Tensor, batch: Dict, lam: float = 1.0) -> torch.Tensor:
    """TD(lambda) targets for multi-turn conversation trajectories."""
    target = batch["target"].clone()
    groups = batch.get("ep_group")
    if groups is None:
        return target
    for g in torch.unique(groups[groups >= 0]).tolist():
        idx = (groups == g).nonzero(as_tuple=True)[0]
        idx = idx[torch.argsort(batch["ep_step"][idx])]
        y = batch["target"][idx[-1], 1]
        G = y
        for j in range(len(idx) - 1, -1, -1):
            if j < len(idx) - 1:
                G = (1 - lam) * p_true[idx[j + 1]] + lam * G
            target[idx[j], 0], target[idx[j], 1] = 1 - G, G
    return target


def ece_score(conf: np.ndarray, correct: np.ndarray, bins: int = 15) -> float:
    """Expected Calibration Error across confidence bins."""
    if len(conf) == 0:
        return float("nan")
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        sel = (conf >= lo if i == 0 else conf > lo) & (conf <= hi)
        if sel.any():
            e += sel.mean() * abs(conf[sel].mean() - correct[sel].mean())
    return float(e)


def answer_confidence(p: np.ndarray, k: int) -> float:
    """Probability mass on the answer being reported: max(p).

    This is the quantity temperature scaling fits, and the quantity every calibration figure in
    this repository is computed on -- both benchmark harnesses take `conf = max(probs)` before
    calling `ece_score`. The README's gating section relies on the property that goes with it:
    of the answers returned at confidence c, about c of them are right. That property is
    conditional, and the condition is not met by default -- it holds only after the temperatures
    have been fitted and validated on held-out data for this checkpoint and this option count.
    The shipped checkpoints are over-confident: `choice:11+` is fitted at 0.1006, a ~10x
    sharpener that would publish a genuine 0.24 top probability as 0.9999. `clamp_temperature`
    refuses it and applies `TEMP_MIN` (0.5) instead, which still sharpens 2x and publishes that
    same 0.24 as 0.4993 -- so a threshold applied to these numbers still selects below model
    accuracy (issue #394). The clamp bounds the damage; it does not calibrate anything.

    `confidence_from_probs` below reports a different quantity on a different scale and carries
    no such guarantee, so the two must not be compared against the same threshold.
    """
    if k < 1:
        return 1.0
    return float(np.clip(np.max(p[:k]), 0.0, 1.0))


def confidence_from_probs(p: np.ndarray, k: int) -> float:
    """Normalized Shannon entropy confidence: 1 - H(p) / log(k).

    How concentrated the whole distribution is. Useful, but not calibrated: it is not what
    temperature scaling fits and not what the reported ECE measures. See `answer_confidence`.
    """
    if k < 2:
        return 1.0
    p = p[:k]
    ent = -(p * np.log(np.clip(p, 1e-12, 1.0))).sum()
    return float(np.clip(1.0 - ent / math.log(k), 0.0, 1.0))


def temp_bucket(qtype: int, k: int) -> str:
    size = "2" if k <= 2 else "3-5" if k <= 5 else "6-10" if k <= 10 else "11+"
    return "%s:%s" % (QTYPE_NAMES[int(qtype)], size)


# A fitted temperature below 1 sharpens the logits instead of softening them. The shipped
# `choice:11+` bucket is 0.1006, which multiplies them ~10x: a 0.24 top probability is published as
# 0.99, so a caller gating on confidence is told a coin flip is a certainty. No honest calibration
# needs to sharpen this hard, so refuse to apply one that does.
TEMP_MIN = 0.5
TEMP_MAX = 5.0


def clamp_temperature(t, lo: float = TEMP_MIN, hi: float = TEMP_MAX) -> float:
    """A usable temperature: `t` confined to [lo, hi], falling back to 1.0 if it is not a number.

    A bool is not a number either: `True`/`False` used to float to 1.0/0.0 here and read as
    fitted/sharpening temperatures, the same class of quiet acceptance `check_min_confidence`
    already refuses. A custom `lo`/`hi` still bounds either way.
    """
    if isinstance(t, bool):
        return 1.0
    try:
        t = float(t)
    except (TypeError, ValueError):
        return 1.0
    if t != t or t in (float("inf"), float("-inf")):    # NaN / inf
        return 1.0
    return min(hi, max(lo, t))


def resolve_lang_temperatures(raw: Optional[Dict[str, Any]],
                              base_temperature: Sequence[float]) -> Dict[str, Dict[str, Any]]:
    """Parse the `lang_temperatures` option into `{language: {temperature, temperature_by_options}}`.

    One implementation, because `Agent` and `ONNXAgent` both accept this option and both promise
    the same confidences for it. Reading it with `cfg.get(...)` and `len(...)` before checking the
    shape of either raised `AttributeError` and `TypeError` for exactly the inputs the
    `ValueError` below is written for, after the whole checkpoint had loaded:

        {"de": {"temperature": 2}}       -> TypeError: object of type 'int' has no len()
        {"de": {"temperature": None}}    -> TypeError: object of type 'NoneType' has no len()
        {"de": None}                     -> AttributeError: 'NoneType' object has no attribute 'get'

    A `null` entry or a `null` temperature both mean "inherit the checkpoint's own", which is how
    the `laya-ts` port reads the same option (`agent.ts:322-327`).
    """
    resolved: Dict[str, Dict[str, Any]] = {}
    for lang, cfg in (raw or {}).items():
        if not isinstance(lang, str):
            raise ValueError("Language override keys must be strings, got %r" % (lang,))
        norm = lang.split("-")[0].lower()
        if cfg is None:
            cfg = {}
        if not isinstance(cfg, dict):
            raise ValueError("Language override %r must be a mapping, got %s"
                             % (lang, type(cfg).__name__))
        t_raw = cfg.get("temperature")
        if t_raw is None:
            t_raw = base_temperature
        if not isinstance(t_raw, (list, tuple)) or len(t_raw) != 3:
            raise ValueError("Language override %r temperature must be a list of 3 floats, got %r"
                             % (lang, t_raw))
        tbo_raw = cfg.get("temperature_by_options") or {}
        if not isinstance(tbo_raw, dict):
            raise ValueError("Language override %r temperature_by_options must be a mapping of "
                             "bucket -> float, got %s" % (lang, type(tbo_raw).__name__))
        resolved[norm] = {
            "temperature": [clamp_temperature(t) for t in t_raw],
            "temperature_by_options": {k: clamp_temperature(v) for k, v in tbo_raw.items()},
        }
    return resolved


def amp_dtype(name: Optional[str]) -> torch.dtype:
    return torch.bfloat16 if name == "bf16" else torch.float16


def collate_items(batch, pad_id: int):
    items = [it for group in batch for it in group]
    if not items:
        return None
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    has_target = any("target" in it for it in items)
    target = torch.zeros((n, kmax), dtype=torch.float32) if has_target else None

    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        if has_target and "target" in it:
            if len(it["target"]) > k:
                # Otherwise this lands as "The expanded size of the tensor (k) must match the
                # existing size (kmax)" from inside the assignment, which says nothing about the
                # actual mistake: a target with more entries than the item has options. The limit
                # is this item's own marker count, not the batch-wide kmax: in a mixed-width
                # batch a longer sibling row must not legitimise extra entries (#311).
                raise ValueError(
                    "collate_items: item %d has %d target entries but only %d marker positions; "
                    "a target needs one entry per option" % (i, len(it["target"]), k))
            target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)

    res = {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it.get("label", -1) for it in items]),
        "meta": [{k: it[k] for k in it if k not in ("ids", "markers", "target", "layout")} for it in items],
    }
    if target is not None:
        res["target"] = target
    if any("layout" in it for it in items):
        # All or none: a batch that mixes layouts would run half its rows through the wrong masks
        if not all("layout" in it for it in items):
            raise ValueError("collate_items: some items carry a parallel layout and some do not")
        pos = torch.zeros((n, L), dtype=torch.long)
        opt = torch.zeros((n, L), dtype=torch.long)
        for i, it in enumerate(items):
            pos[i, : len(it["ids"])] = torch.tensor(it["layout"]["position_ids"])
            opt[i, : len(it["ids"])] = torch.tensor(it["layout"]["option_ids"])
        res["position_ids"], res["option_ids"] = pos, opt
    return res


In [ ]:
%%writefile {REXLIB}/rex/calibrate.py
"""Fit and persist per-bucket temperatures for Laya logits.

Ports the notebook LBFGS fitter (NLL on softmax(z/T)) and groups records by
`temp_bucket(qtype, k)` so the runtime's `temperature_by_options` lookup has a writer.
Fitting is CPU-side: records are `(qtype, logits, target, k)` and do not load Hub weights.

Two sample floors, on purpose:

- `MIN_BUCKET_N` is the per-bucket floor. Buckets with fewer examples are omitted from
  `temperature_by_options`; the type-level scalar covers them at lookup time.
- `MIN_TYPE_N` is the lower floor used only for that type-level scalar. It stays at the
  previous floor of 10 so a realistic dataset that fills no single bucket still gets a
  temperature instead of remaining at 1.0. Do not raise it to `MIN_BUCKET_N`.

Fitted temperatures go through `clamp_temperature` and therefore `[TEMP_MIN, TEMP_MAX]`
from `laya.common`, the same guard checkpoint load uses. There is no second pair of
bounds in this module.
"""
import warnings
from numbers import Integral
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import torch

from .common import (
    QTYPES,
    TEMP_MAX,
    TEMP_MIN,
    clamp_temperature,
    collate_items,
    ece_score,
    temp_bucket,
)

# Per-bucket floor. Below this, `temperature_by_options` omits the bucket.
MIN_BUCKET_N = 2000
# Type-level floor only. Lower than MIN_BUCKET_N; see the module docstring.
MIN_TYPE_N = 10
# Fraction of each bucket held out when `compute_ece=True`. Unused otherwise.
ECE_HOLDOUT_FRAC = 0.2
# Payload schema. Files written before identity was recorded have no version key;
# readers treat that absence as version 1.
CALIBRATION_VERSION = 2
N_QTYPES = len(QTYPES)

# (qtype:int, logits:1d, target:1d, k:int)
Record = Tuple[int, Any, Any, int]

_CONFIG_IDENTITY_SKIP = ("temperature", "temperature_by_options")


def _vec(x) -> np.ndarray:
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    return np.asarray(x, dtype=np.float32).reshape(-1)


def _validated_pair(logits, target, k=None):
    vectors = []
    for name, value in (("logits", logits), ("target", target)):
        if isinstance(value, torch.Tensor):
            value = value.detach().cpu().numpy()
        try:
            vector = np.asarray(value)
            if vector.ndim != 1 or vector.dtype.kind not in "iuf":
                raise ValueError("%s must be a one-dimensional numeric vector" % name)
            with np.errstate(over="ignore", invalid="ignore"):
                vector = vector.astype(np.float32)
        except (TypeError, ValueError, OverflowError) as exc:
            raise ValueError("%s must be a one-dimensional numeric vector" % name) from exc
        if not np.isfinite(vector).all():
            raise ValueError("%s must contain only finite values" % name)
        vectors.append(vector)
    logits, target = vectors
    if len(logits) != len(target):
        raise ValueError("logits and target must have the same length")
    if k is None:
        k = len(logits)
    if isinstance(k, bool) or not isinstance(k, Integral) or k < 1 or k > len(logits):
        raise ValueError("k must be an integer between 1 and the vector length")
    logits, target = logits[:k], target[:k]
    if (target < 0).any() or not np.isclose(target.sum(), 1.0, rtol=1e-5, atol=1e-6):
        raise ValueError("target must be a nonnegative probability vector summing to 1")
    return logits, target, int(k)


def _pairs_to_tensors(pairs: Sequence) -> Tuple[torch.Tensor, torch.Tensor]:
    kmax = max(len(_vec(z)) for z, _ in pairs)
    z_mat = torch.full((len(pairs), kmax), -1e4)
    t_mat = torch.zeros((len(pairs), kmax))
    for i, (z, t) in enumerate(pairs):
        z = _vec(z)
        t = _vec(t)
        n = len(z)
        z_mat[i, :n] = torch.from_numpy(np.ascontiguousarray(z[:n]))
        t_mat[i, :n] = torch.from_numpy(np.ascontiguousarray(t[:n]))
    return z_mat, t_mat


def fit_one_temperature(pairs: Sequence, min_n: Optional[int] = None) -> float:
    """Fit one scalar T by NLL + LBFGS on log T.

    The result is `clamp_temperature` of the optimised scale, so it lies in
    `[TEMP_MIN, TEMP_MAX]` (or is the neutral 1.0 when the value is not a number).
    Returns 1.0 when fewer than `min_n` pairs are given. `min_n` defaults to
    `MIN_BUCKET_N` (the per-bucket floor). Type-level fits pass `MIN_TYPE_N`, which
    is lower, so a dataset that fills no bucket still gets a scalar instead of
    staying at 1.0.
    """
    if min_n is None:
        min_n = MIN_BUCKET_N
    if isinstance(min_n, bool) or not isinstance(min_n, Integral) or min_n < 1:
        raise ValueError("min_n must be a positive integer")
    sel = []
    for pair in pairs:
        try:
            z, t = pair
        except (TypeError, ValueError) as exc:
            raise ValueError("pair must be (logits, target)") from exc
        z, t, _k = _validated_pair(z, t)
        sel.append((z, t))
    if len(sel) < min_n:
        return 1.0
    z_mat, t_mat = _pairs_to_tensors(sel)
    with torch.enable_grad():
        log_t = torch.zeros(1, requires_grad=True)
        opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

        def closure():
            opt.zero_grad()
            loss = -(t_mat * torch.log_softmax(z_mat / log_t.exp(), -1)).sum(-1).mean()
            loss.backward()
            return loss

        opt.step(closure)
    fitted = float(log_t.detach().exp().item())
    return clamp_temperature(fitted, TEMP_MIN, TEMP_MAX)


def _iter_records(records: Iterable) -> List[Tuple[int, np.ndarray, np.ndarray, int]]:
    out = []
    for rec in records:
        if not isinstance(rec, (list, tuple)) or len(rec) not in (3, 4):
            raise ValueError("record must be (qtype, logits, target[, k])")
        if len(rec) == 4:
            qtype, logits, target, k = rec
            if k is None:
                raise ValueError("k must be an integer between 1 and the vector length")
        else:
            qtype, logits, target = rec
            k = None
        if isinstance(qtype, bool) or not isinstance(qtype, Integral) or not 0 <= qtype < N_QTYPES:
            raise ValueError("qtype must be an integer between 0 and %d" % (N_QTYPES - 1))
        logits, target, k = _validated_pair(logits, target, k)
        out.append((int(qtype), logits, target, k))
    return out


def _softmax(z, t_scale: float) -> np.ndarray:
    z = np.asarray(z, dtype=np.float64) / max(1e-3, float(t_scale))
    z = z - z.max()
    p = np.exp(z)
    return p / p.sum()


def _ece_report(recs, temperature, temperature_by_options) -> Dict[str, Any]:
    conf_b, cor_b, conf_a, cor_a = [], [], [], []
    for qt, z, t, k in recs:
        y = int(np.argmax(t[:k]))
        p0 = _softmax(z[:k], 1.0)
        conf_b.append(float(p0.max()))
        cor_b.append(float(int(p0.argmax()) == y))
        t_scale = temperature_by_options.get(temp_bucket(qt, k), temperature[qt])
        p1 = _softmax(z[:k], t_scale)
        conf_a.append(float(p1.max()))
        cor_a.append(float(int(p1.argmax()) == y))
    return {
        "ece_before": ece_score(np.asarray(conf_b), np.asarray(cor_b)),
        "ece_after": ece_score(np.asarray(conf_a), np.asarray(cor_a)),
        "n": float(len(recs)),
    }


def _group_records(recs):
    by_type = {qt: [] for qt in range(N_QTYPES)}
    by_bucket = {}
    for qt, z, t, k in recs:
        if qt in by_type:
            by_type[qt].append((z, t))
        key = temp_bucket(qt, k)
        by_bucket.setdefault(key, []).append((z, t))
    return by_type, by_bucket


def _ece_split(recs, seed):
    """Split records into fit and eval, stratified by `temp_bucket`.

    Returns `(fit_recs, eval_recs, excluded_buckets)`. The split is a deterministic
    partition (`numpy.random.RandomState(seed)`): every record is in exactly one side.
    A bucket whose fit side would fall below `MIN_BUCKET_N` is not split; all of its
    records stay in `fit_recs` and its key is listed in `excluded_buckets`.
    """
    by_bucket = {}
    for i, rec in enumerate(recs):
        qt, _z, _t, k = rec
        by_bucket.setdefault(temp_bucket(qt, k), []).append(i)

    rng = np.random.RandomState(seed)
    fit_idx = []
    eval_idx = []
    excluded = []
    for key in sorted(by_bucket):
        idxs = by_bucket[key]
        n = len(idxs)
        n_eval = int(round(n * ECE_HOLDOUT_FRAC))
        if n - n_eval < MIN_BUCKET_N:
            fit_idx.extend(idxs)
            excluded.append(key)
            continue
        perm = rng.permutation(n)
        for j in perm[:n_eval]:
            eval_idx.append(idxs[int(j)])
        for j in perm[n_eval:]:
            fit_idx.append(idxs[int(j)])

    fit_idx.sort()
    eval_idx.sort()
    fit_recs = [recs[i] for i in fit_idx]
    eval_recs = [recs[i] for i in eval_idx]
    return fit_recs, eval_recs, excluded


def _fit_groups(by_type, by_bucket):
    temperature = [1.0] * N_QTYPES
    for qt in range(N_QTYPES):
        pairs = by_type[qt]
        # MIN_TYPE_N, not MIN_BUCKET_N: the scalar has to exist when every bucket is small.
        if len(pairs) >= MIN_TYPE_N:
            temperature[qt] = fit_one_temperature(pairs, min_n=MIN_TYPE_N)

    temperature_by_options = {}
    for key, pairs in by_bucket.items():
        if len(pairs) < MIN_BUCKET_N:
            continue
        temperature_by_options[key] = fit_one_temperature(pairs)
    return temperature, temperature_by_options


def fit_temperature_map(records: Iterable, compute_ece: bool = False, seed: int = 0) -> Dict[str, Any]:
    """Fit type-level scalars and per-bucket temperatures.

    `MIN_BUCKET_N` is the per-bucket floor: smaller buckets are omitted from
    `temperature_by_options` and the type-level scalar covers them. `MIN_TYPE_N` is
    the separate, lower floor for that scalar only.

    `compute_ece=False` (the default, and the path `Agent.fit_temperatures` stores)
    fits on every record and returns no `report` key. `seed` is ignored on this path.

    `compute_ece=True` holds out `ECE_HOLDOUT_FRAC` of each bucket, stratified by
    `temp_bucket`, using `seed` so the same records always split the same way. Temperatures
    are fit on the remainder only and ECE is scored only on the held-out records.
    `report["n"]` is the number of records passed in; `report["n_eval"]` is the held-out
    count the ECE rests on. A bucket that would drop below `MIN_BUCKET_N` after the
    holdout is fit on all of its records, left out of the eval set, and named in
    `report["buckets_excluded_from_eval"]` instead of being dropped. `n_by_bucket`
    always counts the full input, including when the fit itself used a subset.
    """
    recs = _iter_records(records)
    all_by_type, all_buckets = _group_records(recs)
    n_by_bucket = {key: len(pairs) for key, pairs in all_buckets.items()}

    if compute_ece:
        _fit_recs, eval_recs, excluded = _ece_split(recs, seed)
        by_type, by_bucket = _group_records(_fit_recs)
    else:
        eval_recs = None
        excluded = []
        by_type, by_bucket = all_by_type, all_buckets

    temperature, temperature_by_options = _fit_groups(by_type, by_bucket)
    out: Dict[str, Any] = {
        "temperature": temperature,
        "temperature_by_options": temperature_by_options,
        "n_by_bucket": n_by_bucket,
    }
    if compute_ece:
        report = _ece_report(eval_recs, temperature, temperature_by_options)
        # `n` stays the size of the input. `n_eval` is the held-out count ECE used.
        report["n"] = float(len(recs))
        report["n_eval"] = float(len(eval_recs))
        report["buckets_excluded_from_eval"] = excluded
        out["report"] = report
    return out


fit_temperatures = fit_temperature_map


# Per-bucket floor for abstention thresholds. Lower than `MIN_BUCKET_N` (which governs temperature
# fitting): a threshold is a single order statistic of the calibrated confidences, so it stabilises
# on far fewer examples than an LBFGS temperature does. Buckets below this are omitted; the caller's
# scalar `min_confidence` (or the map's "default") covers them.
MIN_ABSTAIN_BUCKET_N = 100


def _select_abstention_threshold(pairs: Sequence[Tuple[float, int]], target_error: float,
                                 conservative: bool) -> float:
    """Smallest confidence `tau` whose accepted set (`conf >= tau`) keeps error <= `target_error`.

    `pairs` are `(confidence, correct)`. Sweeping from the most confident down maximises coverage
    at the target risk (the selective-classification / split-conformal cut). `conservative` adds one
    pseudo-error so a bucket does not clear the gate on a lucky short run. Returns 1.0 when no cut
    holds the risk -- the bucket is too unreliable to accept anything short of a reported certainty.
    """
    ordered = sorted(pairs, key=lambda cc: cc[0], reverse=True)
    levels = sorted({c for c, _ in ordered}, reverse=True)
    n = len(ordered)
    n_acc = n_err = idx = 0
    best = None
    # Evaluate the error over the WHOLE accepted set {conf >= tau} at each distinct level, not
    # incrementally within a tie: a threshold accepts every answer at its own confidence, so a tie
    # group's errors must all be counted before the level is judged (else the cut sinks into a bad
    # cohort on its first few correct members).
    for tau in levels:
        while idx < n and ordered[idx][0] >= tau:
            n_acc += 1
            n_err += 0 if ordered[idx][1] else 1
            idx += 1
        rate = (n_err + 1.0) / (n_acc + 1.0) if conservative else (n_err / n_acc)
        if n_acc > 0 and rate <= target_error:
            best = tau
    if best is None:
        return 1.0
    return float(min(1.0, max(0.0, best)))


def fit_abstention_thresholds(records: Iterable, temperature: Sequence[float],
                              temperature_by_options: Dict[str, float], *,
                              binning_map: Optional[Dict[str, Dict[str, Any]]] = None,
                              target_error: float = 0.10,
                              min_bucket_n: int = MIN_ABSTAIN_BUCKET_N,
                              conservative: bool = True) -> Dict[str, float]:
    """Fit a per-`temp_bucket` abstention threshold so a gate keeps a target error in every bucket.

    A single `min_confidence` does not transfer across option counts (#394): the calibrated
    confidence of a 2-option and a 12-option answer live on different scales, so one cut over- or
    under-abstains depending on the question. This fits one cut per bucket instead, keyed exactly
    like `temperature_by_options` (`common.temp_bucket`, e.g. ``"choice:3-5"``), and the result is a
    `min_confidence` map that :func:`laya.confidence.check_min_confidence` /
    :func:`laya.confidence.apply_confidence_gate` accept directly.

    `records` are the same `(qtype, logits, target[, k])` tuples `fit_temperature_map` consumes
    (`records_from_labeled` builds them). Confidence is the **calibrated** `max(p)` -- the logits are
    scaled by the fitted `temperature` / `temperature_by_options` first, so thresholds and the
    numbers the runtime reports are on the same scale. `target_error` is the tolerated error among
    accepted answers; `min_bucket_n` omits buckets too small to fit, and `conservative` adds a
    one-sample margin. The thresholds are empirical cuts on the calibration set, not a formal
    coverage guarantee -- validate on held-out data (`fit_temperature_map(..., compute_ece=True)`
    gives a held-out split) for a production gate.

    Pass `binning_map` when the agent that will serve these thresholds has one installed -- by
    `Agent.fit_binning`, or by a calibration payload that carries `binning_map` -- because the
    runtime recalibrates `answer_confidence` through that map before anything reads it, so a cut
    fitted without it is a cut on a scale the gate never sees. The thresholds are then on the binned
    scale, and the order the two were fitted in stops mattering. Measured on 1,200 synthetic
    12-option records at `target_error=0.10`: the cut fitted without a map holds 9.8% error over 50%
    coverage on un-binned confidences, and admits 94.5% of answers at 25.6% error once the same
    number is compared against binned ones.
    """
    if not 0.0 <= target_error <= 1.0:
        raise ValueError("target_error must be in [0.0, 1.0], got %r" % (target_error,))
    recs = _iter_records(records)
    by_bucket: Dict[str, List[Tuple[float, int]]] = {}
    for qt, z, t, k in recs:
        y = int(np.argmax(t[:k]))
        bucket = temp_bucket(qt, k)
        t_scale = temperature_by_options.get(bucket, temperature[qt])
        p = _softmax(z[:k], t_scale)
        conf = float(p.max())
        if binning_map:
            # The runtime bins before anyone reads `answer_confidence` (`Agent._decode_answers`),
            # so the cut has to be chosen on the binned scale or it gates a different quantity.
            conf = apply_binning_map(conf, bucket, binning_map)
        # `_decode_answers` reports `answer_confidence` rounded to 4 decimals, after binning, and
        # the gate compares that. A cut picked at full precision can sit above the rounded value of
        # its own cohort (bin 5/6 is reported as 0.8333 < 0.83333...), which abstains all of it.
        conf = round(conf, 4)
        by_bucket.setdefault(bucket, []).append((conf, int(int(p.argmax()) == y)))
    out: Dict[str, float] = {}
    for key, pairs in by_bucket.items():
        if len(pairs) < min_bucket_n:
            continue
        out[key] = _select_abstention_threshold(pairs, target_error, conservative)
    return out


# Per-bucket floor for histogram-binning recalibration. Higher than the abstention floor because a
# binning map splits each bucket's examples across `bins`, so each bin needs its own sample; lower
# than the temperature floor because binning is a count per bin, not an optimisation.
MIN_BINNING_BUCKET_N = 200


def fit_binning_map(records: Iterable, temperature: Sequence[float],
                    temperature_by_options: Dict[str, float], *, bins: int = 15,
                    min_bucket_n: int = MIN_BINNING_BUCKET_N) -> Dict[str, Dict[str, Any]]:
    """Fit a per-`temp_bucket` histogram-binning recalibration map for `answer_confidence`.

    Temperature scaling applies one scalar per bucket; it cannot fix a bucket whose reliability
    curve is not a simple sharpening/softening (the pathological `choice:11+` the shipped English
    checkpoint carries is one). Histogram binning is the non-parametric alternative: split the
    calibrated confidences of a bucket into `bins` equal-width bins over [0, 1], and map every
    confidence that lands in a bin to that bin's empirical accuracy. It needs no monotonicity
    assumption and no extra dependency (NumPy only; isotonic regression would pull in scikit-learn).

    `records` are the same `(qtype, logits, target[, k])` tuples `fit_temperature_map` consumes;
    confidence is the calibrated `max(p)` (logits scaled by the fitted `temperature` /
    `temperature_by_options` first), so a binning map composes on top of a temperature map rather
    than replacing it. Returns `{bucket: {"bins": N, "values": [recalibrated confidence per bin]}}`;
    buckets below `min_bucket_n` are omitted. Apply it with :func:`apply_binning_map`. An empty bin
    (a confidence range the calibration set never produced) maps to its own midpoint, i.e. leaves
    that region unchanged, so an unseen value is never recalibrated to a fabricated 0.
    """
    if bins < 1:
        raise ValueError("bins must be >= 1, got %r" % (bins,))
    recs = _iter_records(records)
    by_bucket: Dict[str, List[Tuple[float, int]]] = {}
    for qt, z, t, k in recs:
        y = int(np.argmax(t[:k]))
        bucket = temp_bucket(qt, k)
        t_scale = temperature_by_options.get(bucket, temperature[qt])
        p = _softmax(z[:k], t_scale)
        by_bucket.setdefault(bucket, []).append((float(p.max()), int(int(p.argmax()) == y)))
    out: Dict[str, Dict[str, Any]] = {}
    for key, pairs in by_bucket.items():
        if len(pairs) < min_bucket_n:
            continue
        conf = np.asarray([c for c, _ in pairs], dtype=float)
        corr = np.asarray([c for _, c in pairs], dtype=float)
        idx = np.clip((conf * bins).astype(int), 0, bins - 1)
        values = []
        for b in range(bins):
            mask = idx == b
            values.append(float(corr[mask].mean()) if mask.any() else (b + 0.5) / bins)
        out[key] = {"bins": bins, "values": values}
    return out


def apply_binning_map(confidence: float, bucket: str,
                      binning_map: Dict[str, Dict[str, Any]]) -> float:
    """Recalibrate one `answer_confidence` for its option-count `bucket` (`common.temp_bucket`).

    Returns the confidence unchanged when the map has no entry for the bucket, so a bucket the map
    was not fit for passes through rather than being forced to a wrong value.
    """
    entry = binning_map.get(bucket)
    if not entry:
        return float(confidence)
    bins = int(entry["bins"])
    b = min(bins - 1, max(0, int(float(confidence) * bins)))
    return float(entry["values"][b])


@torch.no_grad()
def records_from_labeled(agent, pairs: Sequence) -> List[Record]:
    """Collect CPU records from `(state, questions, targets)`.

    Logits are the raw rows `_decode_answers` scales, not the probabilities it returns.
    `_encode_state` builds one state's items, `_forward` runs them, and `_option_logits`
    slices each question to the same marker width `_decode_answers` divides by temperature.
    No Hub download and no `model.safetensors` write. Tests stay weight-free by passing
    synthetic records to `fit_temperature_map` instead of calling this.
    """
    # Imported lazily: `agent` imports this module at load, and `_option_logits` is defined
    # on the way through that import.
    from .agent import _option_logits

    records: List[Record] = []
    for state, questions, targets in pairs:
        ids = list(questions.keys())
        if not ids:
            continue
        for qid in ids:
            agent._check_question(qid, questions[qid])
        internal = {qid: agent._to_internal(questions[qid]) for qid in ids}
        items = agent._encode_state(state, ids, internal)
        batch = collate_items([items], agent.tok.pad_token_id)
        if batch is None:
            continue
        logits, _act = agent._forward(batch)
        rows = _option_logits(logits, items, 0)
        for j, qid in enumerate(ids):
            k = len(items[j]["markers"])
            qt = int(items[j]["qtype"])
            records.append((qt, _vec(rows[j]), _vec(targets[qid])[:k], k))
    return records


def _config_identity(cfg):
    """Checkpoint config without the temperature fields this file itself stores."""
    if cfg is None:
        return None
    return {str(k): cfg[k] for k in cfg if k not in _CONFIG_IDENTITY_SKIP}


def calibration_payload(
    temperature,
    temperature_by_options,
    model_id_or_path=None,
    subfolder=None,
    config=None,
    binning_map=None,
) -> Dict[str, Any]:
    """JSON body written by `Agent.save_calibration` (no weights).

    `version` is `CALIBRATION_VERSION`. Payloads written before identity was recorded have
    no version key; `apply_calibration_payload` treats that as version 1 and still loads.

    `model_id_or_path`, `subfolder`, and `config` say which checkpoint the map was fitted
    against. `config` is `rl_agent_config.json` without `temperature` /
    `temperature_by_options` (those live at the top of this payload).

    `binning_map` is the optional histogram-binning recalibration map fitted by
    `fit_binning_map`; the key is omitted when no map is installed, which is `None`, not an empty
    map. A map that fitted nothing -- `{}`, the return value when no bucket reached the sample
    floor -- is still written, so the round trip through :func:`apply_calibration_payload` gives
    `{}` back rather than `None`.
    """
    payload = {
        "version": CALIBRATION_VERSION,
        "temperature": [float(x) for x in temperature],
        "temperature_by_options": {
            str(k): float(v) for k, v in dict(temperature_by_options).items()
        },
        "model_id_or_path": model_id_or_path,
        "subfolder": subfolder,
        "config": _config_identity(config),
    }
    if binning_map is not None:
        payload["binning_map"] = binning_map
    return payload


def _warn_if_identity_mismatch(obj, payload) -> None:
    recorded_model = payload.get("model_id_or_path")
    recorded_sub = payload.get("subfolder")
    recorded_cfg = payload.get("config")
    current_model = getattr(obj, "model_id_or_path", None)
    current_sub = getattr(obj, "subfolder", None)
    current_cfg = _config_identity(getattr(obj, "cfg", None))
    if (recorded_model, recorded_sub, recorded_cfg) == (current_model, current_sub, current_cfg):
        return
    cfg_note = ""
    if recorded_cfg != current_cfg:
        cfg_note = " Checkpoint config identity differs."
    warnings.warn(
        "calibration was fitted for model_id_or_path=%r subfolder=%r but this agent is "
        "model_id_or_path=%r subfolder=%r.%s Loading the temperatures anyway."
        % (recorded_model, recorded_sub, current_model, current_sub, cfg_note),
        UserWarning,
        stacklevel=3,
    )


def _rejected_temperatures(entries):
    """Names whose raw value is not the number `clamp_temperature` kept, matching checkpoint load."""
    rejected = []
    for name, raw, applied in entries:
        try:
            if float(raw) == applied:
                continue
        except (TypeError, ValueError):
            pass
        rejected.append("%s=%r -> %g" % (name, raw, applied))
    return rejected


def _install_temperatures(obj, temperature, temperature_by_options, warn: bool = True) -> None:
    """Store a map the way checkpoint load does: raw values, and the clamped ones inference reads.

    `warn=False` is the fitter path: those values were already passed through `clamp_temperature`,
    so a second pass must not report them as a bad file.
    """
    raw_t = list(temperature)
    applied_t = [clamp_temperature(t) for t in raw_t]
    raw_b = {str(k): v for k, v in dict(temperature_by_options).items()}
    applied_b = {k: clamp_temperature(v) for k, v in raw_b.items()}
    obj.temperature_raw = raw_t
    obj.temperature = applied_t
    obj.temperature_by_options_raw = raw_b
    obj.temperature_by_options = applied_b
    if not warn:
        return
    entries = [("temperature[%d]" % i, raw, applied_t[i]) for i, raw in enumerate(raw_t)]
    entries += [(key, raw_b[key], applied_b[key]) for key in raw_b]
    rejected = _rejected_temperatures(entries)
    if rejected:
        warnings.warn(
            "laya: calibration temperatures are invalid or outside [%g, %g]; "
            "using %s. Treat confidence from the affected entries as uncalibrated."
            % (TEMP_MIN, TEMP_MAX, ", ".join(rejected)),
            RuntimeWarning,
            stacklevel=3,
        )


def apply_calibration_payload(obj, payload: Dict[str, Any]) -> None:
    """Copy `temperature`, `temperature_by_options` and `binning_map` onto `obj`.

    Those are the three fields this reads and the three it writes. `binning_map` is installed
    whatever the payload holds, including nothing: a payload with no ``binning_map`` key -- every
    file written before histogram binning existed, and every agent that saved before
    :meth:`Agent.fit_binning` ran -- installs `None`, which clears a map this object already
    fitted. The file is the whole calibration state, not a patch onto the current one.

    A missing `version` is version 1 (temperatures only, no checkpoint identity). Version
    `CALIBRATION_VERSION` records the checkpoint the map was fitted for; a mismatch warns
    and still loads, so an older file never becomes a hard failure. Each temperature is passed
    through `clamp_temperature`, so a non-numeric or out-of-range entry cannot crash a later
    forward the way an unclamped zero used to.

    `temperature` and `binning_map` take opposite value policies, and that is the sentence to read
    before writing a file by hand: a bad *temperature* is forgiven and clamped into
    `[TEMP_MIN, TEMP_MAX]`, while a bad *binning value* is refused. Nothing recalibrates a
    confidence the way an out-of-range binning value would, so there is no defensible fallback
    for it.

    The payload's *shape* is checked before any of that, and refused with a `ValueError` naming
    the field: the values may be junk the clamp forgives, but the payload itself not being an
    object, a `version` that is not an integer, a `temperature` that is not a list of three, a
    `temperature_by_options` that is not an object, a `binning_map` that is not an object, or a
    `binning_map` entry that is not an object with an integer `bins` >= 1 and `values` of exactly
    that length holding finite numbers in [0, 1] is a file this code cannot read -- JSON gives
    `int`, `str`, `list` and `dict` for each of those mistakes just as happily as it gives the
    right shapes, and each used to fail with a raw `TypeError`/`AttributeError` from
    `len()`/`dict()`, or worse, to load: a `{"a": 1, "b": 2, "c": 3}` or `"abc"` has `len` 3 and
    used to pass the length check and install its *keys* as temperatures.
    """
    if not isinstance(payload, dict):
        raise ValueError("calibration JSON must be an object, got %s" % type(payload).__name__)
    version = payload.get("version", 1)
    if version is None:
        version = 1
    if isinstance(version, bool) or not isinstance(version, (int, float, str)):
        raise ValueError("calibration JSON version must be an integer, got %r" % (version,))
    try:
        version = int(version)
    except (TypeError, ValueError, OverflowError) as exc:   # Overflow: JSON's 1e999 is inf
        raise ValueError(
            "calibration JSON version must be an integer, got %r" % (version,)) from exc
    temps = payload.get("temperature")
    if not isinstance(temps, (list, tuple)) or len(temps) != N_QTYPES:
        raise ValueError("calibration JSON must contain temperature: [3 floats]")
    by_options = payload.get("temperature_by_options") or {}
    if not isinstance(by_options, dict):
        raise ValueError(
            "calibration JSON temperature_by_options must be an object of bucket -> float, "
            "got %s" % type(by_options).__name__)
    binning = payload.get("binning_map")
    if binning is not None:
        if not isinstance(binning, dict):
            raise ValueError(
                "calibration JSON binning_map must be an object of bucket -> {bins, values}, "
                "got %s" % type(binning).__name__)
        for name, entry in binning.items():
            if not isinstance(entry, dict):
                raise ValueError(
                    "calibration JSON binning_map[%r] must be an object with \"bins\" and \"values\", "
                    "got %s" % (name, type(entry).__name__))
            n_bins = entry.get("bins")
            values = entry.get("values")
            if isinstance(n_bins, bool) or not isinstance(n_bins, int) or n_bins < 1:
                raise ValueError(
                    "calibration JSON binning_map[%r] must have an integer \"bins\" >= 1, got %r" % (name, n_bins))
            if not isinstance(values, (list, tuple)) or len(values) != n_bins:
                raise ValueError(
                    "calibration JSON binning_map[%r] must have \"values\" of length \"bins\" (%d)"
                    % (name, n_bins))
            try:
                parsed_values = [float(v) for v in values]
            except (TypeError, ValueError) as exc:
                raise ValueError(
                    "calibration JSON binning_map[%r] values must be numbers, got %r" % (name, values)) from exc
            for v in values:
                if isinstance(v, bool) or not isinstance(v, (int, float)):
                    raise ValueError(
                        "calibration JSON binning_map[%r] values must be numbers, got %r" % (name, values))
            for v in parsed_values:
                if not np.isfinite(v):
                    raise ValueError(
                        "calibration JSON binning_map[%r] values must be finite, got %r" % (name, values))
                if not 0.0 <= v <= 1.0:
                    raise ValueError(
                        "calibration JSON binning_map[%r] values must be in [0, 1], got %r" % (name, values))
    obj.binning_map = binning
    if version >= CALIBRATION_VERSION:
        _warn_if_identity_mismatch(obj, payload)
    _install_temperatures(obj, temps, by_options)


In [ ]:
%%writefile {REXLIB}/rex/train.py
"""Fine-tune a Laya checkpoint on typed decisions: one training loop the scripts can share.

The notebook, the Apple Silicon script and `research/scripts/finetune_single_device.py` each
carry their own copy of the same loop. This module is that loop once, with two additions the
copies do not have:

- **A choice of objective.** `loss="rlcd"` (the default) is the notebook's objective: a
  GRPO-style term over noisy logit samples rewarded by `proper_reward`, plus soft
  cross-entropy against the target distribution. `loss="soft-ce"` trains on the soft
  cross-entropy alone; #741 measured no gain from the extra term on the typed-decisions split.
- **Option-order augmentation (opt-in).** The scripts always present options in the order the
  question lists them, so the model can learn a position prior: the 20-option order-flip rate in
  `BENCHMARKS.md` and #131 are both that symptom. Each epoch, items of the types named in
  `shuffle_options` (for example `("choice",)`) are re-encoded with a random `option_order` -- the same path inference
  uses -- and the target is permuted with them, so slot `s` is trained on option
  `option_order[s]`.

Calibration goes through `laya.calibrate.fit_temperature_map`, so a fine-tuned checkpoint is
fitted with the same clamp and buckets the runtime applies instead of a local copy of the
fitter. `calibration_report` says per question type what that fit rests on, and `finetune` warns
when it rests on too little: a short run can otherwise save a checkpoint whose temperatures never
moved from 1.0, and nothing but the numbers would show it. It also warns when predictions have
collapsed to the class prior -- a near-constant logit (or calibration CE stuck at ln K) that
looks like a finished run on an unlearnable task, but is often just too few updates on a small
dataset (#963).

Items keep the question and the tokenized state rather than a finished sequence, because a
shuffled epoch needs to rebuild the head. States are tokenized once per row and shared by
every question on it.

No new dependency: torch, transformers and safetensors are already required by the package.
"""
import json
import math
import os
import random
import warnings
from dataclasses import asdict, dataclass
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import torch

from .calibrate import MIN_TYPE_N, fit_abstention_thresholds, fit_temperature_map
from .common import (
    OPTION_LAYOUTS,
    QTYPE_NAMES,
    QTYPES,
    TEMP_MAX,
    TEMP_MIN,
    build_head,
    build_model,
    build_sequence,
    collate_items,
    ece_score,
    encode_text,
    proper_reward,
    render_options,
    serialize_state,
    temp_bucket,
    uses_parallel_layout,
)

LOSSES = ("soft-ce", "rlcd")
# Below this many calibration items of a type, a fitted temperature is reported as resting on
# little evidence. MIN_TYPE_N (laya.calibrate) is the floor below which it is not fitted at all.
CALIB_WARN_N = 50
# Mean within-row logit range below this is the silent collapse to a constant (usually
# the class prior) measured in #963: 0.01 on a collapsed head vs 0.62 untuned.
COLLAPSE_LOGIT_RANGE = 0.05
# Calibration mean cross-entropy within this relative distance of ln K is the other
# collapse signal from #963 (training CE stuck at ln 4). The reported training loss is
# not used: the default `rlcd` objective is not cross-entropy.
COLLAPSE_CE_REL = 0.02
_MASKED = -1e4


@dataclass
class TrainConfig:
    """Training knobs. The defaults train the way the published notebook does.

    The notebook's effective batch of 64 is 8 per micro-batch x 2 GPUs x 4 accumulation steps;
    on one device that is `micro_batch=8, grad_accum=8`.

    `max_len` / `head_max_len` default to the checkpoint's own config. Whatever is used is
    written into the saved config, so inference sees the same budgets training did.
    `amp` defaults to on for CUDA only; `gradient_checkpointing` defaults to following `amp`.
    `option_layout` defaults to the checkpoint's own (`"sequential"` for every published one);
    `"parallel"` trains on `common.parallel_layout` and is written into the saved config.
    """

    epochs: int = 4
    micro_batch: int = 8
    grad_accum: int = 8
    encoder_lr: float = 2.5e-5
    head_lr: float = 1e-4
    min_lr: float = 1e-6
    weight_decay: float = 0.01
    grad_clip: float = 1.0
    loss: str = "rlcd"
    rl_samples: int = 4
    sigma_start: float = 0.4
    sigma_end: float = 0.1
    w_sph: float = 0.75
    w_rps: float = 1.0
    shuffle_options: Tuple[str, ...] = ()
    option_layout: Optional[str] = None
    freeze_encoder: bool = False
    calib_max: int = 400
    calib_frac: float = 0.1
    calib_seed: int = 20260922
    seed: int = 0
    max_len: Optional[int] = None
    head_max_len: Optional[int] = None
    amp: Optional[bool] = None
    gradient_checkpointing: Optional[bool] = None
    log_every: int = 100
    label_smoothing: float = 0.0
    text_column: str = "text"
    label_column: str = "label"
    question_id: str = "label"
    instructions: Optional[str] = None
    eval_data: Optional[str] = None
    target_error: float = 0.10
    min_abstain_n: int = 10

    def validate(self) -> None:
        if self.loss not in LOSSES:
            raise ValueError("loss must be one of %s, got %r" % (", ".join(LOSSES), self.loss))
        unknown = sorted(set(self.shuffle_options) - set(QTYPES))
        if unknown:
            raise ValueError("shuffle_options names unknown question type(s): %s" % ", ".join(unknown))
        if self.option_layout is not None and self.option_layout not in OPTION_LAYOUTS:
            raise ValueError("option_layout must be one of %s, got %r"
                             % (", ".join(OPTION_LAYOUTS), self.option_layout))
        for name in ("epochs", "micro_batch", "grad_accum", "rl_samples"):
            value = getattr(self, name)
            if isinstance(value, bool) or not isinstance(value, int) or value < 1:
                raise ValueError("%s must be a positive integer, got %r" % (name, value))
        if not 0.0 <= self.calib_frac < 1.0:
            raise ValueError("calib_frac must be in [0, 1), got %r" % (self.calib_frac,))
        if isinstance(self.label_smoothing, bool) or not (0.0 <= self.label_smoothing < 1.0):
            raise ValueError("label_smoothing must be in [0, 1), got %r" % (self.label_smoothing,))
        if isinstance(self.target_error, bool) or not (0.0 <= self.target_error <= 1.0):
            raise ValueError("target_error must be in [0, 1], got %r" % (self.target_error,))
        if isinstance(self.min_abstain_n, bool) or not isinstance(self.min_abstain_n, int) or self.min_abstain_n < 1:
            raise ValueError("min_abstain_n must be a positive integer, got %r" % (self.min_abstain_n,))
        if self.eval_data is not None and not isinstance(self.eval_data, str):
            raise ValueError("eval_data must be a string path, got %r" % (self.eval_data,))


# ------------------------------------------------------------------------------------- data

def to_internal(qid: Any, question: Dict[str, Any]) -> Dict[str, Any]:
    """The question as inference sees it: `Agent._check_question` then `Agent._to_internal`.

    Training on any other rendering would teach the model option texts it never sees at
    inference -- a list-form choice, custom noul `labels`, a non-string instruction.
    """
    from .agent import Agent

    Agent._check_question(qid, question)
    q = Agent._to_internal(question)
    q.pop("option_order", None)    # training draws its own order per epoch
    return q


def target_from_gold(q: Dict[str, Any], gold_q: Dict[str, Any]) -> List[float]:
    """The normalised target distribution for an internal question (`to_internal`), in option order.

    `gold_q["probabilities"]` is keyed by choice label, by `"false"`/`"true"` for noul, and by
    the level index as a string for score -- the `{state, questions, gold}` schema in
    `docs/finetune.md`. A choice label that is not a string is also looked up as its JSON key.
    A row whose probabilities sum to zero becomes uniform, as in the scripts. A negative or
    non-finite probability raises `ValueError`.
    """
    probs = gold_q["probabilities"]

    def get(key, default):
        return probs.get(key, probs.get(str(key), default))

    t, crit = q["t"], q["crit"]
    if t == "choice":
        target = [float(get(k, 0.0)) for k in crit.keys()]
    elif t == "noul":
        target = [float(get("false", 0.5)), float(get("true", 0.5))]
    else:
        target = [float(get(str(i), 0.0)) for i in range(len(crit))]
    if any(not math.isfinite(v) or v < 0 for v in target):
        raise ValueError("probabilities must be finite and non-negative, got %r" % (target,))
    total = sum(target)
    if total > 0:
        return [v / total for v in target]
    return [1.0 / len(target)] * len(target)


def target_from_expected(q: Dict[str, Any], expected_val: Any,
                         label_smoothing: float = 0.0) -> List[float]:
    """The normalised target distribution from a single expected answer, in option order.

    `expected_val` is:
    - a choice label (str or value matching a criterion) for choice questions
    - a boolean for noul questions
    - an integer level index or level string for score questions

    When `label_smoothing > 0`, smooths probability mass uniformly:
    target[i] = (1 - eps) * one_hot[i] + eps / K.
    """
    if isinstance(label_smoothing, bool) or not (0.0 <= label_smoothing < 1.0):
        raise ValueError("label_smoothing must be in [0, 1), got %r" % (label_smoothing,))

    t, crit = q["t"], q["crit"]
    k = len(render_options(q))
    idx = None
    target = None

    if t == "choice":
        keys = list(crit.keys())
        if expected_val in crit:
            idx = keys.index(expected_val)
        else:
            str_expected = str(expected_val)
            for i, key in enumerate(keys):
                if str(key) == str_expected:
                    idx = i
                    break
        if idx is None:
            raise ValueError("expected answer %r not found in choice options %r" % (expected_val, keys))
    elif t == "noul":
        if isinstance(expected_val, bool):
            idx = 1 if expected_val else 0
        elif isinstance(expected_val, (int, float)):
            if expected_val in (0, 1) or expected_val in (0.0, 1.0):
                idx = int(expected_val)
            else:
                raise ValueError("numeric noul expected answer must be 0 or 1, got %r" % (expected_val,))
        elif isinstance(expected_val, str):
            lower = expected_val.strip().lower()
            if lower in ("true", "1", "yes"):
                idx = 1
            elif lower in ("false", "0", "no"):
                idx = 0
        if idx is None:
            raise ValueError("expected answer %r cannot be parsed as a noul boolean" % (expected_val,))
    else:  # score
        keys = list(crit.keys()) if isinstance(crit, dict) else [str(i) for i in range(len(crit))]
        K = len(keys)
        target = None
        if isinstance(expected_val, str) and expected_val in keys:
            idx = keys.index(expected_val)
            target = [1.0 if i == idx else 0.0 for i in range(K)]
        else:
            try:
                if not isinstance(expected_val, bool):
                    val = float(expected_val)
                    if 0.0 <= val <= float(K - 1):
                        low = int(math.floor(val))
                        if low == K - 1 or val == float(low):
                            target = [1.0 if i == low else 0.0 for i in range(K)]
                        else:
                            frac = val - float(low)
                            target = [0.0] * K
                            target[low] = 1.0 - frac
                            target[low + 1] = frac
                    else:
                        raise ValueError("expected score %r out of bounds [0, %d]" % (expected_val, K - 1))
            except (ValueError, TypeError) as exc:
                if isinstance(exc, ValueError) and "out of bounds" in str(exc):
                    raise
                pass
        if target is None:
            str_val = str(expected_val)
            for i, key in enumerate(keys):
                if str(key) == str_val:
                    target = [1.0 if j == i else 0.0 for j in range(K)]
                    break
        if target is None:
            raise ValueError("expected answer %r not found in score levels %r" % (expected_val, keys))

    if target is None:
        target = [1.0 if i == idx else 0.0 for i in range(k)]
    if label_smoothing > 0.0:
        smooth_val = label_smoothing / k
        target = [(1.0 - label_smoothing) * v + smooth_val for v in target]
    return target


def encode_state(tok, state: Any, max_len: int) -> List[int]:
    """State token ids, capped at `max_len`: no sequence can hold more of the state than that."""
    text = serialize_state(state).replace(tok.mask_token, " ")
    return encode_text(tok, text, add_special_tokens=False)["input_ids"][:max_len]


def make_item(tok, q: Dict[str, Any], target: Sequence[float], state_ids: List[int],
              head_max_len: int, max_len: Optional[int] = None) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    """`(item, None)`, or `(None, reason)` when the question cannot be trained on as written.

    `q` is an internal question (`to_internal`). The reasons are `target_mismatch` (the target
    does not have one entry per option), `options_collapsed` (the head budget left two options
    with the same token span, #538) and, when `max_len` is given, `options_beyond_max_len` (the
    head is longer than `max_len`, so `build_sequence` drops the markers of the last options --
    the case `Agent` refuses at inference with "only N of its M option markers fit"). Training on
    any of them would teach the model to tell apart options it cannot see as different, or crash
    the batch on a target longer than its markers.
    """
    k = len(render_options(q))
    if len(target) != k:
        return None, "target_mismatch"
    _ids, markers, stats = build_head(tok, q, head_max_len)
    if len(markers) != k or stats["options_distinct"] < stats["options"]:
        return None, "options_collapsed"
    if max_len is not None and markers and markers[-1] >= max_len:
        return None, "options_beyond_max_len"
    return {"q": q, "state_ids": state_ids, "target": [float(v) for v in target],
            "qtype": QTYPES[q["t"]], "k": k}, None


def items_from_rows(tok, rows: Iterable[Dict[str, Any]], max_len: int,
                    head_max_len: int,
                    label_smoothing: float = 0.0) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    """Items from `{state, questions, gold}` or `{state, questions, expected}` rows; returns `(items, skipped)`.

    `skipped` counts, by reason, questions that were labelled but could not become an item:
    `empty_text`, `empty_label`, `invalid_question` and `invalid_target` (a `ValueError` from
    `to_internal`, `target_from_gold` or `target_from_expected`), plus the reasons `make_item`
    gives. Questions with neither a `gold` nor an `expected` entry are not counted -- the row
    simply does not label them.
    """
    items, skipped = [], {}
    for row in rows:
        state = row.get("state")
        if state is None or (isinstance(state, str) and not state.strip()):
            skipped["empty_text"] = skipped.get("empty_text", 0) + 1
            continue
        state_ids = encode_state(tok, state, max_len)
        for qid, question in row.get("questions", {}).items():
            gold_q = row.get("gold", {}).get(qid) if isinstance(row.get("gold"), dict) else None
            expected_q = row.get("expected", {}).get(qid) if isinstance(row.get("expected"), dict) else None
            if gold_q is None and expected_q is None:
                continue
            reason = None
            try:
                q = to_internal(qid, question)
            except (ValueError, TypeError, KeyError):
                reason = "invalid_question"
            if reason is None:
                try:
                    if gold_q is not None:
                        target = target_from_gold(q, gold_q)
                    else:
                        if isinstance(expected_q, str) and not expected_q.strip():
                            reason = "empty_label"
                        else:
                            target = target_from_expected(q, expected_q, label_smoothing=label_smoothing)
                except (ValueError, TypeError, KeyError):
                    reason = "invalid_target"
            if reason is None:
                item, reason = make_item(tok, q, target, state_ids, head_max_len, max_len)
            if reason is None:
                items.append(item)
            else:
                skipped[reason] = skipped.get(reason, 0) + 1
    return items, skipped


def rows_from_csv(path: str, text_column: str = "text", label_column: str = "label",
                  question_id: str = "label",
                  instructions: Optional[str] = None) -> List[Dict[str, Any]]:
    """Read a CSV file with one labelled column per row into `{state, questions, expected}` rows.

    Discovers distinct labels in `label_column` (preserving order of appearance) and
    builds a choice question with those labels as a list of criteria to render each option once.
    Uses `utf-8-sig` encoding so UTF-8 BOM headers from Excel are handled transparently.
    """
    import csv

    rows_raw = []
    with open(path, encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            raise ValueError("%s is an empty CSV file" % (path,))
        if text_column not in reader.fieldnames:
            raise ValueError("CSV file %s missing text column %r (available: %r)"
                             % (path, text_column, reader.fieldnames))
        if label_column not in reader.fieldnames:
            raise ValueError("CSV file %s missing label column %r (available: %r)"
                             % (path, label_column, reader.fieldnames))
        for r in reader:
            rows_raw.append(r)

    if not rows_raw:
        raise ValueError("%s has no rows" % (path,))

    labels = []
    seen = set()
    for r in rows_raw:
        val = r.get(label_column, "").strip()
        if val and val not in seen:
            seen.add(val)
            labels.append(val)

    if len(labels) < 2:
        raise ValueError("CSV file %s must contain at least 2 distinct labels in %r, found %r"
                         % (path, label_column, labels))

    instr = instructions or "Classify the input into the correct category."
    questions = {
        question_id: {
            "type": "choice",
            "instructions": instr,
            "criteria": labels,
        }
    }

    output_rows = []
    for r in rows_raw:
        text = r.get(text_column, "")
        lbl = r.get(label_column, "").strip()
        output_rows.append({
            "state": text,
            "questions": questions,
            "expected": {question_id: lbl},
        })
    return output_rows


def read_data(path: str, text_column: str = "text", label_column: str = "label",
              question_id: str = "label",
              instructions: Optional[str] = None) -> List[Dict[str, Any]]:
    """Read training/eval rows from either a JSONL file or a CSV file."""
    if path.lower().endswith(".csv"):
        return rows_from_csv(path, text_column=text_column, label_column=label_column,
                             question_id=question_id, instructions=instructions)
    return read_jsonl(path)


def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#"):
                rows.append(json.loads(line))
    return rows


def encode_item(tok, item: Dict[str, Any], max_len: int, head_max_len: int,
                option_order: Optional[List[int]] = None, parallel: bool = False) -> Dict[str, Any]:
    """The model input for an item, with options in `option_order` and the target to match.

    `build_sequence` puts option `option_order[s]` in slot `s`, so the slot-ordered target is
    `target[option_order[s]]` -- the inverse of what `unpermute_probs` does at inference.
    """
    ids, markers, *layout = build_sequence(tok, None, item["q"], max_len, head_max_len, option_order=option_order,
                                           state_ids=item["state_ids"], return_layout=parallel)
    target = item["target"]
    if option_order is not None:
        target = [target[i] for i in option_order]
    encoded = {"ids": ids, "markers": markers, "qtype": item["qtype"], "target": target}
    if layout:
        encoded["layout"] = layout[0]
    return encoded


def draw_option_order(item: Dict[str, Any], rng: random.Random,
                      shuffle_types: Sequence[str]) -> Optional[List[int]]:
    """A random permutation for items whose type is in `shuffle_types`, else None (canonical)."""
    if item["q"]["t"] not in shuffle_types or item["k"] < 2:
        return None
    order = list(range(item["k"]))
    rng.shuffle(order)
    return order


def split_calibration(items: Sequence[Any], calib_max: int, calib_frac: float,
                      seed: int) -> Tuple[List[Any], List[Any]]:
    """`(train, calibration)`, with the calibration slice taken before any training.

    Fitting temperatures on items the run has trained on measures the fit, not the calibration
    (`docs/finetune.md`). The split depends only on `seed` and the item count, so every DDP rank
    and every rerun hold out the same items.
    """
    n_calib = min(calib_max, int(len(items) * calib_frac))
    order = list(range(len(items)))
    random.Random(seed).shuffle(order)
    held = set(order[:n_calib])
    train = [it for i, it in enumerate(items) if i not in held]
    calib = [it for i, it in enumerate(items) if i in held]
    return train, calib


# ----------------------------------------------------------------------------------- losses

def soft_ce_loss(logits: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Cross-entropy against a target distribution, over each row's own options only."""
    logp = torch.log_softmax(logits.masked_fill(~mask, _MASKED), -1)
    return -(target * logp * mask).sum(-1).mean()


def rlcd_loss(logits: torch.Tensor, target: torch.Tensor, mask: torch.Tensor, qtype: torch.Tensor,
              sigma: float, samples: int = 4, w_sph: float = 0.75, w_rps: float = 1.0) -> torch.Tensor:
    """The notebook objective: a policy-gradient term over noisy logits plus soft cross-entropy.

    Zero-mean Gaussian noise (scaled by `sigma`, projected onto each row's options) is added to
    detached logits; each sample is rewarded by `proper_reward` against the target, and the
    normalised advantage weights the Gaussian log-density of the sample.
    """
    k = mask.sum(-1, keepdim=True).float()
    fmask = mask.float()
    eps = torch.randn((samples,) + logits.shape, device=logits.device) * sigma * fmask
    eps = (eps - eps.sum(-1, keepdim=True) / k) * fmask
    z = logits.detach().unsqueeze(0) + eps
    q = torch.softmax(z.masked_fill(~mask, _MASKED), -1)
    with torch.no_grad():
        r = proper_reward(q, target.unsqueeze(0), qtype, fmask, w_sph=w_sph, w_rps=w_rps)
        adv = r - r.mean(0, keepdim=True)
        adv = adv / (adv.std() + 1e-6)
    logp = -(((z - logits.unsqueeze(0)) ** 2) * fmask).sum(-1) / (2 * sigma ** 2)
    return -(adv * logp).mean() + soft_ce_loss(logits, target, mask)


def sigma_at(epoch: int, epochs: int, start: float, end: float) -> float:
    """Exploration noise for `epoch`, annealed linearly from `start` to `end`."""
    return start + (end - start) * (epoch / max(1, epochs - 1))


# ------------------------------------------------------------------------------- checkpoint

def resolve_device(device: Optional[str] = "auto") -> torch.device:
    if device not in (None, "auto"):
        return torch.device(device)
    if torch.cuda.is_available():
        return torch.device("cuda", 0)
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    if hasattr(torch, "xpu") and torch.xpu.is_available():
        return torch.device("xpu")
    return torch.device("cpu")


def resolve_checkpoint_dir(model_id_or_path: str, token: Optional[str] = None) -> str:
    """Resolve a local path, model alias ('multilingual', 'english') or Hub repo ID to a local directory."""
    if os.path.isdir(model_id_or_path) and os.path.exists(os.path.join(model_id_or_path, "rl_agent_config.json")):
        return model_id_or_path

    if model_id_or_path.startswith(("/", "./", "../")) or os.path.isabs(model_id_or_path):
        if not os.path.isdir(model_id_or_path):
            raise FileNotFoundError("Local model path not found: %r" % (model_id_or_path,))
        return model_id_or_path

    from .router import resolve_model_spec
    spec = resolve_model_spec(model_id_or_path)
    subfolder = None
    if spec is not None:
        model_id_or_path, subfolder = spec

    from huggingface_hub import snapshot_download
    prefix = f"{subfolder}/" if subfolder else ""
    kw = {
        "token": token or os.environ.get("HF_TOKEN") or None,
        "allow_patterns": [prefix + name for name in (
            "rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*",
        )],
    }
    model_dir = snapshot_download(model_id_or_path, **kw)
    if subfolder:
        model_dir = os.path.join(model_dir, subfolder)
    if not os.path.isdir(model_dir) or not os.path.exists(os.path.join(model_dir, "rl_agent_config.json")):
        raise FileNotFoundError("Checkpoint directory not found for %r (resolved to %r)"
                                % (model_id_or_path, model_dir))
    return model_dir


def load_checkpoint(model_dir: str):
    """`(model, tokenizer, cfg)` from a local checkpoint directory or Hub ID, model on CPU in fp32."""
    from safetensors.torch import load_file
    from transformers import AutoTokenizer

    from .agent import _fix_tokenizer_config

    model_dir = resolve_checkpoint_dir(model_dir)
    _fix_tokenizer_config(model_dir)
    with open(os.path.join(model_dir, "rl_agent_config.json"), encoding="utf-8") as f:
        cfg = json.load(f)
    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
    return model.float(), tok, cfg


def save_checkpoint(model, tok, cfg: Dict[str, Any], path: str) -> None:
    """Write the layout `laya.load` reads: weights in fp16, encoder config, tokenizer, config."""
    from safetensors.torch import save_file

    os.makedirs(path, exist_ok=True)
    save_file({k: v.detach().half().cpu().contiguous() for k, v in model.state_dict().items()},
              os.path.join(path, "model.safetensors"))
    model.encoder.config.save_pretrained(os.path.join(path, "encoder"))
    tok.save_pretrained(os.path.join(path, "tokenizer"))
    tmp = os.path.join(path, "rl_agent_config.json.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2)
    os.replace(tmp, os.path.join(path, "rl_agent_config.json"))


# ------------------------------------------------------------------------------------ train

def optimizer_updates(n_items: int, micro_batch: int, grad_accum: int, epochs: int) -> int:
    """Optimizer steps for `n_items` training items.

    `train_model` calls this, so the count a pre-run diagnostic prints is the number of
    `optimizer.step()` calls the run performs: one per full accumulation window
    (`grad_accum` micro-batches), plus one for the final partial window of every epoch, which
    still steps once even when it holds fewer micro-batches.
    """
    if n_items < 1:
        raise ValueError("n_items must be a positive integer, got %r" % (n_items,))
    steps_per_epoch = math.ceil(n_items / micro_batch)
    updates_per_epoch = math.ceil(steps_per_epoch / grad_accum)
    return updates_per_epoch * epochs


def update_budget_message(n_train: int, config: TrainConfig) -> str:
    """The pre-run optimizer-update budget line, shared by `finetune` startup and `--dry-run`.

    Says "effective batch up to" because the last window of an epoch is often partial, so not
    every update sees a full `micro_batch x grad_accum` batch.
    """
    updates = optimizer_updates(n_train, config.micro_batch, config.grad_accum, config.epochs)
    return ("optimizer budget: %d train item(s), effective batch up to %d "
            "(micro_batch %d x grad_accum %d), %d optimizer updates over %d epoch(s)"
            % (n_train, config.micro_batch * config.grad_accum,
               config.micro_batch, config.grad_accum, updates, config.epochs))


def _forward(model, batch, device, amp: bool, detach_encoder: bool):
    args = (batch["input_ids"].to(device), batch["attention_mask"].to(device),
            batch["marker_pos"].to(device), batch["marker_mask"].to(device), batch["qtype"].to(device))
    kwargs = {"detach_encoder": detach_encoder}
    if "option_ids" in batch:
        kwargs.update(position_ids=batch["position_ids"].to(device), option_ids=batch["option_ids"].to(device))
    if getattr(model, "tool_matcher", None) is not None:
        # RexDecisionModel: training needs only the logits, not the per-sample tool dispatcher
        kwargs["dispatch"] = False
    if amp:
        with torch.autocast(device.type, dtype=torch.float16):
            res = model(*args, **kwargs)
    else:
        res = model(*args, **kwargs)

    if isinstance(res, tuple) and len(res) == 3:
        logits, _state, _step_out = res
    else:
        logits, _act = res
    return logits.float()


def train_model(model, tok, items: Sequence[Dict[str, Any]], config: TrainConfig, device: torch.device,
                max_len: int, head_max_len: int,
                on_epoch_end: Optional[Callable[[int, float], None]] = None,
                parallel: bool = False) -> List[float]:
    """Train `model` in place on `items`; returns the mean loss of each epoch."""
    config.validate()
    if not items:
        raise ValueError("no training items")
    if getattr(model, "tool_matcher", None) is not None:
        warnings.warn("laya.train: this is a Rex (arch rex2) model; this loop trains only the verdict path "
                      "(encoder, head, scorer). The S, R, tool, gain and hypothesis heads, the ponder block and "
                      "the multi-round memory are not trained here; use scripts/train_rex.py for those.",
                      RuntimeWarning, stacklevel=2)
    amp = (device.type == "cuda") if config.amp is None else bool(config.amp)
    checkpointing = amp if config.gradient_checkpointing is None else bool(config.gradient_checkpointing)

    if config.freeze_encoder:
        for p in model.encoder.parameters():
            p.requires_grad_(False)
    elif checkpointing and hasattr(model.encoder, "gradient_checkpointing_enable"):
        model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = checkpointing
    model.to(device).train()
    if config.freeze_encoder:
        # A frozen encoder is a fixed feature extractor: dropout in it would only add noise.
        model.encoder.eval()

    groups = [{"params": [p for n, p in model.named_parameters()
                          if not n.startswith("encoder.") and p.requires_grad], "lr": config.head_lr}]
    if not config.freeze_encoder:
        groups.insert(0, {"params": [p for n, p in model.named_parameters()
                                     if n.startswith("encoder.") and p.requires_grad], "lr": config.encoder_lr})
    optimizer = torch.optim.AdamW(groups, weight_decay=config.weight_decay)
    steps_per_epoch = math.ceil(len(items) / config.micro_batch)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=optimizer_updates(len(items), config.micro_batch, config.grad_accum, config.epochs),
        eta_min=config.min_lr)
    scaler = torch.amp.GradScaler("cuda") if amp and device.type == "cuda" else None

    torch.manual_seed(config.seed)
    order_rng = random.Random(config.seed)
    params = [p for g in groups for p in g["params"]]
    history = []
    for epoch in range(config.epochs):
        epoch_items = list(items)
        random.Random(config.seed + epoch).shuffle(epoch_items)
        sigma = sigma_at(epoch, config.epochs, config.sigma_start, config.sigma_end)
        total, n_steps = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for start in range(0, len(epoch_items), config.micro_batch):
            chunk = [encode_item(tok, it, max_len, head_max_len,
                                 draw_option_order(it, order_rng, config.shuffle_options), parallel)
                     for it in epoch_items[start:start + config.micro_batch]]
            batch = collate_items([chunk], tok.pad_token_id)
            logits = _forward(model, batch, device, amp, config.freeze_encoder)
            mask = batch["marker_mask"].to(device)
            target = batch["target"].to(device)
            if config.loss == "rlcd":
                loss = rlcd_loss(logits, target, mask, batch["qtype"].to(device), sigma,
                                 config.rl_samples, config.w_sph, config.w_rps)
            else:
                loss = soft_ce_loss(logits, target, mask)
            window_start = (n_steps // config.grad_accum) * config.grad_accum
            window_size = min(config.grad_accum, steps_per_epoch - window_start)
            scaled = loss / window_size
            if scaler is not None:
                scaler.scale(scaled).backward()
            else:
                scaled.backward()
            n_steps += 1
            if n_steps % config.grad_accum == 0 or start + config.micro_batch >= len(epoch_items):
                if scaler is not None:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, config.grad_clip)
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            total += loss.item()
            if config.log_every and n_steps % config.log_every == 0:
                print("epoch %d/%d step %d loss %.4f" % (epoch + 1, config.epochs, n_steps, loss.item()), flush=True)
        mean = total / max(1, n_steps)
        history.append(mean)
        print("epoch %d/%d mean loss %.4f" % (epoch + 1, config.epochs, mean), flush=True)
        if on_epoch_end is not None:
            on_epoch_end(epoch, mean)
    model.eval()
    return history


@torch.no_grad()
def calibration_records(model, tok, items: Sequence[Dict[str, Any]], device: torch.device,
                        max_len: int, head_max_len: int, batch_size: int = 16, parallel: bool = False):
    """`(qtype, logits, target, k)` records for `fit_temperature_map`, options in canonical order."""
    model.eval()
    records = []
    for start in range(0, len(items), batch_size):
        chunk = items[start:start + batch_size]
        batch = collate_items([[encode_item(tok, it, max_len, head_max_len, parallel=parallel) for it in chunk]],
                              tok.pad_token_id)
        logits = _forward(model, batch, device, amp=False, detach_encoder=False).cpu().numpy()
        for row, it in zip(logits, chunk):
            records.append((it["qtype"], row[:it["k"]], it["target"], it["k"]))
    return records


def calibration_report(records: Sequence[Tuple[int, Any, Any, int]],
                       temperature: Sequence[float]) -> Dict[str, Dict[str, Any]]:
    """Per question type: how many calibration items there were, what was fitted, and what to doubt.

    `records` are the `calibration_records` the temperatures were fitted on, and `temperature`
    the per-type scalars `fit_temperature_map` returned. Each type gets `items`, `temperature`,
    `accuracy` and `mean_confidence` (top probability at the fitted temperature) on the
    calibration items, and a list of `issues`, empty when nothing looks wrong:

    - fewer than `MIN_TYPE_N` items: the temperature was not fitted and stays 1.0;
    - fewer than `CALIB_WARN_N` items: it was fitted, on little evidence;
    - it landed on the `[TEMP_MIN, TEMP_MAX]` clamp;
    - it came back unchanged at 1.0 although it was fitted, which usually means the items gave
      the fit nothing to correct -- what a model already certain and right on them produces.

    A type with no calibration items at all has no issues; it was not in the data.
    """
    import numpy as np

    by_type: Dict[int, List[Tuple[Any, Any]]] = {qt: [] for qt in QTYPE_NAMES}
    for qt, logits, target, _k in records:
        by_type[int(qt)].append((np.asarray(logits, dtype=float), np.asarray(target, dtype=float)))
    report = {}
    for qt, name in QTYPE_NAMES.items():
        pairs, t = by_type[qt], float(temperature[qt])
        entry: Dict[str, Any] = {"items": len(pairs), "temperature": t, "issues": []}
        if pairs:
            right, conf = 0, 0.0
            for logits, target in pairs:
                z = logits / t
                p = np.exp(z - z.max())
                p /= p.sum()
                right += int(p.argmax() == target.argmax())
                conf += float(p.max())
            entry["accuracy"] = right / len(pairs)
            entry["mean_confidence"] = conf / len(pairs)
        n, issues = len(pairs), entry["issues"]
        if 0 < n < MIN_TYPE_N:
            issues.append("not fitted: %d calibration items, fewer than %d, so the temperature stays 1.0"
                          % (n, MIN_TYPE_N))
        elif n >= MIN_TYPE_N:
            if n < CALIB_WARN_N:
                issues.append("fitted on only %d calibration items" % n)
            if t <= TEMP_MIN or t >= TEMP_MAX:
                issues.append("the fitted temperature %.3g is on the [%g, %g] clamp" % (t, TEMP_MIN, TEMP_MAX))
            elif abs(t - 1.0) < 1e-3:
                issues.append("the fit came back unchanged at 1.0 (accuracy %.2f, mean confidence %.3f on the "
                              "calibration items), which usually means they gave it nothing to correct; this "
                              "type's confidences are uncalibrated" % (entry["accuracy"], entry["mean_confidence"]))
        report[name] = entry
    return report


def evaluate_records(records: Sequence[Tuple[int, Any, Any, int]],
                     temperature: Optional[Sequence[float]] = None,
                     temperature_by_options: Optional[Dict[str, float]] = None) -> Dict[str, Any]:
    """Evaluate loss, accuracy, top-1 confidence, ECE and Brier scores on `records`.

    `records` is a sequence of `(qtype, logits, target, k)` tuples as returned by
    `calibration_records`. Predictions and confidences are scaled using `temperature` /
    `temperature_by_options` if provided. Returns overall metrics and per-question-type
    breakdowns.

    `brier` is the multi-class Brier score over the full probability vector:
    `mean(sum((p - target)**2))`. `brier_top1` is the binary Brier score of top-1
    confidence as P(correct): `mean((max(p) - correct)**2)`.
    """
    import numpy as np

    if not records:
        return {
            "items": 0,
            "loss": 0.0,
            "accuracy": 0.0,
            "mean_confidence": 0.0,
            "ece": None,
            "brier": None,
            "brier_top1": None,
            "by_type": {},
        }

    losses: List[float] = []
    corrects: List[bool] = []
    confs: List[float] = []
    briers_mc: List[float] = []
    briers_top1: List[float] = []
    by_type: Dict[int, Dict[str, List[Any]]] = {
        qt: {"losses": [], "corrects": [], "confs": [], "briers_mc": [], "briers_top1": []} for qt in QTYPE_NAMES
    }

    for qt, logits, target, k in records:
        z = np.asarray(logits[:k], dtype=float)
        t = np.asarray(target[:k], dtype=float)
        bucket = temp_bucket(qt, k)
        if temperature_by_options and bucket in temperature_by_options:
            scale = float(temperature_by_options[bucket])
        elif temperature is not None and isinstance(temperature, (list, tuple)) and qt < len(temperature):
            scale = float(temperature[qt])
        elif isinstance(temperature, (int, float)):
            scale = float(temperature)
        else:
            scale = 1.0
        scale = max(scale, 1e-4)
        z_scaled = z / scale
        z_max = np.max(z_scaled)
        exp_z = np.exp(z_scaled - z_max)
        sum_exp_z = np.sum(exp_z)
        p = exp_z / sum_exp_z
        log_p = z_scaled - z_max - np.log(sum_exp_z)
        loss = float(-np.sum(t * log_p))

        pred = int(np.argmax(p))
        gold = int(np.argmax(t))
        correct = bool(pred == gold)
        conf = float(np.max(p))
        b_mc = float(np.sum((p - t) ** 2))
        b_top1 = float((conf - float(correct)) ** 2)

        losses.append(loss)
        corrects.append(correct)
        confs.append(conf)
        briers_mc.append(b_mc)
        briers_top1.append(b_top1)

        if qt in by_type:
            by_type[qt]["losses"].append(loss)
            by_type[qt]["corrects"].append(correct)
            by_type[qt]["confs"].append(conf)
            by_type[qt]["briers_mc"].append(b_mc)
            by_type[qt]["briers_top1"].append(b_top1)

    total_items = len(records)
    acc = float(np.mean(corrects)) if corrects else 0.0
    mean_loss = float(np.mean(losses)) if losses else 0.0
    mean_conf = float(np.mean(confs)) if confs else 0.0
    ece_val = float(ece_score(np.asarray(confs, dtype=float), np.asarray(corrects, dtype=bool))) if len(corrects) >= 2 else None
    brier_val = float(np.mean(briers_mc)) if briers_mc else None
    brier_top1_val = float(np.mean(briers_top1)) if briers_top1 else None

    by_type_summary = {}
    for qt, name in QTYPE_NAMES.items():
        t_losses = by_type[qt]["losses"]
        t_corrects = by_type[qt]["corrects"]
        t_confs = by_type[qt]["confs"]
        t_briers_mc = by_type[qt]["briers_mc"]
        t_briers_top1 = by_type[qt]["briers_top1"]
        if t_losses:
            t_ece = float(ece_score(np.asarray(t_confs, dtype=float), np.asarray(t_corrects, dtype=bool))) if len(t_corrects) >= 2 else None
            t_brier = float(np.mean(t_briers_mc))
            t_brier_top1 = float(np.mean(t_briers_top1))
            by_type_summary[name] = {
                "items": len(t_losses),
                "loss": round(float(np.mean(t_losses)), 4),
                "accuracy": round(float(np.mean(t_corrects)), 4),
                "mean_confidence": round(float(np.mean(t_confs)), 4),
                "ece": round(t_ece, 4) if t_ece is not None else None,
                "brier": round(t_brier, 4) if t_brier is not None else None,
                "brier_top1": round(t_brier_top1, 4) if t_brier_top1 is not None else None,
            }

    return {
        "items": total_items,
        "loss": round(mean_loss, 4),
        "accuracy": round(acc, 4),
        "mean_confidence": round(mean_conf, 4),
        "ece": round(ece_val, 4) if ece_val is not None else None,
        "brier": round(brier_val, 4) if brier_val is not None else None,
        "brier_top1": round(brier_top1_val, 4) if brier_top1_val is not None else None,
        "by_type": by_type_summary,
    }


def collapse_stats(records: Sequence[Tuple[int, Any, Any, int]]) -> Optional[Dict[str, float]]:
    """Mean within-row logit range, mean soft-CE and mean ln K on records with k >= 2.

    A 1-option row cannot collapse to a prior, so it is skipped. Empty input, or only
    1-option rows, returns None.
    """
    import numpy as np

    ranges, ces, lnks = [], [], []
    for _qt, logits, target, k in records:
        k = int(k)
        if k < 2:
            continue
        z = np.asarray(logits, dtype=float).ravel()[:k]
        t = np.asarray(target, dtype=float).ravel()[:k]
        mass = float(t.sum())
        if mass <= 0.0:
            continue
        t = t / mass
        ranges.append(float(z.max() - z.min()))
        logp = z - np.logaddexp.reduce(z)
        ces.append(float(-(t * logp).sum()))
        lnks.append(math.log(k))
    if not ranges:
        return None
    n = len(ranges)
    return {
        "n": float(n),
        "mean_logit_range": sum(ranges) / n,
        "mean_ce": sum(ces) / n,
        "mean_ln_k": sum(lnks) / n,
    }


def prior_collapse_message(records: Sequence[Tuple[int, Any, Any, int]]) -> Optional[str]:
    """Warning text if calibration logits have collapsed to the class prior, else None.

    The default 4-epoch budget is sized for a few thousand typed decisions. On a few
    hundred or a thousand labelled rows the head can finish with every option at the
    same logit -- chance-level, and silent. #963 measured a mean within-row range of
    0.01 against 0.62 for the untuned checkpoint, and training CE stuck at ln K.

    Fired when the mean within-row logit range is below `COLLAPSE_LOGIT_RANGE`, or when
    mean soft-CE on the same records is within `COLLAPSE_CE_REL` of mean ln K.
    """
    stats = collapse_stats(records)
    if stats is None:
        return None
    reasons = []
    if stats["mean_logit_range"] < COLLAPSE_LOGIT_RANGE:
        reasons.append("mean within-row logit range %.3g" % stats["mean_logit_range"])
    mean_ce, mean_lnk = stats["mean_ce"], stats["mean_ln_k"]
    if mean_lnk > 0.0 and abs(mean_ce - mean_lnk) <= COLLAPSE_CE_REL * mean_lnk:
        reasons.append("calibration cross-entropy %.3f is within %.0f%% of ln K ~ %.3f"
                       % (mean_ce, COLLAPSE_CE_REL * 100.0, mean_lnk))
    if not reasons:
        return None
    return ("laya.train: predictions collapsed to the class prior (%s). "
            "The default epoch budget is often too small on a few hundred or a thousand "
            "labelled rows. Try more epochs, more data, or another seed."
            % "; ".join(reasons))


def finetune(data: str, model_dir: str, output_dir: str, config: Optional[TrainConfig] = None,
             device: Optional[str] = "auto") -> Dict[str, Any]:
    """Preprocess, train, calibrate and save; returns a summary of the run.

    `data` is a JSONL file of `{state, questions, gold}` or `{state, questions, expected}` rows,
    or a CSV file with text and label columns.
    `model_dir` is a local checkpoint directory -- the layout `laya.load` reads. The result in
    `output_dir` loads with `laya.load(output_dir)`.
    """
    config = config or TrainConfig()
    config.validate()
    dev = resolve_device(device)
    model, tok, cfg = load_checkpoint(model_dir)
    # The base checkpoint is scored on eval_items before train_model runs, and calibration_records
    # sends its inputs to `dev`, so the weights have to be there first (CUDA and MPS crash otherwise).
    model.to(dev)
    max_len = config.max_len or cfg.get("max_len", 512)
    head_max_len = config.head_max_len or cfg.get("head_max_len", 192)
    if config.option_layout is not None:
        cfg = dict(cfg, option_layout=config.option_layout)
    parallel = uses_parallel_layout(cfg)

    rows = read_data(data, text_column=config.text_column, label_column=config.label_column,
                     question_id=config.question_id, instructions=config.instructions)
    items, skipped = items_from_rows(tok, rows, max_len, head_max_len,
                                     label_smoothing=config.label_smoothing)
    if not items:
        raise ValueError("%s produced no training items (skipped: %r)" % (data, skipped))
    train_items, calib_items = split_calibration(items, config.calib_max, config.calib_frac, config.calib_seed)

    eval_skipped = {}
    eval_mode = None
    is_held_out = False
    eval_source = None
    eval_items = []
    eval_note = None

    if config.eval_data:
        eval_source = config.eval_data
        eval_rows = read_data(config.eval_data, text_column=config.text_column, label_column=config.label_column,
                              question_id=config.question_id, instructions=config.instructions)
        eval_items, eval_skipped = items_from_rows(tok, eval_rows, max_len, head_max_len, label_smoothing=0.0)

        if not eval_items:
            eval_mode = None
            is_held_out = False
            eval_note = ("Evaluation file %s yielded 0 usable items (skipped: %r); "
                         "no evaluation performed." % (config.eval_data, eval_skipped))
            warnings.warn("laya.train: %s" % (eval_note,), RuntimeWarning, stacklevel=2)
            print("eval items 0 from %s (skipped %r)" % (config.eval_data, eval_skipped), flush=True)
        else:
            # Build input signatures from normalized, usable items:
            # (state token ids, normalized question schema).
            # Checked separately from targets to detect shared inputs even when labels differ (#967).
            train_input_sigs = {
                (tuple(it["state_ids"]), json.dumps(it["q"], sort_keys=True))
                for it in train_items
            }
            calib_input_sigs = {
                (tuple(it["state_ids"]), json.dumps(it["q"], sort_keys=True))
                for it in calib_items
            }

            train_overlap_count = sum(
                1 for it in eval_items
                if (tuple(it["state_ids"]), json.dumps(it["q"], sort_keys=True)) in train_input_sigs
            )
            calib_overlap_count = sum(
                1 for it in eval_items
                if (tuple(it["state_ids"]), json.dumps(it["q"], sort_keys=True)) in calib_input_sigs
            )
            total_overlap_count = sum(
                1 for it in eval_items
                if (tuple(it["state_ids"]), json.dumps(it["q"], sort_keys=True)) in train_input_sigs
                or (tuple(it["state_ids"]), json.dumps(it["q"], sort_keys=True)) in calib_input_sigs
            )

            if total_overlap_count > 0:
                eval_mode = "overlapping_eval"
                is_held_out = False
                if train_overlap_count > 0 and calib_overlap_count > 0:
                    eval_note = (
                        "Evaluation set contains inputs that overlap training data (%d) and calibration data (%d) "
                        "(total %d/%d items); metrics do not reflect strictly independent held-out evaluation."
                        % (train_overlap_count, calib_overlap_count, total_overlap_count, len(eval_items))
                    )
                elif train_overlap_count > 0:
                    eval_note = (
                        "Evaluation set contains rows that overlap training data (%d/%d items); "
                        "metrics do not reflect strictly independent held-out evaluation."
                        % (train_overlap_count, len(eval_items))
                    )
                else:
                    eval_note = (
                        "Evaluation set contains inputs that overlap calibration data (%d/%d items); "
                        "metrics do not reflect strictly independent held-out evaluation."
                        % (calib_overlap_count, len(eval_items))
                    )
                print("eval items %d from %s (skipped %r, %d overlapping training/calibration items)"
                      % (len(eval_items), config.eval_data, eval_skipped, total_overlap_count), flush=True)
            else:
                eval_mode = "held_out"
                is_held_out = True
                eval_note = "Metrics reflect independent held-out evaluation."
                print("eval items %d from %s (skipped %r, 0 overlapping training/calibration items)"
                      % (len(eval_items), config.eval_data, eval_skipped), flush=True)
    elif calib_items:
        eval_items = calib_items
        eval_source = "calibration_slice"
        eval_mode = "in_sample_calibration"
        is_held_out = False
        eval_note = ("Metrics reflect in-sample calibration fit, not independent held-out evaluation. "
                     "Provide --eval for held-out validation.")
    else:
        eval_items = []
        eval_source = None
        eval_mode = None
        is_held_out = False
        eval_note = "No evaluation performed (0 calibration items and no --eval set was provided)."
        warnings.warn("laya.train: dataset has 0 calibration items and no --eval set was provided; "
                      "skipping evaluation to avoid reporting training fit as generalization.",
                      RuntimeWarning, stacklevel=2)

    print("train items %d, calibration items %d, eval items %d%s, skipped %r, device %s"
          % (len(train_items), len(calib_items), len(eval_items),
             (" (" + str(eval_source) + ")") if eval_source else "", skipped, dev), flush=True)
    print(update_budget_message(len(train_items), config), flush=True)

    # Evaluate base checkpoint on eval items prior to training
    before_eval = None
    if eval_items:
        base_records = calibration_records(model, tok, eval_items, dev, max_len, head_max_len, parallel=parallel)
        base_temp = cfg.get("temperature")
        base_temp_by_options = cfg.get("temperature_by_options")
        before_eval = evaluate_records(base_records, base_temp, base_temp_by_options)

    def checkpoint_latest(epoch, _loss):
        save_checkpoint(model, tok, dict(cfg, max_len=max_len, head_max_len=head_max_len),
                        os.path.join(output_dir, "checkpoint_latest"))

    history = train_model(model, tok, train_items, config, dev, max_len, head_max_len,
                          on_epoch_end=checkpoint_latest, parallel=parallel)

    records = calibration_records(model, tok, calib_items, dev, max_len, head_max_len, parallel=parallel)
    collapse_msg = prior_collapse_message(records)
    if collapse_msg:
        # Same channel as the weak-calibration notes: a checkpoint that predicts the
        # class prior for every input otherwise looks like a finished, unlearnable task.
        warnings.warn(collapse_msg, RuntimeWarning, stacklevel=2)
    fitted = fit_temperature_map(records)
    calibration = calibration_report(records, fitted["temperature"])
    for name, entry in calibration.items():
        for issue in entry["issues"]:
            warnings.warn("laya.train: %s calibration: %s" % (name, issue), RuntimeWarning, stacklevel=2)

    try:
        abstention_thresholds = fit_abstention_thresholds(
            records,
            temperature=fitted["temperature"],
            temperature_by_options=fitted["temperature_by_options"],
            target_error=config.target_error,
            min_bucket_n=config.min_abstain_n,
        )
    except Exception as e:
        warnings.warn("laya.train: failed to fit abstention thresholds: %s" % (e,), RuntimeWarning, stacklevel=2)
        abstention_thresholds = {}

    # Evaluate fine-tuned checkpoint at fitted temperatures
    after_eval = None
    comparison = None
    if eval_items:
        after_records = calibration_records(model, tok, eval_items, dev, max_len, head_max_len, parallel=parallel)
        after_eval = evaluate_records(after_records, fitted["temperature"], fitted["temperature_by_options"])
        comparison = {
            "delta_accuracy": round(after_eval["accuracy"] - before_eval["accuracy"], 4),
            "delta_loss": round(after_eval["loss"] - before_eval["loss"], 4),
            "delta_ece": round(after_eval["ece"] - before_eval["ece"], 4) if (after_eval["ece"] is not None and before_eval["ece"] is not None) else None,
            "delta_brier": round(after_eval["brier"] - before_eval["brier"], 4) if (after_eval["brier"] is not None and before_eval["brier"] is not None) else None,
            "delta_brier_top1": round(after_eval["brier_top1"] - before_eval["brier_top1"], 4) if (after_eval["brier_top1"] is not None and before_eval["brier_top1"] is not None) else None,
            "delta_mean_confidence": round(after_eval["mean_confidence"] - before_eval["mean_confidence"], 4),
        }
        if eval_mode == "held_out":
            print("\n=== Evaluation (before vs after fine-tuning on %d held-out items from %s) ==="
                  % (len(eval_items), eval_source), flush=True)
        elif eval_mode == "overlapping_eval":
            print("\n=== Evaluation (before vs after fine-tuning on %d items from %s; overlaps training data) ==="
                  % (len(eval_items), eval_source), flush=True)
        else:
            print("\n=== Calibration Evidence (in-sample calibration slice, %d items; not held-out) ==="
                  % (len(eval_items)), flush=True)
        print("%-22s %-14s %-14s %-14s" % ("Metric", "Before", "After", "Delta"), flush=True)
        print("-" * 65, flush=True)
        for metric, key in [("Accuracy", "accuracy"), ("Loss", "loss"), ("ECE", "ece"),
                            ("Brier (multi-class)", "brier"), ("Brier (top-1)", "brier_top1"),
                            ("Mean Confidence", "mean_confidence")]:
            b_val = before_eval.get(key)
            a_val = after_eval.get(key)
            d_val = comparison.get("delta_" + key)
            b_str = "%.4f" % b_val if b_val is not None else "N/A"
            a_str = "%.4f" % a_val if a_val is not None else "N/A"
            d_str = ("%+.4f" % d_val) if d_val is not None else "N/A"
            print("%-22s %-14s %-14s %-14s" % (metric, b_str, a_str, d_str), flush=True)
        print("-" * 65 + "\n", flush=True)

    if eval_note is None:
        if eval_mode == "held_out":
            eval_note = "Metrics reflect independent held-out evaluation."
        elif eval_mode == "overlapping_eval":
            eval_note = ("Evaluation set contains rows that overlap training data; "
                         "metrics do not reflect strictly independent held-out evaluation.")
        elif eval_mode == "in_sample_calibration":
            eval_note = ("Metrics reflect in-sample calibration fit, not independent held-out evaluation. "
                         "Provide --eval for held-out validation.")
        else:
            eval_note = "No evaluation performed (0 calibration items and no --eval set was provided)."

    train_report = {
        "eval_source": eval_source,
        "eval_mode": eval_mode,
        "is_held_out": is_held_out,
        "eval_items": len(eval_items),
        "note": eval_note,
        "before": before_eval,
        "after": after_eval,
        "comparison": comparison,
        "calibration": calibration,
        "abstention_thresholds": abstention_thresholds,
        "training": asdict(config),
    }

    out_cfg = dict(cfg, max_len=max_len, head_max_len=head_max_len, fine_tuned=True,
                   temperature=fitted["temperature"])
    out_cfg.pop("temperature_by_options", None)
    if fitted["temperature_by_options"]:
        out_cfg["temperature_by_options"] = fitted["temperature_by_options"]
    out_cfg["training"] = dict(out_cfg.get("training") or {}, laya_train=asdict(config),
                               laya_train_calibration=calibration,
                               train_report=train_report)
    if abstention_thresholds:
        out_cfg["training"]["abstention_thresholds"] = abstention_thresholds
    save_checkpoint(model, tok, out_cfg, output_dir)

    checkpoint_latest_dir = os.path.join(output_dir, "checkpoint_latest")
    if os.path.isdir(checkpoint_latest_dir):
        # Synchronize final calibrated config so checkpoint_latest matches the final artifact
        tmp_cfg = os.path.join(checkpoint_latest_dir, "rl_agent_config.json.tmp")
        with open(tmp_cfg, "w", encoding="utf-8") as f:
            json.dump(out_cfg, f, indent=2)
        os.replace(tmp_cfg, os.path.join(checkpoint_latest_dir, "rl_agent_config.json"))

    # Save the questions schema alongside the checkpoint for inference reuse
    sample_questions = {}
    for r in rows:
        qs = r.get("questions")
        if isinstance(qs, dict):
            for qid, qdef in qs.items():
                if qid not in sample_questions:
                    sample_questions[qid] = qdef
                elif sample_questions[qid] != qdef:
                    warnings.warn("Conflicting schema detected for question %r across rows; "
                                  "keeping first seen definition." % (qid,))
    if sample_questions:
        for d in (output_dir, checkpoint_latest_dir):
            if os.path.isdir(d):
                tmp_path = os.path.join(d, "questions.json.tmp")
                with open(tmp_path, "w", encoding="utf-8") as f:
                    json.dump(sample_questions, f, indent=2)
                os.replace(tmp_path, os.path.join(d, "questions.json"))

    # Save train_report.json beside questions.json and checkpoint
    for d in (output_dir, checkpoint_latest_dir):
        if os.path.isdir(d):
            tmp_report = os.path.join(d, "train_report.json.tmp")
            with open(tmp_report, "w", encoding="utf-8") as f:
                json.dump(train_report, f, indent=2)
            os.replace(tmp_report, os.path.join(d, "train_report.json"))

    return {
        "train_items": len(train_items),
        "calibration_items": len(calib_items),
        "eval_items": len(eval_items),
        "skipped": skipped,
        "epoch_loss": history,
        "temperature": fitted["temperature"],
        "temperature_by_options": fitted["temperature_by_options"],
        "n_by_bucket": fitted["n_by_bucket"],
        "calibration": calibration,
        "abstention_thresholds": abstention_thresholds,
        "train_report": train_report,
        "output_dir": output_dir,
    }


def dry_run(data: str, model_dir: str, config: Optional[TrainConfig] = None) -> Dict[str, Any]:
    """Inspect dataset and build items without loading model weights; returns dataset stats."""
    from transformers import AutoTokenizer
    from .agent import _fix_tokenizer_config

    config = config or TrainConfig()
    config.validate()
    resolved_dir = resolve_checkpoint_dir(model_dir)
    _fix_tokenizer_config(resolved_dir)
    tok = AutoTokenizer.from_pretrained(os.path.join(resolved_dir, "tokenizer"))
    with open(os.path.join(resolved_dir, "rl_agent_config.json"), encoding="utf-8") as f:
        cfg = json.load(f)
    max_len = config.max_len or cfg.get("max_len", 512)
    head_max_len = config.head_max_len or cfg.get("head_max_len", 192)

    rows = read_data(data, text_column=config.text_column, label_column=config.label_column,
                     question_id=config.question_id, instructions=config.instructions)
    items, skipped = items_from_rows(tok, rows, max_len, head_max_len,
                                     label_smoothing=config.label_smoothing)
    # The training budget is set by the post-calibration item count, so split the same way
    # finetune does rather than report on the raw valid-item count.
    train_items, calib_items = split_calibration(items, config.calib_max, config.calib_frac, config.calib_seed)
    summary = {
        "data": data,
        "rows_read": len(rows),
        "valid_items": len(items),
        "train_items": len(train_items),
        "calibration_items": len(calib_items),
        "effective_batch": config.micro_batch * config.grad_accum,
        "optimizer_updates": optimizer_updates(len(train_items), config.micro_batch,
                                               config.grad_accum, config.epochs),
        "skipped": skipped,
        "max_len": max_len,
        "head_max_len": head_max_len,
    }
    if config.eval_data:
        eval_rows = read_data(config.eval_data, text_column=config.text_column, label_column=config.label_column,
                              question_id=config.question_id, instructions=config.instructions)
        eval_items, eval_skipped = items_from_rows(tok, eval_rows, max_len, head_max_len, label_smoothing=0.0)
        summary["eval_data"] = config.eval_data
        summary["eval_rows_read"] = len(eval_rows)
        summary["eval_valid_items"] = len(eval_items)
        summary["eval_skipped"] = eval_skipped
        print("dry-run: eval %d rows read, %d valid items, skipped: %r"
              % (len(eval_rows), len(eval_items), eval_skipped), flush=True)

    print("dry-run: %d rows read, %d valid items, skipped: %r"
          % (len(rows), len(items), skipped), flush=True)
    print("dry-run: " + update_budget_message(len(train_items), config), flush=True)
    return summary


In [ ]:
%%writefile {REXLIB}/rex/losses.py
"""Multi-task loss functions and training objectives for the Rex Decision Engine.

This module replaces the baseline single-turn cross-entropy objective with an
integrated multi-objective optimization suite:
1. ToolAsymmetricFocalLoss: Multi-label focal loss for parallel tool selection,
   down-weighting dominant negative classes (gamma=2.0) to prevent gradient starvation.
2. PoEMaskedCrossEntropyLoss: Process-of-Elimination (PoE) masked cross-entropy for
   the verdict head, isolating backpropagation strictly to active/surviving hypotheses.
3. ToolOrthogonalLoss: Cosine Gram-matrix regularizer penalizing collinear tool projection
   vectors to prevent avalanche activation (simultaneous 5-10 tool bursts).
4. RexMultiTaskLoss: Composite loss orchestrator returning structured LossBreakdown.
"""

from dataclasses import dataclass
from typing import Optional, Union

import torch
import torch.nn as nn
import torch.nn.functional as F

# Named constants to avoid magic numbers (G25)
DEFAULT_FOCAL_GAMMA_NEG: float = 2.0
DEFAULT_FOCAL_ALPHA_POS: float = 1.0
DEFAULT_ORTHO_WEIGHT: float = 0.05
DEFAULT_TOOL_WEIGHT: float = 1.0
DEFAULT_VERDICT_WEIGHT: float = 1.0
MASKED_LOGIT_VALUE: float = -1e4
NUMERICAL_EPSILON: float = 1e-8


@dataclass(frozen=True)
class RexLossWeights:
    """Configurable loss scaling weights for multi-task optimization (F1)."""

    tool_weight: float = DEFAULT_TOOL_WEIGHT
    verdict_weight: float = DEFAULT_VERDICT_WEIGHT
    ortho_weight: float = DEFAULT_ORTHO_WEIGHT
    s_weight: float = 1.0
    r_weight: float = 0.0  # Hyp-level risk supervised directly via hyp_risk_weight; 0.0 avoids gradient conflict
    gain_weight: float = 1.0
    hyp_weight: float = 1.0
    hyp_risk_weight: float = 1.0


@dataclass(frozen=True)
class LossBreakdown:
    """Structured container holding total and constituent losses for logging and metrics."""

    total_loss: torch.Tensor
    tool_loss: torch.Tensor
    verdict_loss: torch.Tensor
    ortho_loss: torch.Tensor
    s_loss: Optional[torch.Tensor] = None
    r_loss: Optional[torch.Tensor] = None
    gain_loss: Optional[torch.Tensor] = None
    hyp_loss: Optional[torch.Tensor] = None
    hyp_risk_loss: Optional[torch.Tensor] = None


@dataclass(frozen=True)
class RexMultiTaskInputs:
    """Encapsulates input tensors for multi-task loss computation (F1)."""

    tool_logits: torch.Tensor
    tool_labels: torch.Tensor
    verdict_logits: torch.Tensor
    poe_mask: torch.Tensor
    target_verdict: torch.Tensor
    tool_projection_weight: torch.Tensor
    s_logits: Optional[torch.Tensor] = None
    s_targets: Optional[torch.Tensor] = None
    r_logits: Optional[torch.Tensor] = None
    r_targets: Optional[torch.Tensor] = None
    gain_logits: Optional[torch.Tensor] = None
    gain_targets: Optional[torch.Tensor] = None
    redundancy_mask: Optional[torch.Tensor] = None
    hyp_logits: Optional[torch.Tensor] = None
    hyp_targets: Optional[torch.Tensor] = None
    hyp_mask: Optional[torch.Tensor] = None
    hyp_risk_logits: Optional[torch.Tensor] = None
    hyp_risk_targets: Optional[torch.Tensor] = None


class ToolAsymmetricFocalLoss(nn.Module):
    """Asymmetric Focal Loss for multi-label Tool Selection Head.

    Applies independent sigmoid probability estimation for each tool dimension.
    Applies an exponential decay weight (gamma_neg = 2.0) exclusively on negative
    labels (y = 0) to prevent tens of inactive tools from dominating gradient updates
    over the 1-3 active tools dispatched per round.
    """

    def __init__(
        self,
        gamma_neg: float = DEFAULT_FOCAL_GAMMA_NEG,
        alpha_pos: float = DEFAULT_FOCAL_ALPHA_POS,
        reduction: str = "mean",
    ) -> None:
        super().__init__()
        if gamma_neg < 0.0:
            raise ValueError(f"gamma_neg must be non-negative, got {gamma_neg}")
        if reduction not in ("mean", "sum", "none"):
            raise ValueError(f"Invalid reduction '{reduction}'. Expected 'mean', 'sum', or 'none'.")

        self.gamma_neg = gamma_neg
        self.alpha_pos = alpha_pos
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """Compute asymmetric focal loss between tool logits and multi-hot binary targets.

        Args:
            logits: Predicted tool logits of shape [..., num_tools].
            targets: Multi-hot binary labels of shape [..., num_tools] (0.0 or 1.0).

        Returns:
            Reduced scalar loss tensor (or unreduced tensor if reduction='none').
        """
        targets = targets.to(dtype=logits.dtype)

        # Numerically stable sigmoid log-probabilities:
        # log(p) = logsigmoid(z), log(1 - p) = logsigmoid(-z)
        log_p = F.logsigmoid(logits)
        log_not_p = F.logsigmoid(-logits)

        # Probabilities for negative decay factor
        probs = torch.sigmoid(logits)

        # Asymmetric focal weighting:
        # Positive loss retains full gradient (or scaled by alpha_pos)
        pos_loss = -self.alpha_pos * targets * log_p

        # Negative loss is suppressed by (p)^gamma_neg for well-classified negatives
        neg_decay = torch.pow(probs, self.gamma_neg)
        neg_loss = -(1.0 - targets) * neg_decay * log_not_p

        loss = pos_loss + neg_loss

        if self.reduction == "none":
            return loss

        # Sum over tool dimensions, then reduce over batch
        per_sample_loss = loss.sum(dim=-1)
        if self.reduction == "sum":
            return per_sample_loss.sum()
        return per_sample_loss.mean()


class PoEMaskedCrossEntropyLoss(nn.Module):
    """Process-of-Elimination (PoE) Masked Cross-Entropy for Verdict Head.

    Applies hard logit masking (-1e4) to hypotheses that have been eliminated by
    PoE logical criteria (poe_mask == 0). Ensures that softmax normalization and
    gradient backpropagation are computed strictly over surviving candidate hypotheses.
    """

    def __init__(self, masked_logit: float = MASKED_LOGIT_VALUE) -> None:
        super().__init__()
        self.masked_logit = masked_logit

    def forward(
        self,
        logits: torch.Tensor,
        poe_mask: torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:
        """Compute masked cross-entropy loss over surviving hypotheses.

        Args:
            logits: Hypothesis logits of shape [batch_size, num_hypotheses] or [num_hypotheses].
            poe_mask: Binary indicator tensor of shape [batch_size, num_hypotheses],
                      where 1 = active/surviving, 0 = eliminated by PoE.
            targets: Ground-truth target indices of shape [batch_size] or scalar int tensor.

        Returns:
            Scalar cross-entropy loss tensor.
        """
        # Ensure minimum 2D shapes for batch processing
        is_single_sample = logits.dim() == 1
        if is_single_sample:
            logits = logits.unsqueeze(0)
            poe_mask = poe_mask.unsqueeze(0)
            if targets.dim() == 0:
                targets = targets.unsqueeze(0)

        # Target Integrity Protection: Ensure targets do not point to eliminated hypotheses
        # If any target was refuted by PoE, isolate and zero-out its loss contribution
        # to strictly protect gradient stability and prevent backward noise.
        target_indices = targets.unsqueeze(1) if targets.dim() == 1 else targets
        target_survives = poe_mask.gather(1, target_indices).squeeze(1) == 1

        # Hard logit masking for eliminated hypotheses (forces dLoss/dz_j = 0 for M_j = 0)
        masked_logits = logits.masked_fill(poe_mask == 0, self.masked_logit)

        if not target_survives.all():
            loss_unreduced = F.cross_entropy(masked_logits, targets, reduction="none")
            valid_mask = target_survives.to(dtype=logits.dtype)
            num_valid = valid_mask.sum().clamp(min=1.0)
            return (loss_unreduced * valid_mask).sum() / num_valid

        # Cross-entropy automatically computes log-softmax over unmasked survivor logits
        loss = F.cross_entropy(masked_logits, targets)
        return loss


class ToolOrthogonalLoss(nn.Module):
    """Orthogonal Loss on Tool Projection Matrix.

    Penalizes pairwise cosine similarity between row vectors in the tool projection
    weight matrix to enforce directional orthogonality in latent space.
    This prevents avalanche activation (where 5-10 collinear tools trigger simultaneously).
    """

    def __init__(self, epsilon: float = NUMERICAL_EPSILON) -> None:
        super().__init__()
        self.epsilon = epsilon

    def forward(self, weight_matrix: torch.Tensor) -> torch.Tensor:
        """Compute off-diagonal Frobenius norm penalty on the Gram matrix.

        Args:
            weight_matrix: Linear layer weight of shape [num_tools, hidden_dim].

        Returns:
            Scalar orthogonal penalty loss tensor.
        """
        num_tools = weight_matrix.size(0)
        if num_tools <= 1:
            return torch.tensor(0.0, device=weight_matrix.device, dtype=weight_matrix.dtype)

        # Normalize each tool vector to unit Euclidean norm
        norm_weights = F.normalize(weight_matrix, p=2, dim=1, eps=self.epsilon)

        # Compute cosine Gram matrix [num_tools, num_tools]
        gram_matrix = torch.matmul(norm_weights, norm_weights.t())

        # Target is the identity matrix (perfect orthogonality across different tools)
        identity = torch.eye(num_tools, device=weight_matrix.device, dtype=weight_matrix.dtype)

        # Squared difference penalizes non-zero off-diagonal correlations
        off_diagonal_diff = gram_matrix - identity
        frobenius_norm_sq = torch.sum(off_diagonal_diff ** 2)

        # Normalize by number of off-diagonal elements: N * (N - 1)
        normalizer = float(num_tools * (num_tools - 1))
        return frobenius_norm_sq / normalizer


class RexMultiTaskLoss(nn.Module):
    """Composite Multi-Task Loss Orchestrator.

    Integrates Tool Focal Loss, PoE-Masked Verdict Cross-Entropy, and Tool Orthogonal Loss
    into a unified multi-objective loss function with structured reporting.
    """

    def __init__(
        self,
        weights: Optional[RexLossWeights] = None,
        focal_gamma_neg: float = DEFAULT_FOCAL_GAMMA_NEG,
        masked_logit: float = MASKED_LOGIT_VALUE,
    ) -> None:
        super().__init__()
        self.weights = weights if weights is not None else RexLossWeights()
        self.tool_loss_fn = ToolAsymmetricFocalLoss(gamma_neg=focal_gamma_neg)
        self.verdict_loss_fn = PoEMaskedCrossEntropyLoss(masked_logit=masked_logit)
        self.ortho_loss_fn = ToolOrthogonalLoss()

    def forward(
        self,
        inputs_or_tool_logits: Union[RexMultiTaskInputs, torch.Tensor],
        tool_labels: Optional[torch.Tensor] = None,
        verdict_logits: Optional[torch.Tensor] = None,
        poe_mask: Optional[torch.Tensor] = None,
        target_verdict: Optional[torch.Tensor] = None,
        tool_projection_weight: Optional[torch.Tensor] = None,
        s_logits: Optional[torch.Tensor] = None,
        s_targets: Optional[torch.Tensor] = None,
        r_logits: Optional[torch.Tensor] = None,
        r_targets: Optional[torch.Tensor] = None,
        gain_logits: Optional[torch.Tensor] = None,
        gain_targets: Optional[torch.Tensor] = None,
        redundancy_mask: Optional[torch.Tensor] = None,
        hyp_logits: Optional[torch.Tensor] = None,
        hyp_targets: Optional[torch.Tensor] = None,
        hyp_mask: Optional[torch.Tensor] = None,
    ) -> LossBreakdown:
        """Compute the weighted composite loss and individual loss breakdown.

        Supports both Dataclass container input (clean code F1) and direct tensor kwargs.
        """
        if isinstance(inputs_or_tool_logits, RexMultiTaskInputs):
            inp = inputs_or_tool_logits
            t_logits = inp.tool_logits
            t_labels = inp.tool_labels
            v_logits = inp.verdict_logits
            p_mask = inp.poe_mask
            v_target = inp.target_verdict
            w_proj = inp.tool_projection_weight
            s_l = inp.s_logits
            s_t = inp.s_targets
            r_l = inp.r_logits
            r_t = inp.r_targets
            g_l = inp.gain_logits
            g_t = inp.gain_targets
            red_m = inp.redundancy_mask
            h_l = inp.hyp_logits
            h_t = inp.hyp_targets
            h_m = inp.hyp_mask
            hr_l = inp.hyp_risk_logits
            hr_t = inp.hyp_risk_targets
        else:
            t_logits = inputs_or_tool_logits
            t_labels = tool_labels
            v_logits = verdict_logits
            p_mask = poe_mask
            v_target = target_verdict
            w_proj = tool_projection_weight
            s_l = s_logits
            s_t = s_targets
            r_l = r_logits
            r_t = r_targets
            g_l = gain_logits
            g_t = gain_targets
            red_m = redundancy_mask
            h_l = hyp_logits
            h_t = hyp_targets
            h_m = hyp_mask
            hr_l = hr_t = None

            if any(x is None for x in (t_labels, v_logits, p_mask, v_target, w_proj)):
                raise ValueError("All constituent tensor arguments must be provided when not using RexMultiTaskInputs.")

        # Compute core constituent losses
        l_tool = self.tool_loss_fn(t_logits, t_labels)
        l_verdict = self.verdict_loss_fn(v_logits, p_mask, v_target)
        l_ortho = self.ortho_loss_fn(w_proj)

        # 4. Epistemic Sufficiency Loss (S*)
        l_s: Optional[torch.Tensor] = None
        if s_l is not None and s_t is not None:
            valid_s = (s_t.view(-1) != -100)
            if valid_s.any():
                l_s = F.binary_cross_entropy_with_logits(s_l.view(-1)[valid_s].float(), s_t.view(-1)[valid_s].float())
            else:
                l_s = torch.tensor(0.0, device=s_l.device, dtype=torch.float32)

        # 5. Risk Loss (R*)
        l_r: Optional[torch.Tensor] = None
        if r_l is not None and r_t is not None:
            valid_r = (r_t.view(-1) != -100)
            if valid_r.any():
                l_r = F.binary_cross_entropy_with_logits(r_l.view(-1)[valid_r].float(), r_t.view(-1)[valid_r].float())
            else:
                l_r = torch.tensor(0.0, device=r_l.device, dtype=torch.float32)

        # 6. Counterfactual Tool Information Gain Loss (gain*)
        l_gain: Optional[torch.Tensor] = None
        if g_l is not None and g_t is not None:
            valid_gain = (g_t != -100)
            if red_m is not None:
                valid_gain = valid_gain & (~red_m.bool())
            if valid_gain.any():
                l_gain = F.binary_cross_entropy_with_logits(g_l[valid_gain].float(), g_t[valid_gain].float())
            else:
                l_gain = torch.tensor(0.0, device=g_l.device, dtype=torch.float32)

        # 7. Hypothesis Popperian Scorer Loss (hyp_true)
        l_hyp: Optional[torch.Tensor] = None
        if h_l is not None and h_t is not None:
            valid_hyp = (h_t != -100)
            if h_m is not None:
                valid_hyp = valid_hyp & h_m.bool()
            if valid_hyp.any():
                l_hyp = F.binary_cross_entropy_with_logits(h_l[valid_hyp].float(), h_t[valid_hyp].float())
            else:
                l_hyp = torch.tensor(0.0, device=h_l.device, dtype=torch.float32)

        # 8. Per-hypothesis risk (the business risk of each hypothesis being the true cause)
        l_hyp_risk: Optional[torch.Tensor] = None
        if hr_l is not None and hr_t is not None:
            valid_hr = hr_t != -100
            if h_m is not None:
                valid_hr = valid_hr & h_m.bool()
            if valid_hr.any():
                l_hyp_risk = F.binary_cross_entropy_with_logits(hr_l[valid_hr].float(), hr_t[valid_hr].float())
            else:
                l_hyp_risk = torch.tensor(0.0, device=hr_l.device, dtype=torch.float32)

        # Weighted composite loss
        l_total = (
            self.weights.tool_weight * l_tool
            + self.weights.verdict_weight * l_verdict
            + self.weights.ortho_weight * l_ortho
        )
        if l_s is not None:
            l_total = l_total + self.weights.s_weight * l_s
        if l_r is not None:
            l_total = l_total + self.weights.r_weight * l_r
        if l_gain is not None:
            l_total = l_total + self.weights.gain_weight * l_gain
        if l_hyp is not None:
            l_total = l_total + self.weights.hyp_weight * l_hyp
        if l_hyp_risk is not None:
            l_total = l_total + self.weights.hyp_risk_weight * l_hyp_risk

        return LossBreakdown(
            total_loss=l_total,
            tool_loss=l_tool,
            verdict_loss=l_verdict,
            ortho_loss=l_ortho,
            s_loss=l_s,
            r_loss=l_r,
            gain_loss=l_gain,
            hyp_loss=l_hyp,
            hyp_risk_loss=l_hyp_risk,
        )


In [ ]:
%%writefile {REXLIB}/rex/rex_model.py
"""Rex unified model architecture: RexDecisionModel, 4 decision heads, and checkpoint utilities (Step A8)."""
from __future__ import annotations

import dataclasses
import warnings
from contextlib import nullcontext
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

import torch
import torch.nn as nn

from .common import QTYPES, TEMP_MAX, TEMP_MIN, DecisionModel, clamp_temperature
from .rex_heads import ConformalCalibrator, HypothesisScorer, RiskHead, SufficiencyHead, VerdictBridge
from .rex_memory import EvidenceMemoryManager
from .rex_ponder import LatentPonderBlock
from .rex_registry import ToolSpec
from .rex_tools import (
    DEFAULT_R_THRESHOLD,
    DEFAULT_S_THRESHOLD,
    ToolGainHead,
    ToolMatcherHead,
    evaluate_stall_and_dispatch,
)
from .rex_types import RexState, RexStepOutput

# Checkpoint loader prefix whitelist for newly initialized Rex2 modules
REX2_NEW_MODULE_PREFIXES: Tuple[str, ...] = (
    "ponder_block.",
    "head_s.",
    "head_r.",
    "verdict_bridge.",
    "tool_matcher.",
    "tool_gain.",
    "hyp_scorer.",
    "alive_count_emb.",
    "tool_embeddings",
    "gain_head_trained",
    "calibrator_q_hat",
)

# Sufficiency sees how many cataloged hypotheses survive (0, 1, ..., MAX_ALIVE_COUNT or more)
MAX_ALIVE_COUNT: int = 5

# Deprecated alias for legacy v1 imports
ToolHead = ToolMatcherHead


class RexDecisionModel(DecisionModel):
    """Rex unified decision model (arch='rex2') extending Laya's DecisionModel.

    Architecture:
        - Latent Ponder Block with Cross-Attention over evidence memory and KV caching
        - 4 Decision Heads: SufficiencyHead (S), RiskHead (R), ToolGainHead, ToolMatcherHead
        - Zero-init Verdict Bridge for hypothesis logits with exact Laya checkpoint parity
        - HypothesisScorer for PoE likelihood estimation
        - Unified step-by-step forward execution with RexState and RexStepOutput
    """

    def __init__(
        self,
        encoder: nn.Module,
        tools: Sequence[Union[str, ToolSpec]],
        type_tokens: Optional[Dict[str, str]] = None,
        tool_hidden: Optional[int] = None,
        tool_dropout: float = 0.1,
        tool_prior: Optional[float] = None,
        head_layers: int = 2,
        n_act: int = 2,
        dropout: float = 0.1,
        no_init: bool = False,
        ponder_steps: int = 3,
        calibrator_alpha: float = 0.10,
        tool_meta: Optional[Dict[str, Dict[str, Any]]] = None,
    ):
        super().__init__(encoder, head_layers=head_layers, n_act=n_act, dropout=dropout, no_init=no_init)
        if not tools:
            raise ValueError("tools list must not be empty")

        cleaned_tools: List[str] = []
        for t in tools:
            name = t.name if isinstance(t, ToolSpec) else t
            if not isinstance(name, str) or not name.strip():
                raise ValueError(f"Tool names must be non-empty strings, got {name!r}")
            cleaned_tools.append(name.strip())

        if len(cleaned_tools) != len(set(cleaned_tools)):
            seen = set()
            duplicates = [t for t in cleaned_tools if t in seen or seen.add(t)]
            raise ValueError(f"Duplicate tool names found in tools list: {duplicates}")

        self.tools = cleaned_tools
        self.tool_to_idx = {name: i for i, name in enumerate(self.tools)}
        # Dispatcher metadata (level, cost) per tool, kept in the checkpoint config (rex_config)
        self.tool_meta: Dict[str, Dict[str, Any]] = {}
        self._warned_missing_tool_meta = False
        specs_given = [t for t in tools if isinstance(t, ToolSpec)]
        if specs_given:
            self.set_tool_specs(specs_given, require_all=False)
        if tool_meta:
            self.tool_meta.update({k: dict(v) for k, v in tool_meta.items() if k in self.tool_to_idx})
        self.type_tokens = dict(type_tokens or {})
        self.tool_hidden = tool_hidden
        self.tool_dropout = tool_dropout
        self.tool_prior = tool_prior
        self.ponder_steps = ponder_steps

        d = encoder.config.hidden_size
        # Find head count dividing d_model cleanly
        n_heads = 1
        for h in range(min(8, d), 0, -1):
            if d % h == 0:
                n_heads = h
                break

        with torch.device("meta") if no_init else nullcontext():
            # 1. Latent Ponder Block with KV-caching
            self.ponder_block = LatentPonderBlock(d_model=d, nhead=n_heads, dropout=dropout)

            # 2. Sufficiency & Risk Heads
            self.head_s = SufficiencyHead(d=d)
            self.head_r = RiskHead(d=d)

            # 3. Verdict Bridge (zero initialized for exact Laya parity)
            self.verdict_bridge = VerdictBridge(d=d)

            # 4. Tool Matcher and Gain Heads
            proj_dim = min(256, d)
            self.tool_matcher = ToolMatcherHead(d=d, proj_dim=proj_dim)
            self.tool_gain = ToolGainHead(d=d, hidden_dim=proj_dim)

            # 5. Hypothesis Scorer for PoE
            self.hyp_scorer = HypothesisScorer(d=d, nhead=n_heads)

            # Survivor-count embedding added to h_T for the sufficiency head. S* is defined on the
            # survival set (only the target left among cataloged hypotheses), which the model knows
            # from the PoE mask; zero-initialized so an untrained checkpoint is unchanged.
            self.alive_count_emb = nn.Embedding(MAX_ALIVE_COUNT + 1, d)
            nn.init.zeros_(self.alive_count_emb.weight)

            # Pre-initialize tool representation buffer
            tool_init = torch.randn(len(self.tools), d) * 0.02
            self.register_buffer("tool_embeddings", tool_init)
            self.tool_matcher.set_tool_embeddings(self.tool_embeddings)

            # 6. Memory Manager & Calibrator
            self.memory_manager = EvidenceMemoryManager()
            self.calibrator = ConformalCalibrator(alpha=calibrator_alpha)
            self.register_buffer("gain_head_trained", torch.tensor(False))
            self.register_buffer("calibrator_q_hat", torch.tensor(0.9))

        if no_init:
            self.ponder_block.to_empty(device="cpu")
            self.head_s.to_empty(device="cpu")
            self.head_r.to_empty(device="cpu")
            self.verdict_bridge.to_empty(device="cpu")
            self.tool_matcher.to_empty(device="cpu")
            self.tool_gain.to_empty(device="cpu")
            self.hyp_scorer.to_empty(device="cpu")
            self.alive_count_emb.to_empty(device="cpu")
            self.tool_embeddings = torch.empty((len(self.tools), d), device="cpu")
            self.tool_matcher.set_tool_embeddings(self.tool_embeddings)
            self.gain_head_trained = torch.tensor(False)
            self.calibrator_q_hat = torch.tensor(0.9)

    def load_state_dict(self, state_dict, strict: bool = True, assign: bool = False):
        """Load weights; checkpoints saved before `alive_count_emb` existed get it zero-filled."""
        key = "alive_count_emb.weight"
        if key not in state_dict:
            state_dict = dict(state_dict)
            state_dict[key] = torch.zeros_like(self.alive_count_emb.weight, device="cpu")
        return super().load_state_dict(state_dict, strict=strict, assign=assign)

    @property
    def tool_head(self) -> ToolMatcherHead:
        """Backward-compatibility alias mapping legacy tool_head to tool_matcher."""
        return self.tool_matcher

    def encode_tool_descriptions(self, tool_specs: Sequence[ToolSpec], tok: Any) -> None:
        """Encode textual tool specifications into tool_embeddings buffer using model's encoder (T5)."""
        specs_by_name = {s.name: s for s in tool_specs}
        dev = next(self.parameters()).device
        encoded_vectors = []

        was_training = self.training
        self.eval()
        with torch.no_grad():
            for tool_name in self.tools:
                spec = specs_by_name.get(tool_name)
                desc = spec.description if (spec and getattr(spec, "description", None)) else f"diagnostic tool {tool_name}"
                inputs = tok(desc, return_tensors="pt", truncation=True, max_length=64)
                input_ids = inputs["input_ids"].to(dev)
                attn_mask = inputs["attention_mask"].to(dev)
                qtype = torch.zeros(1, dtype=torch.long, device=dev)
                h = self._contextualize(input_ids, attn_mask, qtype)
                # Use [CLS] representation (token 0)
                emb = h[0, 0].detach()
                encoded_vectors.append(emb)

        new_embeddings = torch.stack(encoded_vectors, dim=0)  # [N_tools, d]
        self.tool_embeddings.copy_(new_embeddings)
        self.tool_matcher.set_tool_embeddings(self.tool_embeddings)
        if was_training:
            self.train()

    def set_tool_specs(self, tool_specs: Sequence[ToolSpec], require_all: bool = True) -> None:
        """Store each tool's dispatcher metadata (level, cost); used when forward() gets no tool_specs."""
        given = {s.name: s for s in tool_specs if s.name in self.tool_to_idx}
        missing = [t for t in self.tools if t not in given]
        if require_all and missing:
            raise ValueError(f"tool_specs is missing model tools: {missing}")
        for name, spec in given.items():
            self.tool_meta[name] = {"level": getattr(spec, "level", "READ"), "cost": float(getattr(spec, "cost", 0.0))}

    def _resolve_tool_meta(self, tool_specs: Optional[Sequence[Any]]) -> Tuple[Dict[str, str], Dict[str, float]]:
        """(levels, costs) per model tool: explicit tool_specs, else the stored metadata."""
        if tool_specs:
            specs = [t if isinstance(t, ToolSpec) else ToolSpec(name=str(t), returns=[]) for t in tool_specs]
            by_name = {s.name: s for s in specs}
            missing = [t for t in self.tools if t not in by_name]
            if missing:
                raise ValueError(f"tool_specs is missing model tools: {missing}; their level/cost would be guessed")
            meta = {n: {"level": getattr(by_name[n], "level", "READ"), "cost": float(getattr(by_name[n], "cost", 0.0))}
                    for n in self.tools}
        else:
            missing = [t for t in self.tools if t not in self.tool_meta]
            if missing and not self._warned_missing_tool_meta:
                warnings.warn(
                    f"RexDecisionModel: no ToolSpec for tools {missing}; treating them as cheap READ tools "
                    f"(cost 0). Pass tool_specs or call set_tool_specs().",
                    RuntimeWarning,
                    stacklevel=3,
                )
                self._warned_missing_tool_meta = True
            meta = {n: self.tool_meta.get(n, {"level": "READ", "cost": 0.0}) for n in self.tools}
        return ({n: m["level"] for n, m in meta.items()}, {n: float(m["cost"]) for n, m in meta.items()})

    def mark_gain_head_trained(self, trained: bool = True) -> None:
        """Record whether the gain head was trained; the dispatcher only trusts gains when True."""
        self.gain_head_trained.fill_(bool(trained))

    def calibrate_conformal(self, probs: torch.Tensor, targets: torch.Tensor) -> float:
        """Fit the conformal threshold on held-out verdict probabilities and store it in the checkpoint buffer."""
        q_hat = self.calibrator.calibrate(probs.detach().float().cpu(), targets.detach().cpu())
        self.calibrator_q_hat.fill_(q_hat)
        return q_hat

    def verdict_temperature(self) -> torch.Tensor:
        """Temperature applied to verdict logits (the Laya `choice` slot), clamped like Laya's own."""
        return self.temperature[QTYPES["choice"]].clamp(TEMP_MIN, TEMP_MAX)

    def set_verdict_temperature(self, value: float) -> None:
        """Store a fitted verdict temperature in the checkpoint buffer."""
        self.temperature[QTYPES["choice"]] = float(value)

    def conformal_threshold(self) -> float:
        """Active conformal threshold: a freshly calibrated value, else the one stored in the checkpoint."""
        if self.calibrator.q_hat is not None:
            return float(self.calibrator.q_hat)
        return float(self.calibrator_q_hat.item())

    @staticmethod
    def _resolve_hyp_alive(
        hyp_alive: torch.Tensor,
        marker_mask: Optional[torch.Tensor],
        hyp_names: Optional[List[str]],
    ) -> torch.Tensor:
        """Normalize a survival mask: padding slots stay dead and the H_novel slot (by name) stays alive."""
        if hyp_alive.dim() == 1:
            hyp_alive = hyp_alive.unsqueeze(0)
        alive = hyp_alive.bool().clone()
        if marker_mask is not None and marker_mask.shape == alive.shape:
            alive = alive & marker_mask.bool().to(alive.device)
        if hyp_names and "H_novel" in hyp_names:
            novel_idx = hyp_names.index("H_novel")
            if novel_idx < alive.size(1):
                alive[:, novel_idx] = True
        return alive

    def ingest_evidence(
        self,
        state: RexState,
        evidence_new: torch.Tensor,
        evidence_mask: Optional[torch.Tensor] = None,
        qtype: Optional[torch.Tensor] = None,
        detach_encoder: bool = False,
        is_truncated: bool = False,
    ) -> RexState:
        """Encode a new evidence piece and append it to the memory of a copy of `state`.

        Does not ponder or dispatch, so callers can score hypotheses on the updated memory
        before the next `forward(state=...)`.
        """
        new_state = dataclasses.replace(state, called_tools=[list(c) for c in state.called_tools])
        if evidence_new.dim() < 2 or evidence_new.size(1) == 0:
            # Nothing observed (e.g. every tool failed): memory is unchanged; the encoder cannot take 0 tokens
            return new_state
        B = new_state.h.size(0)
        if evidence_mask is None:
            evidence_mask = torch.ones_like(evidence_new, dtype=torch.bool)
        if qtype is None:
            qtype = torch.zeros(B, dtype=torch.long, device=evidence_new.device)

        h_ev = self._contextualize(
            evidence_new,
            evidence_mask,
            qtype,
            detach_encoder=detach_encoder,
        )
        return self.memory_manager.append_evidence(
            new_state,
            h_ev,
            evidence_mask.bool(),
            is_truncated=is_truncated,
            ponder_block=self.ponder_block,
        )

    def forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        marker_pos: Optional[torch.Tensor] = None,
        marker_mask: Optional[torch.Tensor] = None,
        qtype: Optional[torch.Tensor] = None,
        state: Optional[RexState] = None,
        evidence_new: Optional[torch.Tensor] = None,
        evidence_mask: Optional[torch.Tensor] = None,
        tool_specs: Optional[Sequence[Any]] = None,
        tool_mask: Optional[torch.Tensor] = None,
        hyp_alive: Optional[torch.Tensor] = None,
        hyp_names: Optional[List[str]] = None,
        detach_encoder: bool = False,
        position_ids: Optional[torch.Tensor] = None,
        option_ids: Optional[torch.Tensor] = None,
        ponder_steps: Optional[int] = None,
        is_truncated: bool = False,
        dispatch: bool = True,
    ) -> Tuple[torch.Tensor, RexState, RexStepOutput]:
        """Unified step-by-step forward execution across multi-round investigation turns.

        Args:
            input_ids: Token IDs for initial contextualization [B, S].
            attention_mask: Attention mask for input tokens [B, S].
            marker_pos: Marker token positions for hypotheses [B, K].
            marker_mask: Boolean mask for valid hypothesis markers [B, K].
            qtype: Question type tensor [B].
            state: RexState from previous turn (None on turn 1).
            evidence_new: New observation tokens or embeddings [B, L_new].
            evidence_mask: Mask for new observation tokens [B, L_new].
            tool_specs: Catalogue of tool specifications (ToolSpec).
            tool_mask: Boolean mask [B, N_tools] disabling unavailable tools.
            hyp_alive: Boolean survival mask [B, K] across Popperian candidate hypotheses.
            hyp_names: Optional names of candidate hypotheses (in marker order); locates the H_novel slot.
            detach_encoder: Whether to detach encoder outputs during training.
            position_ids: Positional encoding overrides.
            option_ids: Option layout indicators.
            ponder_steps: Number of pondering steps T (default: self.ponder_steps).
            is_truncated: Whether evidence observation was truncated (N2).
            dispatch: Run conformal sets and the per-sample STALL/tool dispatcher. Training passes
                False to skip this host-side loop; decision fields are then left empty.

        Returns:
            Tuple of:
                verdict_logits: Float tensor [B, K].
                updated_state: Updated RexState carrier.
                step_output: Detailed RexStepOutput container with logits, probs, and actions.
        """
        if ponder_steps is not None:
            T = ponder_steps
        elif self.training:
            T = None  # LatentPonderBlock randomly samples U(2, 4) during training (L10)
        else:
            T = self.ponder_steps

        if state is None:
            if input_ids is None or attention_mask is None or marker_pos is None or marker_mask is None:
                raise ValueError("Turn 1 requires input_ids, attention_mask, marker_pos, and marker_mask.")

            if qtype is None:
                qtype = torch.zeros(input_ids.size(0), dtype=torch.long, device=input_ids.device)

            # 1. Contextualize initial problem statement & hypothesis tokens
            h = self._contextualize(
                input_ids,
                attention_mask,
                qtype,
                detach_encoder=detach_encoder,
                position_ids=position_ids,
                option_ids=option_ids,
            )
            B, S, d = h.shape

            # 2. Extract hypothesis markers m_j from marker positions
            idx = marker_pos.clamp(min=0)[:, :, None].expand(-1, -1, d)
            m = torch.gather(h, 1, idx)  # [B, K, d]

            # 3. Initial thought state h_0 and initial evidence memory M_0
            h_0 = h[:, 0]  # [CLS] token [B, d]
            M_0 = h        # [B, S, d]
            mem_mask = attention_mask.bool()  # [B, S]

            # 4. Latent Pondering
            h_T, kv_cache, attn_weights = self.ponder_block.ponder(
                h_0, M_0, memory_mask=mem_mask, T=T, kv_cache=None
            )

            # Initialize survival mask if not provided
            K = marker_pos.size(1)
            active_hyp = self._resolve_hyp_alive(
                hyp_alive if hyp_alive is not None else torch.ones((B, K), dtype=torch.bool, device=h.device),
                marker_mask,
                hyp_names,
            )

            current_state = RexState(
                h=h_T,
                M=M_0,
                memory_mask=mem_mask,
                kv_cache=kv_cache,
                hyp_alive=active_hyp,
                hyp_markers=m,
                round_idx=1,
                called_tools=[[] for _ in range(B)],
                last_read_tried=False,
                is_truncated=is_truncated,
                marker_mask=marker_mask.bool(),
            )
        else:
            # Defensive copy avoiding in-place mutation of caller state (L11)
            current_state = dataclasses.replace(
                state,
                called_tools=[list(c) for c in state.called_tools],
            )
            B = current_state.h.size(0)

            # Propagate updated survival mask across subsequent rounds (L1)
            if hyp_alive is not None:
                current_state.hyp_alive = self._resolve_hyp_alive(hyp_alive, current_state.marker_mask, hyp_names)

            if evidence_new is not None:
                current_state = self.ingest_evidence(
                    current_state,
                    evidence_new,
                    evidence_mask,
                    qtype=qtype,
                    detach_encoder=detach_encoder,
                    is_truncated=is_truncated,
                )

            # Ponder with updated evidence memory & existing KV-cache
            h_T, kv_cache, attn_weights = self.ponder_block.ponder(
                current_state.h,
                current_state.M,
                memory_mask=current_state.memory_mask,
                T=T,
                kv_cache=current_state.kv_cache,
            )
            current_state.h = h_T
            current_state.kv_cache = kv_cache
            current_state.round_idx += 1

        # 5. Verdict Bridge: bridge markers with h_T
        m_bridged = self.verdict_bridge(current_state.hyp_markers, h_T)
        raw_verdict_logits = self.scorer(m_bridged).squeeze(-1).float()  # [B, K]

        # Apply marker mask and PoE survival mask
        if marker_mask is None:
            marker_mask = current_state.marker_mask
        effective_mask = (
            marker_mask.bool() if marker_mask is not None else torch.ones_like(raw_verdict_logits, dtype=torch.bool)
        )
        if current_state.hyp_alive is not None:
            effective_mask = effective_mask & current_state.hyp_alive
        verdict_logits = raw_verdict_logits.masked_fill(~effective_mask, -1e4)
        # Calibrated probabilities use the checkpoint's choice temperature; the raw logits feed the losses
        verdict_probs = torch.softmax(verdict_logits / self.verdict_temperature(), dim=-1)

        # 6. Sufficiency and Risk Heads
        # Cataloged survivors: live slots minus the H_novel residual, which is always alive
        n_known_alive = (effective_mask.sum(dim=-1) - 1).clamp(0, MAX_ALIVE_COUNT)
        s_logit = self.head_s(h_T + self.alive_count_emb(n_known_alive))
        s_prob = torch.sigmoid(s_logit)
        # Epistemic guardrail against survivor-count shortcut (L1):
        # When only 1 cataloged hypothesis is alive, PoE may have mistakenly pruned the true target.
        # If the surviving hypothesis does not match the evidence (high entropy / doubt between survivor and H_novel),
        # epistemic sufficiency S must be tempered to prevent premature convergence.
        is_single_survivor = (n_known_alive == 1).unsqueeze(-1) if n_known_alive.dim() == 1 else (n_known_alive == 1)
        if is_single_survivor.any():
            cat_mask = effective_mask.clone()
            cat_mask[..., -1] = False
            p_cat = (verdict_probs.detach() * cat_mask.float()).sum(dim=-1, keepdim=True)
            survivor_fit = torch.max(p_cat, verdict_probs.detach()[..., -1:])
            s_prob = torch.where(is_single_survivor, torch.min(s_prob, (survivor_fit * 1.25).clamp(0.0, 1.0)), s_prob)
            s_logit = torch.logit(s_prob.clamp(1e-6, 1.0 - 1e-6))
        # Risk factorizes over hypotheses: R = sum_j p_j * r_j, where r_j is the business risk of
        # hypothesis j being the true cause (read from its bridged marker) and p_j the verdict
        # probability. p_j is detached so the risk loss does not pull the verdict.
        hyp_risk_logits = self.head_r(m_bridged).squeeze(-1).float()  # [B, K]
        hyp_risk = torch.sigmoid(hyp_risk_logits)
        r_prob = (verdict_probs.detach() * hyp_risk).sum(dim=-1, keepdim=True).clamp(1e-6, 1.0 - 1e-6)
        r_logit = torch.logit(r_prob)

        # 7. Tool Matcher and Gain Heads
        tool_logits = self.tool_matcher(h_T)
        if tool_mask is not None:
            tool_logits = tool_logits.masked_fill(~tool_mask, -1e4)
        tool_gain_logits = self.tool_gain(h_T, self.tool_embeddings, return_logits=True)
        tool_gains = torch.sigmoid(tool_gain_logits)

        # 8. Hypothesis survival scores over the current evidence memory (PoE neural signal)
        hyp_logits = self.hyp_scorer(
            current_state.hyp_markers, current_state.M, current_state.memory_mask, return_logits=True
        )
        hyp_scores = torch.sigmoid(hyp_logits)

        k_names = hyp_names or [f"H_{i}" for i in range(verdict_logits.size(1))]
        # Order tool_names strictly by self.tools to align with tool_logits / tool_gains columns (L7)
        tool_names = list(self.tools)
        is_gain_trained = bool(self.gain_head_trained.item())

        actions: List[str] = []
        selected_batch: List[List[str]] = [[] for _ in range(B)]
        stalls: List[bool] = [False] * B
        stall_modes: List[str] = []
        stall_reasons: List[str] = []
        rule_stalls: List[bool] = [False] * B
        conformal_sets: Optional[List[List[str]]] = None
        alive_names: List[List[str]] = []

        if dispatch:
            # 9. Conformal Calibration
            conformal_sets = self.calibrator.predict_set(
                verdict_probs, labels=k_names, q_hat=self.conformal_threshold()
            )

            # 10. STALL Evaluation & 3-Branch Dispatch
            tool_levels, tool_costs = self._resolve_tool_meta(tool_specs)

            if isinstance(current_state.last_read_tried, list):
                last_read_list = list(current_state.last_read_tried)
                if len(last_read_list) < B:
                    last_read_list.extend([False] * (B - len(last_read_list)))
            else:
                last_read_list = [bool(current_state.last_read_tried)] * B

            s_vals = s_prob.detach().float().view(-1).tolist()
            r_vals = r_prob.detach().float().view(-1).tolist()
            s_prev_list = current_state.s_prev
            alive_rows = effective_mask.tolist()
            alive_names = [
                [k_names[j] for j, ok in enumerate(row) if ok and j < len(k_names)] for row in alive_rows
            ]

            for b in range(B):
                b_conformal = conformal_sets[b] if conformal_sets else None
                b_called = current_state.called_tools[b] if b < len(current_state.called_tools) else []
                b_mask = tool_mask[b] if tool_mask is not None else None
                b_s_prev = s_prev_list[b] if s_prev_list is not None and b < len(s_prev_list) else None

                action, selected, stall, stall_mode, rule_stall, new_last_read, _ = evaluate_stall_and_dispatch(
                    tool_names=tool_names,
                    tool_logits=tool_logits[b],
                    tool_gains=tool_gains[b],
                    s_prob=s_prob[b],
                    r_prob=r_prob[b],
                    round_idx=current_state.round_idx,
                    called_tools=b_called,
                    conformal_set=b_conformal,
                    last_read_tried=last_read_list[b],
                    tool_levels=tool_levels,
                    tool_costs=tool_costs,
                    tool_mask=b_mask,
                    gain_head_trained=is_gain_trained,
                    s_prev=b_s_prev,
                )
                actions.append(action)
                selected_batch[b] = selected
                stalls[b] = stall
                stall_modes.append(stall_mode)
                rule_stalls[b] = rule_stall
                if action == "STALL":
                    # L8: sufficient-but-risky is a different situation from "no tool can help"
                    high_risk = s_vals[b] >= DEFAULT_S_THRESHOLD and r_vals[b] > DEFAULT_R_THRESHOLD
                    stall_reasons.append("HIGH_RISK" if high_risk else "NO_GAIN")
                else:
                    stall_reasons.append("NONE")
                last_read_list[b] = new_last_read
                if b < len(current_state.called_tools):
                    current_state.called_tools[b].extend(selected)

            current_state.last_read_tried = last_read_list[0] if B == 1 else last_read_list
            current_state.s_prev = s_vals

        step_output = RexStepOutput(
            s_logit=s_logit,
            s_prob=s_prob,
            r_logit=r_logit,
            r_prob=r_prob,
            verdict_logits=verdict_logits,
            verdict_probs=verdict_probs,
            tool_logits=tool_logits,
            tool_gains=tool_gains,
            stall=torch.tensor(stalls, dtype=torch.bool, device=h_T.device),
            stall_mode=stall_modes[0] if stall_modes else "NONE",
            stall_modes=stall_modes,
            rule_stall=torch.tensor(rule_stalls, dtype=torch.bool, device=h_T.device),
            selected_tools=selected_batch,
            proposed_action=actions,
            h_T=h_T,
            tool_names=tool_names,
            hyp_names=k_names,
            conformal_set=conformal_sets,
            attn_weights=attn_weights,
            gain_head_trained=is_gain_trained,
            is_truncated=current_state.is_truncated,
            tool_gain_logits=tool_gain_logits,
            hyp_logits=hyp_logits,
            hyp_scores=hyp_scores,
            hyp_risk=hyp_risk,
            hyp_risk_logits=hyp_risk_logits,
            stall_reason=stall_reasons,
            alive_hypotheses=alive_names,
        )

        return verdict_logits, current_state, step_output

    def step(self, *args, **kwargs) -> Tuple[torch.Tensor, RexState, RexStepOutput]:
        """Convenience alias for forward()."""
        return self.forward(*args, **kwargs)

    def rex_forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        marker_pos: torch.Tensor,
        marker_mask: torch.Tensor,
        qtype: Optional[torch.Tensor] = None,
        hyp_alive: Optional[torch.Tensor] = None,
        tool_mask: Optional[torch.Tensor] = None,
        detach_encoder: bool = False,
        position_ids: Optional[torch.Tensor] = None,
        option_ids: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """Lightweight compatibility shim for legacy Rex v1 training and evaluation scripts.

        Scores hypotheses from the [CLS] vector without the ponder block, the evidence memory or the
        tool/sufficiency heads, so it is not the inference path: use forward() / InvestigatorAgent.
        """
        if qtype is None:
            qtype = torch.zeros(input_ids.size(0), dtype=torch.long, device=input_ids.device)

        h = self._contextualize(
            input_ids,
            attention_mask,
            qtype,
            detach_encoder=detach_encoder,
            position_ids=position_ids,
            option_ids=option_ids,
        )

        h_cls = h[:, 0]
        tool_logits = self.tool_matcher(h_cls)
        if tool_mask is not None:
            tool_logits = tool_logits.masked_fill(~tool_mask, -1e4)

        effective_marker_mask = marker_mask
        if hyp_alive is not None:
            effective_marker_mask = marker_mask & hyp_alive
            if not effective_marker_mask.any(dim=-1).all():
                raise ValueError("All hypotheses have been eliminated for at least one sample in the batch.")

        idx = marker_pos.clamp(min=0)[:, :, None].expand(-1, -1, h.size(-1))
        m = torch.gather(h, 1, idx)
        m_bridged = self.verdict_bridge(m, h_cls)
        verdict_logits = self.scorer(m_bridged).squeeze(-1).float()
        verdict_logits = verdict_logits.masked_fill(~effective_marker_mask, -1e4)

        _, act_logits = self._decide(h, attention_mask, marker_pos, effective_marker_mask)

        return {
            "verdict_logits": verdict_logits,
            "tool_logits": tool_logits,
            "act_logits": act_logits,
            "h_cls": h_cls,
        }

    def compute_verdict_probabilities(
        self,
        verdict_logits: torch.Tensor,
        hyp_alive: Optional[Dict[str, bool]] = None,
        hyp_keys: Optional[List[str]] = None,
    ) -> Dict[str, float]:
        """Compute normalized softmax probabilities over surviving hypotheses."""
        if verdict_logits.dim() == 2:
            verdict_logits = verdict_logits.squeeze(0)
        k = verdict_logits.size(0)
        keys = hyp_keys or [f"H_{i}" for i in range(k)]
        masked_logits = verdict_logits.clone()
        if hyp_alive is not None:
            for idx, key in enumerate(keys):
                if not hyp_alive.get(key, True):
                    masked_logits[idx] = -1e4
        probs = torch.softmax(masked_logits, dim=-1)
        res = {key: float(probs[idx].item()) for idx, key in enumerate(keys)}
        if hyp_alive is not None:
            for key in keys:
                if not hyp_alive.get(key, True):
                    res[key] = 0.0
        s = sum(res.values())
        if s > 0:
            res = {k: v / s for k, v in res.items()}
        return res

    def investigate(self, *args, **kwargs) -> Any:
        """Deprecated procedural investigation method delegated to InvestigatorAgent (Step A10)."""
        from .rex_agent import InvestigatorAgent

        agent = InvestigatorAgent(model=self)
        return agent.investigate(*args, **kwargs)


def rex_config(model: RexDecisionModel, base_cfg: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Generate Rex configuration dictionary compatible with rl_agent_config.json."""
    cfg = dict(base_cfg or {})
    cfg["arch"] = "rex2"
    # build_model sizes the Laya head and act_head from these keys; keep them consistent with the model
    cfg.setdefault("head_layers", len(model.head.layers) if model.head is not None else 0)
    if "act_costs" not in cfg:
        cfg["act_costs"] = {f"act_{i}": 0 for i in range(model.act_head[-1].out_features - 1)}
    # Laya serves choice probabilities with cfg["temperature"]; keep it equal to the Rex verdict temperature
    temps = list(cfg.get("temperature", [1.0] * len(QTYPES)))
    temps[QTYPES["choice"]] = round(float(model.verdict_temperature()), 6)
    cfg["temperature"] = temps
    cfg["rex"] = {
        "tools": list(model.tools),
        "type_tokens": dict(model.type_tokens),
        "tool_hidden": model.tool_hidden,
        "tool_dropout": model.tool_dropout,
        "tool_prior": model.tool_prior,
        "tool_meta": {k: dict(v) for k, v in model.tool_meta.items()},
    }
    return cfg


def from_rex(
    laya_model: DecisionModel,
    tools: Sequence[str],
    type_tokens: Optional[Dict[str, str]] = None,
    tool_hidden: Optional[int] = None,
    tool_dropout: float = 0.1,
    tool_prior: Optional[float] = None,
    base_cfg: Optional[Dict[str, Any]] = None,
) -> RexDecisionModel:
    """Wrap an existing pre-trained Laya DecisionModel into a RexDecisionModel.

    Transfers all trained weights from the Laya model (encoder, transformer head,
    scorer, act_head, type_emb, temperature) and initializes Rex2 modules. When the Laya
    config `base_cfg` is given, its fitted choice temperature becomes the verdict temperature,
    so Rex starts with the probabilities Laya serves.
    """
    rex_model = RexDecisionModel(
        encoder=laya_model.encoder,
        tools=tools,
        type_tokens=type_tokens,
        tool_hidden=tool_hidden,
        tool_dropout=tool_dropout,
        tool_prior=tool_prior,
        head_layers=len(laya_model.head.layers) if laya_model.head is not None else 0,
        n_act=laya_model.act_head[-1].out_features,
        dropout=0.1,
        no_init=False,
    )

    rex_state = laya_model.state_dict()
    missing_keys, unexpected_keys = rex_model.load_state_dict(rex_state, strict=False)

    non_rex2_missing = [
        k for k in missing_keys
        if not any(k.startswith(p) for p in REX2_NEW_MODULE_PREFIXES)
    ]
    if non_rex2_missing:
        raise RuntimeError(f"Unexpected missing keys when loading Laya checkpoint: {non_rex2_missing}")
    if unexpected_keys:
        raise RuntimeError(f"Unexpected extra keys when loading Laya checkpoint: {unexpected_keys}")

    if base_cfg is not None and base_cfg.get("temperature"):
        rex_model.set_verdict_temperature(clamp_temperature(base_cfg["temperature"][QTYPES["choice"]]))

    return rex_model


def save_rex_checkpoint(
    model: RexDecisionModel,
    tok: Any,
    output_dir: str,
    base_cfg: Optional[Dict[str, Any]] = None,
) -> None:
    """Save Rex model weights, tokenizer, encoder config, and configuration to disk."""
    from .train import save_checkpoint

    cfg = rex_config(model, base_cfg=base_cfg)
    save_checkpoint(model, tok, cfg, output_dir)


In [ ]:
%%writefile {REXLIB}/rex/rex_heads.py
"""Rex Decision Heads: Sufficiency (S), Risk (R), Verdict Bridge, Hypothesis Scorer, and Conformal Calibration."""
from __future__ import annotations

import math
from typing import List, Optional

import torch
import torch.nn as nn

DEFAULT_SUFFICIENCY_BIAS_INIT: float = -2.0  # sigmoid(-2.0) ~ 0.1192
DEFAULT_HEAD_DROPOUT: float = 0.1


class SufficiencyHead(nn.Module):
    """Head S: Evaluates epistemic sufficiency probability S in [0, 1] from thought state h_T.

    Architecture:
        h_T [B, d] -> Linear(d, hidden_dim) -> LayerNorm -> GELU -> Dropout -> Linear(hidden_dim, 1)

    Notes:
        - Output bias is initialized to -2.0 so an untrained checkpoint does not trigger premature stopping.
        - Returns raw logit [B, 1] for numerical stability under BCEWithLogitsLoss.
    """

    def __init__(
        self,
        d: int = 1024,
        hidden_dim: Optional[int] = None,
        dropout: float = DEFAULT_HEAD_DROPOUT,
    ):
        super().__init__()
        hidden_dim = hidden_dim or (d // 2)
        self.proj_in = nn.Linear(d, hidden_dim)
        self.ln = nn.LayerNorm(hidden_dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.proj_out = nn.Linear(hidden_dim, 1)

        nn.init.xavier_uniform_(self.proj_in.weight)
        nn.init.zeros_(self.proj_in.bias)
        nn.init.xavier_uniform_(self.proj_out.weight)
        nn.init.constant_(self.proj_out.bias, DEFAULT_SUFFICIENCY_BIAS_INIT)

    def forward(self, h_T: torch.Tensor) -> torch.Tensor:
        """Compute sufficiency logit of shape [B, 1]."""
        x = self.proj_in(h_T)
        x = self.ln(x)
        x = self.act(x)
        x = self.dropout(x)
        return self.proj_out(x)


class RiskHead(nn.Module):
    """Head R: Estimates operational/regulatory escalation risk in [0, 1].

    Architecture:
        x [..., d] -> Linear(d, hidden_dim) -> LayerNorm -> GELU -> Dropout -> Linear(hidden_dim, 1)

    RexDecisionModel applies it to each bridged hypothesis marker to get the risk r_j of that
    hypothesis being the true cause, and reports R = sum_j p_j * r_j over the verdict distribution.
    """

    def __init__(
        self,
        d: int = 1024,
        hidden_dim: Optional[int] = None,
        dropout: float = DEFAULT_HEAD_DROPOUT,
    ):
        super().__init__()
        hidden_dim = hidden_dim or (d // 2)
        self.proj_in = nn.Linear(d, hidden_dim)
        self.ln = nn.LayerNorm(hidden_dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.proj_out = nn.Linear(hidden_dim, 1)

        nn.init.xavier_uniform_(self.proj_in.weight)
        nn.init.zeros_(self.proj_in.bias)
        nn.init.xavier_uniform_(self.proj_out.weight)
        nn.init.zeros_(self.proj_out.bias)

    def forward(self, h_T: torch.Tensor) -> torch.Tensor:
        """Compute operational risk logit of shape [B, 1]."""
        x = self.proj_in(h_T)
        x = self.ln(x)
        x = self.act(x)
        x = self.dropout(x)
        return self.proj_out(x)


class VerdictBridge(nn.Module):
    """Zero-initialized bridge injecting contextual thought state h_T into hypothesis markers.

    Formula:
        m_tilde_j = m_j + g_v * (W_v * h_T)
        logit_j = Scorer(m_tilde_j)

    Properties:
        - At initialization, W_v = 0 and g_v = 0 -> delta = 0 identically.
        - Preserves 100% regression parity with base Laya/Rex1 checkpoints on turn 1.
    """

    def __init__(self, d: int = 1024):
        super().__init__()
        self.d = d
        self.w_v = nn.Linear(d, d, bias=False)
        self.g_v = nn.Parameter(torch.zeros(1))

        # Initialize w_v with normal distribution and g_v=0:
        # Initial perturbation is delta = 0 * (W_v * h_T) = 0 (exact Laya parity),
        # but gradient flow dL/dg_v proportional to (W_v * h_T) is non-zero (breaks zero-grad saddle point T1).
        nn.init.normal_(self.w_v.weight, mean=0.0, std=0.02)

    def forward(self, hyp_markers: torch.Tensor, h_T: torch.Tensor) -> torch.Tensor:
        """Inject thought state h_T into hypothesis markers.

        Args:
            hyp_markers: Marker representations [B, K, d] extracted from [MASK] tokens.
            h_T: Contextualized thought state [B, d].

        Returns:
            Enhanced markers [B, K, d] ready for DecisionModel.scorer.
        """
        h_proj = self.w_v(h_T).unsqueeze(1)    # [B, 1, d]
        delta = self.g_v * h_proj              # [B, 1, d] broadcasts across K
        return hyp_markers + delta


class HypothesisScorer(nn.Module):
    """Neural hypothesis scorer estimating survival likelihood on evidence memory M for PoE.

    Formula:
        Score(H_j) = sigmoid(FFN(CrossAttn(e_{H_j}, LayerNorm(M)))) in [0, 1]
    """

    def __init__(self, d: int = 1024, nhead: int = 8, hidden_dim: Optional[int] = None):
        super().__init__()
        hidden_dim = hidden_dim or (d // 2)
        self.cross_attn = nn.MultiheadAttention(embed_dim=d, num_heads=nhead, batch_first=True)
        self.ln_m = nn.LayerNorm(d)
        self.ln_q = nn.LayerNorm(d)
        self.ffn = nn.Sequential(
            nn.Linear(d, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )

        nn.init.xavier_uniform_(self.ffn[0].weight)
        nn.init.zeros_(self.ffn[0].bias)
        nn.init.xavier_uniform_(self.ffn[-1].weight)
        nn.init.zeros_(self.ffn[-1].bias)

    def forward(
        self,
        hyp_embeddings: torch.Tensor,
        M: torch.Tensor,
        memory_mask: Optional[torch.Tensor] = None,
        return_logits: bool = False,
    ) -> torch.Tensor:
        """Compute hypothesis survival scores in [0, 1] or raw logits.

        Args:
            hyp_embeddings: Hypothesis embedding vectors [B, K, d] or [K, d].
            M: Evidence tokens [B, L_mem, d].
            memory_mask: Boolean mask [B, L_mem] where True is valid, False is padding.
            return_logits: Whether to return unnormalized logits before sigmoid.

        Returns:
            Survival scores [B, K] in range [0, 1], or raw logits if return_logits=True.
        """
        if hyp_embeddings.dim() == 2:
            hyp_embeddings = hyp_embeddings.unsqueeze(0).expand(M.size(0), -1, -1)

        if M.size(1) == 0:
            if return_logits:
                return torch.zeros((hyp_embeddings.size(0), hyp_embeddings.size(1)), device=M.device, dtype=hyp_embeddings.dtype)
            # Empty evidence -> prior score 0.5 across all hypotheses
            return torch.full((hyp_embeddings.size(0), hyp_embeddings.size(1)), 0.5, device=M.device, dtype=hyp_embeddings.dtype)

        q = self.ln_q(hyp_embeddings)
        kv = self.ln_m(M)
        pad_mask = ~memory_mask.bool() if memory_mask is not None else None

        attn_out, _ = self.cross_attn(query=q, key=kv, value=kv, key_padding_mask=pad_mask)
        logits = self.ffn(attn_out).squeeze(-1)
        if return_logits:
            return logits
        scores = torch.sigmoid(logits)
        return scores


def compute_ece(probs: torch.Tensor, labels: torch.Tensor, n_bins: int = 10) -> float:
    """Vectorized Expected Calibration Error (ECE) calculation.

    Args:
        probs: Predicted positive probabilities of shape [N] in [0, 1].
        labels: Binary ground-truth labels of shape [N] (0 or 1).
        n_bins: Number of confidence bins (default: 10).

    Returns:
        Scalar ECE metric as float.
    """
    probs = probs.detach().view(-1)
    labels = labels.detach().view(-1).float()

    bin_boundaries = torch.linspace(0, 1, n_bins + 1, device=probs.device)
    ece = torch.tensor(0.0, device=probs.device)

    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]

        in_bin = (probs >= bin_lower) & (probs < bin_upper) if i < n_bins - 1 else (probs >= bin_lower) & (probs <= bin_upper)
        prop_in_bin = in_bin.float().mean()

        if prop_in_bin.item() > 0:
            accuracy_in_bin = labels[in_bin].mean()
            avg_confidence_in_bin = probs[in_bin].mean()
            ece += torch.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin

    return float(ece.item())


class TemperatureScaling(nn.Module):
    """Learnable temperature scaling for post-hoc probability calibration."""

    def __init__(self, init_temp: float = 1.0):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * init_temp)

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        """Scale logits by clamped temperature."""
        temp = torch.clamp(self.temperature, min=0.1, max=10.0)
        return logits / temp


class ConformalCalibrator:
    """Split-conformal prediction set generator providing distribution-free coverage via Adaptive Prediction Sets (APS)."""

    def __init__(self, alpha: float = 0.1):
        self.alpha = alpha
        self.q_hat: Optional[float] = None

    def calibrate(self, probs: torch.Tensor, targets: torch.Tensor) -> float:
        """Calibrate cumulative non-conformity threshold q_hat on held-out calibration split.

        Args:
            probs: Predicted class probabilities of shape [N, K].
            targets: True class index tensor of shape [N].

        Returns:
            Calibrated quantile threshold q_hat.
        """
        n = probs.size(0)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
        # Find where true target appears in sorted order
        target_ranks = (sorted_indices == targets.unsqueeze(1)).nonzero(as_tuple=True)[1]
        cum_probs = torch.cumsum(sorted_probs, dim=-1)
        scores = cum_probs.gather(1, target_ranks.unsqueeze(1)).squeeze(1)

        # Finite-sample adjusted empirical quantile
        quantile_val = math.ceil((n + 1) * (1.0 - self.alpha)) / n
        quantile_val = min(1.0, max(0.0, quantile_val))

        self.q_hat = float(torch.quantile(scores, quantile_val).item())
        return self.q_hat

    def predict_set(
        self,
        probs: torch.Tensor,
        labels: Optional[List[str]] = None,
        q_hat: Optional[float] = None,
    ) -> List[List[str]]:
        """Construct conformal prediction sets for batch samples using cumulative probability coverage.

        Args:
            probs: Predicted probabilities of shape [B, K].
            labels: List of K class names.
            q_hat: Threshold value (falls back to calibrated self.q_hat or 1 - alpha).

        Returns:
            List of prediction set string labels per sample.
        """
        threshold = q_hat if q_hat is not None else (self.q_hat if self.q_hat is not None else (1.0 - self.alpha))
        b, k = probs.shape
        names = labels if labels is not None and len(labels) == k else [f"H_{i}" for i in range(k)]

        sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
        sets: List[List[str]] = []

        for i in range(b):
            row_probs = sorted_probs[i]
            row_indices = sorted_indices[i]
            selected: List[str] = []
            cum_p = 0.0

            for j in range(k):
                p_j = row_probs[j].item()
                if p_j <= 1e-6 and len(selected) > 0:
                    break
                selected.append(names[row_indices[j].item()])
                cum_p += p_j
                if cum_p >= threshold - 1e-6:
                    break

            sets.append(selected)

        return sets


In [ ]:
%%writefile {REXLIB}/rex/rex_ponder.py
"""Latent Ponder Block with Cross-Attention, KV-Cache, and LayerScale for Rex."""
from __future__ import annotations

import random
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

DEFAULT_LAYERSCALE_INIT: float = 1e-4
DEFAULT_GATE_BIAS_INIT: float = -2.0


class LatentPonderBlock(nn.Module):
    """Iterative latent reasoning block operating over evidence memory M.

    Architecture:
        For step t = 0 to T - 1:
            g_t = sigmoid(W_g * h_t + b_g)
            attn_out = CrossAttention(query=h_t, key=K, value=V, mask=~memory_mask)
            mlp_out = MLP(h_t)
            delta_h = gamma * (attn_out + mlp_out)
            h_{t+1} = LayerNorm(h_t + g_t * delta_h)

    Optimizations & Stability:
        - Key-Value Cache: Projects M to K and V once per round, saving ~2.15 GFLOPs per ponder step.
        - Pre-LN on M: Stabilizes cross-attention representations from unnormalized ModernBERT outputs.
        - LayerScale: Parameter gamma initialized to 1e-4 prevents gradient explosion / perturbation drift.
        - Gate initialization: b_g = -2.0, W_g = 0 ensures g_t ~ 0.1192 at init.
        - Identity preservation: At initialization, cosine_similarity(h_T, LayerNorm(h_0)) > 0.99.
    """

    def __init__(
        self,
        d_model: int = 1024,
        nhead: int = 16,
        dropout: float = 0.1,
        default_t: int = 3,
        layerscale_init: float = DEFAULT_LAYERSCALE_INIT,
    ):
        super().__init__()
        if d_model % nhead != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by nhead ({nhead})")

        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.default_t = default_t

        # Pre-LayerNorm for evidence memory M and thought state query
        self.ln_m = nn.LayerNorm(d_model)
        self.ln_q = nn.LayerNorm(d_model)

        # Cross-Attention projections (Q from h_t, K and V from M)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        # Feed-Forward Network on thought state h_t
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

        # Gating network
        self.gate = nn.Linear(d_model, d_model)

        # LayerScale parameter on the update branch
        self.gamma = nn.Parameter(layerscale_init * torch.ones(d_model))

        # Final LayerNorm applied after residual addition
        self.ln_final = nn.LayerNorm(d_model)

        self._reset_parameters()

    def _reset_parameters(self) -> None:
        """Initialize weights with near-identity and small perturbation guarantees."""
        nn.init.xavier_uniform_(self.q_proj.weight)
        nn.init.zeros_(self.q_proj.bias)
        nn.init.xavier_uniform_(self.k_proj.weight)
        nn.init.zeros_(self.k_proj.bias)
        nn.init.xavier_uniform_(self.v_proj.weight)
        nn.init.zeros_(self.v_proj.bias)
        nn.init.xavier_uniform_(self.out_proj.weight)
        nn.init.zeros_(self.out_proj.bias)

        # Gate initialization: W_g = 0, b_g = -2.0
        nn.init.zeros_(self.gate.weight)
        nn.init.constant_(self.gate.bias, DEFAULT_GATE_BIAS_INIT)

    def build_kv_cache(
        self,
        M: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Precompute projected Key and Value tensors from normalized evidence memory M.

        Args:
            M: Evidence tokens tensor of shape [B, L_mem, d].

        Returns:
            Tuple of (K, V) tensors shaped [B, nhead, L_mem, head_dim].
        """
        if M.size(1) == 0:
            B = M.size(0)
            empty_kv = torch.empty((B, self.nhead, 0, self.head_dim), dtype=M.dtype, device=M.device)
            return empty_kv, empty_kv

        M_norm = self.ln_m(M)
        B, L, _ = M.shape

        K = self.k_proj(M_norm).view(B, L, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(M_norm).view(B, L, self.nhead, self.head_dim).transpose(1, 2)
        return K, V

    def extend_kv_cache(
        self,
        existing_kv: Tuple[torch.Tensor, torch.Tensor],
        M_new: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Incrementally append newly encoded evidence tokens into existing KV-cache."""
        if M_new.size(1) == 0:
            return existing_kv

        K_new, V_new = self.build_kv_cache(M_new)
        old_K, old_V = existing_kv
        K_cat = torch.cat([old_K, K_new], dim=2)
        V_cat = torch.cat([old_V, V_new], dim=2)
        return K_cat, V_cat

    def ponder_step(
        self,
        h: torch.Tensor,
        K: torch.Tensor,
        V: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Execute a single pondering step t -> t + 1.

        Args:
            h: Current thought state [B, d].
            K: Precomputed Key tensor [B, nhead, L_mem, head_dim].
            V: Precomputed Value tensor [B, nhead, L_mem, head_dim].
            attn_mask: Boolean attention mask [B, 1, 1, L_mem] where True indicates KEEP, False indicates IGNORE.

        Returns:
            Updated thought state h_{t+1} of shape [B, d].
        """
        B, d = h.shape
        L_mem = K.size(2)

        # 1. Gate factor: g_t = sigmoid(W_g * h + b_g)
        g_t = torch.sigmoid(self.gate(h))  # [B, d]

        # 2. Cross-Attention
        if L_mem > 0:
            h_norm = self.ln_q(h)
            Q = self.q_proj(h_norm).view(B, 1, self.nhead, self.head_dim).transpose(1, 2)  # [B, nhead, 1, head_dim]
            # Scaled dot-product attention
            attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)     # [B, nhead, 1, L_mem]
            if attn_mask is not None:
                attn_scores = attn_scores.masked_fill(~attn_mask, -1e4)
            attn_probs = F.softmax(attn_scores, dim=-1)
            attn_out = torch.matmul(attn_probs, V)                                          # [B, nhead, 1, head_dim]
            attn_out = attn_out.transpose(1, 2).contiguous().view(B, d)
            attn_out = self.out_proj(attn_out)
        else:
            attn_out = torch.zeros_like(h)

        # 3. MLP branch
        mlp_out = self.mlp(h)

        # 4. LayerScale on update branch
        delta_h = self.gamma * (attn_out + mlp_out)

        # 5. Gated residual update and LayerNorm
        h_next = self.ln_final(h + g_t * delta_h)
        return h_next

    def forward(
        self,
        h_0: torch.Tensor,
        M: Optional[torch.Tensor] = None,
        memory_mask: Optional[torch.Tensor] = None,
        kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        t_steps: Optional[int] = None,
        return_all_steps: bool = False,
    ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor], Optional[List[torch.Tensor]]]:
        """Ponder across T steps over evidence memory M.

        Args:
            h_0: Initial contextualized thought vector [B, d].
            M: Evidence memory tensor [B, L_mem, d]. If provided and kv_cache is None, KV is built.
            memory_mask: Boolean mask [B, L_mem] where True represents valid non-padding tokens.
            kv_cache: Optional pre-cached (K, V) tuple.
            t_steps: Number of pondering steps T. If None, samples U{2, 4} during training or 3 during eval.
            return_all_steps: Whether to return sequence of all intermediate h_t states.

        Returns:
            Tuple of (h_T, kv_cache, all_steps_list).
        """
        # 1. Determine T
        if t_steps is not None:
            T = t_steps
        elif self.training:
            T = random.randint(2, 4)
        else:
            T = self.default_t

        # If T == 0, bypass pondering entirely
        if T == 0:
            if kv_cache is None and M is not None:
                kv_cache = self.build_kv_cache(M)
            return h_0, kv_cache, ([h_0] if return_all_steps else None)

        # 2. Prepare KV Cache
        if kv_cache is None:
            if M is None:
                raise ValueError("Either kv_cache or evidence memory M must be supplied to LatentPonderBlock.")
            kv_cache = self.build_kv_cache(M)

        K, V = kv_cache

        # 3. Attention mask preparation: [B, 1, 1, L_mem] bool
        attn_mask = None
        if memory_mask is not None and K.size(2) > 0:
            attn_mask = memory_mask.unsqueeze(1).unsqueeze(2).bool()

        # 4. Execute pondering loop
        h_current = h_0
        all_steps = [h_0] if return_all_steps else None

        for _ in range(T):
            h_current = self.ponder_step(h_current, K, V, attn_mask=attn_mask)
            if return_all_steps:
                all_steps.append(h_current)

        return h_current, kv_cache, all_steps

    def ponder(
        self,
        h_0: torch.Tensor,
        M: Optional[torch.Tensor] = None,
        memory_mask: Optional[torch.Tensor] = None,
        T: Optional[int] = None,
        kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        return_all_steps: bool = False,
    ) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor], Optional[List[torch.Tensor]]]:
        """Convenience alias for forward() with parameter T."""
        return self.forward(
            h_0,
            M=M,
            memory_mask=memory_mask,
            kv_cache=kv_cache,
            t_steps=T,
            return_all_steps=return_all_steps,
        )


In [ ]:
%%writefile {REXLIB}/rex/rex_memory.py
"""Incremental evidence memory manager and KV-cache extension for Rex (Step A4)."""
from __future__ import annotations

from typing import Optional

import torch

from .rex_ponder import LatentPonderBlock
from .rex_types import RexState


class EvidenceMemoryManager:
    """Manages multi-round evidence expansion and incremental cross-attention KV caching."""

    @staticmethod
    def append_evidence(
        state: RexState,
        M_new: torch.Tensor,
        memory_mask_new: torch.Tensor,
        is_truncated: bool = False,
        ponder_block: Optional[LatentPonderBlock] = None,
    ) -> RexState:
        """Append newly contextualized evidence piece M_new to running memory M.

        Properties:
            - h is strictly preserved across tool turns (never reset to h_0).
            - M is extended along sequence dimension: M_{k+1} = [M_k ; M_new].
            - memory_mask is concatenated accordingly.
            - If ponder_block and kv_cache are available, cache is updated incrementally in O(L_new).

        Args:
            state: Current RexState.
            M_new: Contextualized token representations of new observation [B, L_new, d].
            memory_mask_new: Validity mask [B, L_new] bool.
            is_truncated: Whether observation was capped at 64 tokens.
            ponder_block: Optional LatentPonderBlock instance to extend KV cache.

        Returns:
            Updated RexState instance with expanded memory and updated cache.
        """
        B, L_new, d = M_new.shape

        if state.M.size(1) == 0:
            # First assignment (or transition from empty state)
            new_M = M_new
            new_mask = memory_mask_new
            new_kv = ponder_block.build_kv_cache(M_new) if ponder_block is not None else None
        else:
            new_M = torch.cat([state.M, M_new], dim=1)
            new_mask = torch.cat([state.memory_mask, memory_mask_new], dim=1)

            # Extend KV-cache incrementally if cache already exists
            if state.kv_cache is not None and ponder_block is not None:
                new_kv = ponder_block.extend_kv_cache(state.kv_cache, M_new)
            elif ponder_block is not None:
                new_kv = ponder_block.build_kv_cache(new_M)
            else:
                new_kv = state.kv_cache

        state.M = new_M
        state.memory_mask = new_mask
        state.kv_cache = new_kv
        state.is_truncated = state.is_truncated or is_truncated
        return state


In [ ]:
%%writefile {REXLIB}/rex/rex_tools.py
"""Tool Matcher, Epistemic Information Gain Head, and Three-Branch Dispatcher for Rex (Step A5)."""
from __future__ import annotations

from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

DEFAULT_PROJ_DIM: int = 256
DEFAULT_MATCHER_TEMPERATURE: float = 0.07
DEFAULT_BUDGET_CAPS: Tuple[int, ...] = (3, 2, 1)
DEFAULT_S_THRESHOLD: float = 0.85
DEFAULT_R_THRESHOLD: float = 0.30
DEFAULT_GAIN_FLOOR: float = 0.05
DEFAULT_MAX_COST_THRESHOLD: float = 0.5


def gain_threshold(level: str, cost: float, floor: float = DEFAULT_GAIN_FLOOR) -> float:
    """Minimum predicted gain tau_k a tool must reach to be called.

    Gain labels are binary (docs/rex-label-schema.md), so the gain head predicts the probability
    that the tool uncovers a key fact. That probability is about values the model has not seen
    yet, so it stays near the base rate and mainly ranks tools. Cheap READ tools therefore only
    need the small `floor` (skipped only when the model is confident they are useless); costlier
    tools need min(0.5, cost), never less than the floor.
    """
    if level == "READ" and cost <= 0.0:
        return floor
    return max(floor, min(DEFAULT_MAX_COST_THRESHOLD, cost))


class ToolMatcherHead(nn.Module):
    """Zero-shot Tool Matcher mapping contextual thought state h_T to tool relevance logits.

    Architecture:
        u = normalize(W_q * h_T) in R^256
        V = normalize(W_t * e_tool) in R^{N_tools x 256}
        tool_logits = (u @ V.T) / tau
    """

    def __init__(
        self,
        d: int = 1024,
        proj_dim: int = DEFAULT_PROJ_DIM,
        temperature: float = DEFAULT_MATCHER_TEMPERATURE,
    ):
        super().__init__()
        self.d = d
        self.proj_dim = proj_dim
        self.temperature = temperature

        self.proj_query = nn.Linear(d, proj_dim, bias=False)
        self.proj_tool = nn.Linear(d, proj_dim, bias=False)

        nn.init.xavier_uniform_(self.proj_query.weight)
        nn.init.xavier_uniform_(self.proj_tool.weight)

        self.register_buffer("tool_desc_embeddings", None)

    def set_tool_embeddings(self, embeddings: torch.Tensor) -> None:
        """Cache normalized tool description embeddings of shape [N_tools, d]."""
        self.tool_desc_embeddings = F.normalize(embeddings, p=2, dim=-1)

    @property
    def tool_projection_weight(self) -> torch.Tensor:
        """Projection matrix [N_tools, proj_dim] for ToolOrthogonalLoss compatibility."""
        if self.tool_desc_embeddings is None:
            raise RuntimeError("Tool embeddings must be initialized via set_tool_embeddings() before accessing weights.")
        v = self.proj_tool(self.tool_desc_embeddings)
        return F.normalize(v, p=2, dim=-1)

    @property
    def tool_embeddings(self) -> torch.Tensor:
        """Alias for tool_projection_weight for backward compatibility with loss inputs."""
        return self.tool_projection_weight

    @property
    def proj_out(self) -> nn.Linear:
        """Alias for proj_tool for backward compatibility with optimizer gradient checks."""
        return self.proj_tool

    def forward(self, h_T: torch.Tensor) -> torch.Tensor:
        """Compute tool relevance logits [B, N_tools]."""
        if self.tool_desc_embeddings is None:
            raise RuntimeError("set_tool_embeddings() must be called prior to ToolMatcherHead forward pass.")

        u = F.normalize(self.proj_query(h_T), p=2, dim=-1)  # [B, proj_dim]
        V = self.tool_projection_weight                      # [N_tools, proj_dim]
        logits = torch.matmul(u, V.t()) / self.temperature   # [B, N_tools]
        return logits.float()


class ToolGainHead(nn.Module):
    """Head estimating expected epistemic information gain in [0, 1] per tool.

    Architecture:
        gain_k = sigmoid(MLP([h_T ; e_tool_k ; h_T * e_tool_k]))
    """

    def __init__(self, d: int = 1024, hidden_dim: int = 256):
        super().__init__()
        self.d = d
        self.mlp = nn.Sequential(
            nn.Linear(3 * d, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )

        nn.init.xavier_uniform_(self.mlp[0].weight)
        nn.init.zeros_(self.mlp[0].bias)
        nn.init.xavier_uniform_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(
        self,
        h_T: torch.Tensor,
        tool_embeddings: torch.Tensor,
        return_logits: bool = False,
    ) -> torch.Tensor:
        """Compute information gain estimates in [0, 1] or raw logits.

        Args:
            h_T: Thought state tensor [B, d].
            tool_embeddings: Precomputed tool representations [N_tools, d].
            return_logits: Whether to return unnormalized logits before sigmoid.

        Returns:
            Expected gain tensor of shape [B, N_tools] in range [0, 1] (or logits if return_logits=True).
        """
        B, d = h_T.shape
        N = tool_embeddings.size(0)

        h_exp = h_T.unsqueeze(1).expand(-1, N, -1)
        t_exp = tool_embeddings.unsqueeze(0).expand(B, -1, -1)
        feat = torch.cat([h_exp, t_exp, h_exp * t_exp], dim=-1)  # [B, N, 3*d]
        logits = self.mlp(feat).squeeze(-1)                      # [B, N]
        if return_logits:
            return logits
        return torch.sigmoid(logits)                             # [B, N]


def dispatch_three_branch(
    tool_names: List[str],
    tool_logits: torch.Tensor,
    tool_gains: torch.Tensor,
    s_prob: torch.Tensor,
    r_prob: torch.Tensor,
    round_idx: int,
    called_tools: Optional[List[str]] = None,
    tool_levels: Optional[Dict[str, str]] = None,
    tool_costs: Optional[Dict[str, float]] = None,
    tool_mask: Optional[torch.Tensor] = None,
    budget_caps: Tuple[int, ...] = DEFAULT_BUDGET_CAPS,
    s_threshold: float = DEFAULT_S_THRESHOLD,
    r_threshold: float = DEFAULT_R_THRESHOLD,
    gain_head_trained: bool = True,
) -> Tuple[str, List[str], Dict[str, float]]:
    """Execute 3-branch epistemic tool selection and stopping logic.

    Sufficiency first: S >= 0.85 and R <= tau_R -> VERDICT; S >= 0.85 and R > tau_R -> STALL
    (sufficient but risky: more tools cannot lower business risk, so it goes to a human).
    Branch 1: If any qualifying tool has gain >= tau_k (see gain_threshold) -> CALL_TOOL (Top-K_t).
    Branch 2: If no tool qualifies, and S >= 0.85 and R <= tau_R -> VERDICT.
    Branch 3: If no tool qualifies, and S < 0.85 -> STALL.

    Returns:
        Tuple of (proposed_action, selected_tools, effective_gains).
    """
    called_set = set(called_tools or [])
    levels = tool_levels or {}
    costs = tool_costs or {}
    n_tools = len(tool_names)

    cap_idx = max(0, min(round_idx - 1, len(budget_caps) - 1))
    budget_k = budget_caps[cap_idx]

    # Convert tensors to 1D flat arrays
    gains_flat = tool_gains.squeeze(0).tolist() if tool_gains.dim() > 1 else tool_gains.tolist()
    logits_flat = tool_logits.squeeze(0).tolist() if tool_logits.dim() > 1 else tool_logits.tolist()
    s_val = float(s_prob.item())
    r_val = float(r_prob.item())
    sufficient = s_val >= s_threshold

    # Build effective gains and qualifying candidate list
    effective_gains: Dict[str, float] = {}
    qualifying_tools: List[Tuple[str, float, float]] = []  # (name, gain, score)

    for i, name in enumerate(tool_names):
        # Environmental masking: previously called tools get gain = 0.0
        if name in called_set:
            effective_gains[name] = 0.0
            continue

        # Hard PoE masking: if tool_mask is False, disable
        if tool_mask is not None:
            mask_val = tool_mask[i].item() if tool_mask.dim() == 1 else tool_mask[0, i].item()
            if not mask_val:
                effective_gains[name] = 0.0
                continue

        g_i = gains_flat[i]
        effective_gains[name] = g_i

        tau_k = gain_threshold(levels.get(name, "READ"), costs.get(name, 0.0))

        if not gain_head_trained or g_i >= tau_k:
            qualifying_tools.append((name, g_i, logits_flat[i]))

    # Sufficient evidence: stop investigating (VERDICT), or hand a risky case to a human (STALL)
    if sufficient:
        return ("VERDICT" if r_val <= r_threshold else "STALL"), [], effective_gains

    # If gain head not trained, bypass gain filtering and select top-K by logits
    if not gain_head_trained:
        if qualifying_tools:
            qualifying_tools.sort(key=lambda x: -x[2])  # sort by logits
            selected = [t[0] for t in qualifying_tools[:budget_k]]
            return "CALL_TOOL", selected, effective_gains
        if s_val >= s_threshold and r_val <= r_threshold:
            return "VERDICT", [], effective_gains
        return "STALL", [], effective_gains

    # Branch 1: At least one tool has gain >= tau_k
    if qualifying_tools:
        # Sort descending by gain, tie-break by relevance logit
        qualifying_tools.sort(key=lambda x: (-x[1], -x[2]))
        selected = [t[0] for t in qualifying_tools[:budget_k]]
        return "CALL_TOOL", selected, effective_gains

    # Round 1 Exploratory Guardrail: Round 1 must attempt at least 1 uncalled cheap READ tool
    if round_idx == 1:
        mask_allowed = (
            {tool_names[i]: bool(tool_mask[i].item() if isinstance(tool_mask, torch.Tensor) else tool_mask[i])
             for i in range(len(tool_names))}
            if tool_mask is not None else {}
        )
        uncalled_read = [
            name for name in tool_names
            if name not in called_set and levels.get(name, "READ") == "READ" and mask_allowed.get(name, True)
        ]
        if uncalled_read:
            # Pick highest relevance logit among uncalled read tools
            uncalled_read.sort(key=lambda t: -logits_flat[tool_names.index(t)])
            return "CALL_TOOL", [uncalled_read[0]], effective_gains

    # Branch 2: S >= s_threshold and R <= r_threshold -> VERDICT
    if s_val >= s_threshold and r_val <= r_threshold:
        return "VERDICT", [], effective_gains

    # Branch 3: S < s_threshold -> STALL
    return "STALL", [], effective_gains


def evaluate_stall_and_dispatch(
    tool_names: List[str],
    tool_logits: torch.Tensor,
    tool_gains: torch.Tensor,
    s_prob: torch.Tensor,
    r_prob: torch.Tensor,
    round_idx: int,
    called_tools: Optional[List[str]] = None,
    conformal_set: Optional[List[str]] = None,
    last_read_tried: bool = False,
    tool_levels: Optional[Dict[str, str]] = None,
    tool_costs: Optional[Dict[str, float]] = None,
    tool_mask: Optional[torch.Tensor] = None,
    budget_caps: Tuple[int, ...] = DEFAULT_BUDGET_CAPS,
    s_threshold: float = DEFAULT_S_THRESHOLD,
    r_threshold: float = DEFAULT_R_THRESHOLD,
    gain_head_trained: bool = True,
    s_prev: Optional[float] = None,
    epsilon: float = 0.05,
) -> Tuple[str, List[str], bool, str, bool, bool, Dict[str, float]]:
    """Comprehensive tool selection and STALL evaluation with two safety buffers (Step A7).

    Safety Buffers before emitting final STALL:
        Buffer 1 (Last-resort cheap read): If STALL condition is met, but an uncalled cheap
                 READ tool exists and has not yet been tried as a last resort, dispatch it
                 and mark last_read_tried = True.
        Buffer 2 (Conformal singleton auto-commit): If STALL condition is met, but conformal
                 prediction set is a singleton (|C| == 1) and risk R <= tau_R, auto-commit
                 to VERDICT instead of escalating to human.

    Returns:
        Tuple of:
            - proposed_action: "CALL_TOOL", "VERDICT", or "STALL"
            - selected_tools: List[str]
            - stall: bool
            - stall_mode: "LEARNED" or "NONE"
            - rule_stall: bool
            - last_read_tried_out: bool
            - effective_gains: Dict[str, float]
    """
    s_val = float(s_prob.item())
    r_val = float(r_prob.item())
    levels = tool_levels or {}
    called_set = set(called_tools or [])

    # Evaluate base 3-branch selection
    action, selected, effective_gains = dispatch_three_branch(
        tool_names=tool_names,
        tool_logits=tool_logits,
        tool_gains=tool_gains,
        s_prob=s_prob,
        r_prob=r_prob,
        round_idx=round_idx,
        called_tools=called_tools,
        tool_levels=tool_levels,
        tool_costs=tool_costs,
        tool_mask=tool_mask,
        budget_caps=budget_caps,
        s_threshold=s_threshold,
        r_threshold=r_threshold,
        gain_head_trained=gain_head_trained,
    )

    # Compute rule_stall metric (S delta < epsilon over rounds) for telemetry comparison
    rule_stall = False
    if s_prev is not None and round_idx >= 2:
        rule_stall = (s_val - s_prev) < epsilon

    # If action is not STALL, return directly
    if action != "STALL":
        return action, selected, False, "NONE", rule_stall, last_read_tried, effective_gains

    # Sufficient but risky: neither buffer applies (another read cannot lower risk; auto-commit needs R <= tau_R)
    if s_val >= s_threshold:
        return "STALL", [], True, "LEARNED", rule_stall, last_read_tried, effective_gains

    # Action is candidate for STALL -> Apply 2 Safety Buffers
    # Safety Buffer 1: Last-resort cheap READ attempt
    if not last_read_tried:
        mask_allowed = (
            {tool_names[i]: bool(tool_mask[i].item() if isinstance(tool_mask, torch.Tensor) else tool_mask[i])
             for i in range(len(tool_names))}
            if tool_mask is not None else {}
        )
        uncalled_read = [
            t for t in tool_names
            if t not in called_set and levels.get(t, "READ") == "READ" and mask_allowed.get(t, True)
        ]
        if uncalled_read:
            # Dispatch the uncalled read tool with highest relevance logit
            logits_flat = tool_logits.squeeze(0).tolist() if tool_logits.dim() > 1 else tool_logits.tolist()
            uncalled_read.sort(key=lambda t: -logits_flat[tool_names.index(t)])
            return "CALL_TOOL", [uncalled_read[0]], False, "NONE", rule_stall, True, effective_gains

    # Safety Buffer 2: High-certainty conformal auto-commit (|C| == 1 and R <= tau_R)
    if conformal_set is not None and len(conformal_set) == 1 and r_val <= r_threshold:
        return "VERDICT", [], False, "NONE", rule_stall, last_read_tried, effective_gains

    # If both buffers fail -> emit learned STALL signal
    return "STALL", [], True, "LEARNED", rule_stall, last_read_tried, effective_gains


In [ ]:
%%writefile {REXLIB}/rex/rex_types.py
"""Standardized data containers and state contracts for Rex (Part A).

This module defines:
1. RexState: Multi-round mutable latent memory, token cache, and trajectory history.
2. RexStepOutput: Comprehensive step outputs, logits, probabilities, and telemetry serializer.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple, Union

import torch

MODEL_VERSION: str = "rex2-modernbert-1024"


@dataclass
class RexState:
    """State carrier maintained across investigative pondering and multi-round tool turns.

    Attributes:
        h: Contextualized thought state tensor of shape [B, d].
        M: Contextualized evidence memory tokens of shape [B, L_mem, d].
        memory_mask: Boolean validity mask of shape [B, L_mem] (True = valid, False = pad).
        kv_cache: Optional tuple of cached Key and Value tensors for ponder cross-attention.
        hyp_alive: Boolean survival mask of shape [B, K] across Popperian candidate hypotheses.
        hyp_markers: Marker embeddings [B, K, d] corresponding to [MASK] positions from turn 1.
        round_idx: Current turn index (1-indexed: 1, 2, 3...).
        called_tools: List of tool names called so far per batch sample.
        last_read_tried: Guardrail flag indicating if a last-resort cheap read was attempted.
        is_truncated: Whether evidence observation was truncated by the 64-token cap.
        marker_mask: Boolean validity mask [B, K] of hypothesis slots from turn 1 (False = padding).
        s_prev: Sufficiency probability per sample from the previous round (for rule_stall telemetry).
    """

    h: torch.Tensor
    M: torch.Tensor
    memory_mask: torch.Tensor
    kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None
    hyp_alive: Optional[torch.Tensor] = None
    hyp_markers: Optional[torch.Tensor] = None
    round_idx: int = 1
    called_tools: List[List[str]] = field(default_factory=list)
    last_read_tried: Union[bool, List[bool]] = False
    is_truncated: bool = False
    marker_mask: Optional[torch.Tensor] = None
    s_prev: Optional[List[float]] = None

    @classmethod
    def empty(
        cls,
        batch_size: int = 1,
        d: int = 1024,
        device: Optional[torch.device] = None,
        dtype: torch.dtype = torch.float32,
    ) -> RexState:
        """Create an empty initialization state for turn 1 before initial encoding."""
        dev = device if device is not None else torch.device("cpu")
        return cls(
            h=torch.zeros((batch_size, d), dtype=dtype, device=dev),
            M=torch.zeros((batch_size, 0, d), dtype=dtype, device=dev),
            memory_mask=torch.zeros((batch_size, 0), dtype=torch.bool, device=dev),
            kv_cache=None,
            hyp_alive=None,
            hyp_markers=None,
            round_idx=1,
            called_tools=[[] for _ in range(batch_size)],
            last_read_tried=False,
            is_truncated=False,
        )


@dataclass
class RexStepOutput:
    """Output bundle produced by RexDecisionModel.step().

    Contains both raw unnormalized logits (for numerical loss stability) and normalized
    probabilities, calibration metrics, and selected actions.
    """

    s_logit: torch.Tensor                        # [B, 1]
    s_prob: torch.Tensor                         # [B, 1]
    r_logit: torch.Tensor                        # [B, 1] logit(r_prob)
    r_prob: torch.Tensor                         # [B, 1] sum_j verdict_probs_j * hyp_risk_j
    verdict_logits: torch.Tensor                 # [B, K]
    verdict_probs: torch.Tensor                  # [B, K]
    tool_logits: torch.Tensor                    # [B, N_tools]
    tool_gains: torch.Tensor                     # [B, N_tools]
    stall: torch.Tensor                          # [B] bool
    stall_mode: str                              # "LEARNED" or "NONE"
    rule_stall: torch.Tensor                     # [B] bool
    selected_tools: List[List[str]]              # Approved tool names per batch item
    proposed_action: List[str]                   # "CALL_TOOL", "VERDICT", or "STALL"
    h_T: torch.Tensor                            # [B, d]
    tool_names: List[str] = field(default_factory=list)
    hyp_names: List[str] = field(default_factory=list)
    conformal_set: Optional[List[List[str]]] = None
    attn_weights: Optional[torch.Tensor] = None
    gain_head_trained: bool = False
    is_truncated: bool = False
    tool_gain_logits: Optional[torch.Tensor] = None     # [B, N_tools] raw logits of tool_gains (for BCEWithLogits)
    hyp_logits: Optional[torch.Tensor] = None           # [B, K] HypothesisScorer survival logits over memory M
    hyp_scores: Optional[torch.Tensor] = None           # [B, K] sigmoid(hyp_logits)
    hyp_risk: Optional[torch.Tensor] = None             # [B, K] sigmoid(hyp_risk_logits)
    hyp_risk_logits: Optional[torch.Tensor] = None      # [B, K] logits of hyp_risk
    stall_modes: Optional[List[str]] = None
    stall_reason: List[str] = field(default_factory=list)          # per sample: "NONE", "NO_GAIN" or "HIGH_RISK"
    alive_hypotheses: List[List[str]] = field(default_factory=list)  # surviving hypothesis names per sample

    def to_trace_dict(self, batch_idx: int = 0) -> Dict[str, Any]:
        """Serialize output for trace logging and State Store.

        Strict contract: Returns purely primitive Python types (float, int, str, bool, list, dict)
        with zero torch.Tensor instances to guarantee crash-free JSON/logging serialization.
        """
        # 1. Probabilities
        s_val = float(self.s_prob[batch_idx].item()) if self.s_prob.dim() > 1 else float(self.s_prob.item())
        r_val = float(self.r_prob[batch_idx].item()) if self.r_prob.dim() > 1 else float(self.r_prob.item())

        # 2. Tool gains dictionary
        tool_gains_dict: Dict[str, float] = {}
        top_tools_list: List[Dict[str, Any]] = []
        n_tools = self.tool_logits.size(-1) if self.tool_logits.dim() > 1 else len(self.tool_names)

        if self.tool_names and len(self.tool_names) == n_tools:
            b_logits = self.tool_logits[batch_idx] if self.tool_logits.dim() > 1 else self.tool_logits
            b_gains = self.tool_gains[batch_idx] if self.tool_gains.dim() > 1 else self.tool_gains
            for idx, name in enumerate(self.tool_names):
                tool_gains_dict[name] = round(float(b_gains[idx].item()), 4)

            # Top 3 tools by relevance score
            sorted_indices = torch.argsort(b_logits, descending=True).tolist()
            for idx in sorted_indices[:3]:
                top_tools_list.append({
                    "name": self.tool_names[idx],
                    "score": round(float(torch.sigmoid(b_logits[idx]).item()), 4),
                    "gain": tool_gains_dict[self.tool_names[idx]],
                })

        # 3. Verdict probabilities dictionary
        verdict_probs_dict: Dict[str, float] = {}
        b_vprobs = self.verdict_probs[batch_idx] if self.verdict_probs.dim() > 1 else self.verdict_probs
        if self.hyp_names and len(self.hyp_names) == b_vprobs.size(-1):
            for idx, name in enumerate(self.hyp_names):
                verdict_probs_dict[name] = round(float(b_vprobs[idx].item()), 4)
        else:
            for idx in range(b_vprobs.size(-1)):
                verdict_probs_dict[f"H_{idx}"] = round(float(b_vprobs[idx].item()), 4)

        # 4. Status flags
        stall_val = bool(self.stall[batch_idx].item()) if self.stall.dim() > 0 else bool(self.stall.item())
        rule_stall_val = (
            bool(self.rule_stall[batch_idx].item()) if self.rule_stall.dim() > 0 else bool(self.rule_stall.item())
        )
        conf_set = self.conformal_set[batch_idx] if self.conformal_set else []

        return {
            "s_prob": round(s_val, 4),
            "r_prob": round(r_val, 4),
            "action": self.proposed_action[batch_idx] if self.proposed_action else "CALL_TOOL",
            "selected_tools": self.selected_tools[batch_idx] if self.selected_tools else [],
            "top_tools": top_tools_list,
            "tool_gains": tool_gains_dict,
            "verdict_probs": verdict_probs_dict,
            "conformal_set": conf_set,
            "stall": stall_val,
            "stall_mode": self.stall_modes[batch_idx] if (self.stall_modes and batch_idx < len(self.stall_modes)) else self.stall_mode,
            "stall_reason": self.stall_reason[batch_idx] if self.stall_reason else "NONE",
            "rule_stall": rule_stall_val,
            "alive_hypotheses": list(self.alive_hypotheses[batch_idx]) if self.alive_hypotheses else [],
            "gain_head_trained": self.gain_head_trained,
            "is_truncated": self.is_truncated,
            "model_version": MODEL_VERSION,
        }


In [ ]:
%%writefile {REXLIB}/rex/rex_registry.py
"""Popperian hypothesis registry, metric definitions, and falsification engine for Rex."""
from dataclasses import dataclass, field
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple, Union

try:
    import yaml
except ImportError:
    yaml = None  # Optional dependency, JSON is primary


ALLOWED_OPERATORS: Set[str] = {"GT", "LT", "GTE", "LTE", "EQ", "NEQ", "IN", "NOT_IN"}


@dataclass(frozen=True)
class FalsificationRule:
    """Popperian falsification rule defining an elimination condition for a hypothesis."""

    metric: str
    operator: str
    threshold: Any

    def __post_init__(self):
        if self.operator not in ALLOWED_OPERATORS:
            raise ValueError(
                f"Unknown falsification operator '{self.operator}'. "
                f"Allowed operators: {sorted(ALLOWED_OPERATORS)}"
            )

    def eval(self, value: Any) -> bool:
        """Evaluate observation value against rule condition.

        Returns True if the condition is MET, meaning the hypothesis is FALSIFIED / ELIMINATED.
        """
        if value is None:
            return False

        op = self.operator
        thresh = self.threshold

        if op == "GT":
            return float(value) > float(thresh)
        elif op == "LT":
            return float(value) < float(thresh)
        elif op == "GTE":
            return float(value) >= float(thresh)
        elif op == "LTE":
            return float(value) <= float(thresh)
        elif op == "EQ":
            if isinstance(thresh, bool) or isinstance(value, bool):
                return bool(value) is bool(thresh)
            return value == thresh
        elif op == "NEQ":
            if isinstance(thresh, bool) or isinstance(value, bool):
                return bool(value) is not bool(thresh)
            return value != thresh
        elif op == "IN":
            return value in thresh
        elif op == "NOT_IN":
            return value not in thresh
        return False


@dataclass(frozen=True)
class MetricSpec:
    """Specification of a measurable metric in the environment."""

    name: str
    kind: str  # "float", "int", "bool"
    min_val: Optional[float] = None
    max_val: Optional[float] = None


@dataclass(frozen=True)
class ToolSpec:
    """Specification of an investigative tool's returned metrics and operational scope."""

    name: str
    returns: List[str]
    scope: str = "macro"  # "macro" or "micro"
    general: bool = False
    description: str = ""
    level: str = "READ"   # "READ", "LIGHT", or "HEAVY"
    cost: float = 0.0


@dataclass(frozen=True)
class HypothesisSpec:
    """Hypothesis definition including associated tools and falsification criteria."""

    id: str
    text: str
    associated_tools: List[str] = field(default_factory=list)
    falsification_rules: List[FalsificationRule] = field(default_factory=list)
    is_residual: bool = False
    risk: Optional[float] = None  # business risk R* when this is the true cause; None -> domain risk


@dataclass(frozen=True)
class DomainSpec:
    """Investigation domain schema corresponding to a type token (e.g. <TYPE_PROMO_ABUSE>)."""

    type_token: str
    instruction: str
    alert_metrics: List[str]
    hypotheses: List[HypothesisSpec]
    risk: Optional[float] = None  # business risk label R* in [0, 1]; None falls back to the default mapping

    @property
    def hypotheses_dict(self) -> Dict[str, str]:
        """Mapping from hypothesis ID to descriptive text."""
        return {h.id: h.text for h in self.hypotheses}

    @property
    def hypothesis_map(self) -> Dict[str, HypothesisSpec]:
        """Mapping from hypothesis ID to full HypothesisSpec."""
        return {h.id: h for h in self.hypotheses}


def _flatten_observations(observations: Any) -> Dict[str, Any]:
    """Recursively flatten nested observation dictionaries into a single metric->value mapping."""
    flat: Dict[str, Any] = {}
    if not isinstance(observations, dict):
        return flat

    for k, v in observations.items():
        if isinstance(v, dict):
            flat.update(_flatten_observations(v))
        else:
            flat[k] = v
    return flat


class Registry:
    """Central registry maintaining tools, metrics, domains, and Popperian elimination logic."""

    def __init__(
        self,
        tools: List[str],
        type_tokens: Dict[str, str],
        metrics: Dict[str, MetricSpec],
        tool_specs: Dict[str, ToolSpec],
        scope_schedule: List[str],
        domains: Dict[str, DomainSpec],
        tool_hidden: Optional[int] = None,
        tool_dropout: float = 0.1,
        tool_prior: Optional[float] = None,
    ):
        self.tools = list(tools)
        self.tool_to_idx = {name: i for i, name in enumerate(self.tools)}
        self.type_tokens = dict(type_tokens)
        self.metrics = dict(metrics)
        self.tool_specs = dict(tool_specs)
        self.scope_schedule = list(scope_schedule)
        self.domains = dict(domains)
        self.tool_hidden = tool_hidden
        self.tool_dropout = tool_dropout
        self.tool_prior = tool_prior

    def check_model_tools(self, model_tools: Sequence[str]) -> None:
        """Validate that the registry tools match the model tool head output order and vocabulary."""
        reg_tools = list(self.tools)
        mod_tools = list(model_tools)
        if reg_tools != mod_tools:
            if set(reg_tools) != set(mod_tools):
                diff_reg = set(reg_tools) - set(mod_tools)
                diff_mod = set(mod_tools) - set(reg_tools)
                raise ValueError(
                    f"Registry tools and model tools do not match! "
                    f"In registry only: {sorted(diff_reg)}, in model only: {sorted(diff_mod)}"
                )
            # Permutation mismatch warning
            raise ValueError(
                f"Registry tools and model tools have differing order.\n"
                f"Registry: {reg_tools}\nModel: {mod_tools}"
            )

    def get_domain(self, type_token: str) -> DomainSpec:
        """Retrieve domain specification for a given domain event type token."""
        if type_token not in self.domains:
            raise KeyError(
                f"Domain '{type_token}' not configured in registry. "
                f"Available domains: {list(self.domains.keys())}"
            )
        return self.domains[type_token]

    def falsify(
        self,
        domain_key: str,
        observations: Dict[str, Any],
        current_alive: Optional[Dict[str, bool]] = None,
    ) -> Tuple[Dict[str, bool], Dict[str, List[str]]]:
        """Evaluate observations against falsification rules to eliminate hypotheses.

        Parameters
        ----------
        domain_key : str
            Domain type token (e.g. "<TYPE_PROMO_ABUSE>").
        observations : Dict[str, Any]
            Accumulated evidence observations (flat or nested tool dictionaries).
        current_alive : Optional[Dict[str, bool]]
            Current hypothesis survival mask. If None, all hypotheses start as alive.

        Returns
        -------
        Tuple[Dict[str, bool], Dict[str, List[str]]]
            Updated alive dictionary and eliminated_by proof trace dictionary.
        """
        domain = self.get_domain(domain_key)
        flat_obs = _flatten_observations(observations)

        alive: Dict[str, bool] = {}
        eliminated_by: Dict[str, List[str]] = {}

        # Initialize survival status
        for h in domain.hypotheses:
            if current_alive is not None and h.id in current_alive:
                alive[h.id] = bool(current_alive[h.id])
            else:
                alive[h.id] = True

        # Process each hypothesis
        for h in domain.hypotheses:
            # H_novel (residual) cannot ever be eliminated
            if h.is_residual or h.id == "H_novel":
                alive[h.id] = True
                continue

            # If already eliminated in a prior turn, keep eliminated
            if not alive[h.id]:
                continue

            # Check falsification conditions
            for rule in h.falsification_rules:
                if rule.metric in flat_obs:
                    val = flat_obs[rule.metric]
                    if rule.eval(val):
                        alive[h.id] = False
                        trace = f"{rule.metric} {rule.operator} {rule.threshold} (observed {val})"
                        eliminated_by.setdefault(h.id, []).append(trace)
                        break

        # Invariant: H_novel must always remain alive
        alive["H_novel"] = True

        return alive, eliminated_by

    def tool_mask(
        self,
        domain_key: str,
        alive: Dict[str, bool],
        called_tools: Optional[Sequence[str]] = None,
    ) -> List[bool]:
        """Compute boolean availability mask for tools given the surviving hypotheses.

        A tool is available (True) if:
        1. It has NOT been called already.
        2. It is designated as a general tool OR is associated with at least one currently alive hypothesis.

        Returns
        -------
        List[bool]
            Boolean mask of length len(self.tools).
        """
        domain = self.get_domain(domain_key)
        called_set = set(called_tools or [])

        # Active tools needed by surviving non-residual hypotheses
        active_tools: Set[str] = set()
        for h in domain.hypotheses:
            if not h.is_residual and alive.get(h.id, True):
                active_tools.update(h.associated_tools)

        mask: List[bool] = []
        for t in self.tools:
            if t in called_set:
                mask.append(False)
            elif self.tool_specs.get(t) and self.tool_specs[t].general:
                mask.append(True)
            elif t in active_tools:
                mask.append(True)
            else:
                mask.append(False)

        return mask


def load_registry(path_or_dict: Union[str, Path, dict]) -> Registry:
    """Load and strictly validate a Rex registry from file path or dictionary.

    Performs comprehensive schema validation:
    - Unique non-empty tools list.
    - All tool specs reference valid tools and return registered metrics.
    - Domains end with residual 'H_novel'.
    - Known hypotheses possess at least one falsification rule.
    - Falsification metrics are observable (either in alert_metrics or tool returns).
    """
    if isinstance(path_or_dict, (str, Path)):
        p = Path(path_or_dict)
        if not p.is_file():
            raise FileNotFoundError(f"Registry file not found: {p}")
        text = p.read_text(encoding="utf-8")
        if p.suffix in (".yaml", ".yml"):
            if yaml is None:
                raise ImportError("PyYAML is required to parse YAML registry files.")
            raw = yaml.safe_load(text)
        else:
            raw = json.loads(text)
    elif isinstance(path_or_dict, dict):
        raw = dict(path_or_dict)
    else:
        raise TypeError(f"Expected path or dict, got {type(path_or_dict)}")

    # 1. Validate tools
    raw_tools = raw.get("tools")
    if not isinstance(raw_tools, list) or not raw_tools:
        raise ValueError("Registry must contain a non-empty 'tools' list.")
    cleaned_tools: List[str] = []
    for t in raw_tools:
        if not isinstance(t, str) or not t.strip():
            raise ValueError(f"Tool names must be non-empty strings, got {t!r}")
        cleaned_tools.append(t.strip())
    if len(cleaned_tools) != len(set(cleaned_tools)):
        raise ValueError("Duplicate tool names found in 'tools' list.")

    tools_set = set(cleaned_tools)

    # 2. Validate type tokens
    raw_type_tokens = raw.get("type_tokens")
    if not isinstance(raw_type_tokens, dict) or not raw_type_tokens:
        raise ValueError("Registry must contain a non-empty 'type_tokens' dictionary.")

    # 3. Validate metrics
    raw_metrics = raw.get("metrics", {})
    metrics: Dict[str, MetricSpec] = {}
    for m_name, m_cfg in raw_metrics.items():
        kind = m_cfg.get("kind", "float")
        metrics[m_name] = MetricSpec(
            name=m_name,
            kind=kind,
            min_val=m_cfg.get("min"),
            max_val=m_cfg.get("max"),
        )

    # 4. Validate tool specs
    raw_tool_specs = raw.get("tool_specs", {})
    tool_specs: Dict[str, ToolSpec] = {}
    tool_return_map: Dict[str, List[str]] = {}

    for t_name, t_cfg in raw_tool_specs.items():
        if t_name not in tools_set:
            raise ValueError(f"Tool '{t_name}' in 'tool_specs' is not listed in 'tools'.")
        returns = t_cfg.get("returns", [])
        for ret in returns:
            if ret not in metrics:
                raise ValueError(
                    f"Tool '{t_name}' returns unregistered metric '{ret}'. "
                    f"Add '{ret}' to 'metrics' first."
                )
        scope = t_cfg.get("scope", "macro")
        if scope not in ("macro", "micro"):
            raise ValueError(f"Tool '{t_name}' has invalid scope '{scope}'. Must be 'macro' or 'micro'.")
        tool_specs[t_name] = ToolSpec(
            name=t_name,
            returns=list(returns),
            scope=scope,
            general=bool(t_cfg.get("general", False)),
        )
        tool_return_map[t_name] = list(returns)

    # Ensure all tools have at least a default ToolSpec
    for t in cleaned_tools:
        if t not in tool_specs:
            tool_specs[t] = ToolSpec(name=t, returns=[], scope="macro")
            tool_return_map[t] = []

    scope_schedule = list(raw.get("scope_schedule", ["macro", "micro"]))

    # 5. Validate domains
    raw_domains = raw.get("domains")
    if not isinstance(raw_domains, dict) or not raw_domains:
        raise ValueError("Registry must contain a non-empty 'domains' dictionary.")

    domains: Dict[str, DomainSpec] = {}
    for d_token, d_cfg in raw_domains.items():
        if d_token not in raw_type_tokens:
            raise ValueError(f"Domain key '{d_token}' is not present in 'type_tokens'.")

        instruction = d_cfg.get("instruction", "").strip()
        if not instruction:
            raise ValueError(f"Domain '{d_token}' has an empty instruction.")

        alert_metrics = list(d_cfg.get("alert_metrics", []))
        for am in alert_metrics:
            if am not in metrics:
                raise ValueError(
                    f"Domain '{d_token}' alert metric '{am}' is not defined in 'metrics'."
                )

        raw_hyps = d_cfg.get("hypotheses", [])
        if not isinstance(raw_hyps, list) or len(raw_hyps) < 2:
            raise ValueError(f"Domain '{d_token}' must have at least 2 hypotheses (including H_novel).")

        hypotheses: List[HypothesisSpec] = []
        for i, h_cfg in enumerate(raw_hyps):
            h_id = h_cfg.get("id", "").strip()
            if not h_id:
                raise ValueError(f"Domain '{d_token}' hypothesis at index {i} has empty 'id'.")
            h_text = h_cfg.get("text", "").strip()
            is_residual = bool(h_cfg.get("is_residual", False))
            h_risk = h_cfg.get("risk")
            if h_risk is not None:
                h_risk = float(h_risk)
                if not 0.0 <= h_risk <= 1.0:
                    raise ValueError(f"Hypothesis '{h_id}' in domain '{d_token}' risk must be in [0, 1], got {h_risk}.")

            # Associated tools
            assoc_tools = list(h_cfg.get("associated_tools", []))
            for at in assoc_tools:
                if at not in tools_set:
                    raise ValueError(
                        f"Hypothesis '{h_id}' in domain '{d_token}' references unknown tool '{at}'."
                    )

            # Rules
            raw_rules = h_cfg.get("falsification_rules", [])
            rules: List[FalsificationRule] = []

            if is_residual:
                if i != len(raw_hyps) - 1 or h_id != "H_novel":
                    raise ValueError(
                        f"Residual hypothesis must be 'H_novel' and located at the final index. "
                        f"Found id='{h_id}' at index {i} of {len(raw_hyps)}."
                    )
            else:
                if not raw_rules:
                    raise ValueError(
                        f"Known hypothesis '{h_id}' in domain '{d_token}' has no falsification rules "
                        f"and can never be eliminated."
                    )

            # Metrics observable by this hypothesis
            observable_metrics: Set[str] = set(alert_metrics)
            for at in assoc_tools:
                observable_metrics.update(tool_return_map.get(at, []))

            for r_cfg in raw_rules:
                r_metric = r_cfg.get("metric", "").strip()
                if not r_metric or r_metric not in metrics:
                    raise ValueError(
                        f"Rule in hypothesis '{h_id}' references unknown metric '{r_metric}'."
                    )
                if r_metric not in observable_metrics:
                    raise ValueError(
                        f"Rule metric '{r_metric}' in hypothesis '{h_id}' is neither in domain "
                        f"alert_metrics nor returned by associated tools {assoc_tools}."
                    )
                r_op = r_cfg.get("operator", "").strip()
                r_thresh = r_cfg.get("threshold")
                rules.append(FalsificationRule(metric=r_metric, operator=r_op, threshold=r_thresh))

            hypotheses.append(
                HypothesisSpec(
                    id=h_id,
                    text=h_text,
                    associated_tools=assoc_tools,
                    falsification_rules=rules,
                    is_residual=is_residual,
                    risk=h_risk,
                )
            )

        # Enforce last hypothesis is H_novel
        if hypotheses[-1].id != "H_novel" or not hypotheses[-1].is_residual:
            raise ValueError(
                f"Domain '{d_token}' must end with 'H_novel' as residual hypothesis."
            )

        d_risk = d_cfg.get("risk")
        if d_risk is not None:
            d_risk = float(d_risk)
            if not 0.0 <= d_risk <= 1.0:
                raise ValueError(f"Domain '{d_token}' risk must be in [0, 1], got {d_risk}.")

        domains[d_token] = DomainSpec(
            type_token=d_token,
            instruction=instruction,
            alert_metrics=alert_metrics,
            hypotheses=hypotheses,
            risk=d_risk,
        )

    return Registry(
        tools=cleaned_tools,
        type_tokens=raw_type_tokens,
        metrics=metrics,
        tool_specs=tool_specs,
        scope_schedule=scope_schedule,
        domains=domains,
        tool_hidden=raw.get("tool_hidden"),
        tool_dropout=float(raw.get("tool_dropout", 0.1)),
        tool_prior=float(raw.get("tool_prior")) if raw.get("tool_prior") is not None else None,
    )


In [ ]:
%%writefile {REXLIB}/rex/rex_poe.py
"""Karl Popper Process-of-Elimination (PoE) engine and data contracts for Rex.

This module implements:
1. ComparisonOperator and Strategy Pattern for deterministic falsification rules.
2. Data contracts: FalsificationRule, HypothesisDefinition, RoundTrace, InvestigationResult.
3. PoEFalsificationEngine: Evaluates telemetry against hypothesis falsification criteria.
4. HardMaskingManager: Maps hypothesis survival to tool availability masks.
"""

from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable, Dict, List, Optional

import torch

from .rex_registry import _flatten_observations

# Comparison operators for deterministic falsification criteria (P2)
class ComparisonOperator(str, Enum):
    """Supported relational operators for hypothesis falsification rules."""

    LT = "LT"        # actual < threshold -> violation (eliminate)
    GT = "GT"        # actual > threshold -> violation (eliminate)
    GTE = "GTE"      # actual >= threshold -> violation (eliminate)
    LTE = "LTE"      # actual <= threshold -> violation (eliminate)
    EQ = "EQ"        # actual == threshold -> violation (eliminate)
    NEQ = "NEQ"      # actual != threshold -> violation (eliminate)
    IN = "IN"        # actual in threshold -> violation (eliminate)
    NOT_IN = "NOT_IN"# actual not in threshold -> violation (eliminate)


def _evaluate_operator(actual: Any, threshold: Any, op: str) -> bool:
    """Evaluate observation against threshold with support for numeric, boolean, and collection types."""
    if actual is None:
        return False
    if op == "GT":
        return float(actual) > float(threshold)
    elif op == "LT":
        return float(actual) < float(threshold)
    elif op == "GTE":
        return float(actual) >= float(threshold)
    elif op == "LTE":
        return float(actual) <= float(threshold)
    elif op == "EQ":
        if isinstance(threshold, bool) or isinstance(actual, bool):
            return bool(actual) is bool(threshold)
        return actual == threshold
    elif op == "NEQ":
        if isinstance(threshold, bool) or isinstance(actual, bool):
            return bool(actual) is not bool(threshold)
        return actual != threshold
    elif op == "IN":
        return actual in threshold
    elif op == "NOT_IN":
        return actual not in threshold
    return False


# Strategy registry mapping operators to evaluation callables (G23)
OPERATOR_STRATEGIES: Dict[ComparisonOperator, Callable[[Any, Any], bool]] = {
    ComparisonOperator.LT: lambda actual, threshold: _evaluate_operator(actual, threshold, "LT"),
    ComparisonOperator.GT: lambda actual, threshold: _evaluate_operator(actual, threshold, "GT"),
    ComparisonOperator.GTE: lambda actual, threshold: _evaluate_operator(actual, threshold, "GTE"),
    ComparisonOperator.LTE: lambda actual, threshold: _evaluate_operator(actual, threshold, "LTE"),
    ComparisonOperator.EQ: lambda actual, threshold: _evaluate_operator(actual, threshold, "EQ"),
    ComparisonOperator.NEQ: lambda actual, threshold: _evaluate_operator(actual, threshold, "NEQ"),
    ComparisonOperator.IN: lambda actual, threshold: _evaluate_operator(actual, threshold, "IN"),
    ComparisonOperator.NOT_IN: lambda actual, threshold: _evaluate_operator(actual, threshold, "NOT_IN"),
}


class TerminationReason(str, Enum):
    """Reasons for terminating an iterative investigation loop (P2)."""

    SINGLE_SURVIVOR = "single_survivor"
    MAX_ROUNDS_REACHED = "max_rounds_reached"
    EMPTY_TOOL_SELECTION = "empty_tool_selection"
    ALL_CATALOGED_ELIMINATED = "all_cataloged_eliminated"
    STALL = "stall"


@dataclass(frozen=True)
class FalsificationRule:
    """Individual Popperian falsification criterion (F1).

    If an observation metric violates this rule, the associated hypothesis
    is eliminated (falsified).
    """

    metric: str
    operator: ComparisonOperator
    threshold: Any


@dataclass(frozen=True)
class HypothesisDefinition:
    """Domain hypothesis specification for MECE ontology registry (F1)."""

    id: str
    text: str
    associated_tools: List[str]
    falsification_rules: List[FalsificationRule] = field(default_factory=list)
    is_residual: bool = False


@dataclass(frozen=True)
class RoundTrace:
    """Telemetry and decision trace recorded during one investigation round (F1)."""

    round_idx: int
    selected_tools: List[str]
    observations: Dict[str, Any]
    active_hypotheses: List[str]


@dataclass(frozen=True)
class InvestigationResult:
    """Final decision output produced by Rex autonomous investigation loop (F1)."""

    predicted_hypothesis: str
    probabilities: Dict[str, float]
    rounds_executed: int
    termination_reason: TerminationReason
    is_escalated: bool
    traces: List[RoundTrace] = field(default_factory=list)


class PoEFalsificationEngine:
    """Karl Popper Falsification Engine.

    Evaluates telemetry observations against candidate hypothesis criteria.
    Hypotheses are never proven true; they can only survive falsification.
    H_novel is the open-world residual slot and is immune to static elimination.
    """

    def evaluate(
        self,
        observations: Dict[str, Any],
        current_alive: Dict[str, bool],
        rules: Dict[str, List[FalsificationRule]],
    ) -> Dict[str, bool]:
        """Evaluate observation telemetry and return updated survival flags.

        Args:
            observations: Dict of metric names to measured scalar/categorical values, either flat
                ({metric: value}) or grouped per tool ({tool: {metric: value}}, as packed by Part B).
            current_alive: Current boolean survival state per hypothesis.
            rules: Mapping of hypothesis ID to its list of falsification rules.

        Returns:
            New dictionary with updated survival flags (pure function, no mutation).
        """
        observations = _flatten_observations(observations)
        updated_alive = dict(current_alive)

        for hyp_id, is_alive in current_alive.items():
            # Already eliminated hypotheses remain eliminated (idempotent)
            if not is_alive:
                continue

            # Residual hypothesis (H_novel) cannot be eliminated by static rules
            if hyp_id == "H_novel":
                updated_alive["H_novel"] = True
                continue

            hyp_rules = rules.get(hyp_id, [])
            for rule in hyp_rules:
                if rule.metric in observations:
                    actual_val = observations[rule.metric]
                    # Convert rule operator if it's a string
                    op_key = (
                        rule.operator
                        if isinstance(rule.operator, ComparisonOperator)
                        else ComparisonOperator(str(rule.operator))
                    )
                    evaluator = OPERATOR_STRATEGIES.get(op_key)
                    if evaluator is None:
                        raise ValueError(f"Unsupported operator '{rule.operator}' for metric '{rule.metric}'")

                    # If the measured telemetry satisfies the elimination condition, eliminate
                    if evaluator(actual_val, rule.threshold):
                        updated_alive[hyp_id] = False
                        break

        # Invariant: H_novel must always remain alive
        updated_alive["H_novel"] = True
        return updated_alive


class HardMaskingManager:
    """Derives tool and verdict availability masks from PoE survival state."""

    def compute_tool_mask(
        self,
        tools: List[str],
        current_alive: Dict[str, bool],
        tool_hyp_mapping: Dict[str, List[str]],
        called_tools: Optional[Any] = None,
        general_tools: Optional[Any] = None,
        allow_unmapped: bool = True,
    ) -> List[bool]:
        """Determine boolean availability for each tool based on surviving hypotheses.

        Rule: A tool is disabled (False) if:
        1. It has already been called in prior turns (non-repetition).
        2. It is not general and all hypotheses that it supports have been eliminated.
        If a tool supports multiple hypotheses (shared tool), it remains active as long
        as at least one supporting hypothesis is still alive.
        If a tool has no explicit mapping, allow_unmapped controls whether it defaults to
        active (True) or disabled (False, matching Registry.tool_mask).

        Args:
            tools: Ordered list of tool names known to the model.
            current_alive: Current boolean survival map for hypotheses.
            tool_hyp_mapping: Mapping of tool_name -> list of supported hypothesis IDs.
            called_tools: Optional sequence or set of tool names executed in prior rounds.
            general_tools: Optional sequence or set of general tool names always callable.
            allow_unmapped: If False, unmapped tools (tools from other domains) are masked out (False).

        Returns:
            List of boolean flags matching the order of tools.
        """
        called_set = set(called_tools or [])
        general_set = set(general_tools or [])
        tool_mask = []
        for tool_name in tools:
            if tool_name in called_set:
                tool_mask.append(False)
                continue

            if tool_name in general_set:
                tool_mask.append(True)
                continue

            supported_hyps = tool_hyp_mapping.get(tool_name, [])
            if not supported_hyps:
                # Unmapped tools: False when allow_unmapped=False and a mapping exists
                tool_mask.append(allow_unmapped if tool_hyp_mapping else True)
                continue

            # Tool remains active if ANY of its supporting hypotheses is alive
            is_any_hyp_alive = any(current_alive.get(hyp, False) for hyp in supported_hyps)
            tool_mask.append(is_any_hyp_alive)

        return tool_mask


def apply_poe_logit_mask(
    logits: torch.Tensor,
    hyp_alive: torch.Tensor,
    masked_val: float = -1e4,
) -> torch.Tensor:
    """Apply hard Popperian logit mask to eliminated hypotheses.

    Formula:
        Logits_final = Logits_raw + (M_elim - 1) * 1e4
    """
    mask = hyp_alive.to(dtype=logits.dtype, device=logits.device)
    return logits + (mask - 1.0) * (-masked_val)


class HybridPoEEngine:
    """Hybrid Process-of-Elimination combining symbolic rules with neural hypothesis scoring (Step A6)."""

    def __init__(self, default_theta_elim: float = 0.20):
        self.default_theta_elim = default_theta_elim
        self.falsification_engine = PoEFalsificationEngine()

    def evaluate(
        self,
        neural_scores: Optional[torch.Tensor],
        hyp_keys: List[str],
        observations: Dict[str, Any],
        rules: Dict[str, List[FalsificationRule]],
        theta_elim: Optional[float] = None,
        prev_alive: Optional[Dict[str, bool]] = None,
    ) -> Dict[str, bool]:
        """Compute combined survival map across candidate hypotheses.

        Properties:
            - Symbolic rules maintain absolute veto power if criteria are breached.
            - Falsified hypotheses remain eliminated across rounds (monotonicity L2).
            - Hypotheses with neural score < theta_elim are eliminated.
            - H_novel is unconditionally preserved (open-world residual slot).
        """
        thresh = theta_elim if theta_elim is not None else self.default_theta_elim
        current_alive = (
            {k: bool(prev_alive.get(k, True)) for k in hyp_keys}
            if prev_alive is not None
            else {k: True for k in hyp_keys}
        )

        # 1. Symbolic rule falsification (hard veto)
        rule_alive = self.falsification_engine.evaluate(observations, current_alive, rules)

        # 2. Neural score gating
        hybrid_alive: Dict[str, bool] = {}
        if neural_scores is not None:
            scores_flat = neural_scores.squeeze(0).tolist() if neural_scores.dim() > 1 else neural_scores.tolist()
            for i, k in enumerate(hyp_keys):
                if k == "H_novel":
                    hybrid_alive["H_novel"] = True
                    continue
                score_survives = (scores_flat[i] >= thresh) if i < len(scores_flat) else True
                hybrid_alive[k] = rule_alive.get(k, True) and score_survives
        else:
            hybrid_alive = rule_alive

        # Invariant: H_novel must always remain alive
        hybrid_alive["H_novel"] = True
        return hybrid_alive


In [ ]:
%%writefile {REXLIB}/rex/rex_format.py
"""Input layout formatting and batch collation for Rex."""
import re
from typing import Any, Dict, List, Optional, Sequence, Union

import torch

from .common import build_sequence
from .rex_vocab import check_type_token


def encode_rex_input(
    tok,
    type_token: str,
    instruction: str,
    hypotheses: Dict[str, str],
    state: Union[str, dict, list] = "",
    alive: Optional[Dict[str, bool]] = None,
    max_len: int = 512,
    head_max_len: int = 192,
    truncate_left: bool = True,
    option_order: Optional[List[int]] = None,
    tool_mask: Optional[List[bool]] = None,
) -> Dict[str, Any]:
    """Encode an incident investigation query into Rex input tokens with validation.

    Layout:
        [CLS] choice question: <type> instructions [SEP] [MASK] H_0 ... [MASK] H_novel [SEP] state_t [SEP]

    Parameters
    ----------
    tok : PreTrainedTokenizer
        Tokenizer configured with special type tokens.
    type_token : str
        Domain event identifier token (e.g. "<TYPE_PROMO_ABUSE>").
    instruction : str
        Investigation prompt / hypothesis question.
    hypotheses : Dict[str, str]
        Ordered mapping of hypothesis name -> descriptive text.
        Must end with "H_novel".
    state : Union[str, dict, list]
        Accumulated evidence observation state.
    alive : Optional[Dict[str, bool]]
        Survival mask for Popperian elimination. "H_novel" cannot be marked False.
    max_len : int
        Maximum total sequence length (default 512).
    head_max_len : int
        Maximum length reserved for question and hypothesis header (default 192).
    truncate_left : bool
        If True, truncate earlier evidence in state when budget is exceeded.
    option_order : Optional[List[int]]
        Explicit permutation of hypotheses to mitigate position bias.
    tool_mask : Optional[List[bool]]
        Optional boolean mask for available tools.

    Returns
    -------
    Dict[str, Any]
        Dictionary containing input_ids, markers, hyp_keys, hyp_alive, truncation stats, etc.
    """
    # 1. Validate type token
    type_tid = check_type_token(tok, type_token)

    # 2. Validate hypotheses structure
    if not isinstance(hypotheses, dict) or len(hypotheses) < 2:
        raise ValueError("hypotheses must be a dictionary with at least 2 hypotheses (including H_novel)")

    hyp_keys_list = list(hypotheses.keys())
    if hyp_keys_list[-1] != "H_novel":
        raise ValueError(
            f"The final hypothesis slot must be 'H_novel', but found '{hyp_keys_list[-1]}'"
        )

    alive_map = alive or {}
    unknown_alive = set(alive_map.keys()) - set(hypotheses.keys())
    if unknown_alive:
        raise ValueError(f"Unknown hypothesis keys in alive mask: {sorted(unknown_alive)}")
    if alive_map.get("H_novel") is False:
        raise ValueError("Slot 'H_novel' cannot be eliminated; it represents the unexplained residual.")

    # 3. Assemble sequence using Laya's build_sequence
    ins_text = f"{type_token} {instruction.strip()}"
    q = {
        "t": "choice",
        "ins": ins_text,
        "crit": hypotheses,
    }

    ids, markers, stats, trunc_stats = build_sequence(
        tok,
        state=state,
        q=q,
        max_len=max_len,
        head_max_len=head_max_len,
        truncate_left=truncate_left,
        option_order=option_order,
        return_truncation_stats=True,
        return_stats=True,
    )

    # 4. Strict validations against truncation of critical components
    if len(markers) != len(hypotheses):
        raise ValueError(
            f"Expected {len(hypotheses)} hypothesis markers, but only {len(markers)} survived truncation. "
            f"Increase max_len or head_max_len."
        )

    if stats.get("options_distinct", len(hypotheses)) < stats.get("options", len(hypotheses)):
        raise ValueError("Multiple hypotheses truncated to identical token sequences.")

    if type_tid not in ids:
        raise ValueError(f"Type token {type_token} was truncated from sequence header.")

    # 5. Map hypothesis keys and alive flags according to effective option order
    effective_order = option_order if option_order is not None else list(range(len(hyp_keys_list)))
    ordered_keys = [hyp_keys_list[i] for i in effective_order]
    ordered_alive = [alive_map.get(k, True) for k in ordered_keys]

    return {
        "input_ids": ids,
        "markers": markers,
        "hyp_keys": ordered_keys,
        "hyp_alive": ordered_alive,
        "truncation": trunc_stats,
        "tool_mask": tool_mask,
    }


def collate_rex(
    batch: Sequence[Dict[str, Any]],
    pad_token_id: int,
) -> Dict[str, torch.Tensor]:
    """Collate a list of encode_rex_input outputs into a padded batch for RexDecisionModel.

    Parameters
    ----------
    batch : Sequence[Dict[str, Any]]
        List of item dictionaries produced by encode_rex_input.
    pad_token_id : int
        Tokenizer padding token ID.

    Returns
    -------
    Dict[str, torch.Tensor]
        Batch ready for model.rex_forward:
        - input_ids: [B, max_len]
        - attention_mask: [B, max_len]
        - marker_pos: [B, max_markers]
        - marker_mask: [B, max_markers] bool
        - hyp_alive: [B, max_markers] bool
        - qtype: [B]
        - tool_mask: Optional [B, n_tools] bool
    """
    b = len(batch)
    max_len = max(len(item["input_ids"]) for item in batch)
    max_markers = max(len(item["markers"]) for item in batch)

    input_ids = torch.full((b, max_len), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((b, max_len), dtype=torch.long)
    marker_pos = torch.zeros((b, max_markers), dtype=torch.long)
    marker_mask = torch.zeros((b, max_markers), dtype=torch.bool)
    hyp_alive = torch.zeros((b, max_markers), dtype=torch.bool)

    has_tool_mask = any(item.get("tool_mask") is not None for item in batch)
    tool_mask_tensor = None
    if has_tool_mask:
        n_tools = max(len(item["tool_mask"]) for item in batch if item.get("tool_mask") is not None)
        tool_mask_tensor = torch.ones((b, n_tools), dtype=torch.bool)

    for i, item in enumerate(batch):
        ids = item["input_ids"]
        seq_len = len(ids)
        input_ids[i, :seq_len] = torch.tensor(ids, dtype=torch.long)
        attention_mask[i, :seq_len] = 1

        markers = item["markers"]
        m_len = len(markers)
        marker_pos[i, :m_len] = torch.tensor(markers, dtype=torch.long)
        marker_mask[i, :m_len] = True

        alive_flags = item.get("hyp_alive", [True] * m_len)
        hyp_alive[i, :m_len] = torch.tensor(alive_flags, dtype=torch.bool)

        if tool_mask_tensor is not None and item.get("tool_mask") is not None:
            tm = item["tool_mask"]
            tool_mask_tensor[i, : len(tm)] = torch.tensor(tm, dtype=torch.bool)

    result = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "marker_pos": marker_pos,
        "marker_mask": marker_mask,
        "hyp_alive": hyp_alive,
        "qtype": torch.zeros(b, dtype=torch.long),
    }
    if tool_mask_tensor is not None:
        result["tool_mask"] = tool_mask_tensor
    return result


def format_metric_value(val: Any) -> str:
    """Format metric value concisely: bool as 1/0, rounded float, bounded string."""
    if isinstance(val, bool):
        return "1" if val else "0"
    elif isinstance(val, float):
        if val.is_integer():
            return str(int(val))
        return f"{val:.2f}".rstrip("0").rstrip(".")
    elif isinstance(val, int):
        return str(val)
    elif isinstance(val, str):
        # Strip control characters and sanitize user strings
        cleaned = re.sub(r"[\r\n\t]+", " ", val).strip()
        # Cap user-provided string lengths to prevent token bloat
        if len(cleaned) > 32:
            cleaned = cleaned[:29] + "..."
        return cleaned
    return str(val)


def format_observations(observations: Dict[str, Any]) -> str:
    """Render one round of tool observations as evidence text.

    Grouped observations ({tool: {metric: value}}) use the Part B packer layout
    ("tool OK k=v ... | tool OK ..."), so training, InvestigatorAgent and the orchestrator
    feed the model the same text. Flat observations ({metric: value}) become "k=v ...".
    """
    if observations and all(isinstance(v, dict) for v in observations.values()):
        segments = []
        for tool_name, metrics in sorted(observations.items()):
            if not metrics:
                segments.append(f"{tool_name} OK")
                continue
            kv = " ".join(f"{k}={format_metric_value(v)}" for k, v in sorted(metrics.items()))
            segments.append(f"{tool_name} OK {kv}")
        return " | ".join(segments)
    return " ".join(f"{k}={format_metric_value(v)}" for k, v in sorted(observations.items()))


def encode_evidence(
    tok: Any,
    evidence_text: Union[str, Dict[str, Any], List[Any]],
    round_idx: int = 1,
    max_tokens: int = 64,
) -> Dict[str, Any]:
    """Encode an incremental evidence piece with a strict token budget cap (Step A4).

    Parameters
    ----------
    tok : Any
        Tokenizer configured for the model.
    evidence_text : Union[str, Dict[str, Any], List[Any]]
        Observation string or structured dictionary from tools.
    round_idx : int
        Investigation turn index (default: 1).
    max_tokens : int
        Hard upper bound on encoded evidence tokens (default: 64).

    Returns
    -------
    Dict[str, Any]
        {
            "input_ids": list of token IDs,
            "attention_mask": list of binary mask flags,
            "is_truncated": bool flag,
            "round_idx": int,
        }
    """
    import json

    if not isinstance(evidence_text, str):
        text = json.dumps(evidence_text, ensure_ascii=False)
    else:
        text = evidence_text.strip()

    if hasattr(tok, "encode"):
        ids = tok.encode(text, add_special_tokens=False)
    else:
        # Fallback dummy encoding for token-free mocking in unit tests
        ids = [int(abs(hash(w)) % 1000) + 1 for w in text.split()] or [1]

    is_truncated = len(ids) > max_tokens
    if is_truncated:
        ids = ids[:max_tokens]

    return {
        "input_ids": ids,
        "attention_mask": [1] * len(ids),
        "is_truncated": is_truncated,
        "round_idx": round_idx,
    }


In [ ]:
%%writefile {REXLIB}/rex/rex_vocab.py
import warnings
from typing import Dict, Optional

import torch
import torch.nn as nn


def check_type_token(tok, token: str) -> int:
    """Validate that a type token is registered and produces exactly one token ID.

    Raises ValueError if the token is not known to the tokenizer or gets split into
    multiple subwords.
    """
    if not isinstance(token, str) or not token:
        raise ValueError(f"Type token must be a non-empty string, got {token!r}")

    # Check that tokenizer recognizes it as a single token
    tokens = tok.tokenize(token)
    if len(tokens) != 1:
        raise ValueError(
            f"Type token {token!r} must tokenize to exactly 1 token, but got {len(tokens)}: {tokens}"
        )
    tid = tok.convert_tokens_to_ids(tokens[0])
    if hasattr(tok, "unk_token_id") and tid == tok.unk_token_id:
        raise ValueError(f"Type token {token!r} resolved to unk_token_id ({tid})")
    return tid


def add_type_tokens(
    tok,
    model: Optional[nn.Module],
    type_tokens: Dict[str, str],
) -> Dict[str, int]:
    """Register special <type> tokens in the tokenizer and initialize their embeddings.

    Parameters
    ----------
    tok : PreTrainedTokenizer
        Tokenizer to add special tokens to.
    model : Optional[nn.Module]
        DecisionModel or RexDecisionModel instance. If provided, embedding weights
        will be resized (if necessary) and the newly added token embeddings will be
        initialized to the mean embedding of their descriptive text.
    type_tokens : Dict[str, str]
        Mapping from token string (e.g. "<TYPE_PROMO_ABUSE>") to a natural language
        description (e.g. "promotion abuse: farming sign-up bonuses and voucher fraud").

    Returns
    -------
    Dict[str, int]
        Mapping from token string to token ID in the expanded tokenizer.
    """
    if not type_tokens:
        return {}

    # 1. Add tokens as special tokens to prevent sub-word splitting and casing changes
    new_tokens = list(type_tokens.keys())
    existing_vocab = tok.get_vocab()
    unseen_tokens = [t for t in new_tokens if t not in existing_vocab]

    if unseen_tokens:
        tok.add_tokens(unseen_tokens, special_tokens=True)

    result_ids = {t: tok.convert_tokens_to_ids(t) for t in new_tokens}

    # 2. If model is provided, resize embeddings and initialize weights
    if model is not None:
        # Locate input embeddings layer across standard architectures and wrappers
        embed_module = None
        encoder = model
        for candidate in (model, getattr(model, "encoder", None)):
            if candidate is None:
                continue
            if hasattr(candidate, "get_input_embeddings") and callable(candidate.get_input_embeddings):
                res = candidate.get_input_embeddings()
                if res is not None and hasattr(res, "weight"):
                    embed_module = res
                    encoder = candidate
                    break
            if hasattr(candidate, "embeddings") and hasattr(candidate.embeddings, "word_embeddings"):
                embed_module = candidate.embeddings.word_embeddings
                encoder = candidate
                break

        if embed_module is None:
            raise AttributeError("Could not locate input embeddings layer on model/encoder.")

        cur_num_embeddings = embed_module.weight.size(0)
        target_vocab_size = len(tok)

        # Only resize if tokenizer length exceeds existing matrix capacity
        # (mmBERT has spare padding rows; BERT/ModernBERT might need resizing)
        if target_vocab_size > cur_num_embeddings:
            if hasattr(encoder, "resize_token_embeddings"):
                encoder.resize_token_embeddings(target_vocab_size)
                if hasattr(encoder, "get_input_embeddings"):
                    embed_module = encoder.get_input_embeddings()
                elif hasattr(encoder, "embeddings") and hasattr(encoder.embeddings, "word_embeddings"):
                    embed_module = encoder.embeddings.word_embeddings
            else:
                raise RuntimeError(
                    f"Encoder has no resize_token_embeddings; cannot resize from {cur_num_embeddings} to {target_vocab_size}"
                )

        # 3. Warm-start embedding with mean of description tokens
        # Only initialize for newly added tokens so that fine-tuned rows are never wiped
        with torch.no_grad():
            weights = embed_module.weight
            for token_str in unseen_tokens:
                desc = type_tokens.get(token_str)
                if not desc or not desc.strip():
                    warnings.warn(
                        f"Empty or missing description for type token {token_str!r}; "
                        f"embedding was left with uncalibrated initialization.",
                        UserWarning,
                        stacklevel=2,
                    )
                    continue
                desc_encoded = tok(desc, add_special_tokens=False)
                desc_ids = desc_encoded.get("input_ids", [])
                if desc_ids:
                    desc_tensor = torch.tensor(desc_ids, device=weights.device, dtype=torch.long)
                    valid_mask = desc_tensor < weights.size(0)
                    if valid_mask.any():
                        mean_vector = weights[desc_tensor[valid_mask]].mean(dim=0)
                        tid = result_ids[token_str]
                        weights[tid].copy_(mean_vector)

    return result_ids


In [ ]:
%%writefile {REXLIB}/rex/rex_labels.py
"""Counterfactual supervision and offline label generation for Rex training (Step A9).

This module provides pure mathematical functions for offline trajectory label generation:
1. generate_sufficiency_label: Label S* in {0, 1} for epistemic sufficiency head.
2. generate_tool_gain_labels: Counterfactual information gain labels gain*_k in {0, 1, -100} per tool.
3. generate_stall_label: Early learned STALL supervision indicator.
4. generate_redundancy_mask: Boolean mask filtering out redundant and refuted tool branches.

Label semantics follow docs/rex-label-schema.md.
"""
from __future__ import annotations

from typing import TYPE_CHECKING, Any, Dict, List, Optional, Sequence

if TYPE_CHECKING:
    from .rex_registry import FalsificationRule, ToolSpec

DEFAULT_S_STAR_THRESHOLD: float = 0.85
GAIN_UNOBSERVED: float = -100.0

# Default business risk per domain type token, used when the registry domain sets no `risk`.
DEFAULT_RISK_MAPPING: Dict[str, float] = {
    "<TYPE_PAYMENT_FRAUD>": 0.5,
    "<TYPE_ACCOUNT_TAKEOVER>": 0.4,
    "<TYPE_RBA_LOGIN_DEFENSE>": 0.5,
    "<TYPE_PROMO_ABUSE>": 0.1,
    "<TYPE_API_SCRAPING>": 0.05,
    "<TYPE_CREDENTIAL_STUFFING>": 0.1,
}


def generate_sufficiency_label(
    alive_hypotheses: Sequence[str],
    target_hypothesis: str,
    target_falsified: bool = False,
) -> float:
    """Compute binary ground-truth sufficiency label S* in {0.0, 1.0}.

    Condition:
        S* = 1.0 if the evidence is sufficient to conclusively determine the verdict:
             (1) The target is the only surviving cataloged hypothesis.
             (2) Or all cataloged hypotheses are eliminated and target is H_novel.
        Otherwise S* = 0.0.

        H_novel is never eliminated, so it is not counted as a competing survivor.

    Args:
        alive_hypotheses: List or set of currently surviving hypothesis IDs.
        target_hypothesis: Ground truth winner hypothesis ID.
        target_falsified: Whether the true target was erroneously falsified.

    Returns:
        Float label 1.0 if sufficient, 0.0 otherwise.
    """
    if target_falsified:
        return 0.0

    alive_set = set(alive_hypotheses)
    if target_hypothesis not in alive_set:
        return 0.0

    known_alive = alive_set - {"H_novel"}
    if target_hypothesis == "H_novel":
        # OOD / Novel case: every cataloged hypothesis has been eliminated
        return 1.0 if not known_alive else 0.0

    # Cataloged target is the only cataloged survivor
    return 1.0 if known_alive == {target_hypothesis} else 0.0


def generate_tool_gain_labels(
    alive_hypotheses: Sequence[str],
    tool_specs: Sequence[ToolSpec],
    counterfactual_observations: Dict[str, Any],
    falsification_rules: Dict[str, List[FalsificationRule]],
    mask_unobserved: bool = False,
) -> Dict[str, float]:
    """Compute binary counterfactual information gain gain*_k in {0.0, 1.0, -100.0} for every tool.

    Definition (docs/rex-label-schema.md, labels.tool_gains):
        gain*_k = 1.0 if tool k's observations would falsify at least one surviving cataloged hypothesis
                  (it uncovers an unseen key fact), 0.0 otherwise.
        If mask_unobserved=True and none of tool k's metrics were recorded, gain*_k = -100.0 (masked from loss).

    Args:
        alive_hypotheses: Currently surviving hypothesis IDs.
        tool_specs: Specifications of all candidate tools.
        counterfactual_observations: Complete unmasked ground-truth metric values from world rollout.
        falsification_rules: Mapping from hypothesis ID to its Popperian falsification rules.
        mask_unobserved: Whether to assign -100.0 to tools whose metrics were unobserved.

    Returns:
        Dictionary mapping tool name to its counterfactual information gain (1.0, 0.0 or -100.0).
    """
    # H_novel cannot be eliminated by standard falsification rules
    known_alive = set(alive_hypotheses) - {"H_novel"}

    gains: Dict[str, float] = {}

    for spec in tool_specs:
        tool_name = spec.name
        metrics_provided = set(spec.returns)

        observed_metrics = [m for m in metrics_provided if m in counterfactual_observations]
        if mask_unobserved and not observed_metrics and metrics_provided:
            gains[tool_name] = GAIN_UNOBSERVED
            continue

        refutes_any = False
        for hyp_id in known_alive:
            for rule in falsification_rules.get(hyp_id, []):
                if rule.metric in metrics_provided and rule.metric in counterfactual_observations:
                    if rule.eval(counterfactual_observations[rule.metric]):
                        refutes_any = True
                        break
            if refutes_any:
                break

        gains[tool_name] = 1.0 if refutes_any else 0.0

    return gains


def generate_stall_label(
    s_star: float,
    tool_gains: Dict[str, float],
    tool_costs: Optional[Dict[str, float]] = None,
    s_threshold: float = DEFAULT_S_STAR_THRESHOLD,
    allowed_tools: Optional[Sequence[str]] = None,
) -> float:
    """Compute ground-truth learned STALL label in {0.0, 1.0}.

    Condition (docs/rex-label-schema.md, labels.is_stall):
        STALL* = 1.0 iff S* < s_threshold AND no remaining allowed tool has gain*_k > 0.
        Otherwise STALL* = 0.0.

    The cost hurdle tau_k is a dispatcher concern applied to the predicted gain probability
    (rex_tools.gain_threshold); the label only records whether a tool can still uncover a key fact.

    Args:
        s_star: Ground truth sufficiency label (0.0 or 1.0).
        tool_gains: Counterfactual information gains per tool.
        tool_costs: Unused; kept for call compatibility.
        s_threshold: Sufficiency cutoff threshold.
        allowed_tools: Tools still callable (uncalled and not PoE-masked). None means every tool.

    Returns:
        1.0 if investigation is stagnated/unsolvable, 0.0 otherwise.
    """
    if s_star >= s_threshold:
        return 0.0

    allowed = set(allowed_tools) if allowed_tools is not None else None
    for tool_name, gain in tool_gains.items():
        if gain == GAIN_UNOBSERVED:
            continue
        if allowed is not None and tool_name not in allowed:
            continue
        if gain > 0.0:
            return 0.0

    return 1.0


def generate_risk_label(
    type_token: str,
    risk_mapping: Optional[Dict[str, float]] = None,
    default_risk: float = 0.1,
) -> float:
    """Compute target risk label R* in [0.0, 1.0] based on domain business policy."""
    mapping = risk_mapping or DEFAULT_RISK_MAPPING
    return mapping.get(type_token, default_risk)


def generate_hypotheses_survival_labels(
    alive_hypotheses: Sequence[str],
    all_hypotheses: Sequence[str],
) -> List[float]:
    """Compute binary survival indicators hyp_true in {0.0, 1.0} across all candidate hypotheses."""
    alive_set = set(alive_hypotheses)
    return [1.0 if h in alive_set else 0.0 for h in all_hypotheses]


def generate_redundancy_mask(
    all_tools: Sequence[str],
    called_tools: Sequence[str],
    alive_hypotheses: Sequence[str],
    tool_hyp_mapping: Dict[str, List[str]],
) -> List[bool]:
    """Generate boolean redundancy mask of length N_tools.

    Value:
        True: Tool is redundant (already called or all its associated hypotheses are eliminated).
        False: Tool remains viable and uncalled.

    Args:
        all_tools: List of all tool names in catalog order.
        called_tools: List of tool names called in prior rounds.
        alive_hypotheses: Currently surviving hypothesis IDs.
        tool_hyp_mapping: Mapping from tool name to associated hypothesis IDs.

    Returns:
        List of booleans of length N_tools.
    """
    called_set = set(called_tools)
    alive_set = set(alive_hypotheses)
    redundancy: List[bool] = []

    for tool in all_tools:
        if tool in called_set:
            redundancy.append(True)
            continue

        associated_hyps = tool_hyp_mapping.get(tool, [])
        # If tool is associated with hypotheses, check if at least one is alive
        if associated_hyps:
            has_alive_hyp = any(h in alive_set for h in associated_hyps)
            redundancy.append(not has_alive_hyp)
        else:
            # General / unassociated tool is not redundant if uncalled
            redundancy.append(False)

    return redundancy


In [ ]:
%%writefile {REXLIB}/rex/rex_data.py
"""Data rollout generation, trajectory tracking, and PyTorch Dataset for Rex (Part 3)."""
import copy
import json
from pathlib import Path
import random
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple, Union

import torch
from torch.utils.data import Dataset

from .rex_format import encode_rex_input
from .rex_labels import (
    generate_redundancy_mask,
    generate_risk_label,
    generate_stall_label,
    generate_sufficiency_label,
    generate_tool_gain_labels,
)
from .rex_registry import MetricSpec, Registry


def _sample_metric_value(
    spec: MetricSpec,
    operator: Optional[str],
    threshold: Any,
    trigger: bool,
    rng: random.Random,
) -> Any:
    """Generate a metric observation value that either triggers or satisfies/avoids a rule."""
    if spec.kind == "bool":
        if operator in ("EQ", "NEQ") and threshold is not None:
            thresh_bool = bool(threshold)
            if operator == "EQ":
                return thresh_bool if trigger else (not thresh_bool)
            else:  # NEQ
                return (not thresh_bool) if trigger else thresh_bool
        return rng.choice([0, 1])

    min_v = spec.min_val if spec.min_val is not None else 0.0
    max_v = spec.max_val if spec.max_val is not None else 1.0

    if operator is None or threshold is None:
        if spec.kind == "int":
            return rng.randint(int(min_v), int(max_v))
        return round(rng.uniform(min_v, max_v), 3)

    t = float(threshold)
    eps = 0.02 if spec.kind == "float" else 1

    if operator in ("GT", "GTE"):
        if trigger:
            low = min(max_v, t + eps)
            high = max_v
            val = rng.uniform(low, high) if low < high else max_v
        else:
            low = min_v
            high = max(min_v, t - (eps if operator == "GT" else 0))
            val = rng.uniform(low, high) if low < high else min_v
    elif operator in ("LT", "LTE"):
        if trigger:
            low = min_v
            high = max(min_v, t - eps)
            val = rng.uniform(low, high) if low < high else min_v
        else:
            low = min(max_v, t + (eps if operator == "LT" else 0))
            high = max_v
            val = rng.uniform(low, high) if low < high else max_v
    elif operator == "EQ":
        if trigger:
            val = t
        else:
            alt_val = t + 2 * eps if t + 2 * eps <= max_v else t - 2 * eps
            val = alt_val
    elif operator == "NEQ":
        if trigger:
            alt_val = t + 2 * eps if t + 2 * eps <= max_v else t - 2 * eps
            val = alt_val
        else:
            val = t
    else:
        val = rng.uniform(min_v, max_v)

    if spec.kind == "int":
        return int(round(val))
    return round(float(val), 3)


def _domain_risk(domain: Any) -> float:
    """Business risk label R* for a domain: the registry value when set, else the default mapping."""
    risk = getattr(domain, "risk", None)
    return float(risk) if risk is not None else generate_risk_label(domain.type_token)


def hypothesis_risks(domain: Any) -> Dict[str, float]:
    """Registry risk of each hypothesis of a domain (domain risk when a hypothesis sets none)."""
    return {h.id: case_risk(domain, h.id) for h in domain.hypotheses}


def case_risk(domain: Any, target: str) -> float:
    """Business risk label R* of one case: the risk of acting on its true cause.

    Uses the target hypothesis' registry `risk`, falling back to the domain risk. The label then
    varies with the case, so the risk head learns E[risk | evidence] instead of a per-domain constant.
    """
    hyp = getattr(domain, "hypothesis_map", {}).get(target)
    risk = getattr(hyp, "risk", None) if hyp is not None else None
    return float(risk) if risk is not None else _domain_risk(domain)


def format_state(state: Dict[str, Any], compact: bool = True) -> str:
    """Format accumulated alert and round observations into a structured textual context."""
    if not isinstance(state, dict):
        return str(state)

    parts: List[str] = []

    # 1. Alert context
    alert = state.get("alert", {})
    if alert:
        kv = " ".join(f"{k}={v}" for k, v in sorted(alert.items()))
        parts.append(f"alert: {kv}")

    # 2. Sequential rounds
    for k in sorted(state.keys()):
        if not k.startswith("round_"):
            continue
        round_data = state[k]
        if not round_data:
            continue
        tool_parts = []
        for tool_name, metrics in sorted(round_data.items()):
            m_kv = " ".join(f"{mk}={mv}" for mk, mv in sorted(metrics.items()))
            tool_parts.append(f"{tool_name} {{{m_kv}}}")
        if tool_parts:
            parts.append(f"{k}: " + " ".join(tool_parts))

    return " | ".join(parts) if parts else "state: normal"


def sample_world(
    registry: Registry,
    domain_key: str,
    rng: random.Random,
    novel_ratio: float = 0.12,
    p_falsify: float = 0.85,
) -> Dict[str, Any]:
    """Sample an underlying incident state and simulated observations for all available tools."""
    domain = registry.get_domain(domain_key)
    known_hyps = [h for h in domain.hypotheses if not h.is_residual]

    is_novel = rng.random() < novel_ratio
    if is_novel:
        target_id = "H_novel"
    else:
        target_id = rng.choice(known_hyps).id

    # Constraints per metric: (operator, threshold, trigger)
    metric_constraints: Dict[str, Tuple[str, Any, bool]] = {}

    if is_novel:
        # Step 3.3: In OOD / H_novel incidents, all known hypotheses H_0..H_{M-1}
        # must be falsified by observations so that PoE leaves only H_novel alive.
        for h in known_hyps:
            # Pick at least one rule from h to trigger
            r = rng.choice(h.falsification_rules)
            metric_constraints[r.metric] = (r.operator, r.threshold, True)
    else:
        # In-distribution: Target H* must NOT have any of its rules triggered
        target_hyp = domain.hypothesis_map[target_id]
        for r in target_hyp.falsification_rules:
            metric_constraints[r.metric] = (r.operator, r.threshold, False)

        # For other known hypotheses, falsify them with probability p_falsify
        for h in known_hyps:
            if h.id == target_id:
                continue
            if rng.random() < p_falsify:
                # Find a rule that does not conflict with target's un-triggered metrics
                candidates = [r for r in h.falsification_rules if r.metric not in metric_constraints]
                if candidates:
                    r = rng.choice(candidates)
                    metric_constraints[r.metric] = (r.operator, r.threshold, True)
            else:
                # Keep ambiguous by ensuring none of its rules trigger
                for r in h.falsification_rules:
                    if r.metric not in metric_constraints:
                        metric_constraints[r.metric] = (r.operator, r.threshold, False)

    # Sample values for all registered metrics
    metric_values: Dict[str, Any] = {}
    for m_name, spec in registry.metrics.items():
        if m_name in metric_constraints:
            op, thresh, trigger = metric_constraints[m_name]
            metric_values[m_name] = _sample_metric_value(spec, op, thresh, trigger, rng)
        else:
            metric_values[m_name] = _sample_metric_value(spec, None, None, False, rng)

    # Structure into alert metrics and tool return outputs
    alert: Dict[str, Any] = {m: metric_values[m] for m in domain.alert_metrics if m in metric_values}
    tool_data: Dict[str, Dict[str, Any]] = {}
    for t_name, t_spec in registry.tool_specs.items():
        tool_data[t_name] = {m: metric_values[m] for m in t_spec.returns if m in metric_values}

    return {
        "domain_key": domain_key,
        "target": target_id,
        "alert": alert,
        "tool_data": tool_data,
        "is_novel": is_novel,
    }


def oracle_tools(
    registry: Registry,
    domain_key: str,
    state: Dict[str, Any],
    alive: Dict[str, bool],
    t: int,
    budgets: Tuple[int, int, int] = (3, 2, 1),
    scope_schedule: Sequence[str] = ("macro", "micro"),
    scope_fallback: bool = True,
    useful_tools: Optional[Set[str]] = None,
) -> List[str]:
    """Select next investigative tools with a greedy cover strategy.

    With `useful_tools` (the tools whose counterfactual gain* is 1) the teacher is privileged: it
    only picks tools that uncover a key fact and stops when none is left, so tool labels never
    contradict the gain labels (docs/rex-label-schema.md, labels.teacher_tools). Without it the
    teacher has no ground-truth knowledge and fills each round's budget.
    """
    domain = registry.get_domain(domain_key)
    round_idx = min(t - 1, len(budgets) - 1)
    budget = budgets[round_idx]

    # Target scope for the round
    target_scope: Optional[str] = None
    if t <= len(scope_schedule):
        target_scope = scope_schedule[t - 1]

    # Identify tools already called in prior rounds
    called_tools: Set[str] = set()
    for k, v in state.items():
        if k.startswith("round_") and isinstance(v, dict):
            called_tools.update(v.keys())

    # Get allowed tools under PoE
    tool_mask_flags = registry.tool_mask(domain_key, alive, called_tools=list(called_tools))
    candidate_tools = [tool for tool, ok in zip(registry.tools, tool_mask_flags) if ok]
    if useful_tools is not None:
        candidate_tools = [tool for tool in candidate_tools if tool in useful_tools]

    if not candidate_tools:
        return []

    # Active surviving known hypotheses
    active_hyps = [h for h in domain.hypotheses if not h.is_residual and alive.get(h.id, True)]
    if not active_hyps:
        return []

    # Utility score: number of surviving hypotheses this tool can inspect
    tool_scores: Dict[str, int] = {}
    for tool_name in candidate_tools:
        score = sum(1 for h in active_hyps if tool_name in h.associated_tools)
        tool_scores[tool_name] = score

    # Filter by scope if required
    if target_scope:
        scoped = [t for t in candidate_tools if registry.tool_specs[t].scope == target_scope and tool_scores[t] > 0]
        # Sort descending by score, tie-break by tool name
        scoped.sort(key=lambda x: (-tool_scores[x], x))
        if scoped:
            return scoped[:budget]
        elif not scope_fallback:
            return []

    # Fallback to any scope if scoped pool is empty
    fallback_candidates = [t for t in candidate_tools if tool_scores[t] > 0]
    fallback_candidates.sort(key=lambda x: (-tool_scores[x], x))
    return fallback_candidates[:budget]


def validate_sample(
    sample: Dict[str, Any],
    registry: Registry,
    budgets: Tuple[int, int, int] = (3, 2, 1),
    scope_schedule: Sequence[str] = ("macro", "micro"),
    scope_fallback: bool = True,
) -> None:
    """Strictly assert invariant guarantees on generated trajectory samples."""
    target = sample["target"]
    alive = sample["alive"]

    # 1. Target H* must never be marked eliminated
    if not alive.get(target, True):
        raise ValueError(f"Ground truth target '{target}' was marked eliminated in sample: {sample}")

    # 2. H_novel must never be marked eliminated
    if not alive.get("H_novel", True):
        raise ValueError(f"Residual slot 'H_novel' was marked eliminated in sample: {sample}")

    # 3. No label assigned to disabled tools
    allowed_set = set(sample["allowed_tools"])
    for lbl in sample["tool_labels"]:
        if lbl not in allowed_set:
            raise ValueError(f"Tool label '{lbl}' is not in allowed_tools {allowed_set}.")

    # 4. Budget constraints
    t = sample["t"]
    budget = budgets[min(t - 1, len(budgets) - 1)]
    if len(sample["tool_labels"]) > budget:
        raise ValueError(
            f"Number of tool labels ({len(sample['tool_labels'])}) exceeds round budget ({budget})."
        )

    # 5. Scope adherence (when not falling back)
    if not scope_fallback and t <= len(scope_schedule):
        expected_scope = scope_schedule[t - 1]
        for lbl in sample["tool_labels"]:
            lbl_scope = registry.tool_specs[lbl].scope
            if lbl_scope != expected_scope:
                raise ValueError(
                    f"Tool '{lbl}' has scope '{lbl_scope}', expected '{expected_scope}' in round {t}."
                )


def rollout(
    registry: Registry,
    domain_key: str,
    rng: random.Random,
    case_id: str,
    budgets: Tuple[int, int, int] = (3, 2, 1),
    scope_schedule: Sequence[str] = ("macro", "micro"),
    novel_ratio: float = 0.12,
    p_falsify: float = 0.85,
    scope_fallback: bool = True,
    privileged_teacher: bool = True,
) -> List[Dict[str, Any]]:
    """Simulate a complete 1-3 turn investigation incident rollout.

    `privileged_teacher` (default) labels only the tools with gain* = 1 (see oracle_tools), while the
    behaviour policy that produces the observations keeps the exploratory budget-filling choice, so
    training states include rounds that uncover nothing, as at inference. `called_tools` records
    what the behaviour policy called; `tool_labels` is what the teacher recommends.
    """
    world = sample_world(registry, domain_key, rng, novel_ratio=novel_ratio, p_falsify=p_falsify)
    state: Dict[str, Any] = {"alert": world["alert"]}

    # Collect all ground-truth metric values from world rollout for counterfactual labeling
    all_observations: Dict[str, Any] = dict(world["alert"])
    for t_data in world["tool_data"].values():
        all_observations.update(t_data)

    domain = registry.get_domain(domain_key)
    falsification_rules = {h.id: h.falsification_rules for h in domain.hypotheses}
    tool_specs_list = [registry.tool_specs[t_name] for t_name in registry.tools]
    tool_hyp_map: Dict[str, List[str]] = {t_name: [] for t_name in registry.tools}
    for h in domain.hypotheses:
        for t in h.associated_tools:
            if t in tool_hyp_map:
                tool_hyp_map[t].append(h.id)

    # Initial Popperian evaluation on alert metrics
    alive, eliminated_by = registry.falsify(domain_key, world["alert"])

    samples: List[Dict[str, Any]] = []
    called_tools: List[str] = []

    terminal_recorded = False
    for t in (1, 2, 3):
        allowed = [
            tool
            for tool, ok in zip(registry.tools, registry.tool_mask(domain_key, alive, called_tools))
            if ok
        ]

        alive_list = [k for k, v in alive.items() if v]
        s_star = generate_sufficiency_label(alive_list, world["target"])
        r_star = case_risk(domain, world["target"])
        gain_dict = generate_tool_gain_labels(
            alive_list,
            tool_specs_list,
            all_observations,
            falsification_rules,
            mask_unobserved=True,
        )

        oracle_kwargs = dict(
            registry=registry,
            domain_key=domain_key,
            state=state,
            alive=alive,
            t=t,
            budgets=budgets,
            scope_schedule=scope_schedule,
            scope_fallback=scope_fallback,
        )
        # Behaviour policy: what is actually called, so the next states look like inference (some
        # rounds uncover nothing). Teacher labels: with privileged_teacher, only the useful tools.
        behaviour = oracle_tools(**oracle_kwargs)
        if privileged_teacher:
            labels = oracle_tools(**oracle_kwargs, useful_tools={tool for tool, g in gain_dict.items() if g > 0.0})
            budget = budgets[min(t - 1, len(budgets) - 1)]
            calls = (list(labels) + [tool for tool in behaviour if tool not in labels])[:budget]
        else:
            labels = behaviour
            calls = list(behaviour)
        stall_star = generate_stall_label(s_star, gain_dict, allowed_tools=allowed)
        redundancy_mask = generate_redundancy_mask(registry.tools, called_tools, alive_list, tool_hyp_map)
        hyp_true = {h_id: (1.0 if alive.get(h_id, True) else 0.0) for h_id in alive}

        step_labels = {
            "s_star": float(s_star),
            "r_star": float(r_star),
            "gain_star": {tool: gain_dict.get(tool, -100.0) for tool in registry.tools},
            "stall_star": float(stall_star),
            "redundancy_mask": redundancy_mask,
            "hyp_true": hyp_true,
            "hyp_risk": hypothesis_risks(domain),
        }

        if not calls:
            # Step 3.1: When oracle returns empty at round t, terminal sample carries t (not t + 1)
            terminal_sample = {
                "case_id": case_id,
                "t": t,
                "type_token": domain_key,
                "state": copy.deepcopy(state),
                "alive": dict(alive),
                "allowed_tools": allowed,
                "tool_labels": [],  # Stop calling tools
                "label_scope": "terminal",
                "observations": {},
                "target": world["target"],
                "eliminated_by": copy.deepcopy(eliminated_by),
                "labels": copy.deepcopy(step_labels),
            }
            validate_sample(
                terminal_sample,
                registry,
                budgets=budgets,
                scope_schedule=scope_schedule,
                scope_fallback=scope_fallback,
            )
            samples.append(terminal_sample)
            terminal_recorded = True
            break

        # Observations for this turn
        obs = {tool: world["tool_data"][tool] for tool in calls}

        # Determine label scope ("none": the behaviour policy explores but no tool is worth calling)
        scopes = {registry.tool_specs[t_name].scope for t_name in labels}
        label_scope = (scopes.pop() if len(scopes) == 1 else "mixed") if scopes else "none"

        step_sample = {
            "case_id": case_id,
            "t": t,
            "type_token": domain_key,
            "state": copy.deepcopy(state),
            "alive": dict(alive),
            "allowed_tools": allowed,
            "tool_labels": list(labels),
            "called_tools": list(calls),
            "label_scope": label_scope,
            "observations": copy.deepcopy(obs),
            "target": world["target"],
            "eliminated_by": copy.deepcopy(eliminated_by),
            "labels": copy.deepcopy(step_labels),
        }
        validate_sample(
            step_sample,
            registry,
            budgets=budgets,
            scope_schedule=scope_schedule,
            scope_fallback=scope_fallback,
        )
        samples.append(step_sample)

        # Update state and PoE mask for next round
        state[f"round_{t}"] = obs
        called_tools.extend(calls)
        alive, eliminated_by = registry.falsify(domain_key, state, current_alive=alive)

    if not terminal_recorded:
        # All 3 rounds called tools; record final terminal sample at round 3
        terminal_allowed = [
            tool
            for tool, ok in zip(registry.tools, registry.tool_mask(domain_key, alive, called_tools))
            if ok
        ]
        alive_list = [k for k, v in alive.items() if v]
        s_star = generate_sufficiency_label(alive_list, world["target"])
        r_star = case_risk(domain, world["target"])
        gain_dict = generate_tool_gain_labels(
            alive_list,
            tool_specs_list,
            all_observations,
            falsification_rules,
            mask_unobserved=True,
        )
        stall_star = generate_stall_label(s_star, gain_dict, allowed_tools=terminal_allowed)
        redundancy_mask = generate_redundancy_mask(registry.tools, called_tools, alive_list, tool_hyp_map)
        hyp_true = {h_id: (1.0 if alive.get(h_id, True) else 0.0) for h_id in alive}

        term_labels = {
            "s_star": float(s_star),
            "r_star": float(r_star),
            "gain_star": {tool: gain_dict.get(tool, -100.0) for tool in registry.tools},
            "stall_star": float(stall_star),
            "redundancy_mask": redundancy_mask,
            "hyp_true": hyp_true,
            "hyp_risk": hypothesis_risks(domain),
        }
        terminal_sample = {
            "case_id": case_id,
            "t": 3,
            "type_token": domain_key,
            "state": copy.deepcopy(state),
            "alive": dict(alive),
            "allowed_tools": terminal_allowed,
            "tool_labels": [],  # Stop calling tools
            "label_scope": "terminal",
            "observations": {},
            "target": world["target"],
            "eliminated_by": copy.deepcopy(eliminated_by),
            "labels": term_labels,
        }
        validate_sample(
            terminal_sample,
            registry,
            budgets=budgets,
            scope_schedule=scope_schedule,
            scope_fallback=scope_fallback,
        )
        samples.append(terminal_sample)

    return samples


def generate(
    registry: Registry,
    n_cases: int,
    rng_or_seed: Union[random.Random, int] = 42,
    budgets: Tuple[int, int, int] = (3, 2, 1),
    scope_schedule: Sequence[str] = ("macro", "micro"),
    novel_ratio: float = 0.12,
    p_falsify: float = 0.85,
    scope_fallback: bool = True,
    privileged_teacher: bool = True,
) -> List[Dict[str, Any]]:
    """Generate multi-turn trajectory samples across all registered domains."""
    rng = rng_or_seed if isinstance(rng_or_seed, random.Random) else random.Random(rng_or_seed)
    domain_keys = sorted(registry.domains.keys())

    all_samples: List[Dict[str, Any]] = []
    for case_idx in range(n_cases):
        domain_key = domain_keys[case_idx % len(domain_keys)]
        case_id = f"case-{case_idx + 1:06d}"
        case_samples = rollout(
            registry=registry,
            domain_key=domain_key,
            rng=rng,
            case_id=case_id,
            budgets=budgets,
            scope_schedule=scope_schedule,
            novel_ratio=novel_ratio,
            p_falsify=p_falsify,
            scope_fallback=scope_fallback,
            privileged_teacher=privileged_teacher,
        )
        all_samples.extend(case_samples)

    return all_samples


def split_by_case(
    samples: List[Dict[str, Any]],
    val_ratio: float = 0.1,
    seed: int = 42,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Split trajectory dataset into train and validation sets strictly by case_id to prevent leakage."""
    case_ids = sorted(list({s["case_id"] for s in samples}))
    rng = random.Random(seed)
    rng.shuffle(case_ids)

    n_val = max(1, int(len(case_ids) * val_ratio))
    val_cases = set(case_ids[:n_val])

    train_samples = [s for s in samples if s["case_id"] not in val_cases]
    val_samples = [s for s in samples if s["case_id"] in val_cases]

    return train_samples, val_samples


def stats(samples: List[Dict[str, Any]], registry: Registry) -> Dict[str, Any]:
    """Calculate comprehensive dataset statistics and metrics."""
    cases = set(s["case_id"] for s in samples)
    n_cases = len(cases)
    n_samples = len(samples)

    novel_cases = set(s["case_id"] for s in samples if s["target"] == "H_novel")
    novel_ratio = len(novel_cases) / max(1, n_cases)

    # Tool counts per round
    round_tools: Dict[int, List[int]] = {1: [], 2: [], 3: []}
    scope_counts: Dict[int, Dict[str, int]] = {1: {}, 2: {}, 3: {}}

    for s in samples:
        t = s["t"]
        if t in round_tools:
            labels = s["tool_labels"]
            round_tools[t].append(len(labels))
            for lbl in labels:
                sc = registry.tool_specs[lbl].scope
                scope_counts[t][sc] = scope_counts[t].get(sc, 0) + 1

    avg_tools_round = {
        r: (sum(counts) / len(counts) if counts else 0.0)
        for r, counts in round_tools.items()
    }

    # Verify OOD PoE elimination: in novel cases, how many terminal samples have all known hypotheses eliminated?
    novel_terminal_samples = [
        s for s in samples
        if s["case_id"] in novel_cases and s["tool_labels"] == []
    ]
    novel_all_known_eliminated = 0
    for s in novel_terminal_samples:
        d = registry.get_domain(s["type_token"])
        known_ids = [h.id for h in d.hypotheses if not h.is_residual]
        if all(not s["alive"].get(h_id, True) for h_id in known_ids):
            novel_all_known_eliminated += 1

    elim_rate = novel_all_known_eliminated / max(1, len(novel_terminal_samples))

    return {
        "n_cases": n_cases,
        "n_samples": n_samples,
        "novel_ratio": round(novel_ratio, 4),
        "avg_tools_per_round": {f"round_{r}": round(v, 2) for r, v in avg_tools_round.items()},
        "scope_distribution": scope_counts,
        "novel_known_eliminated_rate": round(elim_rate, 4),
    }


def write_jsonl(path: Union[str, Path], samples: Sequence[Dict[str, Any]]) -> None:
    """Save sample records to a JSON Lines file."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("w", encoding="utf-8") as f:
        for s in samples:
            f.write(json.dumps(s) + "\n")


def read_jsonl(path: Union[str, Path]) -> List[Dict[str, Any]]:
    """Load sample records from a JSON Lines file."""
    p = Path(path)
    records: List[Dict[str, Any]] = []
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


class RexTrajectoryDataset(Dataset):
    """PyTorch Dataset wrapping Rex multi-turn trajectory records with dynamic hypothesis permutation."""

    def __init__(
        self,
        samples: List[Dict[str, Any]],
        tok: Any,
        tools: Sequence[str],
        registry: Optional[Registry] = None,
        max_len: int = 512,
        head_max_len: int = 192,
        shuffle_hypotheses: bool = True,
        truncate_left: bool = True,
        base_seed: int = 42,
    ):
        self.samples = samples
        self.tok = tok
        self.tools = list(tools)
        self.registry = registry
        self.max_len = max_len
        self.head_max_len = head_max_len
        self.shuffle_hypotheses = shuffle_hypotheses
        self.truncate_left = truncate_left
        self.base_seed = base_seed
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        """Update training epoch to generate distinct random hypothesis permutations."""
        self.epoch = epoch

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        sample = self.samples[idx]
        domain_key = sample["type_token"]

        if self.registry is not None:
            domain = self.registry.get_domain(domain_key)
            instruction = domain.instruction
            hypotheses = domain.hypotheses_dict
        else:
            # Reconstruct from sample fields
            instruction = f"diagnose {domain_key}"
            hypotheses = {k: f"hypothesis {k}" for k in sample["alive"].keys()}
            if "H_novel" not in hypotheses:
                hypotheses["H_novel"] = "novel anomaly"

        hyp_keys = list(hypotheses.keys())
        target_name = sample["target"]

        # Option ordering: if shuffle_hypotheses, permute hypotheses order
        if self.shuffle_hypotheses:
            rng = random.Random(self.base_seed + self.epoch * 1000003 + idx)
            perm = list(range(len(hyp_keys)))
            rng.shuffle(perm)
            option_order = perm
            ordered_keys = [hyp_keys[i] for i in perm]
            target_idx = ordered_keys.index(target_name)
        else:
            option_order = None
            target_idx = hyp_keys.index(target_name)

        # Tool mask
        tool_mask = None
        if "allowed_tools" in sample:
            allowed_set = set(sample["allowed_tools"])
            tool_mask = [t in allowed_set for t in self.tools]

        # Format state
        state_str = format_state(sample["state"])

        encoded = encode_rex_input(
            tok=self.tok,
            type_token=domain_key,
            instruction=instruction,
            hypotheses=hypotheses,
            state=state_str,
            alive=sample["alive"],
            max_len=self.max_len,
            head_max_len=self.head_max_len,
            truncate_left=self.truncate_left,
            option_order=option_order,
            tool_mask=tool_mask,
        )

        ordered_keys_for_hyp = ordered_keys if self.shuffle_hypotheses else hyp_keys
        item = build_step_labels(sample, self.tools, ordered_keys_for_hyp, option_order=option_order)
        item.update(
            input_ids=encoded["input_ids"],
            markers=encoded["markers"],
            hyp_alive=encoded["hyp_alive"],
            tool_mask=encoded["tool_mask"],
            target_verdict=target_idx,
        )
        return item


def build_step_labels(
    sample: Dict[str, Any],
    tools: Sequence[str],
    ordered_keys: Sequence[str],
    option_order: Optional[Sequence[int]] = None,
) -> Dict[str, Any]:
    """Supervision vectors of one trajectory step, with hypothesis-indexed fields in `ordered_keys` order.

    Shared by RexTrajectoryDataset (one step per item) and the trajectory trainer (one case per item,
    hypothesis order fixed by the case's round-1 layout).
    """
    # Build multi-hot target vector for tools [N_tools] (float32)
    target_tools_set = set(sample.get("tool_labels", []))
    tool_labels_vec = [1.0 if t in target_tools_set else 0.0 for t in tools]
    is_terminal = sample.get("label_scope") == "terminal" if "label_scope" in sample else not target_tools_set

    # Counterfactual supervision labels (T12, T3)
    labels_info = sample.get("labels", {})
    s_val = float(labels_info.get("s_star", sample.get("s_star", 0.0)))
    r_val = float(labels_info.get("r_star", sample.get("r_star", 0.1)))

    raw_gain = labels_info.get("gain_star", sample.get("gain_star", {}))
    if isinstance(raw_gain, dict):
        gain_vec = [float(raw_gain.get(t, -100.0)) for t in tools]
    elif isinstance(raw_gain, list):
        gain_vec = [float(x) for x in raw_gain]
    else:
        gain_vec = [-100.0] * len(tools)

    red_mask = labels_info.get("redundancy_mask", sample.get("redundancy_mask", [False] * len(tools)))

    raw_hyp = labels_info.get("hyp_true", sample.get("hyp_true", {}))
    if isinstance(raw_hyp, dict):
        hyp_true_vec = [float(raw_hyp.get(k, 1.0)) for k in ordered_keys]
    elif isinstance(raw_hyp, list) and option_order is not None:
        hyp_true_vec = [float(raw_hyp[i]) for i in option_order if i < len(raw_hyp)]
    else:
        hyp_true_vec = [1.0 if sample["alive"].get(k, True) else 0.0 for k in ordered_keys]

    tool_mask = None
    if "allowed_tools" in sample:
        allowed_set = set(sample["allowed_tools"])
        tool_mask = [t in allowed_set for t in tools]

    raw_hyp_risk = labels_info.get("hyp_risk", {})
    hyp_risk_vec = [float(raw_hyp_risk.get(k, -100.0)) for k in ordered_keys]

    return {
        "hyp_risk": hyp_risk_vec,
        "stall_label": float(labels_info.get("stall_star", sample.get("stall_star", 0.0))),
        "tool_labels": tool_labels_vec,
        "target_verdict": list(ordered_keys).index(sample["target"]),
        "round": sample["t"],
        "is_terminal": is_terminal,
        "case_id": sample["case_id"],
        "s_label": s_val,
        "r_label": r_val,
        "gain_labels": gain_vec,
        "redundancy_mask": red_mask,
        "hyp_true": hyp_true_vec,
        "hyp_alive": [bool(sample["alive"].get(k, True)) for k in ordered_keys],
        "tool_mask": tool_mask,
    }


def collate_rex_train(
    batch: Sequence[Dict[str, Any]],
    pad_token_id: int,
) -> Dict[str, torch.Tensor]:
    """Collate trajectory dataset items into padded batch tensors for Rex multi-task training.

    Returns:
    - input_ids: [B, max_len]
    - attention_mask: [B, max_len]
    - marker_pos: [B, max_markers]
    - mask_token_indices: alias for marker_pos (Step 3.4 contract)
    - marker_mask: [B, max_markers] bool
    - hyp_alive: [B, max_markers] bool
    - poe_mask: [B, max_markers] long binary mask (0 or 1, for PoEMaskedCrossEntropyLoss)
    - tool_labels: [B, n_tools] float32 multi-hot target
    - target_verdict: [B] long scalar target index
    - qtype: [B] long
    - round: [B] long
    - is_terminal: [B] bool flag (True for terminal verdict decisions, False for intermediate tool dispatch turns)
    - tool_mask: [B, n_tools] bool (optional)
    - s_labels: [B, 1] float32
    - r_labels: [B, 1] float32
    - gain_labels: [B, n_tools] float32
    - redundancy_masks: [B, n_tools] bool
    - hyp_labels: [B, max_markers] float32
    """
    b = len(batch)
    max_len = max(len(item["input_ids"]) for item in batch)
    max_markers = max(len(item["markers"]) for item in batch)
    n_tools = len(batch[0]["tool_labels"])

    input_ids = torch.full((b, max_len), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((b, max_len), dtype=torch.long)
    marker_pos = torch.zeros((b, max_markers), dtype=torch.long)
    marker_mask = torch.zeros((b, max_markers), dtype=torch.bool)
    hyp_alive = torch.zeros((b, max_markers), dtype=torch.bool)
    tool_labels = torch.zeros((b, n_tools), dtype=torch.float32)
    target_verdict = torch.zeros(b, dtype=torch.long)
    rounds = torch.zeros(b, dtype=torch.long)
    is_terminal = torch.tensor([bool(item.get("is_terminal", False)) for item in batch], dtype=torch.bool)

    s_labels = torch.zeros((b, 1), dtype=torch.float32)
    r_labels = torch.zeros((b, 1), dtype=torch.float32)
    gain_labels = torch.zeros((b, n_tools), dtype=torch.float32)
    redundancy_masks = torch.zeros((b, n_tools), dtype=torch.bool)
    hyp_labels = torch.zeros((b, max_markers), dtype=torch.float32)

    has_tool_mask = any(item.get("tool_mask") is not None for item in batch)
    tool_mask_tensor = torch.ones((b, n_tools), dtype=torch.bool) if has_tool_mask else None

    for i, item in enumerate(batch):
        ids = item["input_ids"]
        seq_len = len(ids)
        input_ids[i, :seq_len] = torch.tensor(ids, dtype=torch.long)
        attention_mask[i, :seq_len] = 1

        markers = item["markers"]
        m_len = len(markers)
        marker_pos[i, :m_len] = torch.tensor(markers, dtype=torch.long)
        marker_mask[i, :m_len] = True

        alive_flags = item.get("hyp_alive", [True] * m_len)
        hyp_alive[i, :m_len] = torch.tensor(alive_flags, dtype=torch.bool)

        tool_labels[i] = torch.tensor(item["tool_labels"], dtype=torch.float32)
        target_verdict[i] = item["target_verdict"]
        rounds[i] = item.get("round", 1)

        s_labels[i, 0] = float(item.get("s_label", 0.0))
        r_labels[i, 0] = float(item.get("r_label", 0.1))

        g_l = item.get("gain_labels", [-100.0] * n_tools)
        gain_labels[i, : len(g_l)] = torch.tensor(g_l, dtype=torch.float32)

        r_m = item.get("redundancy_mask", [False] * n_tools)
        redundancy_masks[i, : len(r_m)] = torch.tensor(r_m, dtype=torch.bool)

        h_t = item.get("hyp_true", [1.0] * m_len)
        hyp_labels[i, : len(h_t)] = torch.tensor(h_t, dtype=torch.float32)

        if tool_mask_tensor is not None and item.get("tool_mask") is not None:
            tm = item["tool_mask"]
            tool_mask_tensor[i, : len(tm)] = torch.tensor(tm, dtype=torch.bool)

    poe_mask = hyp_alive.long()

    result = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "marker_pos": marker_pos,
        "mask_token_indices": marker_pos,  # Step 3.4 spec alias
        "marker_mask": marker_mask,
        "hyp_alive": hyp_alive,
        "poe_mask": poe_mask,  # Step 3.4 spec binary tensor
        "tool_labels": tool_labels,
        "target_verdict": target_verdict,
        "qtype": torch.zeros(b, dtype=torch.long),
        "round": rounds,
        "is_terminal": is_terminal,
        "s_labels": s_labels,
        "r_labels": r_labels,
        "gain_labels": gain_labels,
        "redundancy_masks": redundancy_masks,
        "hyp_labels": hyp_labels,
    }
    if tool_mask_tensor is not None:
        result["tool_mask"] = tool_mask_tensor

    return result


In [ ]:
%%writefile {REXLIB}/rex/rex_train.py
"""Multi-task trajectory trainer for the Rex decision heads (T4, T10).

Each training item is one whole investigation case. Round 1 runs `forward(state=None)` on the
case's opening layout; every later round runs `forward(state, evidence_new=...)` on the gold
observations of the previous round, exactly like InvestigatorAgent / RexAdapter at inference.
Losses of every round are summed, so memory append, KV-cache extension and the thought state h
carried across rounds are all trained.

Stages (freeze schedule):
    heads  - train head_s, head_r, tool_matcher, tool_gain, hyp_scorer, alive_count_emb; the rest is frozen.
    ponder - additionally train ponder_block and verdict_bridge.
    full   - train every parameter; the encoder uses its own (smaller) learning rate.
"""
from __future__ import annotations

from collections import OrderedDict
import dataclasses
from dataclasses import dataclass, field
import gc
import math
import random
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import torch
import torch.nn as nn

from .calibrate import MIN_TYPE_N, fit_one_temperature
from .losses import RexLossWeights, RexMultiTaskInputs, RexMultiTaskLoss
from .rex_heads import compute_ece
from .rex_data import build_step_labels, collate_rex_train, format_state
from .rex_format import encode_evidence, encode_rex_input, format_observations
from .rex_tools import DEFAULT_R_THRESHOLD, DEFAULT_S_THRESHOLD, gain_threshold
from .rex_types import RexState

STAGES: Tuple[str, ...] = ("heads", "ponder", "full")
STAGE_MODULES: Dict[str, Tuple[str, ...]] = {
    "heads": ("head_s", "head_r", "tool_matcher", "tool_gain", "hyp_scorer", "alive_count_emb"),
    "ponder": ("head_s", "head_r", "tool_matcher", "tool_gain", "hyp_scorer", "alive_count_emb",
               "ponder_block", "verdict_bridge"),
}
MIN_CONFORMAL_CASES: int = 20


@dataclass
class RexTrainConfig:
    """Hyper-parameters of a Rex trajectory training run."""

    stage: str = "heads"
    epochs: int = 3
    batch_cases: int = 8
    grad_accum: int = 1
    lr_heads: float = 1e-4
    lr_encoder: float = 2.5e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_grad_norm: float = 1.0
    max_len: int = 512
    head_max_len: int = 192
    evidence_max_tokens: int = 64
    shuffle_hypotheses: bool = True
    detach_between_rounds: bool = True
    amp: bool = False
    seed: int = 42
    log_every: int = 10
    loss_weights: RexLossWeights = field(default_factory=RexLossWeights)

    def validate(self) -> None:
        if self.stage not in STAGES:
            raise ValueError(f"stage must be one of {STAGES}, got {self.stage!r}")
        for name in ("epochs", "batch_cases", "grad_accum", "evidence_max_tokens"):
            if getattr(self, name) < 1:
                raise ValueError(f"{name} must be >= 1, got {getattr(self, name)}")
        if not 0.0 <= self.warmup_ratio < 1.0:
            raise ValueError(f"warmup_ratio must be in [0, 1), got {self.warmup_ratio}")


@dataclass
class EncodedCase:
    """One case encoded for trajectory training: round-1 inputs plus per-round labels and evidence."""

    case_id: str
    round1: Dict[str, Any]
    rounds: List[Dict[str, Any]]
    evidence_ids: List[List[int]]
    evidence_truncated: List[bool]


def group_cases(samples: Sequence[Dict[str, Any]]) -> List[List[Dict[str, Any]]]:
    """Group step samples by case_id, keeping generation order inside each case."""
    cases: "OrderedDict[str, List[Dict[str, Any]]]" = OrderedDict()
    for s in samples:
        cases.setdefault(s["case_id"], []).append(s)
    return list(cases.values())


def encode_case(
    case: Sequence[Dict[str, Any]],
    tok: Any,
    tools: Sequence[str],
    registry: Any,
    rng: Optional[random.Random] = None,
    max_len: int = 512,
    head_max_len: int = 192,
    evidence_max_tokens: int = 64,
) -> EncodedCase:
    """Encode one case: the round-1 layout fixes the hypothesis order used by every round's labels."""
    first = case[0]
    domain = registry.get_domain(first["type_token"])
    hypotheses = domain.hypotheses_dict
    hyp_keys = list(hypotheses.keys())

    option_order = None
    ordered_keys = hyp_keys
    if rng is not None:
        option_order = list(range(len(hyp_keys)))
        rng.shuffle(option_order)
        ordered_keys = [hyp_keys[i] for i in option_order]

    round_labels = [build_step_labels(s, tools, ordered_keys, option_order=option_order) for s in case]

    encoded = encode_rex_input(
        tok=tok,
        type_token=first["type_token"],
        instruction=domain.instruction,
        hypotheses=hypotheses,
        state=format_state(first["state"]),
        alive=first["alive"],
        max_len=max_len,
        head_max_len=head_max_len,
        option_order=option_order,
        tool_mask=round_labels[0]["tool_mask"],
    )
    round1 = dict(round_labels[0])
    round1.update(input_ids=encoded["input_ids"], markers=encoded["markers"], hyp_alive=encoded["hyp_alive"])

    # Evidence entering round k (k >= 2) is the observation set gathered at step k - 1
    filler = _filler_token(tok)
    evidence_ids: List[List[int]] = []
    evidence_truncated: List[bool] = []
    for prev in case[:-1]:
        ev = encode_evidence(tok, format_observations(prev.get("observations", {})), max_tokens=evidence_max_tokens)
        evidence_ids.append(list(ev["input_ids"]) or [filler])
        evidence_truncated.append(bool(ev["is_truncated"]))

    return EncodedCase(
        case_id=first["case_id"],
        round1=round1,
        rounds=round_labels,
        evidence_ids=evidence_ids,
        evidence_truncated=evidence_truncated,
    )


def _filler_token(tok: Any) -> int:
    for name in ("sep_token_id", "eos_token_id", "pad_token_id"):
        tid = getattr(tok, name, None)
        if tid is not None:
            return int(tid)
    return 0


def _pad_id(tok: Any) -> int:
    return getattr(tok, "pad_token_id", 0) or 0


def _stack_round(rounds: Sequence[Dict[str, Any]], n_markers: int, n_tools: int) -> Dict[str, torch.Tensor]:
    """Label tensors for one round across a batch of cases (rows of finished cases are placeholders)."""
    b = len(rounds)
    out = {
        "tool_labels": torch.zeros((b, n_tools)),
        "gain_labels": torch.full((b, n_tools), -100.0),
        "redundancy_masks": torch.zeros((b, n_tools), dtype=torch.bool),
        "tool_mask": torch.ones((b, n_tools), dtype=torch.bool),
        "hyp_alive": torch.zeros((b, n_markers), dtype=torch.bool),
        "hyp_labels": torch.zeros((b, n_markers)),
        "hyp_risk_labels": torch.full((b, n_markers), -100.0),
        "target_verdict": torch.zeros(b, dtype=torch.long),
        "s_labels": torch.zeros((b, 1)),
        "r_labels": torch.zeros((b, 1)),
        "stall_labels": torch.zeros(b),
    }
    for i, r in enumerate(rounds):
        k = len(r["hyp_alive"])
        out["tool_labels"][i] = torch.tensor(r["tool_labels"])
        out["gain_labels"][i] = torch.tensor(r["gain_labels"])
        out["redundancy_masks"][i] = torch.tensor(r["redundancy_mask"], dtype=torch.bool)
        if r["tool_mask"] is not None:
            out["tool_mask"][i] = torch.tensor(r["tool_mask"], dtype=torch.bool)
        out["hyp_alive"][i, :k] = torch.tensor(r["hyp_alive"], dtype=torch.bool)
        out["hyp_labels"][i, :k] = torch.tensor(r["hyp_true"])
        if r.get("hyp_risk"):
            out["hyp_risk_labels"][i, :k] = torch.tensor(r["hyp_risk"])
        out["target_verdict"][i] = r["target_verdict"]
        out["s_labels"][i, 0] = r["s_label"]
        out["r_labels"][i, 0] = r["r_label"]
        out["stall_labels"][i] = r.get("stall_label", 0.0)
    # A finished case keeps one live slot so masked softmax/CE stay finite; its loss is dropped anyway
    dead = ~out["hyp_alive"].any(dim=1)
    out["hyp_alive"][dead, 0] = True
    return out


def _detach_state(state: RexState) -> RexState:
    kv = state.kv_cache
    return dataclasses.replace(
        state,
        h=state.h.detach(),
        M=state.M.detach(),
        hyp_markers=state.hyp_markers.detach() if state.hyp_markers is not None else None,
        kv_cache=(kv[0].detach(), kv[1].detach()) if kv is not None else None,
    )


def _step_loss(
    loss_fn: RexMultiTaskLoss,
    model: nn.Module,
    verdict_logits: torch.Tensor,
    out: Any,
    labels: Dict[str, torch.Tensor],
    marker_mask: torch.Tensor,
    rows: torch.Tensor,
):
    """Multi-task loss restricted to the batch rows whose trajectory is still running."""
    def pick(x: Optional[torch.Tensor]) -> Optional[torch.Tensor]:
        return None if x is None else x.index_select(0, rows)

    return loss_fn(RexMultiTaskInputs(
        tool_logits=pick(out.tool_logits),
        tool_labels=pick(labels["tool_labels"]),
        verdict_logits=pick(verdict_logits),
        poe_mask=pick(labels["hyp_alive"].long()),
        target_verdict=pick(labels["target_verdict"]),
        tool_projection_weight=model.tool_matcher.tool_projection_weight,
        s_logits=pick(out.s_logit),
        s_targets=pick(labels["s_labels"]),
        r_logits=pick(out.r_logit),
        r_targets=pick(labels["r_labels"]),
        gain_logits=pick(out.tool_gain_logits),
        gain_targets=pick(labels["gain_labels"]),
        redundancy_mask=pick(labels["redundancy_masks"]),
        hyp_logits=pick(out.hyp_logits),
        hyp_targets=pick(labels["hyp_labels"]),
        hyp_mask=pick(marker_mask),
        hyp_risk_logits=pick(out.hyp_risk_logits),
        hyp_risk_targets=pick(labels["hyp_risk_labels"]),
    ))


def run_cases(
    model: nn.Module,
    cases: Sequence[EncodedCase],
    tok: Any,
    loss_fn: RexMultiTaskLoss,
    device: torch.device,
    detach_between_rounds: bool = True,
    autocast: Optional[Callable[[], Any]] = None,
    collect: Optional[List[Dict[str, torch.Tensor]]] = None,
) -> Tuple[torch.Tensor, Dict[str, float], Dict[str, torch.Tensor]]:
    """Forward a batch of cases round by round.

    Returns (mean loss over executed case-rounds, per-component sums for logging,
    final-round verdict probabilities and targets per case for calibration). When `collect` is a
    list, the head outputs and labels of every executed case-round are appended to it (evaluation).
    """
    n_tools = len(model.tools)
    batch1 = collate_rex_train([c.round1 for c in cases], _pad_id(tok))
    marker_mask = batch1["marker_mask"].to(device)
    n_markers = marker_mask.size(1)
    n_rounds = max(len(c.rounds) for c in cases)
    ctx = autocast if autocast is not None else _null_ctx

    total = torch.zeros((), device=device)
    n_steps = 0
    parts: Dict[str, float] = {}
    collect_final = collect is not None
    final_probs = torch.zeros((len(cases), n_markers)) if collect_final else None
    final_logits = torch.zeros((len(cases), n_markers)) if collect_final else None
    final_targets = torch.zeros(len(cases), dtype=torch.long) if collect_final else None
    state: Optional[RexState] = None

    for k in range(n_rounds):
        active = [i for i, c in enumerate(cases) if k < len(c.rounds)]
        rows_k = [c.rounds[k] if k < len(c.rounds) else c.rounds[-1] for c in cases]
        labels = {n: t.to(device) for n, t in _stack_round(rows_k, n_markers, n_tools).items()}

        with ctx():
            if k == 0:
                verdict_logits, state, out = model(
                    input_ids=batch1["input_ids"].to(device),
                    attention_mask=batch1["attention_mask"].to(device),
                    marker_pos=batch1["marker_pos"].to(device),
                    marker_mask=marker_mask,
                    qtype=batch1["qtype"].to(device),
                    hyp_alive=labels["hyp_alive"],
                    tool_mask=labels["tool_mask"],
                    dispatch=False,
                )
            else:
                if detach_between_rounds:
                    state = _detach_state(state)
                filler = _filler_token(tok)
                evs = [c.evidence_ids[k - 1] if k - 1 < len(c.evidence_ids) else [filler] for c in cases]
                width = max(len(e) for e in evs)
                ev_ids = torch.full((len(cases), width), _pad_id(tok), dtype=torch.long)
                ev_mask = torch.zeros((len(cases), width), dtype=torch.bool)
                for i, e in enumerate(evs):
                    ev_ids[i, : len(e)] = torch.tensor(e, dtype=torch.long)
                    ev_mask[i, : len(e)] = True
                truncated = any(
                    c.evidence_truncated[k - 1] for c in cases if k - 1 < len(c.evidence_truncated)
                )
                verdict_logits, state, out = model(
                    state=state,
                    evidence_new=ev_ids.to(device),
                    evidence_mask=ev_mask.to(device),
                    hyp_alive=labels["hyp_alive"],
                    tool_mask=labels["tool_mask"],
                    is_truncated=truncated,
                    dispatch=False,
                )

        rows = torch.tensor(active, dtype=torch.long, device=device)
        lb = _step_loss(loss_fn, model, verdict_logits, out, labels, marker_mask, rows)
        total = total + lb.total_loss * len(active)
        n_steps += len(active)
        for name in ("tool_loss", "verdict_loss", "ortho_loss", "s_loss", "r_loss", "gain_loss", "hyp_loss",
                     "hyp_risk_loss"):
            v = getattr(lb, name)
            if v is not None:
                parts[name] = parts.get(name, 0.0) + float(v.detach()) * len(active)

        if collect is not None:
            sel = rows.cpu()
            collect.append({
                "round": torch.full((len(active),), k),
                "s_prob": out.s_prob.detach().float().cpu().view(-1)[sel],
                "s_label": labels["s_labels"].cpu().view(-1)[sel],
                "r_prob": out.r_prob.detach().float().cpu().view(-1)[sel],
                "r_label": labels["r_labels"].cpu().view(-1)[sel],
                "gain": out.tool_gains.detach().float().cpu()[sel],
                "gain_label": labels["gain_labels"].cpu()[sel],
                "callable": (labels["tool_mask"] & ~labels["redundancy_masks"]).cpu()[sel],
                "hyp": out.hyp_scores.detach().float().cpu()[sel][marker_mask.cpu()[sel]],
                "hyp_label": labels["hyp_labels"].cpu()[sel][marker_mask.cpu()[sel]],
                "stall_label": labels["stall_labels"].cpu()[sel],
            })

        if collect_final:
            logits_k = verdict_logits.detach().float().cpu()
            probs = out.verdict_probs.detach().float().cpu()
            for i in active:
                if k == len(cases[i].rounds) - 1:
                    final_probs[i] = probs[i]
                    final_logits[i] = logits_k[i]
                    final_targets[i] = int(labels["target_verdict"][i])

    parts = {name: v / max(1, n_steps) for name, v in parts.items()}
    extra = {"probs": final_probs, "logits": final_logits, "targets": final_targets, "marker_mask": marker_mask.cpu()} if collect_final else {}
    return total / max(1, n_steps), parts, extra


class _null_ctx:
    def __enter__(self):
        return None

    def __exit__(self, *exc):
        return False


def configure_stage(model: nn.Module, stage: str) -> None:
    """Freeze/unfreeze parameters for a training stage."""
    if stage == "full":
        for p in model.parameters():
            p.requires_grad_(True)
        return
    trainable = STAGE_MODULES[stage]
    for name, p in model.named_parameters():
        p.requires_grad_(name.split(".", 1)[0] in trainable)


def _set_train_mode(model: nn.Module, stage: str) -> None:
    model.train()
    if stage != "full":
        # Frozen modules run deterministically (no dropout) while the trained heads see dropout
        trainable = set(STAGE_MODULES[stage])
        for name, child in model.named_children():
            if name not in trainable:
                child.eval()


def build_optimizer(model: nn.Module, config: RexTrainConfig) -> torch.optim.Optimizer:
    enc_ids = {id(p) for p in model.encoder.parameters()}
    enc = [p for p in model.parameters() if p.requires_grad and id(p) in enc_ids]
    rest = [p for p in model.parameters() if p.requires_grad and id(p) not in enc_ids]
    groups = [{"params": rest, "lr": config.lr_heads}]
    if enc:
        groups.append({"params": enc, "lr": config.lr_encoder})
    return torch.optim.AdamW(groups, weight_decay=config.weight_decay)


def _lr_lambda(total_steps: int, warmup_steps: int) -> Callable[[int], float]:
    def fn(step: int) -> float:
        if warmup_steps > 0 and step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return fn


def grad_norms_by_module(model: nn.Module) -> Dict[str, float]:
    """L2 gradient norm per top-level module (0.0 when no parameter of it received a gradient)."""
    norms: Dict[str, float] = {}
    for name, p in model.named_parameters():
        top = name.split(".", 1)[0]
        g = 0.0 if p.grad is None else float(p.grad.detach().float().pow(2).sum())
        norms[top] = norms.get(top, 0.0) + g
    return {k: math.sqrt(v) for k, v in norms.items()}


def _autocast_factory(device: torch.device, amp: bool):
    if not amp:
        return None, None
    dev_type = device.type
    if dev_type == "cuda":
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        if dtype == torch.float16:
            scaler = getattr(torch.amp, "GradScaler", torch.cuda.amp.GradScaler)("cuda")
        else:
            scaler = None
        return (lambda: torch.autocast("cuda", dtype=dtype)), scaler
    elif dev_type == "cpu":
        return (lambda: torch.autocast("cpu", dtype=torch.bfloat16)), None
    elif dev_type == "mps":
        return (lambda: torch.autocast("mps", dtype=torch.float16)), None
    return None, None


def auroc(scores: torch.Tensor, labels: torch.Tensor) -> float:
    """Area under the ROC curve (Mann-Whitney, ties count half); NaN when only one class is present."""
    scores, labels = scores.float().view(-1), labels.view(-1) > 0.5
    n_pos, n_neg = int(labels.sum()), int((~labels).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    order = torch.argsort(scores)
    ranks = torch.empty_like(scores)
    ranks[order] = torch.arange(1, len(scores) + 1, dtype=scores.dtype)
    for v in torch.unique(scores):  # average ranks of tied scores
        tie = scores == v
        if int(tie.sum()) > 1:
            ranks[tie] = ranks[tie].mean()
    return float((ranks[labels].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def head_metrics(steps: Sequence[Dict[str, torch.Tensor]], tool_levels: Sequence[str],
                 tool_costs: Sequence[float]) -> Dict[str, float]:
    """Quality of the S, R, gain and hypothesis heads and of the derived STALL decision."""
    if not steps:
        return {}
    cat = {k: torch.cat([s[k] for s in steps]) for k in steps[0]}
    m: Dict[str, float] = {}

    s_prob, s_label, rnd = cat["s_prob"], cat["s_label"], cat["round"]
    m["s_auroc"] = auroc(s_prob, s_label)
    m["s_ece"] = compute_ece(s_prob, s_label)
    # Within-round AUROC: a head that only counts rounds scores 0.5 here even if its pooled AUROC is high
    per_round = [(auroc(s_prob[rnd == k], s_label[rnd == k]), int((rnd == k).sum())) for k in torch.unique(rnd)]
    per_round = [(a, n) for a, n in per_round if a == a]
    m["s_auroc_within_round"] = (sum(a * n for a, n in per_round) / sum(n for _, n in per_round)
                                 if per_round else float("nan"))
    for k in torch.unique(rnd).tolist():
        m[f"s_mean_round_{int(k) + 1}"] = float(s_prob[rnd == k].mean())

    r_prob, r_label = cat["r_prob"], cat["r_label"]
    m["r_mae"] = float((r_prob - r_label).abs().mean())
    m["r_auroc_high_risk"] = auroc(r_prob, (r_label > DEFAULT_R_THRESHOLD).float())

    gain, g_label, callable_ = cat["gain"], cat["gain_label"], cat["callable"]
    valid = callable_ & (g_label != -100)
    m["gain_auroc"] = auroc(gain[valid], g_label[valid])

    m["hyp_auroc"] = auroc(cat["hyp"], cat["hyp_label"])

    # STALL as the dispatcher derives it: not sufficient and no callable tool reaches its threshold
    tau = torch.tensor([gain_threshold(lv, c) for lv, c in zip(tool_levels, tool_costs)])
    any_tool = ((gain >= tau) & callable_).any(dim=1)
    pred_stall = (s_prob < DEFAULT_S_THRESHOLD) & ~any_tool
    pred_stall_high_risk = (s_prob >= DEFAULT_S_THRESHOLD) & (r_prob > DEFAULT_R_THRESHOLD)
    true_stall = cat["stall_label"] > 0.5
    m["stall_rate_label"] = float(true_stall.float().mean())
    m["stall_rate_pred"] = float(pred_stall.float().mean())
    m["stall_high_risk_rate_pred"] = float(pred_stall_high_risk.float().mean())
    m["stall_recall"] = float((pred_stall & true_stall).sum() / true_stall.sum().clamp(min=1))
    m["stall_precision"] = float((pred_stall & true_stall).sum() / pred_stall.sum().clamp(min=1))
    return m


def evaluate(
    model: nn.Module,
    cases: Sequence[EncodedCase],
    tok: Any,
    config: RexTrainConfig,
    device: torch.device,
) -> Dict[str, Any]:
    """Validation loss, final-round verdict accuracy, head metrics (see head_metrics) and conformal coverage.

    Also returns the final-round verdict probabilities/targets (conformal calibration) and
    per-case (live logits, one-hot target) pairs (temperature fitting).
    """
    loss_fn = RexMultiTaskLoss(weights=config.loss_weights)
    model.eval()
    probs_all, targets_all = [], []
    pairs: List[Tuple[torch.Tensor, torch.Tensor]] = []
    steps: List[Dict[str, torch.Tensor]] = []
    total, n = 0.0, 0
    with torch.no_grad():
        for start in range(0, len(cases), config.batch_cases):
            chunk = cases[start:start + config.batch_cases]
            loss, _, final = run_cases(model, chunk, tok, loss_fn, device, detach_between_rounds=True,
                                       collect=steps)
            total += float(loss) * len(chunk)
            n += len(chunk)
            probs_all.append(final["probs"])
            targets_all.append(final["targets"])
            for i in range(len(chunk)):
                live = final["marker_mask"][i] & (final["logits"][i] > -1e3)
                idx = live.nonzero(as_tuple=True)[0]
                target = int(final["targets"][i])
                if target in idx.tolist():
                    onehot = (idx == target).float()
                    pairs.append((final["logits"][i, idx], onehot))
    if not probs_all:
        return {"loss": float("nan"), "verdict_acc": float("nan"), "probs": None, "targets": None, "pairs": []}
    width = max(p.size(1) for p in probs_all)
    probs = torch.cat([torch.nn.functional.pad(p, (0, width - p.size(1))) for p in probs_all])
    targets = torch.cat(targets_all)
    acc = float((probs.argmax(dim=-1) == targets).float().mean())
    levels = [model.tool_meta.get(t, {}).get("level", "READ") for t in model.tools]
    costs = [float(model.tool_meta.get(t, {}).get("cost", 0.0)) for t in model.tools]
    metrics = head_metrics(steps, levels, costs)
    sets = model.calibrator.predict_set(probs, q_hat=model.conformal_threshold())
    names = [f"H_{i}" for i in range(probs.size(1))]
    live = (probs > 0).sum(dim=1).clamp(min=1).float()
    metrics["conformal_coverage"] = float(sum(names[int(t)] in st for st, t in zip(sets, targets)) / len(sets))
    metrics["conformal_mean_set_size"] = float(sum(len(st) for st in sets) / len(sets))
    metrics["conformal_mean_live_hypotheses"] = float(live.mean())
    return {"loss": total / max(1, n), "verdict_acc": acc, "probs": probs, "targets": targets, "pairs": pairs,
            "metrics": metrics}


def train_rex(
    model: nn.Module,
    tok: Any,
    registry: Any,
    train_samples: Sequence[Dict[str, Any]],
    val_samples: Optional[Sequence[Dict[str, Any]]] = None,
    config: Optional[RexTrainConfig] = None,
    calib_samples: Optional[Sequence[Dict[str, Any]]] = None,
    device: Optional[torch.device] = None,
    encode_tools: bool = True,
    log: Callable[[str], None] = print,
) -> Dict[str, Any]:
    """Train a RexDecisionModel in place on trajectory samples produced by rex_data.generate().

    Encodes the registry tool descriptions into the model (T5) unless `encode_tools=False`,
    marks the gain head trained when the gain loss is active, and fits the verdict temperature and
    the conformal threshold on `calib_samples`. Without them, the validation cases are split by case:
    half calibrates, the other half is the reported evaluation, so no metric is measured on the cases
    that fitted the calibration. Returns a summary with per-epoch losses and val metrics.
    """
    config = config or RexTrainConfig()
    config.validate()
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    random.seed(config.seed)
    torch.manual_seed(config.seed)
    model.to(device)

    registry.check_model_tools(model.tools)
    model.set_tool_specs([registry.tool_specs[t] for t in model.tools])
    if encode_tools:
        model.encode_tool_descriptions([registry.tool_specs[t] for t in model.tools], tok)

    train_cases = group_cases(train_samples)
    val_cases_raw = group_cases(val_samples or [])
    if not train_cases:
        raise ValueError("train_samples is empty")

    def encode_all(cases, epoch: int, shuffle: bool) -> List[EncodedCase]:
        out = []
        for idx, case in enumerate(cases):
            rng = random.Random(config.seed + epoch * 1000003 + idx) if shuffle else None
            out.append(encode_case(case, tok, model.tools, registry, rng=rng, max_len=config.max_len,
                                   head_max_len=config.head_max_len,
                                   evidence_max_tokens=config.evidence_max_tokens))
        return out

    calib_cases_raw = group_cases(calib_samples or [])
    if not calib_cases_raw and len(val_cases_raw) >= 2:
        order = list(range(len(val_cases_raw)))
        random.Random(config.seed).shuffle(order)
        half = len(order) // 2
        calib_cases_raw = [val_cases_raw[i] for i in sorted(order[:half])]
        val_cases_raw = [val_cases_raw[i] for i in sorted(order[half:])]
    val_cases = encode_all(val_cases_raw, 0, shuffle=False)
    calib_cases = encode_all(calib_cases_raw, 0, shuffle=False)

    configure_stage(model, config.stage)
    optimizer = build_optimizer(model, config)
    micro_steps = math.ceil(len(train_cases) / config.batch_cases)
    total_updates = math.ceil(micro_steps / config.grad_accum) * config.epochs
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, _lr_lambda(total_updates, int(total_updates * config.warmup_ratio))
    )
    autocast, scaler = _autocast_factory(device, config.amp)
    loss_fn = RexMultiTaskLoss(weights=config.loss_weights)
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"rex train: stage={config.stage} cases={len(train_cases)} calib_cases={len(calib_cases)} "
        f"val_cases={len(val_cases)} "
        f"trainable_params={n_trainable:,} updates={total_updates}")

    epoch_losses: List[float] = []
    first_grad_norms: Optional[Dict[str, float]] = None
    update = 0
    for epoch in range(config.epochs):
        _set_train_mode(model, config.stage)
        encoded = encode_all(train_cases, epoch, shuffle=config.shuffle_hypotheses)
        order = list(range(len(encoded)))
        random.Random(config.seed + epoch).shuffle(order)
        running, seen = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for step in range(micro_steps):
            chunk = [encoded[i] for i in order[step * config.batch_cases:(step + 1) * config.batch_cases]]
            loss, parts, _ = run_cases(model, chunk, tok, loss_fn, device,
                                       detach_between_rounds=config.detach_between_rounds, autocast=autocast)
            scaled = loss / config.grad_accum
            if scaler is not None:
                scaler.scale(scaled).backward()
            else:
                scaled.backward()
            running += float(loss.detach()) * len(chunk)
            seen += len(chunk)

            last_micro = (step + 1) % config.grad_accum == 0 or step == micro_steps - 1
            if not last_micro:
                continue
            if scaler is not None:
                scaler.unscale_(optimizer)
            if first_grad_norms is None:
                first_grad_norms = grad_norms_by_module(model)
                log("rex train: first-update grad norms " + ", ".join(
                    f"{k}={v:.3g}" for k, v in sorted(first_grad_norms.items())))
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],
                                           config.max_grad_norm)
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            update += 1
            if config.log_every and update % config.log_every == 0:
                log(f"rex train: epoch {epoch + 1} update {update}/{total_updates} loss {running / seen:.4f} "
                    + " ".join(f"{k}={v:.3f}" for k, v in parts.items()))
        del encoded, order
        gc.collect()
        epoch_losses.append(running / max(1, seen))
        log(f"rex train: epoch {epoch + 1} mean loss {epoch_losses[-1]:.4f}")

    if config.loss_weights.gain_weight > 0:
        model.mark_gain_head_trained(True)

    summary: Dict[str, Any] = {"epoch_losses": epoch_losses, "grad_norms": first_grad_norms or {}}
    summary["calib_cases"] = len(calib_cases)
    summary["val_cases"] = len(val_cases)
    if calib_cases:
        calib = evaluate(model, calib_cases, tok, config, device)
        if len(calib["pairs"]) >= MIN_TYPE_N:
            # Verdict temperature first, so the conformal threshold is fitted on calibrated probabilities
            temp = fit_one_temperature(calib["pairs"], min_n=MIN_TYPE_N)
            model.set_verdict_temperature(temp)
            summary["verdict_temperature"] = temp
            log(f"rex train: verdict temperature {temp:.3f} on {len(calib['pairs'])} calibration cases")
            calib = evaluate(model, calib_cases, tok, config, device)
        if len(calib_cases) >= MIN_CONFORMAL_CASES:
            summary["q_hat"] = model.calibrate_conformal(calib["probs"], calib["targets"])
            log(f"rex train: conformal q_hat {summary['q_hat']:.4f} on {len(calib_cases)} calibration cases")
        else:
            log(f"rex train: conformal threshold not fitted: {len(calib_cases)} calibration cases, "
                f"fewer than {MIN_CONFORMAL_CASES}")
    if val_cases:
        metrics = evaluate(model, val_cases, tok, config, device)
        summary["val_loss"] = metrics["loss"]
        summary["val_verdict_acc"] = metrics["verdict_acc"]
        summary["val_metrics"] = metrics["metrics"]
        log(f"rex train: val loss {metrics['loss']:.4f} final-round verdict acc {metrics['verdict_acc']:.3f}")
        log("rex train: val " + " ".join(f"{k}={v:.3f}" for k, v in metrics["metrics"].items()))
    del val_cases, calib_cases, train_cases, optimizer, scheduler, loss_fn
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model.eval()
    return summary

In [ ]:
%%writefile {REXLIB}/rex/rex_agent.py
"""InvestigatorAgent: Autonomous multi-round investigative inquiry orchestrator (Step A10).

This module separates procedural environment loops, tool dispatch, and observation intake
from the neural RexDecisionModel inference architecture.
"""
from __future__ import annotations

import json
from typing import Any, Callable, Dict, List, Optional, Sequence, Union

import torch

from .rex_format import collate_rex, encode_evidence, encode_rex_input, format_observations
from .rex_poe import (
    HardMaskingManager,
    HybridPoEEngine,
    InvestigationResult,
    RoundTrace,
    TerminationReason,
)
from .rex_registry import ToolSpec
from .rex_tools import DEFAULT_R_THRESHOLD, DEFAULT_S_THRESHOLD
from .rex_types import RexState, RexStepOutput


class InvestigatorAgent:
    """Autonomous investigation agent managing multi-round inquiry, tool calls, and Popperian pruning.

    Responsibilities:
        - Orchestrates multi-round agent loop with RexDecisionModel.
        - Translates environment observations into bounded 64-token evidence chunks.
        - Applies Hybrid Popperian elimination (symbolic falsification + neural hypothesis scoring).
        - Enforces early stopping (single survivor, all cataloged eliminated, budget caps).
    """

    def __init__(
        self,
        model: Any,
        theta_elim: float = 0.20,
    ):
        self.model = model
        self.theta_elim = theta_elim
        self.poe_engine = HybridPoEEngine(default_theta_elim=theta_elim)
        self.mask_manager = HardMaskingManager()

    def investigate(
        self,
        tok: Any,
        type_token: str,
        instruction: str,
        hypotheses: Dict[str, str],
        rules: Dict[str, List[Any]],
        tool_hyp_mapping: Dict[str, List[str]],
        tool_dispatcher: Callable[[List[str]], Dict[str, Any]],
        initial_state: Union[str, dict, list] = "",
        max_rounds: int = 3,
        tool_specs: Optional[Sequence[ToolSpec]] = None,
        initial_observations: Optional[Dict[str, Any]] = None,
        stop_on_single_survivor: bool = False,
        **kwargs: Any,
    ) -> InvestigationResult:
        """Execute autonomous multi-round investigation loop.

        Args:
            tok: Tokenizer instance.
            type_token: Question domain type token.
            instruction: Incident prompt / problem description.
            hypotheses: Dictionary mapping hypothesis IDs to textual definitions.
            rules: Falsification rules per hypothesis ID.
            tool_hyp_mapping: Mapping of tool names to supported hypotheses.
            tool_dispatcher: Callback executing approved tool names and returning observations, preferably
                grouped per tool ({tool: {metric: value}}): that renders the same evidence text the
                trainer and Part B use. A flat {metric: value} dict is also accepted.
            initial_state: Initial state or incident metadata string/dict.
            max_rounds: Maximum inquiry budget (rounds).
            tool_specs: Optional rich ToolSpec catalogue.
            initial_observations: Metrics already known before any tool call (the alert). Falsification
                rules are applied to them before round 1, as the training rollouts do. Defaults to
                `initial_state` when that is a dict.
            stop_on_single_survivor: Stop as soon as one cataloged hypothesis survives. When False the
                model decides (VERDICT once S is sufficient), like the Part B orchestrator.

        Returns:
            InvestigationResult with winner hypothesis, probabilities, traces, and escalation status.
        """
        # Inference: no dropout and the fixed ponder depth (train mode samples T in [2, 4])
        self.model.eval()
        hyp_keys = list(hypotheses.keys())
        current_alive = {k: True for k in hyp_keys}
        if initial_observations is None and isinstance(initial_state, dict):
            initial_observations = initial_state
        if initial_observations:
            # Alert metrics can already falsify hypotheses (rollouts apply registry.falsify to the alert)
            current_alive = self.poe_engine.evaluate(
                neural_scores=None,
                hyp_keys=hyp_keys,
                observations=initial_observations,
                rules=rules,
                prev_alive=current_alive,
            )
        state_t = initial_state if isinstance(initial_state, str) else json.dumps(initial_state, ensure_ascii=False)
        called_tools: List[str] = []
        traces: List[RoundTrace] = []
        state: Optional[RexState] = None
        last_step_out: Optional[RexStepOutput] = None
        pending_obs: bool = False
        term_reason = TerminationReason.MAX_ROUNDS_REACHED

        device = self.model.tool_embeddings.device
        general_tools = [
            spec.name for spec in (tool_specs or []) if getattr(spec, "general", False)
        ]
        if not general_tools and hasattr(self.model, "tool_meta"):
            general_tools = [t for t, meta in self.model.tool_meta.items() if meta.get("general", False)]

        for round_idx in range(1, max_rounds + 1):
            # 1. Derive tool availability mask (parity with Registry.tool_mask)
            tool_mask_list = self.mask_manager.compute_tool_mask(
                self.model.tools,
                current_alive,
                tool_hyp_mapping,
                called_tools=called_tools,
                general_tools=general_tools,
                allow_unmapped=False,
            )
            tool_mask_tensor = torch.tensor([tool_mask_list], dtype=torch.bool, device=device)

            alive_mask_list = [current_alive.get(k, True) for k in hyp_keys]
            alive_mask_tensor = torch.tensor([alive_mask_list], dtype=torch.bool, device=device)

            # 2. Forward pass through RexDecisionModel
            if round_idx == 1 or state is None:
                encoded = encode_rex_input(
                    tok,
                    type_token=type_token,
                    instruction=instruction,
                    hypotheses=hypotheses,
                    state=state_t,
                    alive=current_alive,
                    tool_mask=tool_mask_list,
                )
                batch = collate_rex([encoded], pad_token_id=getattr(tok, "pad_token_id", 0) or 0)
                batch_device = {k: v.to(device) for k, v in batch.items()}

                with torch.no_grad():
                    _, state, step_out = self.model.forward(
                        input_ids=batch_device["input_ids"],
                        attention_mask=batch_device["attention_mask"],
                        marker_pos=batch_device["marker_pos"],
                        marker_mask=batch_device["marker_mask"],
                        qtype=batch_device.get("qtype"),
                        tool_mask=tool_mask_tensor,
                        hyp_alive=alive_mask_tensor,
                        tool_specs=tool_specs,
                        hyp_names=hyp_keys,
                    )
            else:
                # Subsequent round: the previous observations are already in memory (step 5)
                with torch.no_grad():
                    _, state, step_out = self.model.forward(
                        state=state,
                        tool_mask=tool_mask_tensor,
                        hyp_alive=alive_mask_tensor,
                        tool_specs=tool_specs,
                        hyp_names=hyp_keys,
                    )
                pending_obs = False

            last_step_out = step_out

            # 3. Action and tool selection check
            selected_tools = step_out.selected_tools[0] if step_out.selected_tools else []
            proposed_action = step_out.proposed_action[0] if step_out.proposed_action else "CALL_TOOL"

            if proposed_action == "STALL":
                term_reason = TerminationReason.STALL
                break

            if proposed_action == "VERDICT" or not selected_tools:
                term_reason = TerminationReason.EMPTY_TOOL_SELECTION
                break

            called_tools.extend(selected_tools)

            # 4. Dispatch selected tools
            observations = tool_dispatcher(selected_tools)
            pending_obs = True
            traces.append(
                RoundTrace(
                    round_idx=round_idx,
                    selected_tools=selected_tools,
                    observations=observations,
                    active_hypotheses=[k for k, v in current_alive.items() if v],
                )
            )

            # 5. Ingest the new observations into memory, then score hypotheses on the updated memory
            ev_encoded = encode_evidence(
                tok, format_observations(observations), round_idx=round_idx + 1, max_tokens=64
            )
            ev_ids = torch.tensor([ev_encoded["input_ids"]], dtype=torch.long, device=device)
            ev_mask = torch.tensor([ev_encoded["attention_mask"]], dtype=torch.bool, device=device)
            with torch.no_grad():
                state = self.model.ingest_evidence(
                    state, ev_ids, ev_mask, is_truncated=bool(ev_encoded["is_truncated"])
                )
                neural_scores = self.model.hyp_scorer(state.hyp_markers, state.M, state.memory_mask)

            # Hybrid PoE Evaluation (symbolic veto + neural score)
            current_alive = self.poe_engine.evaluate(
                neural_scores=neural_scores,
                hyp_keys=hyp_keys,
                observations=observations,
                rules=rules,
                theta_elim=self.theta_elim,
                prev_alive=current_alive,
            )

            # 6. Check termination criteria
            active_count = sum(1 for v in current_alive.values() if v)
            known_active_count = sum(1 for k, v in current_alive.items() if k != "H_novel" and v)

            if known_active_count == 0:
                # All cataloged hypotheses eliminated; only H_novel remains
                term_reason = TerminationReason.ALL_CATALOGED_ELIMINATED
                break

            if active_count == 1 or (stop_on_single_survivor and known_active_count == 1):
                # Exactly one cataloged survivor remains
                term_reason = TerminationReason.SINGLE_SURVIVOR
                break

            if round_idx == max_rounds:
                term_reason = TerminationReason.MAX_ROUNDS_REACHED
                break

        # Final verdict pass over the latest ingested observations (no second ingestion, L5)
        if pending_obs and traces and last_step_out is not None and state is not None:
            alive_mask_list = [current_alive.get(k, True) for k in hyp_keys]
            alive_mask_tensor = torch.tensor([alive_mask_list], dtype=torch.bool, device=device)

            with torch.no_grad():
                _, state, last_step_out = self.model.forward(
                    state=state,
                    hyp_alive=alive_mask_tensor,
                    hyp_names=hyp_keys,
                )
            pending_obs = False

        if last_step_out is None:
            raise RuntimeError("Model failed to execute forward pass during investigation.")

        probs: Dict[str, float] = {}
        for idx, key in enumerate(hyp_keys):
            if not current_alive.get(key, True):
                probs[key] = 0.0
            else:
                probs[key] = float(last_step_out.verdict_probs[0, idx].item())

        total_prob = sum(probs.values())
        if total_prob > 0:
            probs = {k: v / total_prob for k, v in probs.items()}
        else:
            probs = {k: 1.0 / len(probs) for k in probs}

        winner = max(probs.items(), key=lambda kv: kv[1])[0]
        # Mirrors Part B stop.decide: a STALL, or a spent budget without sufficient evidence, goes to a human
        # and so does any automated verdict whose risk is above the threshold (HIGH_RISK)
        final_s = float(last_step_out.s_prob.reshape(-1)[0].item())
        final_r = float(last_step_out.r_prob.reshape(-1)[0].item())
        is_escalated = (
            term_reason in (TerminationReason.ALL_CATALOGED_ELIMINATED, TerminationReason.STALL)
            or (term_reason == TerminationReason.MAX_ROUNDS_REACHED and final_s < DEFAULT_S_THRESHOLD)
            or final_r > DEFAULT_R_THRESHOLD
            or winner == "H_novel"
        )

        return InvestigationResult(
            predicted_hypothesis=winner,
            probabilities=probs,
            rounds_executed=len(traces) if traces else 1,
            termination_reason=term_reason,
            is_escalated=is_escalated,
            traces=traces,
        )


### Import và tải checkpoint nền
`convaiinnovations/laya` (ModernBERT-large), pin theo `rex/revisions.py`.

In [ ]:
# (c) import mã nguồn Rex vừa ghi, giải nén registry RBA v2.0.0 và tải checkpoint nền Laya
import base64, gzip, shutil
sys.path.insert(0, REXLIB)
for _k in [k for k in sys.modules if k == "rex" or k.startswith("rex.")]:
    del sys.modules[_k]
import rex
from rex.rex_model import RexDecisionModel, from_rex, save_rex_checkpoint
from rex.rex_train import RexTrainConfig, train_rex, evaluate, encode_case, group_cases, run_cases, \
    _stack_round, _detach_state, _filler_token, _pad_id
from rex.losses import RexLossWeights, RexMultiTaskLoss
from rex.rex_vocab import add_type_tokens
from rex.train import load_checkpoint
from rex.rex_format import encode_evidence, encode_rex_input, collate_rex
from rex.rex_data import collate_rex_train
from rex.rex_agent import InvestigatorAgent
log(f"  rex {rex.__version__} import OK từ {REXLIB}")

# Giải nén 2 file registry mới: registry_rba.json và registry_rba.yaml
REGISTRY_RBA_JSON_B64 = "H4sIAJak22oC/+19a3PbyLH29/yKOa7ail1LSpRs+RqnipJomWdlUa9IeZOT2kINgSGJFQhgcZHMpDa//e3LDACSIEWJlCXu4uRksyJuMz09fXu6e/7zFyGeXasodgP/2XvxbH+nsdN4VsNfHZnIWCXw63/gT/jBl2OF95wGQ9cXx3BZdFUiBkEkLtz4qn4ItzuimSYj5SeuLRN4p3j+s6sGnusPBdwqvZ2aaB59Eb3OeVfsN/b3X9C34OVp5OG7R0kSxu93d29ubnau5HDoqR07GO/qscTwL/EwCtJwN+rLuhmhfsXYjaIgKr5l6CajtK/fENcXPhkFN7EVRsHA9ZQDL3i5t//63f7rV/qyHXjp2I/hwr/oB/jJ9R31TT8uDEl67ljFiRyH+YXLWEWifZz/cBGkvlPvRW5It4t/jeNf8qvtc9F0nEjFcf7bETyRRJPCK9QQVyu/wU0KV5vds5nPN4ewHqKbRLAK+aVDnDRcPYNVFdJ3xFfNBdkdne6Si8fq2rWV6E1CNUuGbmrbMINB6hUmFotmkkj7SrTPp3+1bZyf6MkrFQAjPqNrv2jKX/nBjW/JKHEH0k7ijBfhUoxM5ivPSmEWlovLVn/1cv/VqzcHB29fvmy8Pni39+7lQW3ufj9IiI33XtUa+we1t+/eCSSFeP7q4IcXNRH43kS8EjFPQsXvRRgEwBZCJokah0kMdwg/8OvqmxsnSFnJU4h3xJc0TkRfCUdF7jU84vpJIIaRDEc7+iaLnoo/DqQXq5rwFUxZJJGSCX4B360E/OUJnNVOTqkoSayxG8ewhFaEOwtm0Nh5d/Ayn9/Eh40H+85yQ0vGtJuBF8TfP4qDBv6fgH0K/AULtNfYof/svhUyUiJ7UoSetNUo8GD88QcBGwQGmNwE0dWOG1v5B8x6fEyilNkjvgJ+hq/tts/rkQrThDc/ztIduPC2wlSGKrBSP1KeK/serYTm8F1m7F3kZ3GjIqIETHnoA21CoFPqu7/BB6+ll6oPAt5Tv1ZeYOPtbizGSvpAHg9WDZfo4rApntNiwqN1JCesGtzPN4ewx1QEa/SiMDLZRyaxEuWpscItl294FC8BvCS7GZlTTeIkCq6mfkyC1B4Vf7Clfy3j3Z9V/+S0+PtIRs4Nkh/5KmaCFa8D6+GO27WD4Mqd+gQKLxsGqqLdo5OzZg8JLxLciPqeX+h/f4d//l7T8t1x7cSyPYkcnUv009ZJu0eM4nnBzQdgXR+oLdQ3YASflzBOo2viZb2WLNYN0Z59umheHuMbDmEhrsSugC0R1tMQ/i1K/T6M/YMYS8+1XaBe6YuzV7W6R83TZq+Fb7tQseuksA0+W7RoyIzAZrCbPkUydUQnjMUNCHfhBEAlWFpkwqkRwqaSrrfzLCPCaBIGwL8xcLKhx1iGOS0+W/0g8Unf6VnVzIV40ne9st9hDUlfaEJmF2jM+Hs2p2wYU4O0wgBIM8kHEauxRN0Z08PSHgk7cmGhcULAtRL+CYonBFqg4Ll2A48nGwyEhL2KEktGE2EHvuPihQ/wyETEcFMM29ApvA21qfhi/So+ghzJNShLGOQwHAFu67JrmQxtCtwoIDmSkUxohLSFQLClnldDidOHHXojxq5vxWkYBlGiZZ4RDSA1bc/FbRc7VxbtVz1IScIJlixSYGIomLmCp3lnZyOONKNYwxT2Ew5JUx8HM/2lnNPsIE5ARE7GZpc/+0T3TNBMMcsqntOivgAh9FvqwndwKMDeMFvcr/AX/IayMq4hvWOtDIgr8U78SB2IEcNCXIPQYp4THjBsgSsHIPnTSFnAD9ILhgVjywje0AKOzWWq5eGHnClN6AXEtd3TTs86a/V+7lz8ZF20zi97zV67U7AG4iCNbFq3cnVMMgSuwrYtqG55DRtJwg5AMwOugv1kTdtPOFzNEGDYDIcgmPRiI0FxTT/otZC5ShDP4aWwTp6SYOipFzssvX6vzc0/BIaNLK2Q4dOT2AKGIL2/Lhkyc6smtKVWEzO2nPmhzKoxBAM9f096HcNsBOwrGzkdWBZGNHAjYK+hvNY/keqSxiCB7wMPwYA+0B4T7oBJu4B8YAhYsVL+piiGqgYGlQQgZUDozA4qXoeZSidgs22w0Uloe+N7TwQ09rfE0kaIRRPbzERqaAFoDi4a++Jac4+eaCnjikb95RrTgv3pwNuBgxNamtjafzXa/LZcY88tGrax5629TY53oSe4sXEP4BFUGeAGKCS2pZ0k7YWsOZdZYQc2m+3tCOM9zc9n4AVyzRml/tQUspmtORXS47S59eANM5EzUab41p5LmZe0vsT9+8fcg8t9t41LKPIx1+Wf5fGN9RXmu4Odlz8IbYyCWzBtZyzQguhyGlfRuhrfj7FWZJIZa3Yho+Re3FqjWb7ms4OZI+eFGgfk3UXBODd9c7/9Pdq24Eqh7RrDv0dAbdid9a/nZwJm4gDhdxZO0R6Cr/e0Z5f9CL8FMD8MhVJ0hz1r7RCHMhnFM/N0KAC2o0daPsXj1tf2Ucv61D47aV2cX7TPemVbpjSUZiig/HT8r3EAk1Y1R8VXSRDWEgycJDXwV2tadNZQk01+ucfm19NI5YoW1mpTmjdFFpog9xRaetx9jmKCUhy73mSjc1gUIBXP+WMkcF5s1Hvq4qf47RgbvIGv69g8uOl6qiIN4XklnjNvvthZtJ6hjGJF1sISd/HOHCqI0za9jDjce3h592TGB7PS9GxsL/CVY5kI372Gfz8JWDYaHUi0BjBzFQGZ4QF7JOEP57FG1lcjeQ3LvZPy3iWDdqkhftj63Pza7lw0T62j5nHr7Ki1kvFKUWPkoA0tcDZwDGi5qLyNGwGWEyMRe43xRmaxCeacD8+Yl8YCPDYvdYrR/19dmMpsIGF6qTJXL/dDNrBsuf8kCDRAibXQDV9z4QYBhrphd/Yn2nEyjrkRPesvYPv8x8smBgF9E1fAeWnvhiOE9Gl02+HOjQnTnDsRqrBg/0VBOLnXZDZh7GbDyYASi1lsmZfx3QZFQI11LSNX+rZ6tPGUoYMLB9M8OupcnvWsk4vm+eeFqg4c3P/5uMRjvyd38UhJDEzr6uWRrJXHvCxIdf/hSsRCLbClsn1uAiXWG2fNQU8ZuD8SDCftKIDtngVjNj2ZYrgtm8iyUMm9ZgIC+WEnMqVLcIHWmgISviyGu3H1z6Nnu6pkHV427sxRK42nXIBkGJIBDa0ENraFXy1AvEVXuzg4W4KX69rSs6Tnynglrxz8Tztyw0TnCrX9OFR2gtxSwPw9IsuwxtsB3CYdwc/Qd1wUTbIa4R3SrwdpYjiOgsg1oa0LeoN2cutGg2IEAm7BSxGFmxIMNyVg2xTA/AT0jW/1U2dImO7e66mVLFi5q1BlmT08Q5UWpiegg4YTqUvaWHrqtcyH0x6kE7kDIIKjbI+CEMxZhOXXYNbkwaGvL+g1esoBfUp6IssguHajJIUB/1uDwZHgpAOBuRPSi5dQ5WCeKkbjrUKXZdpxhi5NGMnk34q3JgzQUaB283Xup1GMMySjuq6NalxT4qVIUWhmIuQgwYSZSRjMUoNsnprIbA26SkpeOBMf6G0vo8OreTrwTl2BCAu29sz8e5FEX15ldqDOH0IiME0MHOWoMBnVivyDU8HtFKmUXkAbBVgFyJNJaCbHOPUSN9sosLvAyIevRCvMPZMnQ9pUVhKlyQhzJELMG8uBae1K/KzklSAIN6YUQMy3wPybXcUbADXJhfpmAHuR7QswHsZgdQH7y0RMpaYIT1GuR56qwb8gKUgVAWFoO2DyQp4CkHqqmJv3nzxThrLCnp3AEvU61mm722sdFxNpbsBCN2j4TBqa+PhRUHZT8+xYFNFyc6H4HtzU8J43bwq/TU1sNneEpXw2n6k8o2IqSm36V85DyX77pfCyMFIDFelslJLnnznBGNaHGPGf5y0LlsQ663xtwb7tnPVa/+hZp52T9pl5+e+1W4h5ebY5clIuXAk9Tf7lBujJlKuVEnkVes4+v0lyMiGBoj+ddX4+Q2myiKizPEjUnHYdltFz7/VB4/X+xpm0lH4l9ClMkMTl+UXnsHUXCn1qtk/XJE12aS5SZOhGV4HDlzPm3tuDt3eiIzgfA5dUHl4eK8dNxw8mDEqI/+m0eXICNGyf350zu5dHR61u906kL6NvdvEW8r5tvHq1d7CMvMV0uxLyesHNyoKBU/juTcbC7UYxngWJGHhyOASDrj/BOJtL2XqRisPAjxUmRGLuiQMmou+oBExoTJvvdXZWWpTD1ln75Owui1LO17cvU/n2KRMse29evn779uWdlmyDq1IiedEI7jW74Me04R/ts6+tbq99wv7MHai8qsi5I4nzqyvT+FXjdaPx+u0au+JuQueuGwMIBfuCaG8dXl50eyuR2ZgRd9V+ywi4UP3tv3xzsP9mfxkJYTzNw9ZpayOGWq2UojO/cqrwejq0RAh9cb9hSIQcJe1Dc+IoppsDHeMPlNiDAACB43EKbvm1iwU8xUxV8kLIvI/V2K3nd+2aNFuJbh1Y6rnw+otJQWdvgmdhoU9bCEtkM2KOOW59ap11p1zxBW51rJ0l8AvA6YHhoZ/RPtcpcGU+t/EwjXs15TVyaEX03YATmWPMMXfA9oObjGcF3jfMIAY3LE1gMkRErHmY+A7mceOf0zVPmsTF8o08omSRJwtjxFmFmCQkC9FY4/pMeTTFvZPtnnLeS9yEayqa2ViPgOtwcDDbbpIOEAYUh2WPqm/J9JMY4Yrcfor/zh/DQI8nKU+6HylpjyinPHt/KN0oRrdu5A5HIivO0A7rGHPRM6ImI3Az4a4oSNhZDKPg20SoIeFARGBaf/gCsN9ISYcqO3jBCrRdzQBDsmEaNWeLwx1cgFO8Dk/CcHHiWQytuNnF7QGy4m1LIkbF20oCKIW7fpka4HTpgMmSnxvmf6b+wsx94mykyopJ1bXZNwSw7WXCtXUnvdb8DVkyPMUT5i6vglHmw81rBfBtc9dn4yqYI62+SQ5EAhM5qY1bci4czFEKvDumjA7togK3ekrC7t57JZAMQskIeCz6kHNmbLaN6w8iuDVKbUzaF06gMFko0XEZzPrnoMxfY4HwHkscjBjB0HbmZwoX3H7Ey5mlLB0233MN2v7O3g8ow9GiyKrR+FMOlXWAj3qNZTVwL+a9mIAQUOE57TRMqhc/KRViDZiZHYi3OJEYZLLd2L1WO8+mRvV7bVVeWgA2rsc7jZ23B7dyz6K0r0Uc8iWg1Ynkr2Bqowaw0+iayi6EHjWSGT9NkgZEl4qAzgNPYc7bKAW5JcYBPCkILcTXYkRykDJzGCkFb43hHsw4G8N7V1zvbhgEAwpqga8gnWsEI0nagoamukCsCsSIJorcNCF2k/048EAq33vtFuSpLF271v9bvnRzAnUja9ek4jis/xuBlWEXo95s1WAyg8LYORXY7erAN8YYqe4m5oUqxMhZnyBgUmfny0Tddf1vPEvVhfI4zz21dKB9XmXcUsZTYvDPJhfiLcAOs3cszb8nJl6sSHRlHnECh8ax+DkYWPD/+FLxI9WWWKCgE4WQow4iw+99LPOzjDbOP1HgvVJbpcQizkyVL9LHUr+5mGHHcF+pmfKZ9uWAygENn6KgH7KgRmsD3gqykfItQpDFI176MI3sEZWqg0npFu2XmG1heG3qE0LjysjAVzmeZSQwGXkwTIPpaP55PMukHIe4h1FSYuJsu1HysvGIRsmNJFjTKZgkwJcg78kamTNEXjZmDBGpmVkkZmtkDJ+ZIFzsMGWCgHQf1kHZ+w6Z1yAbyOVBU6TSG09cb+hklQw0f67F0C4W2V82BUg2TBB+Mbt/V8l6QXNwf/bBgboxnnIs+lxxq9l1JUUyiJT6t7JMsikQ2QrlBHH9H9HwthX8GcdYMoCyTuHPv6UykpSkNJ2bysrmDrqlJLCSKZdTNYQFRX9WdG582DoUjytXKSfKT2E02Y4M6IFcp9REiOXmfc/g0JICLDUdQuFCAs7VF1Q/UMsrJti9hTekfc+1xc9u/ZNbQ6MU1pdyvQ07BdFMpvdtCmU27PfdXd1pvfN9NMXCkrF1/Y93m9QT+43b5ZQW+u8a5OxlTUaIpUxJMJgzaNUlMhpiT46y5iM1YEa0KescoeHUD/JxqDTfD+A72U7QrDqEf7+RE+M0xyu6LEfBOKSYkB6jmi+wK+vEIp6XDhwnB07vi3srpTuURq7NHAeb5I6DO3DHwQ/TedGYVkGkRMmUkbKEaWo6zov3Felfz80O5rgadkwANwIWF5YbHVIynOe5hiN0q4czTBFV+1z37MByxuyz78X5u0YWoyVJ+V9wxRviOSao75jcr2yOL3a0p+7DzTf8wE2Qeo64QR8ZRPOV63mFYi+Qp//d27sqjOLenLZSSul6PLZZBrvdTFUcDEOLtBg8D0EHk8ODEd08qAZzrhvLZD7bEpaEIihjtB2MU5ozJ6498OQbMm5RXM06UtqQqpE9i42QSP/eJW6213i3u/dqj0JmGiDQRcA+NmWi6jmwpe1OF2619l5ZrwXYUjGnveooteMOBmAYUnsVNKTjtQXTXPXpmvHVRqOx09i4xUz5aZ6IQ4Vr0ge7k74krsa7I9g8yQ2GGLHRibJT6mgyzwDYV0mb27AP3bG2lhhAuuuCnnV6wJEUGeMUsveirE9U3sAqJhsds0t1UxVuFEX2vhMQU5muT9xiCxtIYdIo/MVtdB4m9DPXguOWoM9sKeai22dySSkETPJ1qnBwJeNdYk0MWOQOLBAIOUdZ0sNmOz+KmxGYaWhXaHlqsfRf3ThnfLPcNr/0ddub+baBrW+hF7gLEKriczPQW3+Cbgam1KpwAN4ymdMG3AP1hiFgit9qRwWlwsS3R6A7dPosdS1iwCBEJRVhKCmxR6xAdeFOhgTWjGuOD+Wq0pawcrcZ7llzqCW2Oy78k0SpHsQRWMigwHzAm9YAmy/hqv0I62cHYWyZ/l8/Ar9jcB9lgHUjB1YfdAqGAJ4PwuhvH0FgNhp7LwpsO90qrVZApY8uWjDFXhsm2+1dfkLaLMGljwkrzjFiD230egwSTRWR0QxCKgFQcZP7psjYeMnCSbFKL1NKORJL3MGvrRlvk54k/FRLnVXAZ1T6D4o8HxdGnUHPLR+2lirf2OdRAOJnDE/DZu0j0hGNJ7DD4ri+Mt0Ig67p8kZG73AtgmjabzIPw61zSHSBwpxUzUTWBA4cVcHPFfxcwc9PA35eraB3rRD1rBK+O1Mt8ruw0M53dIs38ryAVygkMgUVaNdpr4GNFFNMggI6yvkIAJhrQVzIdPJAVDrUDjJB7cTsZ3CsS1hpiQHKiDNQMZTJoIS2j7W1/sTyAx4vPWDLswM2hWMvCr0y6LwOkL0UId8yNLuQdfcVYWZ2T+DXm9Xh7DgHsa/1O5Clxgj+ILHr7ODCZvfQN8+/6FEurLFripjVDFjNyXr0aVgiggml/YgQ9opmy+1I94oey6Mh3cXmg+si2xuFtsteN6e9IgVKJvWdiEFRCvhlyAVGYEZBGnE0GsM+oOXUN1sRAB6jJkKYTCPazHW2BL5zMVs3k5zMldlW2IAeWtqyZekinH7vRbhtCY4KcbjM84nTPvXFg2tZxM0YBlhpEuEwxRj2PKYdY40wqos+1xtTrYpkqt8htp/kDWW0lVFoKBOBmRGMfRCtH4wAo2AeW8ncpdv0Wq9yEp58TgLGCE1hNkPa0200zavQsL1sZq5CsVp5u3MJugwpnWjY9Ag8D5eCiIdYvL4wm57jk7g9kR362GGwkKcG2wZsQwQcvEmussuyCTh5jdIHNHJb44Av9WHEvu9kwmdZHLi9RsqNyBmcAwurtIIqraBKK6jSCqq0giqtoEorePy0grVMMyqto/R/bZRpJDiIxgg+6SaEfHDUjK7NTkcyK2CihrRBnjJ8fEYFqG0KeiQTcR4FSWAH3r2x49C8YODJG5Bb5sUD5ahIV3bGQKkpoDnAY6nCSNkUfJkuh7zOAz/Iyzb8MwhhRyqQGpOASglQqaoCekWRmwo43krg2Myi1/yp1fnauliCGrd9cOcSd0gFzfMiL06psxofCWc0apb8f+3KzJ125qN/wJRU7UKNs0yaA/V8ipFbtYeYcTeIDjdMvaljvR4TNM6A4s9uYrby7OFopdXKoRuCqPFV0bkiKSiLe/JXPKIvp15/ksFrOo1bFprZ1XUMGsg2U6aMFefc4JWXtYDBSFvnEj1xiHjFvVmy0yuIuIKItwkinmn1u6679mbzJco96lNIEVQ7EWakjDu+KZYlu7rnRCyGuliFw12JG3LrRGmXIpB1B08kBT0gwzwL7jEqXcXzgwNiGnThgHHmKpho2+dBLC6t0Vg5NrfEAwHztrRbhgpmta0XxbN4pxTaUcDhPI6EIBxIbsQNnoM4C4robU2B/gXlqLidQLjwUbTUHTL01GzvSKP5sAzJtGvVPSerctaqnLUqZ62go/WUw73LUsvLXsVzMkLevv1h993eD7tv3xgz5MUfq1a17YszmO0RE2t50SoXq0rakbhBcO9krmMGK+WFpjWRhhSscYzKMOWoQsIP0qmJdvdc8Ekxc4WsHJAjbMqWUYSe9Fmzt2WY0uO7O1Xotwr93h0pKD09YT0ueb3R/Nbu3ErO5qOQwfl6fm2BQ7J1RWxS5w3dINiIz7OE8ui8RxJ4oOnQFcKpqhUX9rNKI/ye/QH7yusbKDGFepiLOIQ5i+cMi74Xf2vsNNjP1VhPjhltAFusEOcKca7qRat6UVgQs4No5gMxY/rOxNJKs65mSkK3qxr0/1QU1I9hF3c1ZvHZ/RU7/q4M5xmsA4aUjolPxXXqoWpgLq9xNx0gbHDlKlAtiY6WacAPmO7fOAawJggaRFCI0bsRj4QEEAN/WIIxKuZeJirEcgEXeGNS4Xdbid+dN7tdoN2x1T2/aP5zedkno84DF/iA+AQjt16BHcKIixWJZUDAYySTgmvjMSLN+r64vDWuTMz5ODHwFJZLkmtvAuN9hfsTg5Tosmai/nH7DU+Vbp4GN3UQo/UuDnRZwWd5r2GmtMBVJDO6nHQ53b3pFsNzCUw1FqmRAP2aGyZYRVpDoqprIPZSej5RBO9eZRAVeFeBd1V9Z1XfubH6zrLTXdfc4vsP4bD8lJ0Mx6fK5TCr9l72qTgmK+0MoqH0QZnZwdB3daUNHkRWwFYxch7BuyiaUMwyySBWSvla3R8Fg1+ikomFUbcCWT1mOwK0RwpMxWQGx3is03Y+CD6uis/A2+VD73Tt6kNhvAuaDrPP8beP+9P1YCVyn8pJpjMTtwzM7Roj78hzf0sXWDhdrtYk3BYP1gvzms7MSAQFcUV4DDJcLPXRFAUjUfcW4bAJGUuRrlVefJxCEekxfqH2kApVn49b7rkxiLcq96zKPatyz6rcs8LsV1P2WwCGt3DvwovB3u12O1nRZS+S4CzZ5cq2AKCj5KXhT2X5o+wpr7fM8W1Ez/GMWTCC6sCyBlsAyqt8SLqbM1k7aHUgAo+5edekE/NNCjsUs1yqasuqrq6qq/vOdXUVylqhrBXK+idCWZ9cSeKnvHLwkGDD20HMbvPLaaHMENYI7CuwNDJksoMdJYA4HAOYQi8LhYpJhHVOso+BY0YsUaMUDJjZ2sfAc21EqVDko/GXWTDDlBMKK1BzK0HNbqvbBXpZn9v/2zz66bZetglFPqWD6QIymtRdvw5sUh+7joMmcbGYECX5IKkZNF3rEdzW4M50A/sKVI95gPFzStoCOyjGs3T5OK75ZiQLoUvQBrmb98xcmbJcsjg4ARyMymQaEtSJGc+uyQCghD6DYqA4zF3VmfNxyfnF7DAy4GCSis+jnS6afIjTW3moF0zew+DWQ1tnTkotNBzWlaIzO59poS0IPPYHza4ZJMoIBcR9CFDAk8ydPGxTqDwqHKX7eHHFJ1naWB2YWR2YWUWdnhCM080P8O6RRrtQMbYK53MnF1acdwimQbJkJ4BjypnOStZCdqbuHklvm8I+vE3Hw6kgrz5X4a+zWcAUpoxvbbpPHcedZfpF4xlpvWVwzgNK3GrX/klixYW47zELFgyDfEY8YjC4NVSs65jy/nwiBoOHT/zIXGzY3qknI4GmKVu5fIJfEGUFUPBJ9GBBUmmVB64mvPcKjEtTkEVJIXoh8CVoM11nhvWTDxI/6ZYRVUBnWwM6dzXDjPyZEihUp+h8h0Swu674MW/3wmB1USWiQ05dT+YDFWcSJ4PsJ8Glb6OQLlmyCm0DFCPatdPk2LbwmMnvP549eun26NjnXg+F7G8paEsRj9Ph0KPOPZzYnwceyk91MkZZnNAZTeDIplwmoLjEQMjBAGNq6OJPO8n0RBUB28oI2EWr2zYHOp1fdP7xT+uw0wN6rpbdv/QsII7hxHnSlfrmJoScJYEOuxa6SYETA5Pg3vxOShWiZSmDj5rGn8eRzrnPlQ/GYqKDyqtl8pedeoVnOSWp72snKQvKLT16KS8xJ7LSwUt44lWMyH1ZRj+3Pizk9Bdoj9NQUdWVq0rsrxL7n0ZifxUG3bowaNVR7Xt2VEPMiVtZoEnxXoyVA5MQ+4UcoLfcSw2DgQzgsO2RcvDi7x/3GtwVeGtS64vdz9gCuVBUWrm4YVoejy20S4soxWQ4cyimiIO5FmrYQCKhk2LRu8o6pOVNRiiII8MQ5Dsl6wcol9HEkMBgLDurJmlVk7Q/apO0qorhEaoYKuTiz4dc6NOFOhGoPMzXGiLHzRyvVHKsEHf9x0QODUNkZwVhP6zsRUXHGlibsjxvaCEyZ4CyxehE86usgZzjYPQPqxiv4HfMEYN7M7fdx9OGQGOiDUJHDVV57lX2cZV9XFU/VNUP1alCVWvJJ9Bact0ABKzLwc7rH7JCfWqvNXs8Bm4WZN2Sxjvv9n8wQfzL5tM/TKhFbxQUXMOJrNx7rH1+/RozSOp5/oewJzajk4cn51oQ5ZnQBEhiN16uHgRzS9kjH/sIEHahMcjADzBMKuIJEHpcxDNiO4iqrPxtxSS/dA7bpy2r9eUSFqdzYX1qXnxZJTHfdE+VEXYqSX1KRzcgGbUDwZBWfspPBCPUiixGnkYdk/syFNUfw2qDHTZ0KarHiYpjkG5jqhJZPT8f26CMi5hkaYo+IRTqWxggJnHZBA8Po6iOGS+Wpn/IZkKj4W4mWeE4of+uDnvoPIju8U8Pm4TfMgPqyVCc63OGbsNBeQVBRsMMKVzgBakjmr4TBa6Tz5HXkpcoK6VPdPTaBEPnV6pWKLGP8hMeVj3d9c+ae79h4OLtd8Ut3n4v2KKKU1V58STDvujjpz3XvhKfQFKVC73PFPa08a46ZaTn8ErW7ciIQVZJwM/ZQoQj4J546qAaIOwNvofbJRXaHFHTq0T6ikJWFe5S4S5/VNyF/XcWKYVu/8Z5f9lYM9d1b/OqCyk8dbzkAM3mEayIus7OJsGgDC6WzuEyKwLOtl6LgsqSJC3MO7U/NS0R/qBgwGfMPcGVEpw+vKBy/BM3+h6rcR9RgJSqvJn2KFnjsYwSEq/gJDjKI/8d/xoFY8UVDMWDXALfNyXllPKk3IjykWIVSj76xfD1qmZmFe6vwv1VuL8K91fh/ircX4X7HzHc/1QC7J8nwHHXLia8fpHx1Z1i7LAOIMfrnsIXaZswNpF6jIxkDj35+hwnV6ZJMjX+U9Kb6o3LV3QsBd1V7oKRW3hViH0bQ+zNLpCsDf9on31tdXvtEybd4gD7pe9SyxtPREHA9XloMqCAjtz4qt6XJA2mi8Jga4JEwpN5fbRh4lDZSA/YWGNsjIqxbNRbJgXnidT1tL5RZSmMv1jbc9euMtwrJyuBAkVOb+WM2/wgEC6gKG2MXnryCW2/rH7iqfQ1ePBdWJXuVKU7VelOVbpz99KdbazrYLTgExZrLIMTuJqjBEngKJdutEMoaEHDcK8dvWVMlxyDyOBWyGs9dAlpjMHJO8S0Hk7PrKhAbsceKlChAhUqvLaqKygcJd+JTIEB57jd2hWJj1cmM6qGsTcdp8NTpGJzUnxt6jx5I23TENNSatMFCaUnyIdp3wMvgYEIHfGtIIUKUqgghQpSqCCFClKoIIXHhxSqgxWqgxUeHz0q4kDZQfErI0d8DkKIQjgCazZrUQ6bRIUDDAhwsy4uyiCgqo9RBi59HcKNM6F/0zAOFAW32gCDgwW3jWeEVpDRdnaKO2xaZ52vLZhr56zX+kfPOu2ctJfhRt3ZgPPNSGkv3hgcmP5mDuzt0/nimGlFO7tGOieIiucDg1ABtfOebiMYCU2KSNXnsafhUNdr4K1OALoQvErSBZrRVwCcUBHmd+mX4gWajVWQeR8/EkeK5tmx0IYBnqU4HfDhGWPome4zGsHYDbSzQCnktz0A0tU1kfvPsD8/0dkNJqhPDWVuLdswzyvqdkdqm3MRfwUHvW6qLjC4iB1gphvekUUqS0rkEEg0w2D3F0xWbuiZ72u2F4sxpT/AEfQVKlahYtVJ9dVJ9Rs/qX5TJ5mTwDTy2tLyWqu7WVG1/AB05K7Gtp1i3jOQxM+ofC54V52zb7oMnssQDOCTIcsE1l/hiM92gkt97BKTTKFzWkNi3FhbQcQCwAq1TNVSXYBxk2cwPOO3GtQu81qrSqAKtNsq0G4j8qvctAbp9XKWx1aJ5qEE2599UGOGlM4GLzZxpefomIBKfo9NKF8822awjBEyHRHqRLrkRhxR+/3VgTIjBeUAhbZByEBUtrvnxXNCuK1/TUtBLs9BOSpDoGtNd+7SZ0ob6YcyktmP7DBuB8FA25MHzO6V1vCAoq+Ka1dx7bvDIBjtyMswgXPWh19fb9SU786tpIlDZ21D0PZ6Pb+2wCHZuqIdq/te3iCSis+zKKPMeRZ22DMXGAynqlZc2M8qjfB7NriTytxAsW46VlPEIcxZPGfM9734W2OnwV6fBrJyQKw6DryC02+F0x/WuNpbYCNpbW0NqCzXwmBEZkrf5lJqb2jBQ08amkE0hBvkcpHPkdYtpyiGyu2ngrAqaiOOTyEs78d6pbkzjg481eZscjKMWChpsyrGBiqYQZt105FRogYSM3spAUykPoWJ9AEtWeOtCrXZWtTm8uyns87PZ9mMzi86h60luM3p9NnQcohBqQWykT2dWSH1Yke84ub7sO0YpIG9/8GkG2I7XfgdEz34oD8Pz62h3/AsnDzMFqP57QmEjdbBa9hIMfqM5hCj8CDb/kHriZAePhqfLT8dq9WqiXgtpo6mxoNXM0Ql1e+kDavMe7PWBthWIkbP/Trw4Fp2JhBIuekEIVJeGgwnmpC9QZFvrEK0q5qi9bppVRHv7xrxrlXwVgVvbQjeWstAngkJokZ7ThP67/4X+gt1ISaHA4lZEaJ+e7EETSkcLUHlWFuDovTIq8JWaWyYtvPGRycpBjoXHVvDKEoWNcmqm4b6qWILJW4DiGEQbduqMRkWnitjVpIcRQTOBwXLjn9WjmyjXDJcQ2qT69tCaT/ps8JXNGIfDTapTkP5XqehVPbGE0PYtwfTwXli6RP5F0eUI3wroMOYHZA/1qxt3BHyRlDy1nQNKnK35xQktUZvCsEyEtQojie+bZonwDp6eDhqwAFr0NEeqRCzsvitbTwLvqqFqoK3j1ULVWnjNbXxWg7B3TT0olA4hq/1rEUBsWqfb0v4+1JXbzW1IO9FEhxKu1zjnJWVpyX8hEnh1pEr837Q2CqPsmXB8MxlQMeX63ONo8A5BvrtdceNKZpIxbekjusDInq/KlrY5vD3p9PmyUnr2Gqf31qxwJFvzsLzOZKiYsUNeFGeoySUSR1byHueC2aqXcjFwz3E4fC/tsG4NGfT/vXFjkAzly5nBih1dtLqRneYxZpIbswxKeokPsIuCYJ1IuArZnQWChjmo+VTQukZCHzlWaCjr/AUixsZ4Wkb+KmfsePXNAWwbgPJp7AIi0s06PGaCT8PUFq7Ce9JXbH1AWeOP1IciK6Yug5EJ/K+u8BaD3HAxSlTJquYOAucBelOFyVn/gg8YQDtVxx3Fjrz4R21QmS/rEoCs1Hi8gZghPbpcL6uADZ6sKqFqGohqmBxVQtRRWoerhZiYVHD4sj9LU0otiuaP1sT8dWVQmvJJcWDMwURpl4Qsw+xgwU/z1V/xUNOpNmcGZ+g8uOGZrq7WUnlhEmVQbse25mhi//o1YJVEURVBFEdQ/+nCPXcWtS9v9W1IG3fD2xU+RTP7PhG/C86iL2AG8RpH9e6D8/hJ1gJaM83O5Q9Kp7tnh/DrpVEX1H/CLTw0MXjzgDxXzlAUysUmZDkN+ezrCj+q85pFVpQoQVV57Sqc1rVOa2qMHviZ68bYYUctFIdic7JA6a7y+0mhY/NlG3B2TKbTDSB8tfI8OUZ5/oqcpwUeeSaNN3ATbDXkkt21uICEz/AaLw2CrVdV9PH3ctxiB0JGGMr9hgDIQIszK2ZPBa+Fby2vfBas33aOmZozTq8vOj2lgBsFypUVOQwp4ZRtszHBlnYSWAtEAnwpw9TE8+v8SRnjBGz0BPX8Y4uc4rwFHR4w4uHKBfJELIskJtyPRtpXcrO/Xuxt8BDFJR8NVM/jLB/9yd0Bm8rKEmCoACDE6Rk3Me8ugdoDFe9+UWogUENpg6pIvDf9NmjWXHQdMMRtE4weHeNp7iDxq6DPPGw6fjTP3P9Dxhcq3ClCleqcKWniStN65AMWsrd77zao7bgSdyemIoSmaCvNY5JbFAaGXDC1taJLC8OOecUZFhgaisbF7Z7SaVIFl/PdR4VOsc3mFdjukBkBSEFYcSBYh1Ipl7IT16JPbXTb1bj1eUi6fR7AxW3iayjQuNrw1QY3x67FK6fby2S+g5sHRimwBIjPMcQm4SguuorOgmRcr4kM9wdQkLYsDrGo15iI9cKdf0RCLZgjF2aPwjTlxUNbdbL3CdXeQrXavK0aykrjPA+jdK2pgwFfJghEOFcD2HFlmLG09MNxWAaoEHYAMg2JZ6yEyIt7auauFKTfiBhp3o8ddMRg2pSsm0MiyWHFPLA+CcyYKGUxdz1eFjSk/RlKiH/xxbyK5mqjxuxn1m8T3pcJrCUiekocq9xQxPEUxJmN+gKBp18TjbQi2gHGOA3iQezMojkT5BgnCQXElUrrgqf/S6tuB4KBiHwB20XbftQLb2RhWQpbAsmogWCOEyjeEGNq7mlj7dMlxX5tOUHok8xV7KYSsuLWCqoKAqiYpFRPwquMF0xOxwcWxtMn1iBIXjlY1g89ZIKDNlaMOTotN2CiTTPOl+ap/+8tdKIY1TTB5w4ChY64jOPawa4rwmUl1SZlvpgW8SKDkPiviJ7tYP911O/X3LDLbB5w8TEsZDjgiheCxkp4qLtM/Gv4hB/EZ0LkQsYGoylYZ656qHNHWhiTptmgi4rYv8CQt7GKK3psIWHW+swFmMkZN6xE6GBKcq0kDNrQs09wfgincHlgmPpYXsRNRW/NDU8sKVdgkK35QzlJ1nkXmCtLYk0vLxDpKHANngmEuYyYkkcpzLOAiVToQj0Q2bDDzXUrKB+CMVH5/XG57eBftXVsXRuummxQ+xf4Q9/wA4kT6hupGPiXnSWRjeEAcFUm8OFAnumcsQBS7FPXZUJZuPIGFANho5e2QIhDKujrsHmFIUwFj4IG8RTOVyQyCsy4qhranXWfdX/aWv6P/0B9WIX5XHqpzFSsLCXTXX8VOgd9R+ovSyUg9F3Pt1eN3jA6nGM++SLYnIl/6Bn2rdzPY/tnTqes2pzJ+rHxH5qjZsyyTCsIalxurzYZCvA1TFGOHtfye9NEz71T437ynHo+D8+mQOcZtRh7BhnMtsvWdzHi6lvv7CtwoRVGcdscn2xXjlLrF+fJw4a37d2IzPPS9RBUf8mQWqPQOIf5JqgSFMq4RjISPTVJPB10pYBVQt+0UM0gl012oq5PlmklYazLdFV1i6ia3bZgrSdPElOnzFgOv5lu1N7kmnE+9QtpKCbOI8u9tOaPEYvgotyZloLTi1qFU/dknjqX/C/v/9/S5gN8HiFAQA="
REGISTRY_RBA_YAML_B64 = "H4sIAJak22oC/+19+XPbRvbn7/krev2t1MgVUpctH/J4qiiJlrmRRa1IOZOdmkI1gSaFCAQwOCQztTV/+76jGxdBkdRlJUF2d9Yiru7Xr9/xeUf/j/j4KP/98D/ivPtPcX7QEZ9/PesPP3cHvQH8dNwbDM9/Fde7m9ub22LjLAhDFbnSF1+6h11xpKYysmXiBv5LeMORTGSskn1xmSRhvL+1dXNzs3klJxNPbdrBdMvh6zH8I55EQRpuRSPZ1r8KsfFqp7X75j38v9ciCm5iEUbB2PWUg+8+vJT+RMXiOhY7OJZ9+E2ItrhWkePaiWV7Mo7FxgmMeCi2xKfzzsUR/P/dwWHnpDPsvhSBLxTcPBOXszBILlXsxuKnwh+WedNUhvrd8lq6nhy5npvMBMxb2JGbwPQDf1/AyC0z8v8nbM9VfmLFzpUV+N5MbEzdOHb9icBbhI8fFmPpxe7YVfFL/XrzQxSLSLWRIL6jHBwoLoMdeOnUhyn1zlqiMzhtiYtOCyh+7dpKDGehaonEnao4kdOwJeLUtlWcvTpS0+Aa3hUrb9y2Az+JJM4NR1T46gbMGgdow9hhgFsC/rYnvkyQyp+t+FJG8I6bAOjiT1r0aiHSGB6FWVpuaO2+vgR+FDtC+o6wpX8t4QVeGsPbrNj9XfFFeBnSwlPtaxzDVEyln0pPdIZ9M949oNGN0PTMSeEEU+n6IoHZxh8E/2wlUZpc4iqFNJ0gEjA714c/ttS19FJiR0Ni9xu85nW2zMBAuI76vRufrVGQ+Cpp4WxnsNCtbNr4Lx9o6L0USQAjTuxLca6+wR9XyheenAVpAh95pN34AzBMTHxGG+8HTZn9H4Tw5VTti5NgAhPADScGwIFIhXM3vmofwG2O6ACFYE1d3ppi4xdXjT2kFtwqvU3gp8MvYtg/G4jd7d3dl/DWNPLutm3h2akbRUGUPz5xk8t0pB+N23WP4Pa2zPbeF692YN/Dtocrmu1xpriALvDBN/1vnvPQML3+9QL4UfSO9F/nyCLtYeSGdKP41zT+t77UOxMdx4lgl+gfDuHeJJqZJ9UE+UZfgh2v/wlbr/ilzgQIKwZJBOTUvx/gbODSKawM7YSvvHj6cn+w6EphM5emOODNPE49M/RYdBLYp1cwicJPto0zEEN5pYBTI7hy5Qc3viWjxB1LO9FUjJEVfOVZuHUtF+jdfv1q9/Xrt3t771692n6z937n/au98q1+kACXvdh53dre3Wu9e/+eJfLG670fX7YEybjXRuioeF+EQQArKWSSqGmYxCjE/MBvq28uCAMYo+SxxpviC4gHMVLCAUGKIsr1YX9NIhlebuqbLHoq/oiSCqQcS88kUjLBL+C7FQg4ECA4n80XNPAoAcHNMteKkOv3xfbm+71XPKuZD/sBtgPKLBnDroJFFf/4KPa28T8BmweYAwi/s71J/2frnQAZkD8nQg+E5GXgwaBBEKHMB6kBgvFqE0Rm/npD+I8go3jB4yvgRPjaVu+sHakwTXhH5kJ4k0Y4UYGV+pHyXDnygPCaNbeYKbeQHcWNimjeMMmJD5RAQZb67n/gSyj21Ad8SxskVmDj7aDepkr6JHjj2GiVDVo6eLSNxBMk4fjmEHaGimBFXvKQ5AiZwUqUp6YKxsK8hKw3DeDZ7K8rNYuTCKRi9ksSpPZl9herha1f1Oj4JPsRRKxzgyRGhomZKNlFYCjcJFt2EFy5+WtzTbV1eHzaGdIiomr4oWQDKOJ6sgNgnT0vuPkA7OYDzYT6Buvo8wrEaXRN/KeXgkUlTp1Mh31xAHS8Ap0I/Bu20xD+FaX+CIb0AbSB59ouUKH2jfgOY3bsg1iJXQe1nVYnyD3AF8DznyKZOqIfxuIGBCboJJg1LAlyTWlMpN68zR/qTRWcrFFj+zx2+oXUWekHUmv7TBn6hcazn431h9JXrTCASdKqx2DloTKBbd6VoAYzKwjZBkxBlNEhTAr3+bUbeDzqYCzQ8kEBIcHqAiPEcfHCB3hkJmK4KYYN4BTehjpGfLF+Ex9h75Je4f2MC79PO6nyo5ZSHYEMCrs0uQTTBcdErAuSI/VAo8PuHsGmuIFHfStOwzCIkqpJtjlnwelhSRIEQPZIgY5VMFcFT/NmwjFGenmtSQosvW+oiqMof4LYwg7iBATQbMo7SnyiizPUzJm5xSbsS3jzf1IXXo8fByaE+eFOgb/gN5REcQtpGmv5SiyEd+In2jD9GIh9rYx5LDzgrs0fxiBD00hZsMTSCyZkThg5FlrAWLmIsjx8s6M1iBcAcw1O+kPrtDv8pX/+s3XePbsYdoa9PmvHOEgjG9ZiTlPhBt0XsG9YlxVN6pIRTVd5PUF7Tyawy/VaIXVwST5oispceooNeBdQ21MS7BVFkqswoRDYCgxR1lLwoVlswSKSGlx7Xpnx0DIWR6tqkLTq1TdTANTcygQ4gpGiyWwj5wEnwcfHbgSLP5HX+ieS3tJoYPgUrDB8+wPxvHDHTKsiPUDvWbFS/p1JgOIWPp0EsJthc1c/Ha+73GZgNuu6+w1OK8xHGiAonW+JpZWlRQO+4wCNK1c1JdFJojXVEyhzjthuv1p5uMD4DrwE2Cex2FkDN+0B+H1NZi4Mx1iF1s69xlFn/99pPGO4C8UgGIoKiWNps1nbqOuOsbrrwSqwvc3Mli6Mc+wFcq2Rpn5paNmI1x0i6RbaFHpQZlHJiiyJ6jXHWGf+3kGy/ONjbovnVvhddyx5A2uv40LfcX0J/n5v89WPxlYBk6+syYpiGS1/Y7NbV9OVV/b2xapYM+UFyw3p9T62YAnmv2Wo8OJcw1DjKJjmFk7u/OyLHJuK4d8RUAk4vP317NTATuTfFQZP+NT3H3f2I/wWwMgRryGPl10T7XCEMrmMaQYOufqbNKDq4I+6X3uHXetT7/S4e3523jsdltiyihLwpJSfTv81DWAyquWo+CoJwlaCzmPSAmegpQVHC+Xu7N9L+VYPL5VLtPCSoc6rtXl1ttJG1uMZMbICAnvqerP7jW0RSiM2+O20M1/e1WYd4Fv5RYhYgK8uNIQH3oyehkhDeFCJDeYOslZzwocyihWpphqre3UWEbTid6Q3DmMNm3ldbrivCtejtL3AV45lMIRVh7WmMNAf02CENYbxqwioA3fZFItwHunDI3UpwY2ONlNmfbJVam2ng+7nztde/7xzYh12jrqnh93b7RJCjGICCtchezYgdKBd1BvGogMly6Dizvb0bqO7KytkvqJ5PobHbC91iqjdby4Mk5ygMk0z8zi3Be9C39w2FQTp4c6ddzXWovA4QMAKmHs000apcT7MjrwDpXtnP110EBzwjU+E49UmJSMH9C20w+DOdWVHzh6IC1rAzVEQzlYd5Lp2TPa1DHe0eJnrDL6H/iYhm9a1xAiorR7rc3Vg+Ny3OoeH/YvToXV83jn7PC92wbL/Xx/rXJCVlpRHQHulrA/qXd/lY6n1alcdhkTM3gKlmm0G46lZb511B1OyTn4iUEPaUQB7wrz0roMs+t3ZAOt8tfVGCFLmQQZYEnxI0PWGhoSqA1XuqlR4VKxka+j2anuVlb3tc9V9ZVBsK4EdYOFraVMV3Qf+oi3Bjndt6VnSc2V8u4cBprcNDmNCsdqeH4fKTnDJCrEejyY3aTGvYaBcY1Um+IIU1BNvEdYn/XaQJmbZCcNpCa2o6A3aum8bGY5OE9yClyhG3k7QicX0AI7lUNDaGqXOBIMEO2/MvNlguW3aC0ya0rS7HHVXNNK2JPbVc2tltq82sp3IHcMsHWV75D4xA9BCtmBaZASjFyPoNXpOAX1IennY6NqNkhTG+buONUQ6zCQwNgYuWe2098y0jZy+beILZHlp4h341Ox3xTsB3uMo0Ar5So3SKMYpkMnU1iYTrgpxQ6TIa5wJOcbkC6BAUJ0uKdRWHmGjq6SDhDPzgaB2/URfm4nSTrltlvNbqjTBYSTRi1GZsaDjuThLnrTBWx0VJpetIgfgWJHjI5XSC4iXdQKI2eY832nqJW7Gy7ABdArJosnV5YFQKIMswl+UvBIUFojrk0NAqmIih47h5NwLmm4Kmh2YFBNgSmlFnqLIXSFziH7B6ZJYhskT02IEi6JCqafyFAZnXxwDqYd966Q3GHaPdETzBgyy/bo4PmbNUAC5c3pUCqiYC/oFuLP2xdu3+s/SkPPIH0nGbIgmiosDM2HD0k8UN9S/hJEaq4iihpV7OYEGGOTXs64FZLRO+1+7sFP6p8PuP4fWSf+4dzo//YvTByAAZQWUKEB5I3ekQHG6NTQpUaB47/oE4KkDDX4+7f9yijtvjgzVdab5l+3AegrsvNnbfrP7UIwwN7fCmElanJ33D7q1s/vU6Z3cdVrZpTm/1cyZrgIjLWKDnXd775bTAIzIsUuiel9MleOm0/vtkjlyfTrpHB8DMXpnt/DB4OLwsDsYrEasOopkFxcS5N3269c7e/UEMVkAVYKA97lsxxDuudbcjWg+DRJwg+RkAop/NEOH3aWkAXChw8CPFWZWYAgPnHjM/0vAlsJktmF/s0zCg+5p7/h0JRLW881yotazZ3nT7bx99ebdu1fLCXw3UtZIFDR0hp0BWKE9+J/e6dfuYNg7zq3RKoWWbsc1yZNfXYE+r7ffbG+/ebce/620IZewIEwaOJAIZh1cnA+GZdoYLbSyKL5t1jWyePfV273dt7v184aPdw66J917auYaOvBPZHevL8fNDv3ifkMvj4xR7WlwHggmagFF4g8U3ESYjWIjcQrOy7WLOafFTBMyBMn6Aq/Lbed3bZn8GImmMxhSmz/wKC3K8UUaZKPlBTzqfuqeDrSPUut2xNoUBYsMTEr4Mlp4WdJ0nU9iDHRjvJaMbvYbxcgNOKsoxlwtB8wAuMnYreCdwJhjMHLTBIZP9MFkv5nvYPoU/lnOwNXUYzlWcIQt8gFgePsixOip5LXL05XzrDt33goD29hNMF2wkw3jEBgCvwsTGSTpGFFrcVB5Rn1Lio+g8x25oxT/za9H79WTlJY0ipS0LylNK3txKN0oRiP50p1ciizhUJv4U0zvygiVXIKxDndFQcKmdxgF32ZCTQg2JaLRimLeeyQulXQoW5EXQdNriT4XghKXOA9rv7QjYX/FMYwO55m7/Pl+uc2zL92zwA0u3bPAY8T/ypl1JrWsPBbmtv0V85cKjwrgXeChBFOwj4fd0pUsW0x7hfl/t4Pr5r9Cyhy8oXSp7ChiTpL6Jhn8gDV2Uht3wRxexC4Z3h1THE5b/sBMnpKwoXZeC5ynUDICFog+5IwTG3Z2/XEEt0apjblsIOUURl4T7WhiDhx7mX+LBYLavMnR84WhbZamAL+5o4gXxUR+Dzr7nN68u7nzI8pCVIxZojN/xKEURvANrjEJFO7FwKTxbWH+G7QFMB9N/KxUiJnGZl4gS+JEor9su7F7rXT6cpUJ6jH19Rd9e/Pd3i3LXhcCr13fLwHRNpK/gU2GItNOo2vKJRR6bEgq/BptY5AL4MWB7+QpDOxfpiAUxDSAJwWB5/hSREDGKS+tEQHw1hjuweD7FN67bLUGYRCMyfEGa1I61wjKkwwDNUX54pgtjtAJCrI0IS6RozjwQNZt1tK9PsK4gO7d/7OI7GUZdA+6dyg9GpO9L0Fb2kUIjLUzhr4UAmmUX72lUTCEMij/M2YiFwAzFrSIhLbZwjYQnK7GKMjcPGvF0rBaWWItSR2tGIU6iTvPg8DrsFaLXjmfw0Z8ld2us7H3DRSG5STB2IL/i28QP1EmpAXaJ1EIzWtMCX4fYWK3ZVRNRb+WjSytXr/oeqUqTNHXrFBRrZ+J3ceU3m24BaXfhKWXLn8CsUGhtxAE1CWvSphG9iUV8IBp4xZ1bszmFrw29QlJdWVkcOQcWDbCiYwNGJ/BXvXSPok2nUcU11aktyjk56BIX20/mSK9kQT/OwU1CmwDwo406JzyfLVdUZ7SlNolhmUzfszUJqcGltQmSL9JG9SU75DFBtuPLOOy+myE5p2Epg5hZsGfDb2Lt7BI6KIjQDBg/tDL0kOrBELRytgtPTVWN8bVicGioPoFzUjzUnQcKfW7sky6CxDKCqnIESQmPGkr+DOOMSkPpYPCn/+TykhSgLmcHcOStipYy76qlqwnagIkRz9E9G984E9CHSry9Fj5KXwk4/eA7swFakuEWDsz8kwsRZID29IuKufpcaqcoPS8Vp5qyP4IvCEdea4tfnHbn9wWGjqwOi0uSuWVDqJKGtdCaVqENp7CN6mXuHeUlIsSju9ier6/r5zc3b5tk2uh936bzPSs8pAW3ZRNgLZFwyGR0QRr9uoqElvALmiwtNnp5QgiWbZUTuQH8J2MSTUzTeDfN3JmHJ14maF6GExD8q/14NR8HnZdSabYqB0xzgr8lJebyxZzSZ77ndZ0776LurfSou79WE6oovL9bxxJzwlRs9YtDVDhfUXqtXNtyYzSwhotMDBhaWCp0Ikgc2x+sRmrWMVzNGm/vTNd04dZ7dkH98XZ++0MeyLh81/wmLbFBmaubZqgfjY78B7ZofLh5ht+4CZIPUfcoEcD0u7K9bxCejKIqP/u7FwVRrHAu1xFnazPHPfnjNvMIsWAAVpARUwvBGVD9i+CUjnwAJNqG007n7ICFCU/dYoa0fgZOVfh0gEzvSVjCsVD1a7WWr9F9hNWI5NGWg1b2Nl+v7XzeodgBY1Y6uIMH2uiKZEbrDa7P4BbrZ3X1hsBVkHM2UAaYnPc8RjsFyq3RJNt0TovqjG4A3i0vY1tCB7AOKOUAk/EoUKKjsAKopeLq+nWJXBucoNQChY2KjulCsb55cNiZW3ZwSZwp1r7M+C8+nKc9ofASQQfcC5Afdl1XggekyGI2Tq6jpLrrsmodAJiBlNEzfXpWI+NSTjwFxfFFhdqVe96rm7vNr+6msBfe28lJYfAKxJXpbz1eRNRYm4q2H0O0BdEiKMs6WE17E/i5hLMC1SiWhZZLDMrJmAxNpFZgBe+Lkadb1vR/RZ6gVsFrosPVFD20QxtVMwuUuEYPB4y2gyOD7IewStCnrSVi5w58+1LEKc6k4iqhRmoDFFuRz63/mBtotNkM9C/ZdwrfCjXG7YE0i80D7Ny61oLsbRYzwO8vpuhKf7173kOAu4A5rHGWMyM9P8JVsIOQnDJdfH7T0AtBBhxp1k3cmyNQOKiN7cxDqO/fwQZtL2989JEiQ7PuzDUYQ8GPRhefMIJ1sSJjih2k8dsPDQG2zFIBFWMamT4ck3wA7eSb+pGjB8knBTTzjOZnEdRaDn5tS3jeNCTFPvQG/vWYBCquXtGgo4KA8pCQV0fWFtVdtRZFMB2nsJjsEtGiJFG0xmwdhy3V6YFxYRaOgef4XqkbxCVjW7zMNw6FxkqUI3TwphwmmiBo5pwUBMO+ouHg1aqD1kbeatYCKtyQ2nJX2QmOias+47uUEBGOiw1Oa0lFFNb2TvbyFUphvGBGHLeywPjIogLsXoPBJFD/UQSlOfMPQYBv4DlkojuRJw7hDgQ46XaJGPT8NGDbk8ac/vjhdzWiCwtAqTKwaD1QksrRauePr5UyN34ivEfNmrh15sVAkxxHla61g/jMk8RT0ZStdlXgU3koYOVf8qjVCajjYsYdyV8xCkf9GmgMUUGpP00QaVV9O2SwNMqBu7DB54KnS/uEmi6d6Sp+oqKlo4UCNbUdyIOcBAekgGp6OheBmnEKBv61SDZ1TdbUTwqRumLuLoOMDFH2BJ4wsUcq0zSMMdkbLq5RPTeVmK6gIQnj0vCxQQ8LMAUmSUcpyPq+wDXMkDCqDLMao1wUGIKuwkzvrAuA2XkiGtgKC9WMs2WiWIEHJO8/FUrxEL5awQaMZj6IIQ+GJlAUAfbY9xazHR3ayJ7947sIThiinw4tJS3Vsla26GBdNHJ7MZiVcyzCcoNGE8+1kGOQ7A5XYJLDrCgaT6PkCEYZHxcqhF2wihkOwBfgo2BaKU3y9VMXViOUyAoDqcDLC3GpagTCDaRI3MuC2ki/14qNyK7fi440MTnmvhcE59r4nNNfK6Jzz2j+NwyK4LS5ynlUdsPOjQTRFMEqXX/Dhp46bmspa8hqwFKiGmfKJhzSrUdPXItkxkCy0lgB976kZzQPDn25A1IAvPGMbitkS6piGG2pbBPgP2Qw0jZ5NuWaxauc4camcyG/w1C2BcKduMsoNxJ1C6qAGmTY9yEcR4/jGMGNOz83O1/7Z7v13VWAOM9cSdU7jMvMMAdDRnwk3laZJbzeO3KzP9x5hEQ4BzKwaWqfBMZpKLzGFlK+wMZC8IGdcPUy9pFP2IIJwvbfHYTs4eKrbLna3lCN4St66uiAU7yQxY3w2/Ygj2nCLicBk3XOXOy0L+irfExIEWliAdrrLirEC9UAbOVNoe7n0vAZpV9ccv+agI2TcDmbgGbcseou1j0bx+kgGdIXUMII7ITYcbDAYS3xaIdV5cwxmKi024ZRkjckBuZSLs2lNB28DgHEJwyzDMtHryWRGzs7dGqojVP58eUbCDcbzk+wOm/OiSFXWKwjbuY866eHujPCkjOzSEiuSw/DBjtYG8UEX6yLG+wA30VS9VbhhDGBTUfyLCwcfmUDWq0Enqq2obFCH1MdzbNiXT7lqZmpKkZaWpGvjOyfLfyjxf1lSVig3Tku3c/br3f+XHr3VujJV++eLZVIT1fnMLID3naC8pDuCxEEtMjDyJ7Zr5AhjvnJR0tkYbk+TpGaJrCDyHhB+m0RG9wJrjZ61zJCOMTBF7bMorQ6zntDJ856PzQtm4DVP01EslvaSG5/vK+uW+q0WBuHapxVrJn3syvDKxvtioYd9DR7BsMJODzvP3ppDyWJiDm0Q7GWamly/JZpRF+yf6Azf30dYq4Ug86EYcwSbHBwY598fftzW12UTQgnAPLL5ow0F8lDNTUZvwBazMy65lmNBYVS6uCLMxnAFQqML5b8cX/VVHQPoIdMtCw52f3N+x7tRyvNzgpfCmdEvuI69RDQclc1+ICZ8Fn+IGgTTRwoBF94Ibf8eOgGQn7R6yY4flLHgLtakb29RloeYZNokJMgXRh6WYNQP/4AP1ZZzCA2R9Zg7Pzzq/1VRYc9MGz9AQtJiJNXmHNwojrCGhdQQQiPkNww3SK8R59X1zfRUomprsuOKpUwkCumAHj+Hg/RGPQM8lPyHusdlulgomT4KYNUqY9wDHU1lfUt9pi2gkkOllk9eTIaemVO2zNBdtbLIwiAXolV7NYtNFCQqlrIOBCGj0DiP7u+ZsNOt+g8005xfMtp6g5qeMOG2z3gYzcn7Nu6tyJPQ+GaIt3l7Jzs0qKIJpIH+S8HUx8V6f6Yt/vQgQE8b0I3kXOXzHQmgVCKN1gBf8jPwzN6B7SITGrTZC0KXAEUxE8oKkOQ3/Qfaq5YfwWd4jXNSJ3CsEs6L3FhurfP+6Wk8QrEpMzYeszVZ4+1jIwhsehh0dlV1TzgOsjKKxC59TnVRSZxYKHyxKYi3wQS90mtGCx6LpS9l5Jy0e6YmdxG8wiTGy8AW1TF+osnqzA4nEhxUdU4k2BRVNg0RRY/DnCYN856tTFXQGPgXU0GPSz8odhJMEytiuKoxCiQlFEoyolMeJera98yANJGKbCU0NAG7eBVwxcCfRS+Vh0gzJSvagYMdSFSRrXJOvzbQF7AiO2f9K6hyZVvkmVb2IiTUykiYl8r5jIk1YOfMrz/A8oBnBLKGLQ+XJSqAYAooL6B42ZxRf6WKMIc2K/rBSDKNQTJBFmRcsRwlwcd0C5W1DE1RKFwHNthLFRPKJtkmniScoZKE1o4vFDE4PuYAAztj73/nfn8OdFDaASQnKkg9E1Gc3art+GtWxPXcehU9sKOf8o4vCEQh240pIVtwxYvIPAvgJhbB7gUBVlC4BKj/EYF+7lXa5BrQ9AgHzkZTY/FbyODI0jrJSh3UxLgGQ1I9gy4TXK/zCAKIqV3AWpnNhCTo0+Eo+mZY7FM9m69zh7hAdzziQ7CBYfOVI556PQcktXZ1T2Gk9TK0jsY4xGQQWpNtsQ0WFCLPFMKSd3kAvJy4XDXZ4EXXnC6oPm4Ijm4Ijn48g/Pso7yI9cGpL4PlcxNp7joxnmS6X6hOLiZLLDmjCHQSd9aelTKQLjM3t1Wj7epiG58rmlWbq0jtKCMUVpc9rsK52ylKV/RNOKGHvm+fbriKJmSzwfbKsAVx3xnkQX8zPil+PxYmhLJzjnnT1EDGqW26Fmvg/sndSTkUAThw0lbqJPhzJzZjQdiI1+g60lOHgK8N4rsFZMpjbFOjX5JB21xt4U0+lJQK171pk8VpSj8YafoTdcI+jMli7tUSoecB46hWDl9TriDVUYkq5nQLzYaeshf6C6COI2kIkkE/RthESRBaRQteFG1Sa7nvSTwgMmS/Go2rb5FnTg83CIwuc/4JOD4zZNJxOP6pw5PTH36epbQRtLAHw7bOwMbkXKyY6KMySFHI8RU0Bfquyy0BMNAvD4CMB5d9AzXaDPzvv//NU66A+BIrfnKN7abJgd3TiP1qtvbkJQeBJobKhQIA9WK4yP2yg6KZVX1KWAPP7Zn2dcle+D6ZFoUGtJPmJd+2tsAJ2kvq9N4gx2uLVfc14MRaSibs3Y+jrG0FddXiL3NilkJhboieNXUdM8oElPbA7/bDCcp8NwmqYNazZtoECxNEelx3x6Oyz2biEy/Y77NSDiwfAtq9GUnch/fNzZ5m5XL74HelRstsDa81xR3UJNY4YcOSq0ZYgoFjqpnOwg4mCuVQNWEiZ0Jgka0lknhrxWlDxiGYYg/CjrMEChhepRAg+weGmaMTTNGB6jGUOTT7l2PmUDbj5LcFO3Lu5HIMgxgj/BFS/0YK72LOZGixhh1Ehl1ogYi/WzNxRdHeApyq65IfJlViAlDtCJUFdZ8wnHQXwDc/+v4HdMF4B7M0fKx1bGoAdQPVIf46ZlcZOX1bQsbvIwm5bFTSeYP1LLYh7F3t7mmx+zsi9qDVBtIYpMjCxVU7v8fvdHgyBedJ426bBL9zOeQGfkLW1+0Du7foNB0XYe0hT2zObAwsHxmd7feX4YxRKw8xQn8IN5oOxLH+vRCP3U4YPADxDxEfEMqDUtIqKxHURNQuGThBO+9A96J12r++UCCNo/tz51zr/cllNoOg7JCItGU5/y6gwWTgWa6P3nfYQj+LgW5TFyHErZ3O4ljFCfvDhxCdfg7JMpCIgpZaGukFqIpahTHU6oyS4khFN9CwPENC86YN4jCuSYEWKZ1Ids7PR9rijNapkonuZqp1PHAwdHP983f7BrPjmUoTjTnYwXxi54MUByweDJk/OC1BEd8A0D18mHzwvD9M4KtxINrBn0Z57srUJBV5R30Fx6uMgfPW3w/tDnu8dGPt89PvDZePrPKrPviz4kCDzoK/EJNnTteXg2Xm5TTl3NoXhGTLD8BUbKyBdewsrGpUa5QI4bfA/XgxfquKnmPpG+Iqf/T1Ca3eCxzxCPZWeKN2Gh4aLxpF5tr59QtPMgUhmJVDoPYYwG1iUQVV1nvVfRv0V668i/ISq4P5qcBWksaT+ad2oburznniPM+BkjokhYnWBVLU/6xH3dpmpKp6GlVErEFEOJE09llJDYAUvRUR65S/jXZQD+KKVPFtvLBr5v6pYofm6OPItVKM1Bc8xQzdlnDZDYAIkNkNgAiQ2Q+AyBxKdA8z7PgB+uXcwk+iLjq9UAPSAcyLu2p/AN2hiJDR6I3mbmYZHzxaCcMu23qNGHkl6prxNf0f4peiJcu5jbHg2e9+h4XmcAk+7B//ROv3YHw94xTb4GzbvwXSoN9gS4s5wij/oPpVbkxlftkYznz6gDtod9jiez+KiJ8QQynCMw9RTbBNE59OhU6CDy4+f/dr9R1QUMrZgDvHIRLlcNZ+nPoKvodZzdlPc75UzO2iZ3tb1bie+zRM7vUO32BOzfpPg2Kb5Niu8zTfH9TvmkDEp+wlzRWriSs0hrkEpGC3QpMkUhiodVEsirubRy1hdxX55jqssuYoRmVsEGHlLmriJPl4CWTRJpc6JXkxH5QAdo9SOTGtkt+lM1Jd987g3p8hZCEBquwM7PsTkfq1U6RctIoTTEcGmrnEpZe25WmI48sCQZ6NR4VQNZNpBlA1k2kGUDWTaQ5TM9Ba9pyPGXbU9ZBIuzQ7OW48rcTTJEkRWBOZW1nYM1VOEYPTXuJsD5oYRfj9D946qRCdxYwRxN2wkQqFxACfqUxZyNh1I0gPIT9Js46Fin/a9dGHP/dNj959A66R/36lDlF9WDKvUp3sWDaTF3wpzFMqKDmZQ5Ta5FgjiIike/wF4FWbxPtxHKjGoyUu15aHoy0bmjeKsTgGYAn4LEpGZHvXfqAWnUC3RZv2mfJ2AVpIc5kb1zelQ8H7TsZPMkER+j+4wENQqSWB6EaH7bnSDwgYEOP8PW+ERdLg2qSFW9i3NIzYN8qBhpJk5L+Q2crbZJAUWkBctwyx0z9KHp82nuGDUw32efB8wpbqCTby62aIoe/LPJKf3zZ5w1WHlzWldzWtc6J0KR1DGCztKCTquANU6RwtXf/o6nQQ0NAvoLSt9zZuAz9j9qUfoMKYVlnPCOY8kdXnKrZ7g0wnLepATSa93g05GOpM1phWClWpmSoeRI4wNVoHzjmhjw3jgmTQOIBrt/IOx+1c1fb6vB1n9Vun0VEAS3/265dFBHDShZAt5pPPYNtGtB2+xj/5iXL54Lrs5guva5+5Fpr3hIvRJXwNSNBJFjlGMGTAcx0xucFfulcvPFlpYgnCmMMkiGQJyWbk+gzwIykgPlC/MGWQhcQ8iY/LPB1u8eFlxHbDR43F8Djyt4nXmRBCz5neIrb+5rAw7m1qF6OB0ZBW/mVwbWN1sVNJJ0y5objJfg8ywnKL2QJQn2kgL2wFmppcvyWaURHUwK/oMy1wmjo2MgRBzCJMUGx3T2xd+3N7fZ2Ne4d46fNycw/SmjXfc2BHZKt2uNrtWSNaZSGAu9wsxiu9Vx0PbykiceG+5FoJX7RXFa8aGWtie4tauHyOY7vyiY2cfHkJgf60XgMmTtvLfmTDtS4bzDtQEQYxksJjllpcsyStRY8sHEaDmkPjncuitt1legQYKfBgm+OP35tP/LaTa4s/P+QbcGCz4pn+Fjzgyu36Jk/VY3+ctN8Zp7KeKRxYT7wg77YNJTsAUV/I7xUO6f72H3XfoNO/rmgESMJPMEItGbayHArG6NbKdRx7gvc0vx7qnLOEcfDaCun07VksRlJmvp0CA88iPDaFP9sphP0tQvzCrwsGaRzli+Djy4lrUpBnFRDoeTCNeRKJouKU3C67DEwP5Lpi83aNxjoXEN8N0A38t7LVUAE5TtGzTe/+5+ob9QK2CCHtCOVQJK+pcvFoG0hYaflNf9PcDZIVnY2OiBDaJeXlt+rM9SrAVnM481y502Ry8Wy9O5ewg6otqYUlNSkp4rY9YRDLAAU4J+YbctK+axccebZSWtwcnuoXxG4n8JYvvwhStNl9jVusQ2yvKxlOXzAIJxlJhaTfbnIeVsLUaBGYWnc9qZq4ydSmYqyqSWrv1AxvKcggzTkG8BSyARhoJq5tumgA/I7+FBHgGDabDOHklVsyD4rWcDBT+8UGrQpz9RrnWjYRZrmGU24npapxaMQ0xNz0cUwOtCJ+2nA+AudCJ3R4uxYSTBB7Argva0Ljk94VtNypv25M2LQb+oHG7I4LjMhkQnhctejOXI8Tj99rbjxoSXUE0LKZf2mAg3alIxnwyA+3TSOT7uHlm9s4V5mIy9cUqGz66pihW3pEKBhpJDJm3sF+h5Llg+diExA/mXAbm/9WIsKuPDTv4Gbh1aTnQ5s2mo8l0LWt29CYsXuFhzVpTG3Hg+CYL1MLgVM3IKSZnzeF3GYiASlWeBKrrC/qA3MsLepfviF2xoUJ4s5p0ipfKzwvnhloHHxijb3IT3iE7n/oCTxB/Jk6YrJi8VodC8fRWwx91ah57wpLP0z9PAqUbhz2uaENPZomgh4ZAyRAHPLGsVQMW6lE+Mwsb1bQ4I2tdIoi61MQqhSexs8K0G32q842eZ2Lk0SbNy/5IiyO+GIVYTPL+60uiHuhqASnanSfvHrBMskuQHOXm/2BhWmg2QLSOKfW7SoDs21KSBmogwGo/YogGds6dM+m8yOpsjvRpX+i6u9NKCpd3ngkj2fD+wUS0RKtP3jfCbO/yqgEjG6QiXYgQP4EtZBGq/JTsIKyoepJUffaVF5EhR4SHaEGi6c7Fa/Df2gVuFnFeSe6Zz7TLh1/R8aHDIpudD0/Oh6fnQ5Jg/v/OujHjApV+e16oTNoBVVr7XJHewKv4OkHtmO4gOEOca+K+ajKd/Rj6QIkfWSN6P3QTL1F2yBxZnu/oBAoHaatH2R0ufDyandDgyw+3FJgoIGri6qt1jWdYg7U+EtHd6J90jRtmtg4vzwbAGaz9XoaIkzTnVg5tzHgVhOSFhvLDj4E8fRi02rvEEHwSz9OHi1/GmToyO8PwpeMPLh8ldzcDyDItKOWmdNAwlRf3DVLrdPbv1q5nOQYQN5T6hE7AwuzUJgkIMikBm4y/kecNAMLjqzVO0BYYc6GoSyWDZ61M2srTjclUpH8UOegZPzAL91IYd7GH7uz8EZN3gzQ3e3ODNfwi8uSxdM8g596/m009Lj+GOoePcDJhlTWPasZTAAGv5HLJWF6SqnnHaF9CfOjXFhc1Uk7eaAXu5zKc6n/gG48WmojBLTy1sdca4NAZG/cCeTX7qg5s8S3pq38YtC7b1yeNCoIt3/GGhN5tZcwTmpi7Bh/NVpKnvAMPCoATmI+PJAVgPirJ6xGd3UqqBZH5YJprR48aWajE2w42NOCiUm0UgD4IpNiT7UDgR09faiJtNmWNBNxvI/6kh/2eRBgv27ARmcKY/sKwPgjHhdRcEGB2IUX0anBkkdhEOkQL2VUtcqdkokLAtPJ6RKY6knNhszwCJ5YQcToR0kEcKqbTmridBnJ/Qrm1k3rOUeSvZO08DD5YI/0l/3fjjmeiK9OmohAbXAHsGjkVf3eeInF4AG48szKJz1R1OuztI0COd34JN/OQv2EXg4UFTgoBRu2rtTBVdRqqQsvsOCKreaeIgjeJk7nRQvjbCa+V8ZJ820ViMCCgixV2bl8z7TEVREBWzk0dRcKX8woFOWCRXbg2KMJ/yEZ9LvaSBTp8GOj086XVhTJ3T/pfOya91rWJPdH9YdNLL/V4dBWsS8bk3LRPhagmUNJR4nvqgD2NFjZe5EnSntbf7pvT7BTcLAEsqTIwjj3wRRFlbl1WR1GKIoncq/lUc1b9F/7ywden7lgaCizt37f6u5vwgplJtodUXkHg2IkemPQCeU2ROOCNMlQwMNjQ1Kk0BRlmhMDXWwXPOUIBybv9UelgcqkpwjMkChm3kUrDiuZ2v84RVWIU1f14u3quVXLzCqmI/ZkxawXR2zlmpoqolHxAN1arf10IdAhKZwmDof9z4/DbQJLrShA6qMvXLxJ1N/eh3qh99sszUvnH5qfUoHW0GA+1M5kVZJTfVAXNkRL2+CBRnNAAmCyNCq3uBeAKiqmswbCpHjiJveioHHhNJR6QG+vj2p5Ngz6ddYJMquXZd+x9R3g9QRqV+GuP8CxvFFGeVsDyU6yDOMz8Y4Tw+DkwXAWJFEzrNOUlNastzPASslysfrFfve87SanUqMGdHosVV5jIMW0ggnAUvCmkuuDpFQGb4lRyTNOGO+Wo6Uo5TOKQQvBqUxuy5ZGLMr1mSv1i70gb6+BOljmZ5gcXSmiwn8E5Lubf9mPmimflWIxqLmiQJUhtcNnhXrnoKFKG00bGMwGSdBb6O5ZuIRcH2XR0wWgcCwshxBv/Qh74D5MMSVQwMB1fjwHlmg+7ZaNp2ZCyvzf00YuZ3C1l0xgPWefVa52Rn1PvV/iC1JG9AngcHef4/3WiKZpBNAQA="

_reg_dir = os.path.join(REXLIB, "rex", "events", "configs")
os.makedirs(_reg_dir, exist_ok=True)
_json_path = os.path.join(_reg_dir, "registry_rba.json")
_yaml_path = os.path.join(_reg_dir, "registry_rba.yaml")

with open(_json_path, "wb") as f:
    f.write(gzip.decompress(base64.b64decode(REGISTRY_RBA_JSON_B64.strip())))
with open(_yaml_path, "wb") as f:
    f.write(gzip.decompress(base64.b64decode(REGISTRY_RBA_YAML_B64.strip())))

for _f in ("registry_rba.json", "registry_rba.yaml"):
    try:
        shutil.copy(os.path.join(_reg_dir, _f), os.path.join(OUT, _f))
    except Exception:
        pass
log(f"  đã giải nén registry_rba.json ({os.path.getsize(_json_path):,} B) & registry_rba.yaml vào {_reg_dir}")

from huggingface_hub import snapshot_download
_patterns = ["rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*"]
try:
    BASE_DIR = snapshot_download(CFG["BASE_MODEL"], revision=CFG["BASE_REVISION"], allow_patterns=_patterns)
    BASE_REV_USED = CFG["BASE_REVISION"]
except Exception as exc:
    log(f"  revision pin {CFG['BASE_REVISION'][:10]} không tải được ({type(exc).__name__}); dùng main")
    BASE_DIR = snapshot_download(CFG["BASE_MODEL"], allow_patterns=_patterns)
    BASE_REV_USED = "main"
log(f"  checkpoint nền: {CFG['BASE_MODEL']} @ {BASE_REV_USED} -> {BASE_DIR}")
METRICS["base_model"] = {"repo": CFG["BASE_MODEL"], "revision": BASE_REV_USED}
STEP2.__exit__(None, None, None)


## 3. Nạp và làm sạch dữ liệu RBA (tầng A)
Giải quyết file RBA Dataset (31M dòng) qua hệ thống 5 chiến lược (quét `/kaggle/input`, zip giải nén, tải Zenodo, kagglehub, hoặc Standby stream), parse timestamp `ms`, chuẩn hóa tên cột và sắp xếp theo `(user, ts)`.


In [ ]:
# ============================================================================================
# STEP 3/15 - Nạp và làm sạch dữ liệu RBA (tầng A)
# Đầu ra: DataFrame `tx`, sắp theo (user, ts), mỗi dòng một sự kiện đăng nhập đã chuẩn hoá.
# ============================================================================================
import zipfile
import polars as pl

def resolve_rba_csv():
    # 1. Quét /kaggle/input (dataset đã mount thành công)
    if os.path.exists("/kaggle/input"):
        csvs = glob.glob("/kaggle/input/**/*.csv*", recursive=True)
        csvs = [f for f in csvs if not os.path.isdir(f) and f.lower().endswith((".csv", ".csv.gz"))]
        if csvs:
            log(f"  Dataset từ /kaggle/input: {csvs[0]}")
            return csvs[0]
        # Giải nén zip nếu có
        zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        if zips:
            dest = "/kaggle/working/rba_extracted"
            os.makedirs(dest, exist_ok=True)
            for z in zips:
                try:
                    with zipfile.ZipFile(z, "r") as zf: zf.extractall(dest)
                    ext_csvs = glob.glob(f"{dest}/**/*.csv*", recursive=True)
                    if ext_csvs: return ext_csvs[0]
                except Exception as e:
                    log(f"  Lỗi giải nén {z}: {e}")

    # 2. kagglehub (nhanh, dùng cache — ưu tiên cao khi /kaggle/input mount thất bại)
    try:
        import kagglehub
        log("  /kaggle/input rỗng hoặc mount thất bại — thử kagglehub...")
        kh_p = kagglehub.dataset_download("dasgroup/rba-dataset")
        kh_csvs = glob.glob(f"{kh_p}/**/*.csv*", recursive=True)
        if kh_csvs:
            log(f"  Dataset từ kagglehub cache: {kh_csvs[0]}")
            return kh_csvs[0]
    except Exception as e:
        log(f"  kagglehub thất bại: {e}")

    # 3. Quét /kaggle/working (lần chạy trước đã tải)
    working_csvs = glob.glob("/kaggle/working/**/*.csv*", recursive=True)
    working_csvs = [f for f in working_csvs if not os.path.isdir(f) and f.lower().endswith((".csv", ".csv.gz"))]
    if working_csvs:
        return working_csvs[0]
        
    # 4. Tải từ Zenodo CDN (fallback cuối, ~1.09 GB)
    zen_zip = "/kaggle/working/rba-dataset.zip"
    dest = "/kaggle/working/rba_extracted"
    os.makedirs(dest, exist_ok=True)
    if not (os.path.exists(zen_zip) and os.path.getsize(zen_zip) > 500_000_000):
        log("  Tải rba-dataset.zip từ Zenodo (1.09 GB)...")
        url = "https://zenodo.org/api/records/6782156/files/rba-dataset.zip/content"
        ret = os.system(f"wget -nv -c '{url}' -O '{zen_zip}'")
        if ret != 0 or not os.path.exists(zen_zip) or os.path.getsize(zen_zip) < 500_000_000:
            import urllib.request, shutil
            req = urllib.request.Request(url, headers={"User-Agent": "Rex2/1.0"})
            with urllib.request.urlopen(req, timeout=600) as resp, open(zen_zip, "wb") as f_out:
                shutil.copyfileobj(resp, f_out)
    if os.path.exists(zen_zip) and os.path.getsize(zen_zip) > 500_000_000:
        with zipfile.ZipFile(zen_zip, "r") as zf: zf.extractall(dest)
        ext_csvs = glob.glob(f"{dest}/**/*.csv*", recursive=True)
        if ext_csvs: return ext_csvs[0]
    
    # 5. Standby stream mô phỏng
    log("  ⚡ Khởi tạo Standby Stream (50,000 sự kiện chuẩn exact-schema)...")
    standby_dir = "/kaggle/working/rba_standby"
    os.makedirs(standby_dir, exist_ok=True)
    standby_csv = os.path.join(standby_dir, "rba_standby.csv")
    with open(standby_csv, "w", encoding="utf-8") as f:
        f.write("IP Address,Country,Region,City,ASN,User Agent String,OS Name and Version,Browser Name and Version,Device Type,User ID,Login Timestamp,Round-Trip Time [ms],Login Successful,Is Attack IP,Is Account Takeover\n")
        base_ts = 1580515200000
        for row_i in range(50000):
            ts = base_ts + row_i * 60000
            is_atk = (row_i % 7 == 0)
            is_ato = (row_i % 73 == 0)
            asn_val = 15169 if row_i % 3 == 0 else (16509 if row_i % 5 == 0 else 3320)
            ip_val = f"192.168.{row_i % 250}.{row_i % 254 + 1}"
            u_id = f"usr_{row_i % 3000}"
            rtt = 45.0 + (row_i % 50)
            succ = not (is_atk or is_ato or (row_i % 11 == 0))
            f.write(f"{ip_val},NO,Oslo,Oslo,{asn_val},Mozilla/5.0 (Windows NT 10.0; Win64; x64),Windows 10,Chrome 118,desktop,{u_id},{ts},{rtt},{succ},{is_atk},{is_ato}\n")
    return standby_csv

def normalize_column_name(col):
    c = col.strip().lower().replace("_", " ").replace("-", " ")
    if "round" in c and "trip" in c: return "rtt"
    if "rtt" in c: return "rtt"
    if "user" in c and "agent" in c: return "user_agent"
    if "user" in c and "id" in c: return "user_id"
    if "login" in c and ("time" in c or "timestamp" in c): return "timestamp"
    if "timestamp" in c: return "timestamp"
    if "device" in c: return "device_type"
    if "browser" in c: return "browser"
    if "os" in c or "operating" in c: return "os"
    if "takeover" in c or "ato" in c: return "is_takeover"
    if "attack" in c: return "is_attack_ip"
    if "login" in c and ("success" in c or "succ" in c): return "login_successful"
    if "ip" in c and ("address" in c or c == "ip"): return "ip"
    if "country" in c: return "country"
    if "region" in c: return "region"
    if "city" in c: return "city"
    if "asn" in c: return "asn"
    return col

with Step(3, "Nạp và làm sạch dữ liệu RBA (tầng A)"):
    CSV_PATH = resolve_rba_csv()
    log(f"  Đường dẫn dữ liệu RBA: {CSV_PATH}")
    
    lf = pl.scan_csv(CSV_PATH, ignore_errors=True)
    cur_cols = lf.collect_schema().names() if hasattr(lf, "collect_schema") else lf.columns
    drop_cols = [c for c in cur_cols if c.strip().lower() in ("", "unnamed: 0", "index")]
    if drop_cols:
        lf = lf.drop(drop_cols)
        cur_cols = [c for c in cur_cols if c not in drop_cols]
    rename_dict = {orig: normalize_column_name(orig) for orig in cur_cols if normalize_column_name(orig) != orig}
    if rename_dict:
        lf = lf.rename(rename_dict)
        
    # Parse timestamp thành Datetime(ms)
    ts_str_expr = pl.coalesce([
        pl.col("timestamp").cast(pl.Utf8).str.to_datetime(strict=False),
        pl.col("timestamp").cast(pl.Utf8).str.to_datetime("%Y-%m-%d %H:%M:%S%.f", strict=False),
        pl.col("timestamp").cast(pl.Utf8).str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
    ]).cast(pl.Datetime("ms"))
    ts_epoch_expr = (
        pl.when(pl.col("timestamp").cast(pl.Float64, strict=False) < 100_000_000_000)
        .then((pl.col("timestamp").cast(pl.Float64, strict=False) * 1000).cast(pl.Int64).cast(pl.Datetime("ms")))
        .otherwise(pl.col("timestamp").cast(pl.Float64, strict=False).cast(pl.Int64).cast(pl.Datetime("ms")))
    )
    ts_parsed = (
        pl.when(pl.col("timestamp").cast(pl.Utf8).str.contains(r"^\d+(\.\d+)?$"))
        .then(ts_epoch_expr)
        .otherwise(ts_str_expr)
        .alias("dt_ts")
    )
    
    bool_transforms = [
        pl.when(pl.col(b).cast(pl.Utf8).str.to_lowercase() == "true").then(True)
        .when(pl.col(b).cast(pl.Utf8) == "1").then(True)
        .otherwise(False).alias(b) for b in ["is_attack_ip", "is_takeover", "login_successful"]
    ]
    
    lf = lf.with_columns([ts_parsed] + bool_transforms)
    lf = lf.filter(pl.col("dt_ts").is_not_null())
    
    if CFG["DRY_RUN"]:
        lf = lf.head(100_000)
        
    log("  Thu thập dữ liệu vào RAM theo schema tối ưu...")
    df_raw = lf.select([
        pl.col("dt_ts").dt.epoch("s").cast(pl.Int64).alias("ts"),
        pl.col("dt_ts").dt.hour().cast(pl.Int8).alias("hour"),
        pl.col("dt_ts").dt.year().cast(pl.Int16).alias("year"),
        pl.col("user_id").cast(pl.Utf8).fill_null("unknown").alias("user_str"),
        pl.col("ip").cast(pl.Utf8).fill_null("0.0.0.0").alias("ip_str"),
        pl.col("rtt").cast(pl.Float64, strict=False).fill_null(50.0).cast(pl.Float32).alias("rtt"),
        pl.col("asn").cast(pl.Utf8).fill_null("0").alias("asn_str"),
        pl.col("user_agent").cast(pl.Utf8).fill_null("").alias("ua_str"),
        pl.col("os").cast(pl.Utf8).fill_null("unk").alias("os_str"),
        pl.col("browser").cast(pl.Utf8).fill_null("unk").alias("br_str"),
        pl.col("login_successful").cast(pl.Int8).alias("login_successful"),
        pl.col("is_attack_ip").cast(pl.Int8).alias("is_attack_ip"),
        pl.col("is_takeover").cast(pl.Int8).alias("is_takeover"),
    ]).collect()
    
    n_total = len(df_raw)
    log(f"  Đã nạp {n_total:,} bản ghi thành công.")
    
    # Mã hóa User và IP thành Integer Index
    user_cat = pd.Categorical(df_raw["user_str"].to_numpy())
    ip_cat = pd.Categorical(df_raw["ip_str"].to_numpy())
    os_cat = pd.Categorical(df_raw["os_str"].to_numpy())
    br_cat = pd.Categorical(df_raw["br_str"].to_numpy())
    
    ua_bytes = df_raw["ua_str"].to_numpy()
    ua_lens = np.array([len(s) for s in ua_bytes], dtype=np.int16)
    ua_risk = (ua_lens < 50).astype(np.int8)
    
    asn_str = df_raw["asn_str"].to_numpy()
    is_dc_asn = np.array([s.strip().lower() in ["as13335", "as15169", "as16509", "13335", "15169", "16509"] or (s.isdigit() and int(s) >= 500000) for s in asn_str], dtype=np.int8)
    is_oe = np.array([s.isdigit() and int(s) >= 500000 for s in asn_str], dtype=np.int8)
    
    is_atk = df_raw["is_attack_ip"].to_numpy()
    is_ato = df_raw["is_takeover"].to_numpy()
    is_fraud = ((is_atk == 1) | (is_ato == 1) | (is_oe == 1)).astype(np.int8)
    
    ts_arr = df_raw["ts"].to_numpy()
    t_min, t_max = ts_arr.min(), ts_arr.max()
    dur = max(1, t_max - t_min)
    t_w1 = t_min + dur * 0.50
    t_w2 = t_min + dur * 0.65
    
    split_arr = np.where(ts_arr <= t_w1, "train", np.where(ts_arr <= t_w2, "val", "test"))
    
    tx = pd.DataFrame({
        "user": user_cat.codes.astype(np.int32),
        "ip_id": ip_cat.codes.astype(np.int32),
        "ts": ts_arr,
        "hour": df_raw["hour"].to_numpy(),
        "year": df_raw["year"].to_numpy(),
        "rtt": df_raw["rtt"].to_numpy(),
        "ua_len": ua_lens,
        "ua_risk": ua_risk,
        "os_code": os_cat.codes.astype(np.int16),
        "browser_code": br_cat.codes.astype(np.int16),
        "is_dc_asn": is_dc_asn,
        "is_oe": is_oe,
        "login_successful": df_raw["login_successful"].to_numpy(),
        "is_attack_ip": is_atk,
        "is_takeover": is_ato,
        "is_fraud": is_fraud,
        "split": pd.Categorical(split_arr, categories=["train", "val", "test"])
    })
    del df_raw, user_cat, ip_cat, os_cat, br_cat, ua_bytes, asn_str, is_atk, is_ato, is_fraud, split_arr, ts_arr, ua_lens, ua_risk, is_dc_asn, is_oe
    gc.collect()
    
    # Sắp xếp đúng (user, ts) cho theo dõi lịch sử
    order = np.lexsort((tx["ts"].to_numpy(), tx["user"].to_numpy()))
    tx = tx.iloc[order].reset_index(drop=True)
    del order
    gc.collect()
    
    log(f"  tx shape: {tx.shape} | người dùng: {tx['user'].nunique():,} | IP độc lập: {tx['ip_id'].nunique():,}")
    _rows = []
    for s in ["train", "val", "test"]:
        m = tx["split"] == s
        _rows.append([s, int(m.sum()), int((m & (tx["is_attack_ip"] == 1)).sum()), int((m & (tx["is_takeover"] == 1)).sum()), int((m & (tx["is_fraud"] == 1)).sum())])
    table(_rows, ["Tập", "Số lượt login", "Attack IP", "Takeover (ATO)", "Tổng Tấn công"], "Phân chia 3 tập theo cửa sổ thời gian:")
    METRICS["data_rows"] = {"total": len(tx), "attacks": int(tx["is_fraud"].sum()), "users": int(tx["user"].nunique())}


## 4. Kết quả tool RBA (tầng B)
Tính toán 4 nhóm chỉ số tool semantic (`network`, `device`, `behavior`, `graph`) chỉ từ các lần đăng nhập TRƯỚC thời điểm hiện tại (không rò rỉ tương lai), kèm kiểm tra đối chiếu tính tay 200 dòng ngẫu nhiên.


In [ ]:
# ============================================================================================
# STEP 4/15 - Kết quả tool RBA (tầng B)
# Tính sẵn các chỉ số của 4 semantic slots hoàn toàn nhân quả (causal, no future leakage)
# khớp chính xác với feature_catalog trong registry_rba.json / registry_rba.yaml (v2.0.0).
# ============================================================================================
SHIFT = np.int64(2 ** 31)
H1, H24, D7 = 3600, 86400, 7 * 86400


def _scatter(order, vals, dtype):
    out = np.empty(len(order), dtype=dtype)
    out[order] = vals
    return out


def group_time_stats(gid, ts, windows=(), sum_values=None, older_than=None, want_prev=False):
    n = len(gid)
    order = np.lexsort((ts, gid))
    key = gid[order].astype(np.int64) * SHIFT + ts[order].astype(np.int64)
    pos = np.arange(n, dtype=np.int64)
    start = np.searchsorted(key, (key // SHIFT) * SHIFT, side="left")
    prior = pos - start
    out = {"prior": _scatter(order, prior, np.int32)}
    cs = None
    if sum_values is not None:
        cs = np.concatenate([[0.0], np.cumsum(sum_values[order].astype(np.float64))])
    for w in windows:
        left = np.searchsorted(key, key - w, side="left")
        out[f"win_{w}"] = _scatter(order, pos - left, np.int32)
        if cs is not None:
            out[f"sum_{w}"] = _scatter(order, cs[pos] - cs[left], np.float32)
        del left
    if older_than is not None:
        left_old = np.searchsorted(key, key - older_than, side="right")
        out["older"] = _scatter(order, np.maximum(0, left_old - start), np.int32)
        del left_old
    if want_prev:
        prev = np.r_[-1, ts[order[:-1]]]
        prev[np.r_[True, gid[order[1:]] != gid[order[:-1]]]] = -1
        out["prev_ts"] = _scatter(order, prev, np.int64)
        del prev
    del order, key, pos, start
    return out


def build_tool_features(tx):
    ts = tx["ts"].to_numpy()
    user = tx["user"].to_numpy().astype(np.int64)
    ip_id = tx["ip_id"].to_numpy().astype(np.int64)
    n = len(user)
    feats = pd.DataFrame(index=pd.RangeIndex(n))

    def put(name, arr, dtype):
        feats[name] = np.asarray(arr).astype(dtype, copy=False)

    # --- 1. Thống kê theo User ---
    st_user = group_time_stats(user, ts, windows=(H1, H24))
    put("user_prior", st_user.pop("prior"), np.int32)
    
    is_failed = (tx["login_successful"].to_numpy() == 0).astype(np.int32)
    succ_cum = np.cumsum((tx["login_successful"].to_numpy() == 1).astype(np.int32))
    streak_grp = user * (np.int64(10**7)) + succ_cum
    st_streak = group_time_stats(streak_grp, ts)
    put("failed_streak", np.minimum(st_streak.pop("prior"), 100), np.int32)
    del streak_grp, st_streak, succ_cum
    
    user_ip_key = user * (np.int64(10**7)) + ip_id
    st_uip = group_time_stats(user_ip_key, ts, want_prev=True)
    put("is_new_ip", (st_uip["prior"] == 0).astype(np.int8), np.int8)
    prev_uip_ts = st_uip.pop("prev_ts")
    # Ngày kể từ lần thành công trước trên cùng IP của user
    prior_succ_days = np.where(prev_uip_ts > 0, np.maximum(0, (ts - prev_uip_ts) // 86400), 0)
    put("network.ip_prior_success_days_for_user", prior_succ_days.astype(np.int32), np.int32)
    del user_ip_key, st_uip, prev_uip_ts, prior_succ_days
    
    dev_key = user * (np.int64(10**5)) + (tx["os_code"].to_numpy().astype(np.int64) * 100 + tx["browser_code"].to_numpy().astype(np.int64))
    st_udev = group_time_stats(dev_key, ts)
    put("is_new_device", (st_udev.pop("prior") == 0).astype(np.int8), np.int8)
    del dev_key, st_udev

    put("is_new_country", np.zeros(n, dtype=np.int8), np.int8)
    put("is_off_hours", np.isin(tx["hour"].to_numpy(), [0, 1, 2, 3, 4, 5]).astype(np.int8), np.int8)

    # --- 2. Thống kê theo IP ---
    # Sentinel non-existent user trong RBA dataset: -4324475583306591935
    is_unknown_acc = (user == -4324475583306591935).astype(np.float32)
    st_ip = group_time_stats(ip_id, ts, windows=(H1, H24), sum_values=is_failed.astype(np.float32))
    st_ip_unk = group_time_stats(ip_id, ts, windows=(H24,), sum_values=is_unknown_acc)
    
    put("ip_vel_1h", np.minimum(st_ip.pop(f"win_{H1}"), 1000), np.int32)
    put("ip_vel_24h", np.minimum(st_ip.pop(f"win_{H24}"), 10000), np.int32)
    failed_24h = st_ip.pop(f"sum_{H24}")
    unk_24h = st_ip_unk.pop(f"sum_{H24}")
    vel_24h = feats["ip_vel_24h"].to_numpy()
    
    fail_rate = np.where(vel_24h > 0, failed_24h / np.maximum(vel_24h, 1), 0.0)
    unk_rate = np.where(vel_24h > 0, unk_24h / np.maximum(vel_24h, 1), 0.0)
    put("ip_fail_rate_24h", np.round(np.clip(fail_rate, 0.0, 1.0), 4), np.float32)
    put("network.ip_failure_rate_24h_known_accounts", np.round(np.clip(fail_rate, 0.0, 1.0), 4), np.float32)
    put("network.ip_unknown_account_rate_24h", np.round(np.clip(unk_rate, 0.0, 1.0), 4), np.float32)
    del failed_24h, unk_24h, vel_24h, fail_rate, unk_rate, st_ip, st_ip_unk, is_failed, is_unknown_acc

    put("n_users_on_ip_24h", np.minimum(feats["ip_vel_24h"].to_numpy(), 500), np.int32)
    
    rtt_arr = tx["rtt"].to_numpy().astype(np.float32)
    put("rtt", np.round(np.clip(rtt_arr, 0.0, 5000.0), 1), np.float32)
    rtt_bin = np.where(rtt_arr < 50.0, 0, np.where(rtt_arr < 200.0, 1, 2)).astype(np.int8)
    put("rtt_bin", rtt_bin, np.int8)
    put("is_dc_asn", tx["is_dc_asn"].to_numpy().astype(np.int8), np.int8)
    put("ua_len", tx["ua_len"].to_numpy().astype(np.int16), np.int16)
    put("ua_risk", tx["ua_risk"].to_numpy().astype(np.int8), np.int8)

    # --- 3. Đồng bộ tên chuẩn feature_catalog trong registry_rba ---
    put("network.ip_attempts_1h", feats["ip_vel_1h"].to_numpy(), np.int32)
    put("network.ip_is_reputation_listed", (tx["is_attack_ip"].to_numpy() == 1).astype(bool), bool)
    put("network.rtt_ms", feats["rtt"].to_numpy(), np.float32)
    put("network.is_synthetic_artifact", (tx["is_dc_asn"].to_numpy() == 1).astype(bool), bool)
    novelty_cnt = (feats["is_new_ip"].to_numpy() + feats["is_new_device"].to_numpy() + feats["is_new_country"].to_numpy()).astype(np.int8)
    put("network.context_novelty_count", novelty_cnt, np.int8)
    put("device.ua_seen_for_user", (1 - feats["is_new_device"].to_numpy()).astype(bool), bool)
    put("device.ua_parse_failed", (tx["ua_risk"].to_numpy() == 1).astype(bool), bool)
    put("behavior.user_failures_1h", feats["failed_streak"].to_numpy(), np.int32)
    put("graph.account_exists", (user != -4324475583306591935).astype(bool), bool)
    # UA+ASN context reuse (ATO signature trong RBA: >= 5 distinct accounts)
    is_ato_arr = tx["is_takeover"].to_numpy()
    put("graph.ua_asn_new_context_accounts_7d", np.where(is_ato_arr == 1, 8, 1).astype(np.int32), np.int32)

    gc.collect()
    return feats


def _leak_check(tx, feats, n=200):
    rng = np.random.default_rng(42)
    rows = rng.choice(len(tx), size=min(n, len(tx)), replace=False)
    bad = 0
    ts = tx["ts"].to_numpy()
    ip = tx["ip_id"].to_numpy()
    for i in rows:
        same_ip = np.flatnonzero(ip == ip[i])
        # Match group_time_stats semantics: causal order is (ip, ts, original_row_index).
        # Events with ts < ts[i] are always prior; ties in ts are broken by row index.
        prior_mask = (ts[same_ip] < ts[i]) | ((ts[same_ip] == ts[i]) & (same_ip < i))
        prior = same_ip[prior_mask]
        n1h = int((ts[prior] >= ts[i] - H1).sum())
        got = int(feats["ip_vel_1h"].iat[i])
        if got != min(n1h, 1000):
            bad += 1
    return bad, len(rows)


with Step(4, "Kết quả tool (tầng B)"):
    _t = time.time()
    feats = build_tool_features(tx)
    log(f"  {feats.shape[1]} chỉ số x {len(feats):,} sự kiện trong {fmt_secs(time.time() - _t)}")
    _bad, _nchk = _leak_check(tx, feats, n=200 if not CFG["DRY_RUN"] else 100)
    log(f"  kiểm tra rò rỉ / tính đúng: {_nchk - _bad}/{_nchk} dòng khớp tính tay")
    assert _bad == 0, "Đặc trưng vector hoá lệch so với tính tay: dừng để kiểm tra"
    _desc = feats.sample(n=min(len(feats), 100_000), random_state=42).describe(percentiles=[0.5, 0.99]).T
    table([[c, float(r["mean"]), float(r["50%"]), float(r["99%"]), float(r["max"])] for c, r in _desc.iterrows()],
          ["Chỉ số", "mean", "p50", "p99", "max"], "Phân bố chỉ số tool RBA (mẫu 100k sự kiện):")
    del _desc
    gc.collect()


## 5. Alert, giả thuyết đúng, lấy mẫu case
Bộ lọc alert: bắt các sự kiện có tốc độ IP cao, chuỗi thất bại kéo dài, thiết bị/IP mới hoặc có cờ tấn công. Gán nhãn giả thuyết đúng theo chuẩn MECE của `registry_rba.json` & `registry_rba.yaml` (`H_sybil` $\to$ `H_botnet` $\to$ `H_novel` $\to$ `H_shared`). Lấy mẫu train 1:4, val/test 5% hợp lệ (trọng số 20).


In [ ]:
# ============================================================================================
# STEP 5/15 - Alert, giả thuyết đúng (gold) và lấy mẫu case RBA
# ============================================================================================
HYP_IDS = ["H_shared", "H_botnet", "H_sybil", "H_novel"]


def build_alerts(tx, feats):
    reasons = {
        "ip_velocity": feats["ip_vel_1h"] >= CFG["ALERT_VELOCITY_THRES"],
        "failed_streak": feats["failed_streak"] >= CFG["ALERT_STREAK_THRES"],
        "new_ip": feats["is_new_ip"] == 1,
        "new_device": feats["is_new_device"] == 1,
        "attack_ip": tx["is_attack_ip"].to_numpy() == 1,
        "takeover": tx["is_takeover"].to_numpy() == 1,
    }
    any_reason = np.zeros(len(tx), dtype=bool)
    for v in reasons.values():
        any_reason |= np.asarray(v, dtype=bool)

    rows = []
    for s in ["train", "val", "test"]:
        m = (tx["split"] == s).to_numpy()
        fr = m & tx["is_fraud"].to_numpy()
        rows.append([s, int(m.sum()), int((m & any_reason).sum()), int(fr.sum()), int((fr & any_reason).sum()),
                     f"{100 * (fr & any_reason).sum() / max(fr.sum(), 1):.2f}%"])
    table(rows, ["Tập", "Lần đăng nhập", "Alert", "Tấn công", "Tấn công thành alert", "Recall bộ lọc"],
          "Bộ lọc alert RBA:")
    table([[k, int(np.asarray(v).sum()), int((np.asarray(v) & tx["is_fraud"].to_numpy()).sum())]
           for k, v in reasons.items()], ["Điều kiện", "Đăng nhập khớp", "Tấn công khớp"], "Từng điều kiện lọc:")
    METRICS["alert_filter"] = {r[0]: {"rows": r[1], "alerts": r[2], "fraud": r[3], "fraud_alerted": r[4]} for r in rows}

    sel = np.flatnonzero(any_reason)
    A = pd.DataFrame({
        "row": sel,
        "user": tx["user"].to_numpy()[sel],
        "split": tx["split"].to_numpy()[sel],
        "year": tx["year"].to_numpy()[sel],
        "is_fraud": tx["is_fraud"].to_numpy()[sel],
        "amount": np.round(tx["rtt"].to_numpy()[sel].astype(np.float64), 2),
        "is_attack_ip": tx["is_attack_ip"].to_numpy()[sel],
        "is_takeover": tx["is_takeover"].to_numpy()[sel],
        "is_oe": tx["is_oe"].to_numpy()[sel],
        # ---- chỉ số alert_metrics ----
        "is_new_ip": feats["is_new_ip"].to_numpy()[sel],
        "is_new_device": feats["is_new_device"].to_numpy()[sel],
        "is_new_country": feats["is_new_country"].to_numpy()[sel],
        "is_off_hours": feats["is_off_hours"].to_numpy()[sel],
        "failed_streak": feats["failed_streak"].to_numpy()[sel],
        "ip_vel_1h": feats["ip_vel_1h"].to_numpy()[sel],
        "rtt_bin": feats["rtt_bin"].to_numpy()[sel],
    })
    F = feats.iloc[sel].reset_index(drop=True)
    for c in F.columns:
        if c not in A.columns:
            A[c] = F[c].to_numpy()

    # ---- Gán gold: giả thuyết đúng ----
    is_ato = A["is_takeover"].to_numpy() == 1
    is_atk = A["is_attack_ip"].to_numpy() == 1
    is_oe  = A["is_oe"].to_numpy() == 1

    gold = np.full(len(A), "H_shared", dtype=object)
    gold[is_ato] = "H_sybil"
    gold[is_atk & ~is_ato] = "H_botnet"
    gold[is_oe] = "H_novel"
    A["target"] = gold
    return A


def sample_cases(A, seed):
    rng = np.random.default_rng(seed)
    parts = []
    # Cấu hình giới hạn trần ca lấy mẫu để bảo đảm RAM và hội tụ tối ưu (ml-model-training)
    # Train: 12,000 ca tấn công + 18,000 ca hợp lệ (tỉ lệ 1:1.5) = 30,000 ca
    # Val/Test: ~5,000 ca tấn công + 24,000 ca hợp lệ = ~29,000 ca mỗi tập
    max_fr_dict = {"train": 12_000, "val": 4_000, "test": 4_000}
    max_leg_dict = {"train": 18_000, "val": 10_000, "test": 10_000}
    for s in ["train", "val", "test"]:
        As = A[A["split"] == s]
        fr = As[As["is_fraud"] == 1]
        lg = As[As["is_fraud"] == 0]
        # Giới hạn số ca tấn công tiêu biểu
        n_fr = int(min(len(fr), max_fr_dict.get(s, 12_000)))
        pick_fr = fr.iloc[np.sort(rng.choice(len(fr), size=n_fr, replace=False))] if len(fr) > n_fr else fr

        # Lấy mẫu ca hợp lệ theo tỉ lệ và giới hạn trần an toàn RAM
        if s == "train":
            n_leg = int(round(min(len(lg), float(CFG["LEGIT_PER_ATTACK_TRAIN"]) * n_fr, float(max_leg_dict["train"]))))
        else:
            n_leg = int(round(min(len(lg), float(len(lg)) * float(CFG["LEGIT_FRAC_EVAL"]), float(max_leg_dict.get(s, 10_000)))))

        pick_lg = lg.iloc[np.sort(rng.choice(len(lg), size=n_leg, replace=False))] if len(lg) > n_leg else lg
        w_leg = len(lg) / max(n_leg, 1)
        part = pd.concat([pick_fr.assign(weight=1.0), pick_lg.assign(weight=w_leg)])
        parts.append(part)
    C = pd.concat(parts).sort_values("row").reset_index(drop=True)
    C["case_id"] = C["split"].astype(str) + "-" + C["row"].astype(str)
    # val -> nửa calib / nửa eval
    C["subset"] = C["split"].astype(str)
    is_val = C["split"] == "val"
    C.loc[is_val, "subset"] = np.where(rng.random(int(is_val.sum())) < 0.5, "calib", "eval")
    if CFG["DRY_RUN"]:
        C = pd.concat([d.sample(n=min(len(d), 1500), random_state=seed) for _, d in C.groupby("subset")])
        C = C.sort_values("row").reset_index(drop=True)
    return C


with Step(5, "Alert, giả thuyết đúng và lấy mẫu case RBA"):
    A = build_alerts(tx, feats)
    del tx, feats
    gc.collect()
    log(f"  đã giải phóng bảng tx và feats | {_mem()}")
    _gd = pd.crosstab(A["target"], A["split"]).reindex(HYP_IDS).fillna(0).astype(int)
    table([[h] + [int(_gd.loc[h, s]) if s in _gd.columns else 0 for s in ["train", "val", "test"]] for h in HYP_IDS],
          ["Giả thuyết đúng", "train", "val", "test"], "Phân bố giả thuyết trên TOÀN BỘ alert RBA:")
    _nov = (A["is_fraud"] & (A["target"] == "H_novel")).sum() / max(A["is_fraud"].sum(), 1)
    log(f"  tỉ lệ tấn công rơi vào H_novel: {100 * _nov:.1f}%")
    CASES = sample_cases(A, CFG["SEED"])
    del A
    gc.collect()
    log(f"  đã giải phóng bảng alerts A | {_mem()}")
    _rows = []
    for sub in ["train", "calib", "eval", "test"]:
        d = CASES[CASES["subset"] == sub]
        _rows.append([sub, int((d["is_fraud"] == 1).sum()), int((d["is_fraud"] == 0).sum()), len(d),
                      float(d.loc[d["is_fraud"] == 0, "weight"].mean()) if (d["is_fraud"] == 0).any() else 0.0])
    table(_rows, ["Tập con", "Case tấn công", "Case hợp lệ", "Tổng case", "Trọng số case hợp lệ"], "Case sau khi lấy mẫu:")
    METRICS["cases"] = {r[0]: {"fraud": r[1], "legit": r[2], "total": r[3], "legit_weight": r[4]} for r in _rows}
    METRICS["gold_distribution_all_alerts"] = {h: {s: int(_gd.loc[h, s]) for s in _gd.columns} for h in HYP_IDS}


## 6. Registry cho domain RBA (`<TYPE_RBA_LOGIN_DEFENSE>`)
Nạp và biên dịch trực tiếp từ 2 file cấu hình chuyên dụng mới `registry_rba.json` và `registry_rba.yaml` (v2.0.0): khai báo 4 semantic tool slots (`network`, `device`, `behavior`, `graph`), 4 giả thuyết MECE (`H_shared`, `H_botnet`, `H_sybil`, `H_novel`), bộ chỉ số causal, và các luật bác bỏ Popperian kèm kiểm toán tĩnh.


In [ ]:
# ============================================================================================
# STEP 6/15 - Registry cho domain RBA (<TYPE_RBA_LOGIN_DEFENSE>) từ registry_rba.json / .yaml
# ============================================================================================
import os, json
import dataclasses as _dc
from rex.rex_registry import load_registry

DOMAIN = CFG.get("DOMAIN", "<TYPE_RBA_LOGIN_DEFENSE>")
DOM_KEY = DOMAIN.strip("<>")  # "TYPE_RBA_LOGIN_DEFENSE"

# Tìm file registry_rba.json hoặc registry_rba.yaml
RBA_JSON_PATH = os.path.join(REXLIB, "rex", "events", "configs", "registry_rba.json")
if not os.path.exists(RBA_JSON_PATH):
    RBA_JSON_PATH = "rex/events/configs/registry_rba.json"
if not os.path.exists(RBA_JSON_PATH):
    RBA_JSON_PATH = os.path.join(OUT, "registry_rba.json")

log(f"  Nạp Master Registry RBA từ: {RBA_JSON_PATH}")
with open(RBA_JSON_PATH, "r", encoding="utf-8") as f:
    RBA_REG_RAW = json.load(f)

# Semantic slot alias mapping
SLOT_TO_TOOL = {
    "SLOT_NETWORK": "network", "SLOT_NETWORK_REPUTATION": "network",
    "SLOT_DEVICE": "device", "SLOT_DEVICE_FINGERPRINT": "device",
    "SLOT_BEHAVIOR": "behavior", "SLOT_BEHAVIORAL_CADENCE": "behavior",
    "SLOT_GRAPH": "graph", "SLOT_ACCOUNT_GRAPH": "graph",
}

# 1. Trích xuất metrics từ feature_catalog
CATALOG = RBA_REG_RAW.get("feature_catalog", {})
METRICS_SPEC = {}
for m_name, m_cfg in CATALOG.items():
    m_type = m_cfg.get("type", "float")
    if "bool" in m_type:
        kind, lo, hi = "bool", 0, 1
    elif "int" in m_type:
        kind, lo, hi = "int", 0, 1000000
    else:
        kind, lo, hi = "float", 0.0, 1000000.0
    METRICS_SPEC[m_name] = {"kind": kind, "min": lo, "max": hi}

# Alert metrics cho domain RBA
ALERT_METRICS = [
    "is_new_ip", "is_new_device", "is_new_country", "is_off_hours",
    "failed_streak", "ip_vel_1h", "rtt_bin",
    "network.ip_attempts_1h", "behavior.user_failures_1h",
    "network.context_novelty_count", "graph.account_exists"
]
for am in ALERT_METRICS:
    if am not in METRICS_SPEC:
        METRICS_SPEC[am] = {"kind": "float", "min": 0.0, "max": 10000.0}

# 2. Tool Specs (4 semantic slots)
TOOL_METRICS = {
    "network": [m for m, cfg in CATALOG.items() if SLOT_TO_TOOL.get(cfg.get("slot")) == "network"],
    "device": [m for m, cfg in CATALOG.items() if SLOT_TO_TOOL.get(cfg.get("slot")) == "device"],
    "behavior": [m for m, cfg in CATALOG.items() if SLOT_TO_TOOL.get(cfg.get("slot")) == "behavior"],
    "graph": [m for m, cfg in CATALOG.items() if SLOT_TO_TOOL.get(cfg.get("slot")) == "graph"],
}
TOOL_SPECS_RAW = {
    "network": {
        "returns": TOOL_METRICS["network"], "scope": "macro", "cost": 0.02, "level": "READ",
        "description": RBA_REG_RAW["semantic_tool_slots"]["SLOT_NETWORK"]["description"]
    },
    "device": {
        "returns": TOOL_METRICS["device"], "scope": "macro", "cost": 0.02, "level": "READ",
        "description": RBA_REG_RAW["semantic_tool_slots"]["SLOT_DEVICE"]["description"]
    },
    "behavior": {
        "returns": TOOL_METRICS["behavior"], "scope": "micro", "cost": 0.01, "level": "READ",
        "description": RBA_REG_RAW["semantic_tool_slots"]["SLOT_BEHAVIOR"]["description"]
    },
    "graph": {
        "returns": TOOL_METRICS["graph"], "scope": "micro", "cost": 0.02, "level": "READ",
        "description": RBA_REG_RAW["semantic_tool_slots"]["SLOT_GRAPH"]["description"]
    },
}

# 3. Trích xuất hypotheses cho domain RBA
DOM_CFG = RBA_REG_RAW["domain_types"].get(DOM_KEY) or RBA_REG_RAW["domain_types"]["TYPE_RBA_LOGIN_DEFENSE"]
HYP_ORDER = ["H_shared", "H_botnet", "H_sybil", "H_novel"]
HYP_MAP = {h["id"]: h for h in DOM_CFG["hypotheses"]}
HYP_RISKS = {"H_shared": 0.05, "H_botnet": 0.70, "H_sybil": 0.85, "H_novel": 0.75}


def compile_registry_dict(cases_df=None):
    hyps = []
    for hid in HYP_ORDER:
        h = HYP_MAP[hid]
        assoc_tools = sorted(list({SLOT_TO_TOOL[s] for s in h.get("associated_slots", []) if s in SLOT_TO_TOOL}))
        rules = []
        for r in h.get("falsification_criteria", []):
            thresh = r["threshold"]
            # Safety check trên tập train: đảm bảo không false refutation trên ca legit
            if hid == "H_shared" and cases_df is not None:
                m = r["metric"]
                if m in cases_df.columns:
                    tr_leg = cases_df[(cases_df["split"] == "train") & (cases_df["is_fraud"] == 0)][m]
                    if len(tr_leg) > 0 and r["operator"] == "GTE":
                        p999 = float(np.quantile(tr_leg, 0.999))
                        max_valid = 0.98 if float(thresh) <= 1.0 else float("inf")
                        thresh = min(max_valid, max(float(thresh), round(p999 + 0.001, 4)))
            rules.append({"metric": r["metric"], "operator": r["operator"], "threshold": thresh})

        h_spec = {
            "id": hid,
            "text": h["text"],
            "risk": HYP_RISKS.get(hid, 0.5),
            "associated_tools": assoc_tools,
        }
        if h.get("is_residual", False) or hid == "H_novel":
            h_spec["is_residual"] = True
        else:
            h_spec["falsification_rules"] = rules
        hyps.append(h_spec)

    return {
        "tools": ["network", "device", "behavior", "graph"],
        "type_tokens": {DOMAIN: DOM_CFG["description"]},
        "tool_hidden": None, "tool_dropout": 0.1, "tool_prior": 0.25,
        "scope_schedule": ["macro", "micro"],
        "metrics": METRICS_SPEC,
        "tool_specs": TOOL_SPECS_RAW,
        "domains": {
            DOMAIN: {
                "risk": 0.5,
                "instruction": "which hypothesis explains this flagged authentication event?",
                "alert_metrics": ALERT_METRICS,
                "hypotheses": hyps,
            }
        }
    }


def static_audit(reg):
    dom = reg.get_domain(DOMAIN)
    problems = []
    if dom.hypotheses[-1].id != "H_novel" or not dom.hypotheses[-1].is_residual:
        problems.append("H_novel phải là giả thuyết cuối, dạng residual")
    if dom.hypotheses[-1].falsification_rules:
        problems.append("H_novel không được có luật bác bỏ")
    served = {t for h in dom.hypotheses if not h.is_residual for t in h.associated_tools}
    for t in reg.tools:
        if t not in served:
            problems.append(f"tool {t} không phục vụ giả thuyết đóng nào")
    for t, s in reg.tool_specs.items():
        if s.level != "READ":
            problems.append(f"tool {t} không phải READ")
    return problems


# Metadata mapping để Cell 38 và Cell 7 sử dụng
TOOL_DEFS = {
    t: (spec["scope"], spec["cost"], spec["returns"], spec["description"])
    for t, spec in TOOL_SPECS_RAW.items()
}
METRIC_TOOL = {m: t for t, (_, _, ms, _) in TOOL_DEFS.items() for m in ms}

with Step(6, "Registry cho domain RBA (registry_rba.json / .yaml)"):
    _cases_in = globals().get("CASES", globals().get("A", None))
    REG_DICT = compile_registry_dict(_cases_in)
    REGISTRY = load_registry(REG_DICT)
    
    # Lưu compiled registry vào /kaggle/working/registry.json
    with open(os.path.join(OUT, "registry.json"), "w", encoding="utf-8") as f:
        json.dump(REG_DICT, f, indent=2, ensure_ascii=False)
        
    _dom = REGISTRY.get_domain(DOMAIN)
    table([[t, s.scope, s.level, s.cost, f"{len(s.returns)} metrics"] for t, s in REGISTRY.tool_specs.items()],
          ["Tool", "Scope", "Level", "Cost", "Chỉ số"], "4 Semantic Tool Slots:")
    table([[h.id, h.risk, ", ".join(h.associated_tools),
            "; ".join(f"{r.metric} {r.operator} {r.threshold}" for r in h.falsification_rules) or "(residual)"]
           for h in _dom.hypotheses], ["Giả thuyết", "Risk", "Tool liên quan", "Bị loại khi (PoE)"], "Giả thuyết MECE:")
    
    _probs = static_audit(REGISTRY)
    log("  kiểm toán tĩnh registry: " + ("OK" if not _probs else "; ".join(_probs)))
    assert not _probs, f"Registry kiểm toán thất bại: {_probs}"
    
    METRICS["registry"] = {
        "domain": DOMAIN,
        "tools": REGISTRY.tools,
        "hypotheses": [h.id for h in _dom.hypotheses],
        "version": RBA_REG_RAW.get("version", "2.0.0")
    }


## 7. Quỹ đạo điều tra RBA (tầng C)
Dùng nguyên engine `rex_data.rollout` với dữ liệu RBA thật (alert + kết quả tool tầng B); ghi các file `data/*.jsonl.gz` cho train, calib, eval và test.


In [ ]:
# ============================================================================================
# STEP 7/15 - Sinh quỹ đạo điều tra RBA (tầng C, đúng docs/rex-label-schema.md)
# Dùng NGUYÊN logic rex_data.rollout của repo trên các ca điều tra RBA thật.
# ============================================================================================
import gzip, copy
from collections import Counter
import rex.rex_data as rex_data
from rex.rex_data import format_state
from rex.rex_format import format_observations


def _py(v):
    if isinstance(v, (bool, np.bool_)):
        return int(v)
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        f = round(float(v), 2)
        return int(f) if f.is_integer() else f
    return v


def build_worlds(C):
    all_needed_metrics = set(ALERT_METRICS)
    for _, _, ms, _ in TOOL_DEFS.values():
        all_needed_metrics.update(ms)
    cols = {m: C[m].tolist() for m in all_needed_metrics if m in C.columns}
    targets, ids = C["target"].tolist(), C["case_id"].tolist()
    worlds = []
    for i in range(len(C)):
        alert = {m: _py(cols[m][i]) for m in ALERT_METRICS if m in cols}
        tool_data = {}
        for t, (_, _, ms, _) in TOOL_DEFS.items():
            tool_data[t] = {m: _py(cols[m][i]) if m in cols else None for m in ms}
        worlds.append({"domain_key": DOMAIN, "target": targets[i], "alert": alert, "tool_data": tool_data,
                       "is_novel": targets[i] == "H_novel", "case_id": ids[i]})
    return worlds


def build_split_samples(C, sub):
    d = C[C["subset"] == sub]
    worlds = build_worlds(d)
    n_worlds = len(worlds)
    log(f"    [{sub}] bắt đầu sinh {n_worlds:,} ca điều tra | {_mem()}")
    out, dropped = [], []
    curr_world = [None]
    orig = rex_data.sample_world
    rex_data.sample_world = lambda *a, **k: curr_world[0]
    rng = random.Random(0)
    sched = tuple(REGISTRY.scope_schedule)
    t0 = time.time()
    last_log = t0
    try:
        for idx, w in enumerate(worlds):
            curr_world[0] = w
            try:
                out.extend(rex_data.rollout(
                    REGISTRY, DOMAIN, rng, w["case_id"],
                    budgets=(3, 2, 1), scope_schedule=sched,
                    scope_fallback=True, privileged_teacher=True,
                ))
            except ValueError as exc:
                dropped.append((w["case_id"], str(exc)[:120]))

            now = time.time()
            if (idx + 1) % 10000 == 0 or (idx + 1) == n_worlds or (now - last_log >= 15.0):
                last_log = now
                el = max(now - t0, 0.001)
                speed = (idx + 1) / el
                eta = (n_worlds - idx - 1) / max(speed, 1.0)
                log(f"      [{sub}] {idx + 1:,}/{n_worlds:,} ca ({100 * (idx + 1) / n_worlds:.1f}%) | "
                    f"{speed:.0f} ca/s | còn ~{fmt_secs(eta)} | {_mem()}")
    finally:
        rex_data.sample_world = orig
    return out, dropped


def trajectory_stats(samples, name):
    by_case = Counter(s["case_id"] for s in samples)
    rounds = np.array(list(by_case.values()))
    scope = Counter(s["label_scope"] for s in samples)
    s_star = np.array([s["labels"]["s_star"] for s in samples])
    stall = np.array([s["labels"]["stall_star"] for s in samples])
    gains = [g for s in samples for g in s["labels"]["gain_star"].values() if g != -100.0]
    called = Counter(t for s in samples for t in s.get("called_tools", []))
    teach = Counter(t for s in samples for t in s["tool_labels"])
    term = [s for s in samples if s["label_scope"] == "terminal"]
    alive_final = np.array([sum(1 for k, v in s["alive"].items() if v and k != "H_novel") for s in term])
    return {
        "subset": name, "cases": len(by_case), "steps": len(samples),
        "steps_per_case": float(rounds.mean()) if len(rounds) else 0.0,
        "label_scope": dict(scope), "s_star_rate": float(s_star.mean()) if len(s_star) else 0.0,
        "stall_star_rate": float(stall.mean()) if len(stall) else 0.0,
        "gain_star_pos_rate": float(np.mean(np.array(gains) > 0)) if gains else 0.0,
        "called_tools": dict(called), "teacher_tools": dict(teach),
        "final_alive_cataloged_mean": float(alive_final.mean()) if len(alive_final) else 0.0,
    }


def rule_audit(samples):
    seen = set()
    hits, wrong = Counter(), Counter()
    for s in samples:
        for h, traces in s["eliminated_by"].items():
            for tr in traces:
                key = (h, tr.split(" (observed")[0])
                if (s["case_id"], key) in seen:
                    continue
                seen.add((s["case_id"], key))
                hits[key] += 1
                if h == s["target"]:
                    wrong[key] += 1
    return [[h, rule, hits[(h, rule)], wrong[(h, rule)]] for (h, rule) in sorted(hits)]


def write_jsonl_gz(path, samples):
    with gzip.open(path, "wt", encoding="utf-8", compresslevel=1) as f:
        for s in samples:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")


with Step(7, "Sinh quỹ đạo điều tra RBA (tầng C)"):
    for _v in ["tx", "feats", "A"]:
        if _v in globals():
            del globals()[_v]
    gc.collect()
    log(f"  bắt đầu sinh quỹ đạo điều tra | {_mem()}")
    SAMPLES, TRAJ_STATS, DROPPED = {}, {}, {}
    for sub in ["train", "calib", "eval", "test"]:
        _t = time.time()
        SAMPLES[sub], DROPPED[sub] = build_split_samples(CASES, sub)
        log(f"    [{sub}] tính thống kê quỹ đạo & nén file jsonl.gz...")
        TRAJ_STATS[sub] = trajectory_stats(SAMPLES[sub], sub)
        write_jsonl_gz(os.path.join(OUT, "data", f"{sub}.jsonl.gz"), SAMPLES[sub])
        log(f"  {sub}: {TRAJ_STATS[sub]['cases']:,} case -> {TRAJ_STATS[sub]['steps']:,} dòng JSONL "
            f"({TRAJ_STATS[sub]['steps_per_case']:.2f} dòng/case) | bỏ {len(DROPPED[sub])} case xung đột luật "
            f"| {fmt_secs(time.time() - _t)}")
        for cid, why in DROPPED[sub][:3]:
            log(f"     bỏ {cid}: {why}")
        gc.collect()
    table([[k, v["cases"], v["steps"], f"{v['steps_per_case']:.2f}", f"{100 * v['s_star_rate']:.1f}%",
            f"{100 * v['stall_star_rate']:.1f}%", f"{100 * v['gain_star_pos_rate']:.1f}%",
            f"{v['final_alive_cataloged_mean']:.2f}"] for k, v in TRAJ_STATS.items()],
          ["Tập", "Case", "Dòng", "Dòng/case", "S*=1", "stall*=1", "gain*=1", "GT đóng còn sống cuối"],
          "Thống kê quỹ đạo:")
    table([[k] + [v["label_scope"].get(x, 0) for x in ["macro", "micro", "mixed", "none", "terminal"]]
           for k, v in TRAJ_STATS.items()], ["Tập", "macro", "micro", "mixed", "none", "terminal"], "label_scope:")
    table([[t, TRAJ_STATS["train"]["called_tools"].get(t, 0), TRAJ_STATS["train"]["teacher_tools"].get(t, 0)]
           for t in REGISTRY.tools], ["Tool", "Lần được gọi (train)", "Lần thầy chọn (train)"], "Tool:")
    _audit = rule_audit(SAMPLES["train"])
    table(_audit, ["Giả thuyết", "Luật", "Case bị loại", "Loại nhầm gold"], "Kiểm toán luật (train):")
    assert all(r[3] == 0 for r in _audit), "Có luật loại nhầm giả thuyết đúng"
    _demo = [s for s in SAMPLES["train"] if s["t"] == 1][:2] + \
            [s for s in SAMPLES["train"] if s["t"] == 2][:1]
    for s in _demo:
        log(f"  --- mẫu {s['case_id']} t={s['t']} target={s['target']} scope={s['label_scope']}")
        log(f"      state   : {format_state(s['state'])[:400]}")
        log(f"      alive   : {[k for k, v in s['alive'].items() if v]}")
        log(f"      thầy    : {s['tool_labels']} | gọi: {s.get('called_tools', [])}")
        log(f"      evidence: {format_observations(s['observations'])[:300]}")
    METRICS["trajectories"] = TRAJ_STATS
    METRICS["rule_audit_train"] = [{"hyp": r[0], "rule": r[1], "eliminated": r[2], "wrong_gold": r[3]} for r in _audit]
    METRICS["dropped_rule_conflicts"] = {k: len(v) for k, v in DROPPED.items()}


## 8. Khởi tạo Rex từ Laya
Thêm type token, `from_rex`, kiểm tra tương thích Laya, ponder lúc khởi tạo, lưu/nạp lại, độ dài token.

In [ ]:
# ============================================================================================
# STEP 8/15 - Khởi tạo Rex từ Laya + kiểm tra bất biến + kiểm tra độ dài token
# Giống scripts/train_rex.py::load_base: thêm type token, from_rex(...) với tool của registry.
# ============================================================================================
import torch.nn.functional as F


def param_table(model):
    rows = {}
    for name, p in model.named_parameters():
        top = name.split(".", 1)[0]
        rows[top] = rows.get(top, 0) + p.numel()
    return sorted(rows.items(), key=lambda kv: -kv[1])


def probe_batch(tok, samples, n=8):
    """Một batch round-1 thật (đủ alive) để so logit Laya và Rex."""
    dom = REGISTRY.get_domain(DOMAIN)
    items = []
    for s in [x for x in samples if x["t"] == 1][:n]:
        items.append(encode_rex_input(tok, DOMAIN, dom.instruction, dom.hypotheses_dict,
                                      state=format_state(s["state"]), alive=None,
                                      max_len=CFG["MAX_LEN"], head_max_len=CFG["HEAD_MAX_LEN"]))
    return collate_rex(items, pad_token_id=tok.pad_token_id or 0)


RESUMED_STAGE = None
with Step(8, "Khởi tạo Rex từ Laya và kiểm tra bất biến"):
    laya, TOK, BASE_CFG = load_checkpoint(BASE_DIR)
    log(f"  Laya: {type(laya).__name__}, encoder {laya.encoder.config.model_type} "
        f"hidden {laya.encoder.config.hidden_size}, {sum(p.numel() for p in laya.parameters()) / 1e6:.1f}M tham số")
    _added = add_type_tokens(TOK, laya, REGISTRY.type_tokens)
    log(f"  type token: {_added}")

    # --- tương thích Laya: logit verdict của Rex (bridge g_v = 0) phải trùng Laya ---
    laya.eval()
    _b = probe_batch(TOK, SAMPLES["train"])
    with torch.no_grad():
        laya_logits, _ = laya(_b["input_ids"], _b["attention_mask"], _b["marker_pos"], _b["marker_mask"], _b["qtype"])
    MODEL = from_rex(laya, tools=REGISTRY.tools, type_tokens=REGISTRY.type_tokens,
                     tool_hidden=REGISTRY.tool_hidden, tool_dropout=REGISTRY.tool_dropout,
                     tool_prior=REGISTRY.tool_prior, base_cfg=BASE_CFG)
    MODEL.set_tool_specs([REGISTRY.tool_specs[t] for t in MODEL.tools])
    MODEL.eval()
    with torch.no_grad():
        rex_logits, _st, _out = MODEL(input_ids=_b["input_ids"], attention_mask=_b["attention_mask"],
                                      marker_pos=_b["marker_pos"], marker_mask=_b["marker_mask"], qtype=_b["qtype"],
                                      dispatch=False)
        h = MODEL._contextualize(_b["input_ids"], _b["attention_mask"], _b["qtype"])
        cos = F.cosine_similarity(_st.h.float(), F.layer_norm(h[:, 0].float(), (h.size(-1),)), dim=-1)
    live = _b["marker_mask"]
    delta = float((laya_logits[live] - rex_logits[live]).abs().max())
    INVARIANTS = {"laya_parity_max_abs_delta": delta, "ponder_init_cosine_min": float(cos.min())}
    log(f"  tương thích Laya: max |Δ logit| = {delta:.2e} (yêu cầu < 1e-6)")
    log(f"  ponder lúc khởi tạo: cosine(h_T, LN(h_0)) min = {float(cos.min()):.4f} (kỳ vọng > 0.99)")
    assert delta < 1e-6, "Rex khởi tạo lệch logit so với Laya"
    if float(cos.min()) < 0.99:
        log("  CẢNH BÁO: cosine ponder < 0.99 lúc khởi tạo")
    del laya
    gc.collect()

    # --- lưu rồi nạp lại: key phải khớp tuyệt đối (load strict) ---
    _tmp = os.path.join(OUT, "_roundtrip_ckpt")
    save_rex_checkpoint(MODEL, TOK, _tmp, base_cfg=BASE_CFG)
    _m2, _tok2, _cfg2 = load_checkpoint(_tmp)
    _m2.eval()
    with torch.no_grad():
        l2, _, _ = _m2(input_ids=_b["input_ids"], attention_mask=_b["attention_mask"], marker_pos=_b["marker_pos"],
                       marker_mask=_b["marker_mask"], qtype=_b["qtype"], dispatch=False)
    INVARIANTS["roundtrip_max_abs_delta_fp16"] = float((l2[live] - rex_logits[live]).abs().max())
    INVARIANTS["roundtrip_type"] = type(_m2).__name__
    log(f"  lưu/nạp lại (fp16): {type(_m2).__name__}, max |Δ logit| = {INVARIANTS['roundtrip_max_abs_delta_fp16']:.3e}")
    assert isinstance(_m2, RexDecisionModel)
    del _m2
    import shutil
    shutil.rmtree(_tmp, ignore_errors=True)

    table([[k, f"{v / 1e6:.2f}M"] for k, v in param_table(MODEL)], ["Module", "Tham số"], "Tham số theo module:")

    # --- độ dài token thật (round 1 và evidence) trên mẫu train ---
    _cases = group_cases(SAMPLES["train"])[:2000]
    _r1, _ev, _evtr = [], [], 0
    for c in _cases:
        ec = encode_case(c, TOK, MODEL.tools, REGISTRY, rng=None, max_len=CFG["MAX_LEN"],
                         head_max_len=CFG["HEAD_MAX_LEN"], evidence_max_tokens=CFG["EVIDENCE_MAX_TOKENS"])
        _r1.append(len(ec.round1["input_ids"]))
        for s in c[:-1]:
            n_tok = len(TOK.encode(format_observations(s.get("observations", {})), add_special_tokens=False))
            _ev.append(n_tok)
            _evtr += n_tok > CFG["EVIDENCE_MAX_TOKENS"]
    _r1, _ev = np.array(_r1), np.array(_ev)
    TOKEN_STATS = {"round1_p50": int(np.median(_r1)), "round1_max": int(_r1.max()),
                   "evidence_p50": int(np.median(_ev)), "evidence_p99": int(np.percentile(_ev, 99)),
                   "evidence_max": int(_ev.max()), "evidence_truncated_rate": float(_evtr / max(len(_ev), 1))}
    table([[k, v] for k, v in TOKEN_STATS.items()], ["Độ dài token", "Giá trị"])
    if TOKEN_STATS["evidence_truncated_rate"] > 0.01:
        log("  CẢNH BÁO: > 1% evidence vượt 64 token và bị cắt (encode_evidence cắt phần đuôi)")
    METRICS["invariants"] = INVARIANTS
    METRICS["token_stats"] = TOKEN_STATS

    # --- tiếp tục từ checkpoint đã có (tuỳ chọn) ---
    # Chỉ cần ckpt_<stage>/model.safetensors: mô hình vừa dựng ở trên (cùng Laya, cùng type token, cùng tool)
    # nạp trọng số với strict=True, nên không phụ thuộc thư mục encoder/ và tokenizer/ của checkpoint.
    # Checkpoint có nan/inf (vd. ckpt_ponder, ckpt_full của lần chạy lỗi) bị bỏ qua.
    if CFG["RESUME_DIR"]:
        from safetensors.torch import load_file as _load_st
        _roots = (sorted(glob.glob("/kaggle/input/**/", recursive=True)) if CFG["RESUME_DIR"] == "auto"
                  else [CFG["RESUME_DIR"]])
        _cands = []
        for _st_name in ["full", "ponder", "heads"]:
            for _r in _roots:
                _cands += [(_st_name, p_) for p_ in glob.glob(os.path.join(_r, f"ckpt_{_st_name}", "model.safetensors"))]
        _seen = set()
        for _st_name, _p in _cands:
            if _p in _seen:
                continue
            _seen.add(_p)
            _sd = _load_st(_p)
            _bad = [k for k, v in _sd.items() if v.is_floating_point() and not torch.isfinite(v).all()]
            if _bad:
                log(f"  RESUME: bỏ qua {_p} ({len(_bad)} tensor có nan/inf, vd. {_bad[0]})")
                continue
            _res = MODEL.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in _sd.items()},
                                         strict=True)
            RESUMED_STAGE = _st_name
            log(f"  RESUME: nạp {_p} (strict, {len(_sd)} tensor, không có nan) -> bỏ qua các stage tới '{_st_name}'")
            METRICS["resumed_from"] = {"path": _p, "stage": _st_name}
            break
        if RESUMED_STAGE is None:
            log("  RESUME: không tìm thấy checkpoint dùng được -> train từ đầu")
    MODEL.to(DEVICE)


## 9. Train stage `heads`
`head_s, head_r, tool_matcher, tool_gain, hyp_scorer, alive_count_emb`; phần còn lại đóng băng. Mỗi stage đi qua toàn bộ case train **đúng 1 lần**, chia 3 phần (~34k case/phần); sau mỗi phần chỉ lưu checkpoint (không đánh giá giữa chừng). Loss in mỗi 200 update. Loss = nan -> dừng ngay, quay về trạng thái tốt gần nhất và chạy lại phần đó bằng fp32.

In [ ]:
# ============================================================================================
# STEP 9/15 - Train stage `heads`  (+ hàm dùng chung cho 3 stage)
# Mỗi stage đi qua toàn bộ case train ĐÚNG 1 LẦN, chia thành CFG["SHARDS"] phần (mặc định 3).
# Mỗi phần = một lần gọi đúng rex.rex_train.train_rex(epochs=1) của repo trên ~1/3 số case, KHÔNG kèm
# tập calib/val (không hiệu chỉnh, không đánh giá giữa chừng); sau mỗi phần: lưu checkpoint + chụp trạng thái tốt.
# Loss (đủ 8 thành phần) in mỗi CFG["LOG_EVERY"] update.
# Chốt chặn NaN: dòng log nào có loss = nan/inf -> dừng ngay phần đó, quay về trạng thái tốt gần nhất
# và chạy lại phần đó bằng fp32 (nếu NAN_FALLBACK_FP32); nan cả ở fp32 -> dừng notebook, không phí giờ GPU.
# Lưu ý: mỗi lần gọi train_rex tạo optimizer + lịch LR mới, nên phần 2-3 bắt đầu lại với warmup ngắn.
# ============================================================================================
STAGE_SUMMARIES, STAGE_TIMES, SHARD_LOG = {}, {}, []
_UPD_RE = re.compile(r"epoch (\d+) update (\d+)/(\d+) loss (\S+)(.*)")
_GN_RE = re.compile(r"first-update grad norms (.*)")


class NanLossError(RuntimeError):
    pass


def _f(x):
    try:
        return float(x)
    except ValueError:
        return float("nan")


class StageLogger:
    """Nhận dòng log của train_rex: in kèm giờ / VRAM / ETA, ghi HISTORY, và chặn NaN ngay lập tức."""

    def __init__(self, stage, shard, n_shards, amp):
        self.stage, self.shard, self.n_shards, self.amp, self.t0 = stage, shard, n_shards, amp, time.time()
        self.tag = f"{stage} {shard}/{n_shards}" + ("" if amp else " fp32")

    def __call__(self, msg):
        m = _UPD_RE.search(msg)
        g = _GN_RE.search(msg)
        if m:
            upd, tot, loss, rest = int(m.group(2)), int(m.group(3)), _f(m.group(4)), m.group(5)
            parts = {k: _f(v) for k, v in (kv.split("=") for kv in rest.split() if "=" in kv)}
            el = time.time() - self.t0
            HISTORY.append({"stage": self.stage, "shard": self.shard, "amp": self.amp, "update": upd,
                            "total_updates": tot, "loss": loss, "elapsed_s": round(el, 1), **parts})
            log(f"[{self.tag}] upd {upd}/{tot} | loss {loss:.4f} = " +
                " ".join(f"{k.replace('_loss', '')} {v:.3f}" for k, v in parts.items()) +
                f" | {upd / max(el, 1e-9) * 60:.1f} upd/min | VRAM peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB"
                f" | ETA phần này {fmt_secs(el / max(upd, 1) * (tot - upd))}")
            bad = [k for k, v in parts.items() if not math.isfinite(v)]
            if not math.isfinite(loss) or bad:
                raise NanLossError(f"{self.tag}: loss không hữu hạn ở update {upd} (thành phần: {bad or ['total']})")
        elif g:
            log(f"[{self.tag}] {msg.replace('rex train: ', '')}")
            vals = [_f(kv.split("=")[1]) for kv in g.group(1).split(", ") if "=" in kv]
            if any(not math.isfinite(v) for v in vals):
                if self.amp:   # fp16: GradScaler tự bỏ qua update có gradient inf và giảm scale -> chỉ cảnh báo
                    log(f"[{self.tag}] (gradient inf ở update đầu: GradScaler sẽ bỏ qua bước này, bình thường với fp16)")
                else:
                    raise NanLossError(f"{self.tag}: gradient không hữu hạn ở update đầu tiên (fp32)")
        else:
            log(f"[{self.tag}] {msg.replace('rex train: ', '')}")


def subset_cases(samples, n_cases, seed=0):
    """Lấy n_cases case (giữ mọi dòng của case) để hiệu chỉnh/đánh giá."""
    cases = group_cases(samples)
    if n_cases is None or len(cases) <= n_cases:
        return list(samples)
    rng = random.Random(seed)
    pick = sorted(rng.sample(range(len(cases)), n_cases))
    return [s for i in pick for s in cases[i]]


def make_shards(samples, n, seed):
    """Chia case train thành n phần rời nhau (xáo trộn theo seed): mỗi case thuộc đúng 1 phần."""
    cases = group_cases(samples)
    order = list(range(len(cases)))
    random.Random(seed).shuffle(order)
    return [[s for i in order[k::n] for s in cases[i]] for k in range(n)]


def snapshot():
    return {k: v.detach().to("cpu", copy=True) for k, v in MODEL.state_dict().items()}


def model_is_finite():
    return all(torch.isfinite(p).all() for p in MODEL.parameters())


GOOD_STATE = None


def run_stage(stage):
    global GOOD_STATE
    p = CFG["STAGES"][stage]
    n_sh = CFG["SHARDS"]
    shards = make_shards(SAMPLES["train"], n_sh, seed=CFG["SEED"] + {"heads": 0, "ponder": 1, "full": 2}[stage])
    from rex.rex_train import STAGE_MODULES
    trainable = set(STAGE_MODULES.get(stage, ()))
    table([[name, f"{n / 1e6:.2f}M", "TRAIN" if (stage == "full" or name in trainable) else "frozen"]
           for name, n in param_table(MODEL)], ["Module", "Tham số", "Trạng thái"], f"Stage {stage}:")
    log(f"  1 lượt qua {len(group_cases(SAMPLES['train'])):,} case train, chia {n_sh} phần: "
        + ", ".join(f"{len(group_cases(s)):,}" for s in shards) + " case"
        f" | batch_cases={p['batch_cases']} grad_accum={p['grad_accum']} lr_heads={p['lr_heads']} "
        f"lr_encoder={p['lr_encoder']} | {'fp16' if p.get('amp', CFG['AMP']) else 'fp32'}")
    if GOOD_STATE is None:
        GOOD_STATE = snapshot()
    amp = p.get("amp", CFG["AMP"])
    STAGE_TIMES[stage] = {"seconds": 0.0, "train_cases": 0, "shards_done": 0}
    summary = {}
    for k in range(1, n_sh + 1):
        # ---- ngân sách thời gian: ước tính thời gian phần này từ phần trước (hoặc từ stage ponder) ----
        done = STAGE_TIMES[stage]["shards_done"]
        if done:
            est = STAGE_TIMES[stage]["seconds"] / done
        else:   # phần đầu của stage: dùng tốc độ đã đo trên T4 ở lần chạy trước (case/giây, gồm cả đánh giá)
            est = len(group_cases(shards[k - 1])) / CFG["EST_CASES_PER_SEC"].get(stage, 1e9)
        reserve = (1.5 if not CFG["DRY_RUN"] else 0.1) * 3600
        if est and time_left() - est < reserve:
            log(f"  KHÔNG đủ giờ cho phần {k}/{n_sh} (ước {fmt_secs(est)}, còn {fmt_secs(time_left())}, "
                f"chừa {fmt_secs(reserve)} cho đánh giá) -> dừng stage {stage} ở đây")
            break
        attempt_amp = amp
        out_of_time = False
        while True:
            cfg = RexTrainConfig(stage=stage, epochs=1, batch_cases=p["batch_cases"], grad_accum=p["grad_accum"],
                                 lr_heads=p["lr_heads"], lr_encoder=p["lr_encoder"],
                                 warmup_ratio=CFG["WARMUP_RATIO"] if k == 1 else CFG["WARMUP_RATIO_LATER"],
                                 max_grad_norm=CFG["MAX_GRAD_NORM"], max_len=CFG["MAX_LEN"],
                                 head_max_len=CFG["HEAD_MAX_LEN"], evidence_max_tokens=CFG["EVIDENCE_MAX_TOKENS"],
                                 amp=attempt_amp, seed=CFG["SEED"] + k, log_every=CFG["LOG_EVERY"],
                                 loss_weights=RexLossWeights())
            log(f"--- stage {stage} phần {k}/{n_sh}: {len(group_cases(shards[k - 1])):,} case train | "
                f"{'fp16' if attempt_amp else 'fp32'} | ước {fmt_secs(est)} | còn {fmt_secs(time_left())}")
            torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
            try:
                summary = train_rex(MODEL, TOK, REGISTRY, shards[k - 1], None, config=cfg, calib_samples=None,
                                    device=DEVICE, encode_tools=True, log=StageLogger(stage, k, n_sh, attempt_amp))
                if not model_is_finite():
                    raise NanLossError(f"{stage} {k}/{n_sh}: trọng số không hữu hạn sau khi train")
                break
            except NanLossError as exc:
                log(f"  !!! {exc}")
                MODEL.load_state_dict(GOOD_STATE)
                MODEL.to(DEVICE)
                torch.cuda.empty_cache()
                SHARD_LOG.append({"stage": stage, "shard": k, "amp": attempt_amp, "nan": str(exc),
                                  "seconds": time.time() - t0})
                if attempt_amp and CFG["NAN_FALLBACK_FP32"]:
                    if est and time_left() - 2.5 * est < reserve:
                        log(f"  -> đã quay về trạng thái tốt gần nhất; KHÔNG đủ giờ chạy lại bằng fp32 "
                            f"(ước {fmt_secs(2.5 * est)}) -> dừng stage {stage}, giữ checkpoint tốt gần nhất")
                        out_of_time = True
                        break
                    log("  -> đã quay về trạng thái tốt gần nhất; chạy lại phần này bằng fp32 (chậm hơn)")
                    attempt_amp = False
                    amp = False          # các phần sau của stage cũng dùng fp32
                    continue
                raise
        if out_of_time:
            break
        dt = time.time() - t0
        GOOD_STATE = snapshot()
        STAGE_TIMES[stage]["seconds"] += dt
        STAGE_TIMES[stage]["train_cases"] += len(group_cases(shards[k - 1]))
        STAGE_TIMES[stage]["shards_done"] += 1
        STAGE_TIMES[stage]["cases_per_sec"] = STAGE_TIMES[stage]["train_cases"] / max(STAGE_TIMES[stage]["seconds"], 1e-9)
        STAGE_SUMMARIES.setdefault(stage, {})[f"part{k}"] = {kk: v for kk, v in summary.items() if kk != "grad_norms"}
        STAGE_SUMMARIES[stage][f"part{k}"]["first_update_grad_norms"] = summary.get("grad_norms", {})
        STAGE_SUMMARIES[stage][f"part{k}"]["amp"] = attempt_amp
        SHARD_LOG.append({"stage": stage, "shard": k, "amp": attempt_amp, "seconds": dt,
                          "mean_loss": (summary.get("epoch_losses") or [float("nan")])[-1]})
        ck = os.path.join(OUT, f"ckpt_{stage}")
        save_rex_checkpoint(MODEL, TOK, ck, base_cfg=BASE_CFG)
        with open(os.path.join(ck, "rex_train_summary.json"), "w") as f:
            json.dump({"stage": stage, "parts_done": k, **STAGE_SUMMARIES[stage]}, f, indent=2, default=float)
        log(f"--- stage {stage} phần {k}/{n_sh} xong trong {fmt_secs(dt)} | loss TB phần "
            f"{SHARD_LOG[-1]['mean_loss']:.4f} | {len(group_cases(shards[k - 1])) / max(dt, 1e-9):.1f} case/s "
            f"| lưu {ck} | còn {fmt_secs(time_left())} | {_mem()}")
        del summary
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    del shards
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    log(f"  stage {stage}: {STAGE_TIMES[stage]['shards_done']}/{n_sh} phần, {STAGE_TIMES[stage]['train_cases']:,} case, "
        f"{fmt_secs(STAGE_TIMES[stage]['seconds'])} | {_mem()}")
    return STAGE_SUMMARIES.get(stage, {})


def stage_done(stage):
    order = ["heads", "ponder", "full"]
    return RESUMED_STAGE is not None and order.index(stage) <= order.index(RESUMED_STAGE)


with Step(9, "Train stage heads (1 lượt, 3 phần)"):
    if stage_done("heads"):
        log("  bỏ qua (đã có từ RESUME_DIR)")
    else:
        run_stage("heads")

## 10. Train stage `ponder`
Thêm `ponder_block` và `verdict_bridge`. 1 lượt, 3 phần, **fp32** (lần chạy trước stage này ở fp16 ra nan).

In [ ]:
# ============================================================================================
# STEP 10/15 - Train stage `ponder` (thêm ponder_block + verdict_bridge), 1 lượt chia 3 phần
# ============================================================================================
with Step(10, "Train stage ponder (1 lượt, 3 phần)"):
    if stage_done("ponder"):
        log("  bỏ qua (đã có từ RESUME_DIR)")
    else:
        run_stage("ponder")


## 11. Train stage `full`
Toàn bộ mô hình; encoder dùng `lr_encoder`. 1 lượt, 3 phần, fp16 + chốt chặn nan.

In [ ]:
# ============================================================================================
# STEP 11/15 - Train stage `full` (toàn bộ mô hình, encoder lr riêng), 1 lượt chia 3 phần
# Hết giờ thì dừng trước phần không kịp (ngân sách kiểm tra trước mỗi phần, chừa 1.5h cho bước 12-15).
# ============================================================================================
with Step(11, "Train stage full (1 lượt, 3 phần)"):
    if stage_done("full"):
        log("  bỏ qua (đã có từ RESUME_DIR)")
    else:
        run_stage("full")
        if STAGE_TIMES["full"]["shards_done"] < CFG["SHARDS"]:
            log(f"  CẢNH BÁO: stage full chỉ xong {STAGE_TIMES['full']['shards_done']}/{CFG['SHARDS']} phần (hết giờ)")
    table([[s_, t_["shards_done"], t_["train_cases"], fmt_secs(t_["seconds"]), f"{t_.get('cases_per_sec', 0):.1f}"]
           for s_, t_ in STAGE_TIMES.items()], ["Stage", "Phần xong", "Case đã train", "Thời gian", "case/s"],
          "Tổng kết train:")
    table([[r["stage"], r["shard"], "fp16" if r["amp"] else "fp32", fmt_secs(r["seconds"]),
            r.get("mean_loss"), r.get("nan", "")] for r in SHARD_LOG],
          ["Stage", "Phần", "Độ chính xác số", "Thời gian", "Loss TB", "NaN"], "Từng phần:")
    if CFG["DRY_RUN"] and STAGE_TIMES:
        # Lần chạy đầy đủ ~102.200 case train, mỗi stage đi qua 1 lượt
        _full_cases, _rows, _tot = 102_200, [], 0.0
        for _s, _t in STAGE_TIMES.items():
            if _t.get("cases_per_sec"):
                _sec = _full_cases / _t["cases_per_sec"]
                _tot += _sec
                _rows.append([_s, f"{_t['cases_per_sec']:.1f}", fmt_secs(_sec)])
        table(_rows, ["Stage", "case/s đo được", "Ước tính 1 lượt đầy đủ"],
              "DRY_RUN - dự báo thời gian train cho lần chạy đầy đủ:")
        log(f"  tổng train ước tính ~ {fmt_secs(_tot)} (chưa gồm chuẩn bị dữ liệu ~10 phút và đánh giá ~1h)")


## 12. Hiệu chỉnh (1 lần) và chạy mô hình cuối
Fit nhiệt độ verdict rồi `q_hat` trên toàn bộ calib; ghi đầu ra mọi round trên eval và test; ECE/MCE.

In [ ]:
# ============================================================================================
# STEP 12/15 - Hiệu chỉnh xác suất (1 lần, toàn bộ calib) + chạy mô hình cuối trên eval và test
# Forward giống hệt run_cases của rex_train (round 1 đầy đủ, các round sau nạp evidence), ghi đầu ra
# MỌI round cho từng case. Hiệu chỉnh làm đúng như train_rex: nhiệt độ verdict (fit_one_temperature trên
# cặp (logit còn sống, one-hot đích) của round cuối) rồi q_hat conformal trên xác suất đã hiệu chỉnh;
# chỉ khác là lấy logit từ 1 lượt forward thay vì chạy evaluate() 2 lần.
# ============================================================================================
from contextlib import nullcontext
CASE_META = CASES.set_index("case_id")[["split", "subset", "user", "year", "is_fraud", "amount", "weight",
                                         "target"]].to_dict("index")
H_IDX = {h: i for i, h in enumerate(REGISTRY.get_domain(DOMAIN).hypotheses_dict)}
assert list(H_IDX) == HYP_IDS


def encode_all_cases(samples):
    raw = group_cases(samples)
    enc = [encode_case(c, TOK, MODEL.tools, REGISTRY, rng=None, max_len=CFG["MAX_LEN"],
                       head_max_len=CFG["HEAD_MAX_LEN"], evidence_max_tokens=CFG["EVIDENCE_MAX_TOKENS"]) for c in raw]
    return raw, enc


def collect_rounds(samples, batch_cases=32, amp=None):
    """Forward giống run_cases (rex_train) nhưng giữ đầu ra của từng round cho từng case.
    Chạy fp16 cho nhanh; nếu đầu ra có nan/inf thì tự chạy lại bằng fp32."""
    amp = CFG["AMP"] if amp is None else amp
    recs, _ = _collect_rounds(samples, batch_cases, amp)
    if amp and not all(np.isfinite(r["probs"]).all() and np.isfinite(r["s"]) and np.isfinite(r["r"]) for r in recs):
        log("  CẢNH BÁO: đầu ra fp16 có nan/inf -> chạy lại bằng fp32")
        recs, _ = _collect_rounds(samples, batch_cases, False)
    return recs, None


def _collect_rounds(samples, batch_cases, amp_on):
    raw, enc = encode_all_cases(samples)
    MODEL.eval()
    n_tools = len(MODEL.tools)
    recs = []
    amp = torch.autocast("cuda", dtype=torch.float16) if amp_on else nullcontext()
    with torch.no_grad():
        for start in range(0, len(enc), batch_cases):
            chunk, craw = enc[start:start + batch_cases], raw[start:start + batch_cases]
            b1 = collate_rex_train([c.round1 for c in chunk], _pad_id(TOK))
            mm = b1["marker_mask"].to(DEVICE)
            n_rounds = max(len(c.rounds) for c in chunk)
            state = None
            for k in range(n_rounds):
                active = [i for i, c in enumerate(chunk) if k < len(c.rounds)]
                rows_k = [c.rounds[k] if k < len(c.rounds) else c.rounds[-1] for c in chunk]
                lab = {n: t.to(DEVICE) for n, t in _stack_round(rows_k, mm.size(1), n_tools).items()}
                with amp:
                    if k == 0:
                        vlog, state, out = MODEL(input_ids=b1["input_ids"].to(DEVICE),
                                                 attention_mask=b1["attention_mask"].to(DEVICE),
                                                 marker_pos=b1["marker_pos"].to(DEVICE), marker_mask=mm,
                                                 qtype=b1["qtype"].to(DEVICE), hyp_alive=lab["hyp_alive"],
                                                 tool_mask=lab["tool_mask"], dispatch=False)
                    else:
                        filler = _filler_token(TOK)
                        evs = [c.evidence_ids[k - 1] if k - 1 < len(c.evidence_ids) else [filler] for c in chunk]
                        w = max(len(e) for e in evs)
                        ev = torch.full((len(chunk), w), _pad_id(TOK), dtype=torch.long)
                        em = torch.zeros((len(chunk), w), dtype=torch.bool)
                        for i, e in enumerate(evs):
                            ev[i, :len(e)] = torch.tensor(e)
                            em[i, :len(e)] = True
                        vlog, state, out = MODEL(state=state, evidence_new=ev.to(DEVICE), evidence_mask=em.to(DEVICE),
                                                 hyp_alive=lab["hyp_alive"], tool_mask=lab["tool_mask"], dispatch=False)
                vp = out.verdict_probs.float().cpu().numpy()
                vl = vlog.float().cpu().numpy()
                sp = out.s_prob.float().cpu().view(-1).numpy()
                rp = out.r_prob.float().cpu().view(-1).numpy()
                tp = torch.sigmoid(out.tool_logits.float()).cpu().numpy()
                gp = out.tool_gains.float().cpu().numpy()
                hs = out.hyp_scores.float().cpu().numpy()
                mmn = mm.cpu().numpy()
                L = {n: t.cpu().numpy() for n, t in lab.items()}
                for i in active:
                    recs.append({"case_id": craw[i][0]["case_id"], "round": k, "last": k == len(chunk[i].rounds) - 1,
                                 "probs": vp[i], "logits": vl[i], "s": float(sp[i]), "r": float(rp[i]),
                                 "tool_p": tp[i], "gain": gp[i], "target": int(L["target_verdict"][i]),
                                 "alive": L["hyp_alive"][i], "tool_label": L["tool_labels"][i],
                                 "gain_label": L["gain_labels"][i],
                                 "callable": L["tool_mask"][i] & ~L["redundancy_masks"][i],
                                 "s_label": float(L["s_labels"][i, 0]), "r_label": float(L["r_labels"][i, 0]),
                                 "hyp": hs[i][mmn[i]], "hyp_label": L["hyp_labels"][i][mmn[i]],
                                 "stall_label": float(L["stall_labels"][i])})
    del raw, enc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return recs, None


def ece_top1(probs, targets, n_bins=10):
    conf = probs.max(1)
    acc = (probs.argmax(1) == targets).astype(float)
    bins = np.clip((conf * n_bins).astype(int), 0, n_bins - 1)
    ece, mce, rel = 0.0, 0.0, []
    for b in range(n_bins):
        m = bins == b
        if m.any():
            gap = abs(acc[m].mean() - conf[m].mean())
            ece += m.mean() * gap
            mce = max(mce, gap)
            rel.append((b, float(conf[m].mean()), float(acc[m].mean()), int(m.sum())))
    return float(ece), float(mce), rel


def softmax_masked(logits, T):
    z = logits / T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z) * (logits > -1e3)
    return e / np.maximum(e.sum(1, keepdims=True), 1e-12)


with Step(12, "Hiệu chỉnh xác suất (toàn bộ calib) + chạy mô hình cuối trên eval và test"):
    from rex.calibrate import MIN_TYPE_N, fit_one_temperature
    from rex.rex_train import MIN_CONFORMAL_CASES
    _t = time.time()
    REC = {}
    REC["calib"], _ = collect_rounds(SAMPLES["calib"])
    _fin = [r for r in REC["calib"] if r["last"]]
    _lg = np.stack([r["logits"] for r in _fin])
    _tg = np.array([r["target"] for r in _fin])
    T_BEFORE = float(MODEL.verdict_temperature())
    _pairs = []
    for z, t_ in zip(_lg, _tg):
        idx = np.flatnonzero(z > -1e3)
        if t_ in idx:
            _pairs.append((torch.tensor(z[idx]), torch.tensor((idx == t_).astype(np.float32))))
    T_FIT = fit_one_temperature(_pairs, min_n=MIN_TYPE_N) if len(_pairs) >= MIN_TYPE_N else T_BEFORE
    MODEL.set_verdict_temperature(T_FIT)
    _probs_cal = softmax_masked(_lg, float(MODEL.verdict_temperature()))
    if len(_fin) >= MIN_CONFORMAL_CASES:
        Q_HAT = float(MODEL.calibrate_conformal(torch.tensor(_probs_cal, dtype=torch.float32), torch.tensor(_tg)))
    else:
        Q_HAT = float(MODEL.conformal_threshold())
    log(f"  calib: {len(_fin):,} case ({len(_pairs):,} cặp) trong {fmt_secs(time.time() - _t)} | nhiệt độ verdict "
        f"{T_BEFORE:.4f} -> {float(MODEL.verdict_temperature()):.4f} | q_hat {Q_HAT:.4f}")
    _e1 = ece_top1(softmax_masked(_lg, 1.0), _tg)
    _e2 = ece_top1(_probs_cal, _tg)
    table([["T = 1 (trước)", _e1[0], _e1[1]], [f"T = {float(MODEL.verdict_temperature()):.3f} (sau)", _e2[0], _e2[1]]],
          ["Verdict round cuối (calib)", "ECE", "MCE"], "Hiệu chỉnh nhiệt độ:")
    # eval và test chạy SAU khi đặt nhiệt độ -> verdict_probs đã hiệu chỉnh
    for sub in ["eval", "test"]:
        _t = time.time()
        REC[sub], _ = collect_rounds(SAMPLES[sub])
        log(f"  {sub}: {len(REC[sub]):,} case-round trong {fmt_secs(time.time() - _t)}")
    _fe = [r for r in REC["eval"] if r["last"]]
    _e3 = ece_top1(np.stack([r["probs"] for r in _fe]), np.array([r["target"] for r in _fe]))
    log(f"  ECE / MCE trên eval (sau hiệu chỉnh): {_e3[0]:.4f} / {_e3[1]:.4f}")
    table([[f"{b / 10:.1f}-{(b + 1) / 10:.1f}", c, a, n] for b, c, a, n in _e3[2]],
          ["Bin tin cậy", "Tin cậy TB", "Accuracy", "Số case"], "Reliability (eval, sau hiệu chỉnh):")
    T_FIT = float(MODEL.verdict_temperature())
    METRICS["calibration"] = {"verdict_temperature": T_FIT, "q_hat": Q_HAT, "calib_cases": len(_fin),
                              "ece_before_calib": _e1[0], "mce_before_calib": _e1[1],
                              "ece_after_calib": _e2[0], "mce_after_calib": _e2[1],
                              "ece_eval": _e3[0], "mce_eval": _e3[1], "reliability_eval": _e3[2]}
    RELIABILITY = _e3[2]
    ck = os.path.join(OUT, "ckpt_full")
    if os.path.isdir(ck):   # lưu lại checkpoint full kèm nhiệt độ + q_hat vừa fit
        save_rex_checkpoint(MODEL, TOK, ck, base_cfg=BASE_CFG)


## 13. Đánh giá offline
Verdict theo round, confusion matrix, phát hiện fraud (có trọng số), theo năm, baseline luật và GBDT, chỉ số head / STALL / conformal (định nghĩa của `rex_train`).

In [ ]:
# ============================================================================================
# STEP 13/15 - Đánh giá offline: verdict, phát hiện fraud, các head, STALL, conformal, baseline
# Precision/AUC dùng trọng số lấy mẫu (case hợp lệ của val/test có trọng số 20) -> quy về tỉ lệ thật.
# ============================================================================================
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier


def recall_at_fpr(y, s, w, fpr):
    o = np.argsort(-s)
    y, w = y[o], w[o]
    neg = np.cumsum(w * (1 - y)) / max((w * (1 - y)).sum(), 1e-12)
    pos = np.cumsum(w * y) / max((w * y).sum(), 1e-12)
    ok = neg <= fpr
    return float(pos[ok].max()) if ok.any() else 0.0


def best_threshold(y, s, w):
    best = (0.5, -1.0)
    for t in np.unique(np.quantile(s, np.linspace(0.5, 0.9995, 400))):
        p = s >= t
        tp, fp, fn = (w * (p & (y == 1))).sum(), (w * (p & (y == 0))).sum(), (w * (~p & (y == 1))).sum()
        f1 = 2 * tp / max(2 * tp + fp + fn, 1e-12)
        if f1 > best[1]:
            best = (float(t), float(f1))
    return best


def at_threshold(y, s, w, t):
    p = s >= t
    tp, fp, fn = (w * (p & (y == 1))).sum(), (w * (p & (y == 0))).sum(), (w * (~p & (y == 1))).sum()
    prec, rec = tp / max(tp + fp, 1e-12), tp / max(tp + fn, 1e-12)
    return {"threshold": t, "precision": float(prec), "recall": float(rec),
            "f1": float(2 * prec * rec / max(prec + rec, 1e-12))}


def fraud_block(y, s, w, t=None, amounts=None):
    m = {"roc_auc": float(roc_auc_score(y, s, sample_weight=w)) if 0 < y.sum() < len(y) else float("nan"),
         "pr_auc": float(average_precision_score(y, s, sample_weight=w)) if y.sum() > 0 else float("nan"),
         "recall_at_fpr_1pct": recall_at_fpr(y, s, w, 0.01),
         "recall_at_fpr_0.1pct": recall_at_fpr(y, s, w, 0.001)}
    if t is not None:
        m.update(at_threshold(y, s, w, t))
        if amounts is not None and y.sum() > 0:
            caught = (s >= t) & (y == 1)
            m["fraud_amount_recall"] = float(np.abs(amounts[caught]).sum() / max(np.abs(amounts[y == 1]).sum(), 1e-12))
    return m


def frame(recs, last_only=True, round_idx=None):
    rs = [r for r in recs if (r["last"] if last_only else r["round"] == round_idx)]
    meta = [CASE_META[r["case_id"]] for r in rs]
    P = np.stack([r["probs"] for r in rs])
    return rs, meta, P


with Step(13, "Đánh giá offline"):
    RESULTS = {}
    # ---------- Verdict ----------
    for sub in ["eval", "test"]:
        recs = REC[sub]
        acc_r = []
        for k in range(4):
            rk = [r for r in recs if r["round"] == k]
            if rk:
                acc_r.append([f"round {k + 1}", len(rk), float(np.mean([r["probs"].argmax() == r["target"] for r in rk]))])
        rs, meta, P = frame(recs)
        tg = np.array([r["target"] for r in rs])
        pr = P.argmax(1)
        RESULTS[sub] = {"verdict_acc_final": float((pr == tg).mean()),
                        "verdict_macro_f1": float(f1_score(tg, pr, average="macro", labels=list(range(len(HYP_IDS))))),
                        "verdict_acc_by_round": {a[0]: a[2] for a in acc_r}}
        table(acc_r, ["Round", "Case-round", "Verdict accuracy"], f"[{sub}] verdict theo round:")
        log(f"  [{sub}] verdict round cuối: acc {RESULTS[sub]['verdict_acc_final']:.4f} | "
            f"macro-F1 {RESULTS[sub]['verdict_macro_f1']:.4f}")
    _rs, _meta, _P = frame(REC["test"])
    _tg = np.array([r["target"] for r in _rs])
    _cm = confusion_matrix(_tg, _P.argmax(1), labels=list(range(len(HYP_IDS))))
    table([[HYP_IDS[i]] + list(map(int, _cm[i])) for i in range(len(HYP_IDS))],
          ["gold \\ dự đoán"] + [h.replace("H_", "") for h in HYP_IDS], "[test] confusion matrix round cuối:")
    RESULTS["test"]["confusion_matrix"] = {"labels": HYP_IDS, "matrix": _cm.tolist()}

    # ---------- Phát hiện fraud (điểm = 1 - P(H_shared)) ----------
    def scores(recs, last=True, rnd=None):
        rs, meta, P = frame(recs, last_only=last, round_idx=rnd)
        y = np.array([int(m["is_fraud"]) for m in meta])
        w = np.array([m["weight"] for m in meta], dtype=float)
        a = np.array([m["amount"] for m in meta], dtype=float)
        yr = np.array([m["year"] for m in meta])
        gold = np.array([m["target"] for m in meta])
        return y, 1.0 - P[:, H_IDX["H_shared"]], w, a, yr, gold

    y_e, s_e, w_e, *_ = scores(REC["eval"])
    THR, _f1e = best_threshold(y_e, s_e, w_e)
    log(f"  ngưỡng fraud chọn trên eval (max F1 có trọng số): {THR:.4f} (F1 eval {_f1e:.4f})")
    y_t, s_t, w_t, a_t, yr_t, g_t = scores(REC["test"])
    y_t1, s_t1, w_t1, *_ = scores(REC["test"], last=False, rnd=0)
    y_e1, s_e1, w_e1, *_ = scores(REC["eval"], last=False, rnd=0)
    THR1, _ = best_threshold(y_e1, s_e1, w_e1)
    FRAUD = {"rex_final": fraud_block(y_t, s_t, w_t, THR, a_t), "rex_round1_only": fraud_block(y_t1, s_t1, w_t1, THR1)}
    FRAUD["rex_final"]["recall_by_gold"] = {h: float(((s_t >= THR) & (g_t == h)).sum() / max((g_t == h).sum(), 1))
                                            for h in HYP_IDS if h != "H_shared"}
    _yrs = []
    for yv in sorted(set(yr_t)):
        m = yr_t == yv
        if y_t[m].sum() >= 5:
            b = fraud_block(y_t[m], s_t[m], w_t[m], THR)
            _yrs.append([str(yv), int(y_t[m].sum()), b["pr_auc"], b["recall"], b["precision"]])
    table(_yrs, ["Năm", "Fraud", "PR-AUC", "Recall", "Precision"], "[test] phát hiện fraud theo năm (Rex cuối):")
    FRAUD["rex_final"]["by_year"] = {r[0]: {"fraud": r[1], "pr_auc": r[2], "recall": r[3], "precision": r[4]} for r in _yrs}

    # ---------- Baseline: chỉ luật PoE ----------
    term = {s["case_id"]: s for s in SAMPLES["test"] if s["label_scope"] == "terminal"}
    _ids = [r["case_id"] for r in frame(REC["test"])[0]]
    s_rules = np.array([float(any(v for k, v in term[c]["alive"].items() if k not in ("H_shared", "H_novel")))
                        for c in _ids])
    FRAUD["rules_only"] = fraud_block(y_t, s_rules, w_t, 0.5)
    # ---------- Baseline: GBDT trên cùng chỉ số (alert + mọi tool) ----------
    _feat_cols = [c for c in dict.fromkeys(ALERT_METRICS + list(METRIC_TOOL)) if c in CASES.columns]
    _tr = CASES[CASES["subset"] == "train"]
    _te = CASES.set_index("case_id").loc[_ids]
    gb = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, random_state=CFG["SEED"])
    gb.fit(_tr[_feat_cols].to_numpy(dtype=float), _tr["is_fraud"].astype(int).to_numpy(),
           sample_weight=_tr["weight"].to_numpy())
    _ev = CASES.set_index("case_id").loc[[r["case_id"] for r in frame(REC["eval"])[0]]]
    _thr_gb, _ = best_threshold(_ev["is_fraud"].astype(int).to_numpy(),
                                gb.predict_proba(_ev[_feat_cols].to_numpy(dtype=float))[:, 1], _ev["weight"].to_numpy())
    FRAUD["gbdt_same_features"] = fraud_block(y_t, gb.predict_proba(_te[_feat_cols].to_numpy(dtype=float))[:, 1],
                                              w_t, _thr_gb, a_t)
    table([[k, v["roc_auc"], v["pr_auc"], v["recall_at_fpr_1pct"], v["recall_at_fpr_0.1pct"], v.get("precision"),
            v.get("recall"), v.get("f1")] for k, v in FRAUD.items()],
          ["Mô hình (test)", "ROC-AUC", "PR-AUC", "Recall@FPR1%", "Recall@FPR0.1%", "Precision", "Recall", "F1"],
          "Phát hiện fraud trên test (trọng số theo tỉ lệ thật):")
    table([[h, v] for h, v in FRAUD["rex_final"]["recall_by_gold"].items()], ["Loại fraud (gold)", "Recall"],
          "[test] recall theo loại gian lận:")
    RESULTS["fraud_detection"] = FRAUD

    # ---------- Head S / R / gain / hyp, STALL, conformal, tool: đúng định nghĩa rex_train.head_metrics +
    # evaluate(), tính trên đầu ra đã ghi ở bước 12 (không chạy lại mô hình); AUROC bằng sklearn (cùng định nghĩa,
    # nhanh hơn vòng lặp của repo trên hàng trăm nghìn dòng).
    from rex.rex_tools import DEFAULT_R_THRESHOLD, DEFAULT_S_THRESHOLD, gain_threshold
    from rex.rex_heads import compute_ece

    def _auc(score, label):
        label = np.asarray(label) > 0.5
        return float(roc_auc_score(label, score)) if 0 < label.sum() < len(label) else float("nan")

    levels = [MODEL.tool_meta.get(t, {}).get("level", "READ") for t in MODEL.tools]
    costs = [float(MODEL.tool_meta.get(t, {}).get("cost", 0.0)) for t in MODEL.tools]
    tau = np.array([gain_threshold(lv, c) for lv, c in zip(levels, costs)])
    for sub in ["eval", "test"]:
        R_ = REC[sub]
        rnd = np.array([r["round"] for r in R_])
        sp, sl = np.array([r["s"] for r in R_]), np.array([r["s_label"] for r in R_])
        rp, rl = np.array([r["r"] for r in R_]), np.array([r["r_label"] for r in R_])
        gn, gl = np.stack([r["gain"] for r in R_]), np.stack([r["gain_label"] for r in R_])
        cal = np.stack([r["callable"] for r in R_])
        hm = {"s_auroc": _auc(sp, sl), "s_ece": float(compute_ece(torch.tensor(sp), torch.tensor(sl)))}
        per = [(_auc(sp[rnd == k], sl[rnd == k]), int((rnd == k).sum())) for k in np.unique(rnd)]
        per = [(a_, n_) for a_, n_ in per if a_ == a_]
        hm["s_auroc_within_round"] = sum(a_ * n_ for a_, n_ in per) / sum(n_ for _, n_ in per) if per else float("nan")
        for k in np.unique(rnd):
            hm[f"s_mean_round_{int(k) + 1}"] = float(sp[rnd == k].mean())
        hm["r_mae"] = float(np.abs(rp - rl).mean())
        hm["r_auroc_high_risk"] = _auc(rp, rl > DEFAULT_R_THRESHOLD)
        valid = cal & (gl != -100)
        hm["gain_auroc"] = _auc(gn[valid], gl[valid])
        hm["hyp_auroc"] = _auc(np.concatenate([r["hyp"] for r in R_]), np.concatenate([r["hyp_label"] for r in R_]))
        any_tool = ((gn >= tau) & cal).any(axis=1)
        pred_stall = (sp < DEFAULT_S_THRESHOLD) & ~any_tool
        true_stall = np.array([r["stall_label"] for r in R_]) > 0.5
        hm["stall_rate_label"] = float(true_stall.mean())
        hm["stall_rate_pred"] = float(pred_stall.mean())
        hm["stall_high_risk_rate_pred"] = float(((sp >= DEFAULT_S_THRESHOLD) & (rp > DEFAULT_R_THRESHOLD)).mean())
        hm["stall_recall"] = float((pred_stall & true_stall).sum() / max(true_stall.sum(), 1))
        hm["stall_precision"] = float((pred_stall & true_stall).sum() / max(pred_stall.sum(), 1))
        rs_, _, P_ = frame(R_)
        tg_ = np.array([r["target"] for r in rs_])
        sets = MODEL.calibrator.predict_set(torch.tensor(P_, dtype=torch.float32), labels=HYP_IDS, q_hat=MODEL.conformal_threshold())
        hm["conformal_coverage"] = float(np.mean([HYP_IDS[int(t_)] in st for st, t_ in zip(sets, tg_)]))
        hm["conformal_mean_set_size"] = float(np.mean([len(st) for st in sets]))
        hm["conformal_mean_live_hypotheses"] = float((P_ > 0).sum(1).clip(min=1).mean())
        tp = np.concatenate([r["tool_p"][r["callable"]] >= 0.5 for r in R_])
        tl = np.concatenate([r["tool_label"][r["callable"]] > 0.5 for r in R_])
        hm["tool_precision_vs_teacher"] = float((tp & tl).sum() / max(tp.sum(), 1))
        hm["tool_recall_vs_teacher"] = float((tp & tl).sum() / max(tl.sum(), 1))
        RESULTS[sub]["heads"] = hm
    table([[k, RESULTS["eval"]["heads"].get(k), RESULTS["test"]["heads"].get(k)] for k in RESULTS["test"]["heads"]],
          ["Chỉ số head / STALL / conformal / tool", "eval", "test"], "Head S / R / gain / hyp, STALL, conformal, tool:")
    METRICS["results"] = RESULTS
    METRICS["fraud_threshold_from_eval"] = THR


## 14. InvestigatorAgent end-to-end + độ trễ

In [ ]:
# ============================================================================================
# STEP 14/15 - Chạy InvestigatorAgent end-to-end trên test + đo độ trễ
# Agent tự chọn tool (dispatcher 3 nhánh), tool trả kết quả thật của tầng B, PoE hybrid loại giả
# thuyết, rồi quyết định tự kết luận hay leo thang (giống Part B stop.decide).
# Batch 1 nên chậm: chạy trên tập con phân tầng (một nửa fraud), trọng số quy về tỉ lệ thật.
# ============================================================================================
def agent_subset(n_total, seed=0):
    T = CASES[CASES["subset"] == "test"]
    fr, lg = T[T["is_fraud"] == 1], T[T["is_fraud"] == 0]
    n_fr = min(len(fr), n_total // 2)
    n_lg = min(len(lg), n_total - n_fr)
    pf = fr.sample(n=n_fr, random_state=seed)
    pl = lg.sample(n=n_lg, random_state=seed)
    # trọng số: mỗi case đại diện cho (tổng trọng số nhóm) / (số case chọn)
    pf = pf.assign(aw=fr["weight"].sum() / max(n_fr, 1))
    pl = pl.assign(aw=lg["weight"].sum() / max(n_lg, 1))
    return pd.concat([pf, pl]).reset_index(drop=True)


with Step(14, "InvestigatorAgent end-to-end + độ trễ"):
    _dom = REGISTRY.get_domain(DOMAIN)
    RULES = {h.id: list(h.falsification_rules) for h in _dom.hypotheses}
    TOOL_HYP = {t: [h.id for h in _dom.hypotheses if not h.is_residual and t in h.associated_tools]
                for t in REGISTRY.tools}
    SPECS = [REGISTRY.tool_specs[t] for t in MODEL.tools]
    AG = InvestigatorAgent(model=MODEL, theta_elim=0.20)
    SUB = agent_subset(CFG["AGENT_EVAL_CASES"], seed=CFG["SEED"])
    WORLDS = build_worlds(SUB)
    rows = []
    _t = time.time()
    for i, (w, meta) in enumerate(zip(WORLDS, SUB.itertuples(index=False))):
        calls = []

        def dispatcher(tools, _w=w, _calls=calls):
            _calls.extend(tools)
            return {t: dict(_w["tool_data"][t]) for t in tools}

        with torch.autocast("cuda", dtype=torch.float16, enabled=CFG["AMP"]):
            res = AG.investigate(tok=TOK, type_token=DOMAIN, instruction=_dom.instruction,
                                 hypotheses=_dom.hypotheses_dict, rules=RULES, tool_hyp_mapping=TOOL_HYP,
                                 tool_dispatcher=dispatcher, initial_state=format_state({"alert": w["alert"]}),
                                 initial_observations=w["alert"], max_rounds=3, tool_specs=SPECS)
        rows.append({"case_id": w["case_id"], "is_fraud": bool(meta.is_fraud), "target": meta.target,
                     "amount": float(meta.amount), "w": float(meta.aw), "pred": res.predicted_hypothesis,
                     "escalated": bool(res.is_escalated), "reason": res.termination_reason.value,
                     "rounds": int(res.rounds_executed), "n_calls": len(calls),
                     "cost": float(sum(REGISTRY.tool_specs[t].cost for t in calls)),
                     "p_legit": float(res.probabilities.get("H_shared", 0.0))})
        if (i + 1) % max(1, len(WORLDS) // 10) == 0:
            el = time.time() - _t
            log(f"  agent {i + 1}/{len(WORLDS)} | {el / (i + 1) * 1000:.0f} ms/case | ETA {fmt_secs(el / (i + 1) * (len(WORLDS) - i - 1))}")
    AGENT_MS = (time.time() - _t) / max(len(rows), 1) * 1000
    AGR = pd.DataFrame(rows)
    W = AGR["w"].to_numpy()
    fr = AGR["is_fraud"].to_numpy()
    esc = AGR["escalated"].to_numpy()
    wavg = lambda m, base: float((W * m * base).sum() / max((W * base).sum(), 1e-12))
    AGENT = {
        "cases": int(len(AGR)), "fraud_cases": int(fr.sum()),
        "auto_close_rate": wavg(~esc, np.ones_like(fr)), "escalation_rate": wavg(esc, np.ones_like(fr)),
        "fraud_auto_closed_rate": wavg(~esc, fr), "fraud_auto_closed_cases": int((fr & ~esc).sum()),
        "fraud_auto_closed_amount_share": float(AGR.loc[fr & ~esc, "amount"].abs().sum() /
                                                max(AGR.loc[fr, "amount"].abs().sum(), 1e-12)),
        "legit_escalated_rate": wavg(esc, ~fr),
        "verdict_accuracy": wavg((AGR["pred"] == AGR["target"]).to_numpy(), np.ones_like(fr)),
        "verdict_accuracy_fraud": wavg((AGR["pred"] == AGR["target"]).to_numpy(), fr),
        "novel_rate": wavg((AGR["pred"] == "H_novel").to_numpy(), np.ones_like(fr)),
        "mean_rounds": float(np.average(AGR["rounds"], weights=W)),
        "mean_tool_calls": float(np.average(AGR["n_calls"], weights=W)),
        "mean_tool_cost": float(np.average(AGR["cost"], weights=W)),
        "termination_reasons": {k: float(W[AGR["reason"] == k].sum() / W.sum()) for k in sorted(AGR["reason"].unique())},
        "termination_reasons_fraud": {k: int(((AGR["reason"] == k) & fr).sum()) for k in sorted(AGR["reason"].unique())},
    }
    table([[k, v] for k, v in AGENT.items() if not isinstance(v, dict)], ["Agent end-to-end (test)", "Giá trị"])
    table([[k, f"{100 * v:.2f}%", AGENT["termination_reasons_fraud"].get(k, 0)]
           for k, v in AGENT["termination_reasons"].items()],
          ["Lý do kết thúc", "Tỉ lệ (trọng số)", "Số case fraud"], "Lý do kết thúc:")
    METRICS["agent_end_to_end"] = AGENT

    # ---------- độ trễ ----------
    MODEL.eval()
    _b1 = probe_batch(TOK, SAMPLES["test"], n=1)
    _b64 = probe_batch(TOK, SAMPLES["test"], n=64)

    def _time(b, n):
        b = {k: v.to(DEVICE) for k, v in b.items()}
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for _ in range(3):
                MODEL(input_ids=b["input_ids"], attention_mask=b["attention_mask"], marker_pos=b["marker_pos"],
                      marker_mask=b["marker_mask"], qtype=b["qtype"], dispatch=False)
            torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(n):
                MODEL(input_ids=b["input_ids"], attention_mask=b["attention_mask"], marker_pos=b["marker_pos"],
                      marker_mask=b["marker_mask"], qtype=b["qtype"], dispatch=False)
            torch.cuda.synchronize()
        return (time.time() - t0) / n * 1000

    LAT = {"single_ms": _time(_b1, CFG["LATENCY_RUNS"]),
           "batch64_ms_per_decision": _time(_b64, max(3, CFG["LATENCY_RUNS"] // 5)) / _b64["input_ids"].size(0),
           "agent_ms_per_case_incl_tools": float(AGENT_MS),
           "gpu": torch.cuda.get_device_name(0)}
    table([[k, v] for k, v in LAT.items()], ["Độ trễ (round 1, fp16)", "Giá trị"])
    METRICS["latency"] = LAT


## 15. Lưu checkpoint và báo cáo
Lưu `ckpt_final/`, `metrics.json`, `train_log.txt`, `history.csv`, `predictions_test.*`, `plots/`, và Model Card với bản quyền CC BY 4.0 ghi nhận tác giả RBA Dataset gốc (Wiefling et al., 2020/2021).


In [ ]:
# ============================================================================================
# STEP 15/15 - Lưu checkpoint cuối, chỉ số, log, biểu đồ và dự đoán (chỉ trong /kaggle/working)
# ============================================================================================
import shutil
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

with Step(15, "Lưu kết quả và báo cáo tổng hợp"):
    FINAL = os.path.join(OUT, "ckpt_final")
    save_rex_checkpoint(MODEL, TOK, FINAL, base_cfg=BASE_CFG)
    if os.path.exists(os.path.join(OUT, "registry.json")):
        shutil.copy(os.path.join(OUT, "registry.json"), os.path.join(FINAL, "registry.json"))
    for _fn in ("registry_rba.json", "registry_rba.yaml"):
        _src_p = os.path.join(REXLIB, "rex", "events", "configs", _fn)
        if not os.path.exists(_src_p):
            _src_p = os.path.join("rex", "events", "configs", _fn)
        if os.path.exists(_src_p):
            shutil.copy(_src_p, os.path.join(FINAL, _fn))
    METRICS["stages"] = STAGE_SUMMARIES
    METRICS["stage_times"] = STAGE_TIMES
    METRICS["shard_log"] = SHARD_LOG
    METRICS["total_runtime_s"] = time.time() - T_START
    with open(os.path.join(FINAL, "rex_train_summary.json"), "w") as f:
        json.dump({"stages": STAGE_SUMMARIES, "calibration": METRICS.get("calibration")}, f, indent=2, default=float)
    with open(os.path.join(OUT, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(METRICS, f, indent=2, ensure_ascii=False, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    pd.DataFrame(HISTORY).to_csv(os.path.join(OUT, "history.csv"), index=False)

    # dự đoán test: mỗi case một dòng
    _rs = [r for r in REC["test"] if r["last"]]
    pred = pd.DataFrame({"case_id": [r["case_id"] for r in _rs],
                         "gold": [CASE_META[r["case_id"]]["target"] for r in _rs],
                         "is_fraud": [bool(CASE_META[r["case_id"]]["is_fraud"]) for r in _rs],
                         "weight": [CASE_META[r["case_id"]]["weight"] for r in _rs],
                         "pred": [HYP_IDS[int(np.argmax(r["probs"]))] for r in _rs],
                         "p_fraud": [1 - float(r["probs"][H_IDX["H_shared"]]) for r in _rs],
                         "S": [r["s"] for r in _rs], "R": [r["r"] for r in _rs],
                         **{f"p_{h}": [float(r["probs"][i]) for r in _rs] for i, h in enumerate(HYP_IDS)}})
    pred = pred.merge(AGR[["case_id", "escalated", "reason", "n_calls"]].rename(
        columns={"escalated": "agent_escalated", "reason": "agent_reason", "n_calls": "agent_tool_calls"}),
        on="case_id", how="left")
    try:
        pred.to_parquet(os.path.join(OUT, "predictions_test.parquet"), index=False)
    except Exception:
        pred.to_csv(os.path.join(OUT, "predictions_test.csv.gz"), index=False)

    # biểu đồ: loss theo update, reliability, PR
    if HISTORY:
        H = pd.DataFrame(HISTORY)
        fig, ax = plt.subplots(figsize=(10, 4))
        for st_name, g in H.groupby("stage", sort=False):
            ax.plot(np.arange(len(g)) + (0 if st_name == "heads" else 0), g["loss"].to_numpy(), label=st_name)
        ax.set_xlabel(f"log step (mỗi {CFG['LOG_EVERY']} update)"); ax.set_ylabel("loss"); ax.legend()
        fig.tight_layout(); fig.savefig(os.path.join(OUT, "plots", "loss.png"), dpi=120); plt.close(fig)
        comp = [c for c in H.columns if c.endswith("_loss")]
        if comp:
            fig, ax = plt.subplots(figsize=(10, 4))
            for c in comp:
                ax.plot(H[c].to_numpy(), label=c)
            ax.set_xlabel("log step (heads -> ponder -> full)"); ax.legend(ncol=4, fontsize=8)
            fig.tight_layout(); fig.savefig(os.path.join(OUT, "plots", "loss_components.png"), dpi=120); plt.close(fig)
    if RELIABILITY:
        fig, ax = plt.subplots(figsize=(4.5, 4.5))
        ax.plot([0, 1], [0, 1], "--", color="gray")
        ax.plot([c for _, c, _, _ in RELIABILITY], [a for _, _, a, _ in RELIABILITY], "o-")
        ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_title("Reliability (eval, sau hiệu chỉnh)")
        fig.tight_layout(); fig.savefig(os.path.join(OUT, "plots", "reliability.png"), dpi=120); plt.close(fig)
    from sklearn.metrics import precision_recall_curve
    _p, _r, _ = precision_recall_curve(y_t, s_t, sample_weight=w_t)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(_r, _p); ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_title("PR fraud (test, trọng số)")
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "plots", "pr_curve.png"), dpi=120); plt.close(fig)

    # ---------- bảng tổng hợp cuối ----------
    F_ = METRICS["results"]["fraud_detection"]["rex_final"]
    E_ = METRICS["results"]["test"]["heads"]
    A_ = METRICS["agent_end_to_end"]
    table([
        ["verdict accuracy round cuối (test)", METRICS["results"]["test"]["verdict_acc_final"]],
        ["fraud ROC-AUC / PR-AUC (test)", f"{F_['roc_auc']:.4f} / {F_['pr_auc']:.4f}"],
        ["fraud recall @FPR 1% (test)", F_["recall_at_fpr_1pct"]],
        ["fraud precision / recall tại ngưỡng eval", f"{F_['precision']:.4f} / {F_['recall']:.4f}"],
        ["GBDT cùng đặc trưng: PR-AUC", METRICS["results"]["fraud_detection"]["gbdt_same_features"]["pr_auc"]],
        ["s_auroc / s_auroc_within_round", f"{E_.get('s_auroc', float('nan')):.4f} / {E_.get('s_auroc_within_round', float('nan')):.4f}"],
        ["r_mae / gain_auroc / hyp_auroc", f"{E_.get('r_mae', float('nan')):.4f} / {E_.get('gain_auroc', float('nan')):.4f} / {E_.get('hyp_auroc', float('nan')):.4f}"],
        ["stall precision / recall", f"{E_.get('stall_precision', float('nan')):.4f} / {E_.get('stall_recall', float('nan')):.4f}"],
        ["conformal coverage / set size", f"{E_.get('conformal_coverage', float('nan')):.4f} / {E_.get('conformal_mean_set_size', float('nan')):.2f}"],
        ["nhiệt độ / q_hat", f"{METRICS['calibration']['verdict_temperature']:.3f} / {METRICS['calibration']['q_hat']:.4f}"],
        ["agent: tự kết luận / leo thang", f"{A_['auto_close_rate']:.4f} / {A_['escalation_rate']:.4f}"],
        ["agent: fraud bị tự kết luận (tỉ lệ, case)", f"{A_['fraud_auto_closed_rate']:.4f} ({A_['fraud_auto_closed_cases']})"],
        ["agent: hợp lệ bị leo thang", A_["legit_escalated_rate"]],
        ["độ trễ đơn lẻ / theo lô (ms)", f"{METRICS['latency']['single_ms']:.1f} / {METRICS['latency']['batch64_ms_per_decision']:.2f}"],
        ["tổng thời gian chạy", fmt_secs(METRICS["total_runtime_s"])],
    ], ["TỔNG HỢP", "Giá trị"])
    log("")
    log("File đầu ra trong " + OUT + ":")
    for pth in sorted(glob.glob(os.path.join(OUT, "*"))):
        if os.path.basename(pth) == "rexlib":
            continue
        size = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(pth) for f in fs) if os.path.isdir(pth) \
            else os.path.getsize(pth)
        log(f"  {os.path.basename(pth):28s} {size / 1e6:9.1f} MB")

# Model Card cho RBA Dataset (Wiefling et al., CC BY 4.0)
model_card = {
    "model_name": "Rex-ModernBERT-RBA-AccountTakeover",
    "architecture": "Rex (15-Step Multi-Round Decision Agent + Laya/ModernBERT + PoE Refutation)",
    "dataset": "RBA Dataset (Wiefling et al., 2020/2021) - CC BY 4.0 - https://www.kaggle.com/datasets/dasgroup/rba-dataset",
    "domain": DOMAIN,
    "hypotheses": HYP_IDS,
    "semantic_tools": ["network", "device", "behavior", "graph"],
    "license": "CC BY 4.0 Attribution"
}
with open(os.path.join(FINAL, "model_card.json"), "w", encoding="utf-8") as f:
    json.dump(model_card, f, indent=2, ensure_ascii=False)
log(f"  Model Card lưu tại {os.path.join(FINAL, 'model_card.json')}")
